# NextGen Boardroom

**Where bold ideas meet hard questions.**

## Pipeline

Runs **one detailed debate** for one company. The VC has no question budget: it keeps asking until it has covered everything it needs to decide (market size first, then the ideal customer and beachhead, customers, unit economics, competition, technology, go-to-market, the founders' background and journey, vision, how much they are raising and what the money is for, and risks), follows up as long as an answer leaves a gap, and then says it is ready. The VC addresses the whole team, never a named executive. Answers are not length-limited: every agent is asked to be precise. Asked for TAM, the CFO gives up to three sizing methods, each with its maths (checked in code) and assumptions.

Two more agents sit around the debate: a **Moderator** (JEV) who, after each VC question, says who will take it up ("This question will be taken up by the CFO."), and the **VC's Assistant** (fact-checker: maths settled in code, then JEV, a stronger model only when unsure) who replies after each exchange with just "Facts are correct." or "Facts are incorrect.". The optimized run is compared with a naive run where Opus does everything.

Pipeline: **build corpus → ingest → live debate → (optional) cost baseline → evaluation dashboard → Markdown and HTML export**.

Every turn (with its maths, assumptions and citations), claim, fact-check, retrieval, model call, cost and the
VC decision is stored in `data/debate.sqlite`; checkpoints and VC memory live in Postgres. The last step writes
everything to Markdown in `reports/` and refreshes the local pages under `web/` from the saved SQLite transcript.

**Before you run:** kernel *Python (MultiAgentsForMarketSizing .venv)*; `.env` with `ANTHROPIC_API_KEY`
(and `JEV_API_KEY` for the JEV System-1 classifier, which then takes the first-pass fact-check verdicts); the licensed case PDF at
`case_study/biobot/01.BiobotAnalytics.pdf` and the public ledger `case_study/biobot/public_data.md`, plus the curated
`corpus/biobot/{facts,assumptions,sizing}.yaml`; Docker via Colima (the setup cell starts it).

**Cost:** live runs call Claude. A meeting now runs until the VC is satisfied (typically 25-30 questions over 12 topics) with unlimited answer length, so it costs several times the old 9-question debate, and the all-Opus baseline several times more. Each run is capped at `limits.max_cost_usd_per_run` in `configs/models.yaml` ($25) as a spend guard, and prints its measured cost. The default is a live run. Set `DRY_RUN = True` only when you want scripted placeholder replies and no API spend.

## 1. Settings

In [37]:
COMPANY = "biobot"
MAX_ROUNDS = 60               # runaway guard only: the VC asks until it has what it needs and then decides
PROFILE = "routed"            # "routed" = optimized: JEV for verdicts and the moderator, Haiku for claim extraction, Sonnet/Opus for the rest
DRY_RUN = False              # False = real agent conversation; True = scripted placeholders, no API cost
CLEAR_PREVIOUS_RUNS = True   # delete earlier debates (and reports) before running; ignored when RUN_DEBATE = False

# Turn the debate off to only (re)build the corpus and, if a finished live debate is already stored, evaluate and
# export it. Nothing is deleted and no model is called.
RUN_DEBATE = True             # False = skip the optimized debate (and the reset above) and reuse the latest stored one

# The naive baseline: the same debate with Opus on every call (moderator included). It runs once and the optimized debate
# is never repeated for it; the comparison table below needs both. Set False for exactly one (optimized) debate.
RUN_COST_BASELINE = True

EXPORT_DIR_NAME = "reports"   # Markdown output folder, relative to the project root

## 2. Setup: environment, Postgres, helpers

In [38]:
import contextlib, io, json, shutil, sqlite3, subprocess, sys, time

# A kernel that was started before the code last changed still holds the old `debate` modules. Drop them so this cell
# (and every cell after it) imports the current source; re-running it is enough, no kernel restart needed.
for _name in [m for m in sys.modules if m == "debate" or m.startswith("debate.")]:
    del sys.modules[_name]
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import Markdown, display

from debate.cli import DATA, main as debate_cli
from debate.config import ROOT, load_env

load_env()  # reads .env; values are never printed
import os
missing = [k for k in ("ANTHROPIC_API_KEY",) if not os.environ.get(k)]
if missing and not DRY_RUN:
    raise RuntimeError(f"set {missing} in .env (or use DRY_RUN = True)")
print("project:", ROOT)
print("System-1 verdicts:", f"JEV ({os.environ.get('JEV_MODEL', 'jev-latest')})" if os.environ.get("JEV_API_KEY") and PROFILE == "routed" and not DRY_RUN else "Claude Haiku (no JEV_API_KEY, dry run or opus_low)"); print("System-1 claim extraction: Claude Haiku (JEV only classifies)")

EXPORT_DIR = ROOT / EXPORT_DIR_NAME
EXPORT_DIR.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)


def sh(*cmd, check=True):
    return subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, check=check)


# Docker (Colima) + Postgres/pgvector: checkpoints, long-term memory and the RAG corpus live there.
if shutil.which("docker") is None:
    raise RuntimeError("docker CLI not found: brew install docker docker-compose colima")
if sh("docker", "info", check=False).returncode != 0:
    print("starting Colima ..."); sh("colima", "start", "--cpu", "2", "--memory", "2", "--disk", "10")
sh("docker", "compose", "up", "-d")
for _ in range(30):
    if "healthy" in sh("docker", "compose", "ps", "--format", "{{.Status}}", check=False).stdout:
        break
    time.sleep(2)
print(sh("docker", "compose", "ps", "--format", "{{.Name}}: {{.Status}}").stdout)

SESSION = {"started_at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "company": COMPANY,
           "max_rounds": MAX_ROUNDS, "profile": PROFILE, "dry_run": DRY_RUN, "runs": [], "reports": []}


def debate(*args, save_output_to=None):
    """Run a `debate` CLI command in this kernel; raise if it fails. Optionally save its printout to a file."""
    args = [str(a) for a in args]
    print("$ debate", " ".join(args))
    if save_output_to:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            code = debate_cli(args)
        print(buf.getvalue())
        Path(save_output_to).write_text(buf.getvalue())
    else:
        code = debate_cli(args)
    if code != 0:
        raise RuntimeError(f"`debate {args[0]}` exited with code {code}")


def _max_rowid():
    with sqlite3.connect(DATA / "debate.sqlite") as con:  # read-only: the store owns the schema
        if not con.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='runs'").fetchone():
            return 0
        return con.execute("SELECT COALESCE(MAX(rowid), 0) FROM runs").fetchone()[0]


def run_debate(profile=PROFILE):
    """Run one debate and return its run id (also recorded in SESSION)."""
    before = _max_rowid()
    args = ["run", "--company", COMPANY, "--rounds", MAX_ROUNDS, "--profile", profile]
    args += ["--dry-run", "--with-rag"] if DRY_RUN else []
    code = debate_cli([str(a) for a in args])
    with sqlite3.connect(DATA / "debate.sqlite") as con:
        run_id, status, cost = con.execute(
            "SELECT run_id, status, total_cost_usd FROM runs WHERE rowid > ? ORDER BY rowid DESC LIMIT 1",
            (before,)).fetchone()
    SESSION["runs"].append({"run_id": run_id, "profile": profile, "status": status, "cost_usd": cost})
    if code != 0:
        raise RuntimeError(f"run {run_id} ended with status {status!r}; see the output above")
    return run_id

project: /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing
System-1 verdicts: JEV (jev-latest)
System-1 claim extraction: Claude Haiku (JEV only classifies)
multiagentsformarketsizing-postgres-1: Up 46 hours (healthy)



## 2b. Start clean (optional)

Deletes earlier debates so the database and `reports/` hold only this session's debate. The corpus is kept. Skipped when `RUN_DEBATE = False`.

In [39]:
if CLEAR_PREVIOUS_RUNS and RUN_DEBATE:
    debate("reset", "--yes", "--reports", EXPORT_DIR)
    EXPORT_DIR.mkdir(exist_ok=True)
else:
    print("keeping earlier debates (CLEAR_PREVIOUS_RUNS = False or RUN_DEBATE = False)")

$ debate reset --yes --reports /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports
deleted 5 files/folders, 5 checkpoint threads, 2 memory items


## 3. Build and ingest the corpus

`build-corpus` turns the case PDF into dated topic files, exhibit tables (Exhibit 7 reconciled against its own totals), dated endnotes and an index under `corpus/biobot/`, and turns the public ledger (`case_study/biobot/public_data.md`) into ~100 dated post-case items. It also validates the curated fact table (every quote checked against the PDF), the assumptions (ledger-sourced ones must resolve to exactly one dated item) and the sizing lineage. `ingest` embeds the ~360 chunks locally; `eval-retrieval` scores retrieval on 49 labelled questions. No model API calls.

In [40]:
debate("build-corpus", "--company", COMPANY)
debate("ingest", "--company", COMPANY)
debate("eval-retrieval", "--output", EXPORT_DIR / "retrieval_eval.md")
SESSION["reports"].append("retrieval_eval.md")
display(Markdown((ROOT / "corpus" / COMPANY / "README.md").read_text()))

$ debate build-corpus --company biobot
built /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/corpus/biobot: 58 narrative chunks from 61 paragraphs, 3 exhibit tables, 5 captions, 44 dated endnotes, 152 dated source-ledger items
  exhibit check: Emergency Medical Services, FY16 expenditures: FY16 expenditures printed as '49,63,310' (malformed in the source); implied by the column total: 49,693,309
  exhibit check: FY16 expenditures: bureaus + standalone lines = 95,801,052 -> reconciles
  exhibit check: Total Public Health Service Centers FY16 expenditures: lines sum to 14,322,180, printed 14,302,181
  exhibit check: FY17 budget: bureaus + standalone lines = 98,022,291 -> reconciles
  exhibit check: FY18 budget: bureaus + standalone lines = 101,092,192 -> reconciles
index and sizing lineage: /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/corpus/biobot/README.md
$ debate ingest --company biobot
ingested 418 dated chunks (24466 words) for biobot: 7 assumption, 44 endnote, 5

# Biobot corpus

Generated by `debate build-corpus` from HBS case 821-045, "Biobot Analytics" and the manifest's research ledgers. Every chunk is dated (`as_of`).
Generated files are rebuilt from their sources; `facts.yaml`, `assumptions.yaml` and `sizing.yaml` are curated by hand.
The `dummy` source is invented scenario ground truth, not reported company data. Screenshot funding leads are unverified and cannot support fact-check verdicts.

## Layout

| Path | What | Kind |
|---|---|---|
| `narrative/01_decision_context.md` | 3 paragraphs | generated |
| `narrative/02_founders_team.md` | 14 paragraphs | generated |
| `narrative/03_origin_research.md` | 5 paragraphs | generated |
| `narrative/04_science_and_prior_art.md` | 7 paragraphs | generated |
| `narrative/05_technology_ip.md` | 7 paragraphs | generated |
| `narrative/06_market.md` | 5 paragraphs | generated |
| `narrative/07_customers_go_to_market.md` | 7 paragraphs | generated |
| `narrative/08_competition_and_precedents.md` | 2 paragraphs | generated |
| `narrative/09_risks_and_objections.md` | 8 paragraphs | generated |
| `exhibits/ex5.csv` | Exhibit 5 as rows | generated |
| `exhibits/ex6.csv` | Exhibit 6 as rows | generated |
| `exhibits/ex7.csv` | Exhibit 7 as rows | generated |
| `captions.yaml` | 5 figure captions | generated |
| `sources/endnotes.yaml` | 44 endnotes, each with its publication date | generated |
| `sources/dummy.yaml` | Synthetic Biobot boardroom operating packet: one dated item per table row or bullet of `case_study/biobot/dummy_data.md` | generated |
| `sources/public.yaml` | Biobot public data and source ledger: one dated item per table row or bullet of `case_study/biobot/public_data.md` | generated |
| `facts.yaml` | atomic facts, quotes verified against the PDF | curated |
| `assumptions.yaml` | sizing inputs that are NOT in the case | curated |
| `sizing.yaml` | TAM methods recomputed from facts and assumptions | curated |

## Exhibit 7 reconciliation

- Emergency Medical Services, FY16 expenditures: FY16 expenditures printed as '49,63,310' (malformed in the source); implied by the column total: 49,693,309
- FY16 expenditures: bureaus + standalone lines = 95,801,052 -> reconciles
- Total Public Health Service Centers FY16 expenditures: lines sum to 14,322,180, printed 14,302,181
- FY17 budget: bureaus + standalone lines = 98,022,291 -> reconciles
- FY18 budget: bureaus + standalone lines = 101,092,192 -> reconciles

## Curated facts: 89

- claim: 5
- derived: 7
- estimate: 3
- forecast: 3
- plan: 2
- quote: 16
- reported: 53

## Source-ledger items: 152 (public and synthetic)

| Section | Items | Dated | Reliability |
|---|---:|---|---|
| Historical operating trajectory | 9 | 2018-12-31 to 2026-10-22 | synthetic |
| Historical contract cohorts | 7 | 2019-12-31 to 2025-12-31 | synthetic |
| Historical efficiency and spending | 5 | 2021-12-31 to 2026-09-30 | synthetic |
| Commercial contract book | 6 | 2026-09-30 | synthetic |
| Revenue and retention | 7 | 2025-12-31 to 2026-09-30 | synthetic |
| Pricing and unit economics | 8 | 2026-09-30 | synthetic |
| Procurement and sales cycle | 4 | 2026-09-30 | synthetic |
| Capacity and product performance | 4 | 2026-09-30 | synthetic |
| Market-sizing scenario | 4 | 2026-09-30 | synthetic |
| Check of the requested milestones | 8 | 2017 to 2026-01-07 | sourced |
| History and operating milestones | 9 | 2017 to 2026 | sourced |
| Company and offering | 5 | 2025-11-24 to 2026-10-21 | sourced |
| Current product catalog and delivery model | 12 | 2026-10-22 | sourced |
| Competitors, alternatives, and procurement substitutes | 7 | 2026-10-14 to 2026-10-22 | sourced |
| Funding and investors | 5 | 2018-05 to 2025-06-18 | sourced |
| Contracts, customers, and adoption signals | 6 | 2021-05 to 2026-07 | sourced |
| Public measurements and reusable datasets | 8 | 2018 to 2026-10-22 | sourced |
| Named customer examples and what they show | 7 | 2018 to 2026-10-22 | sourced |
| Technology and intellectual property | 5 | 2019 to 2026-10-22 | sourced |
| Research and performance evidence | 5 | 2021 to 2026-10-22 | sourced |
| Business and market evidence, with limits | 6 | 2024-05-01 to 2026-10-22 | sourced |
| Research findings and measurement cautions | 3 | 2025 to 2026-10-22 | sourced |
| Screenshot-only funding leads supplied with this request | 12 | 2017-09-01 to 2026-10-22 | lead |

How each item was dated: section checked date 19; date column 72; latest year mentioned 24; latest cited publication date 17; year in first cell 2; event year in lead 6; manifest 1; ledger checked date (current-state item) 11.

Sizing inputs backed by public items: `people_per_catchment` <- `biobot_public_history_and_operating_03` (source); `price_per_sample` <- `biobot_public_company_and_offering_05` (cross_check); `municipal_price_floor` <- `biobot_public_company_and_offering_05` (source).

## Sizing lineage

### Beachhead budgets (top-down) = \$511.39M

| Step | Input | Value | Source |
|---|---|---:|---|
|  | `beachhead_municipal_budgets` The four beachhead cities (Boston, San Francisco, New York City, Washi | \$110.00B | fact (reported) |
| × | `public_health_share_estimate` The founders estimated 20% of those municipal budgets went to public h | 20.00% | fact (estimate) |
| × | `boston_recovery_share_fy18` Boston's Recovery Services Bureau was about 4.65% of its FY2018 Public | 4.65% | derived (derived) |
| × | `capture_beachhead` Share of the beachhead cities' recovery-services spend Biobot could wi | 50.00% | assumption (assumption) |

### Share of an existing market (top-down) = \$744.23M

| Step | Input | Value | Source |
|---|---|---:|---|
|  | `water_market_2022_forecast` The water and wastewater treatment market was forecast (in a 2016 repo | \$138.00B | fact (forecast) |
| × | `municipal_share_of_water_market` Municipal buyers were about 53.9% of the 2015 water treatment market. | 53.93% | derived (derived) |
| × | `capture_existing_market` Share of the municipal water-treatment market Biobot could capture. | 1.00% | assumption (assumption) |

### Manhole sampling (bottom-up) = \$6.51B

| Step | Input | Value | Source |
|---|---|---:|---|
|  | `world_population` World population used for the bottom-up sizing. | 7,750,000,000 people | assumption (external_unverified) |
| × | `urban_share_claim` Ghaeli said 70% of the world's population lived in cities (her claim;  | 70.00% | fact (claim) |
| ÷ | `people_per_catchment` People served by one sampling site (catchment) in the bottom-up sizing | 10,000 people | public source (public_source) |
| × | `samples_per_year` Monthly sampling means 12 samples per site per year. | 12 samples per site per year | derived (derived) |
| × | `price_per_sample` Price Biobot charges per analysed sample in the bottom-up sizing. | \$1,000 | assumption (assumption) |

### City contracts at the reported price floor (bottom-up) = \$13.05M

> A floor, not a TAM: it prices one minimum contract per city. The public ledger warns the \$15,000 floor cannot be multiplied by the cumulative number of tested sites to estimate revenue (many sites were pro-bono, federally funded or part of one contract), and it mixes a 2017 city count with a 2025 price.

| Step | Input | Value | Source |
|---|---|---:|---|
|  | `cities_including_smaller` Including smaller US and Canadian cities, the founders counted 870 pot | 870 cities | fact (estimate) |
| × | `municipal_price_floor` Minimum a municipality pays Biobot per year for its service, as report | \$15,000 | public source (public_source) |



## 4. The debate

Set `RUN_DEBATE = False` to skip this step and reuse the latest finished live debate already stored. Each turn prints the point, the maths (re-checked in code) and the assumptions behind it; VC follow-ups are marked `[follow-up]`.

In [41]:
if RUN_DEBATE:
    RUN_ID = run_debate()
else:
    RUN_ID = None  # reuse the latest finished live debate in the store, if there is one (read-only lookup)
    with sqlite3.connect(DATA / "debate.sqlite") as con:
        if con.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='runs'").fetchone():
            for run_id, status, cost, config in con.execute(
                    "SELECT run_id, status, total_cost_usd, config_json FROM runs WHERE status = 'completed' ORDER BY rowid DESC"):
                if not json.loads(config).get("dry_run"):
                    RUN_ID = run_id
                    SESSION["runs"].append({"run_id": run_id, "profile": PROFILE, "status": status, "cost_usd": cost})
                    break
    print(f"RUN_DEBATE = False: {'using stored run ' + RUN_ID if RUN_ID else 'no finished live debate stored; skipping the steps below'}")
RUN_ID


Biobot Analytics | run 4a2867c30c8b

[1] VC -> the panel
    Please give us your pitch: what Biobot does, who it serves and why they buy, and why this is the right moment for the business. Also tell us what you want from this meeting.
    (938 in + 2683 cache-write + 0 cache-read / 199 out, 5042 ms, $0.00765)

[2] MODERATOR -> CEO
    This question will be taken up by the CEO.
    (447 in + 0 cache-write + 0 cache-read / 43 out, 144 ms, $0.00002)

[3] CEO -> VC
    Biobot turns wastewater into public-health data for cities. The evidence supports this origin: an MIT research project that planned to start by helping cities track opioid consumption (case p.1, as of 2017). Why buyers engage: city officials and engineers pulled us toward more uses. Boston's chief engineer asked whether Charles River fecal matter was human or animal, which decides what clean-up is needed. Communities invited other officials to hear the findings, and doors kept opening to deeper engagement (case p.7, 2017). T

'4a2867c30c8b'

## 5. Naive baseline: the same debate with Opus on every call

Runs once, only when `RUN_COST_BASELINE = True`; the optimized debate above is **not** run again (with `RUN_DEBATE = False` it reuses the stored one). Every call (debate, claim extraction, fact-check verdicts, escalations and the moderator who picks who answers) goes to Opus at low effort. Costs are real (the naive run is the expensive one); the per-run spend guard in `configs/models.yaml` is $25.

In [42]:
if RUN_COST_BASELINE and RUN_ID:
    if PROFILE != "routed":
        raise ValueError("the baseline compares against a routed run; set PROFILE = 'routed'")
    OPUS_RUN = run_debate(profile="opus_low")
    cost_report = EXPORT_DIR / "cost_comparison.md"
    debate("compare", "--routed", RUN_ID, "--baseline", OPUS_RUN, save_output_to=cost_report)
    cost_report.write_text("# Cost: routed vs Opus-only\n\n```\n" + cost_report.read_text() + "```\n")
    SESSION["reports"].append(cost_report.name)
else:
    print("skipped (RUN_COST_BASELINE = False, or no optimized debate to compare with)")

Biobot Analytics | run 1f3fc5752d73

[1] VC -> the panel
    Please give us your pitch: what Biobot Analytics does today, who pays for it, and why now is the right moment for this business. In particular, tell us how you reconcile the figures in front of us. The synthetic 2026-09-30 contract book shows $14.384M contracted run-rate ARR across 89 contracts. The synthetic FY2025 recognized revenue was $9.6M, of which $0.7M was one-time services. The unverified Growjo snapshot from 2026-10-22 estimates $3.5M in revenue with a −59% change in headcount. Which numbers should we underwrite, and why?
    cites: biobot_dummy_commercial_contract_book_04, biobot_dummy_revenue_and_retention_01, biobot_public_business_and_market_01
    (1082 in + 2683 cache-write + 0 cache-read / 661 out, 7935 ms, $0.03096)

[2] MODERATOR -> CEO
    This question will be taken up by the CEO.
    (205 in + 619 cache-write + 0 cache-read / 54 out, 2323 ms, $0.00500)

[3] CEO -> VC
    Underwrite the synthetic contract

## 5b. Optimized vs naive: tokens, latency, cost

One table for the whole conversation of each run, written to `reports/optimized_vs_opus_only.md` (plus `reports/inference_comparison.md` and the complete stored results of every live run in `reports/raw/<run_id>.json`). Wall-clock time is start to finish of the debate; total call latency adds up the time of every model call.

In [43]:
res = sh(sys.executable, str(ROOT / "scripts" / "inference_metrics.py"), "--out", str(EXPORT_DIR / "inference_comparison.md"),
         "--versus-out", str(EXPORT_DIR / "optimized_vs_opus_only.md"))
print(res.stdout.strip())
display(Markdown((EXPORT_DIR / "optimized_vs_opus_only.md").read_text()))
SESSION["reports"] += ["optimized_vs_opus_only.md", "inference_comparison.md"]

wrote /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/inference_comparison.md
wrote /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/optimized_vs_opus_only.md
wrote /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/raw/4a2867c30c8b.json
wrote /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/raw/1f3fc5752d73.json


# Optimized (routed) vs Opus-only

Company `biobot`. Both columns are measured from the recorded API usage of each run; nothing is estimated. Each meeting runs until the VC says it is ready to decide, so the two runs can differ in length; compare per-turn figures as well as totals.

| Metric | Optimized (routed) | Opus only |
|---|---:|---:|
| Models | JEV (moderator, fact-check verdicts) + Haiku (claim extraction) + Sonnet (debate, escalations) + Opus (decision) | Opus for every call, the moderator included |
| Run | `4a2867c30c8b` | `1f3fc5752d73` |
| Wall-clock time for the whole debate (s) | 1,305 | 3,438 |
| Model-call latency, summed over all calls (s) | 1,293 | 2,913 |
| VC turns (opening, questions, closing remark) | 31 | 29 |
| Debate turns (VC + executives + devil) | 91 | 85 |
| API requests: Claude | 213 | 237 |
| API requests: JEV | 630 | 0 |
| System-1 requests (claim extraction, verdicts, moderator) | 721 | 0 |
| Claude input tokens (uncached) | 1,325,721 | 2,162,566 |
| Claude cache-write tokens | 19,025 | 37,696 |
| Claude cache-read tokens | 283,435 | 277,130 |
| Claude output tokens (incl. thinking) | 170,598 | 214,951 |
| JEV input tokens (billed) | 1,193,544 | 0 |
| JEV output tokens (reported by JEV, not billed) | 118,456 | 0 |
| Total tokens processed (all of the above) | 3,110,779 | 2,692,343 |
| Cost: Claude (USD) | $4.2684 | $13.1932 |
| Cost: JEV (USD) | $0.050129 | $0.000000 |
| Cost: total (USD) | $4.3185 | $13.1932 |
| Cost check | matches the sum of all recorded calls | matches the sum of all recorded calls |
| Cost per VC turn (USD) | $0.1393 | $0.4549 |
| Avg / max words per answer (executives, devil) | 315 / 488 | 308 / 470 |
| Moderator picks for main questions (CEO / CFO / CTO) | 8 / 4 / 1 | 8 / 4 / 1 |
| Claims: supported / unverifiable / contradicted / miscalculated | 492 / 115 / 41 / 12 | 708 / 64 / 19 / 2 |
| Claims settled in code (calculations) | 43 | 96 |
| Claims escalated to the stronger model | 264 | 101 |
| VC decision | pass | pass |
| Optimized saves (cost) | $8.8747 (67.3% of Opus-only) |  |
| Optimized vs Opus-only (wall-clock) | -2,133s (-62.0%) |  |

Claude input tokens exclude cache writes and reads, which are listed and priced separately; Claude output tokens include thinking. JEV bills input tokens only ($0.042 per million), so its output tokens are shown but not priced.


## 6. Evaluation dashboard

Reads saved records only; no model calls.

In [44]:
if RUN_ID:
    debate("eval-report", "--company", COMPANY, "--output", EXPORT_DIR / "evaluation_dashboard.md")
    SESSION["reports"].append("evaluation_dashboard.md")
else:
    print("skipped: no debate to evaluate")


$ debate eval-report --company biobot --output /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/evaluation_dashboard.md
evaluation dashboard: /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/evaluation_dashboard.md


## 7. Export the transcript and refresh the static pages

Write one Markdown report per run, then embed the optimized and Opus-only conversations and their selected VC decision memo in `web/biobot.html`. Local exports can contain case-derived dialogue; the GitHub Pages workflow replaces them with the public demo.

In [45]:
session_ids = [r["run_id"] for r in SESSION["runs"]]
# Export only this session's debate (plus the baseline, if you switched it on).
if session_ids:
    debate("export", *session_ids, "--out", EXPORT_DIR)
    page_args = [sys.executable, str(ROOT / "scripts" / "build_report_page.py")]
    if DRY_RUN:
        page_args.append("--include-dry-runs")
    print(sh(*page_args).stdout.strip())
else:
    print("skipped: no debate to export")

SESSION["finished_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
SESSION["total_cost_usd"] = round(sum(r["cost_usd"] or 0 for r in SESSION["runs"]), 6)
stamp = SESSION["started_at"].replace(":", "").replace("-", "")[:15]
(EXPORT_DIR / f"session_{stamp}.json").write_text(json.dumps(SESSION, indent=2))
print(f"session cost: ${SESSION['total_cost_usd']:.4f}")
print(*sorted(str(p.relative_to(ROOT)) for p in EXPORT_DIR.rglob("*.md")), sep="\n")

$ debate export 4a2867c30c8b 1f3fc5752d73 --out /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports
wrote 3 Markdown files to /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports
Embedded 2 saved runs in /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/web/biobot.html
session cost: $17.5117
reports/cost_comparison.md
reports/evaluation_dashboard.md
reports/index.md
reports/inference_comparison.md
reports/optimized_vs_opus_only.md
reports/retrieval_eval.md
reports/runs/1f3fc5752d73.md
reports/runs/4a2867c30c8b.md


## 8. Read the results here or open the HTML conversation

In [46]:
if RUN_ID:
    display(Markdown((EXPORT_DIR / "index.md").read_text()))


---
title: "NextGen Boardroom"
---

# NextGen Boardroom

*Where bold ideas meet hard questions.*

## Company conversations

| Company | Status | Decision | Messages | Cost | Started |
|---|---|---|---:|---:|---|
| [Biobot Analytics](runs/4a2867c30c8b.md) | completed | pass | 151 | $4.3185 | 2026-10-23T15:57:00.095+00:00 |
| [Biobot Analytics](runs/1f3fc5752d73.md) | completed | pass | 141 | $13.1932 | 2026-10-23T16:18:45.120+00:00 |

## Reports

- [cost comparison](cost_comparison.md)
- [evaluation dashboard](evaluation_dashboard.md)
- [inference comparison](inference_comparison.md)
- [optimized vs opus only](optimized_vs_opus_only.md)
- [retrieval eval](retrieval_eval.md)


In [47]:
# Full transcript with maths, assumptions, fact-checks and the decision
if RUN_ID:
    display(Markdown((EXPORT_DIR / "runs" / f"{RUN_ID}.md").read_text().split("---", 2)[-1]))




# Biobot Analytics · NextGen Boardroom

*Where bold ideas meet hard questions.*

## Summary

| | |
|---|---|
| Status | completed |
| Decision | pass |
| Turns | 151 |
| Claims | 874 not_checked, 492 supported, 115 unverifiable, 41 contradicted, 12 miscalculated |
| Model calls | 256 (134 System-1) |
| Tokens | 2,519,265 in / 289,054 out |
| Cost | $4.3185 |
| Prompt version | `e58ca84a` |

## Transcript

### 1. VC → the panel (round 1)

Please give us your pitch: what Biobot does, who it serves and why they buy, and why this is the right moment for the business. Also tell us what you want from this meeting.


<sub>claude-sonnet-5-5 · 938 in / 199 out · 5042 ms · $0.00765</sub>

### 2. MODERATOR → CEO (round 1)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 447 in / 43 out · 144 ms · $0.00002</sub>

### 3. CEO → VC (round 1)

Biobot turns wastewater into public-health data for cities. The evidence supports this origin: an MIT research project that planned to start by helping cities track opioid consumption (case p.1, as of 2017). Why buyers engage: city officials and engineers pulled us toward more uses. Boston's chief engineer asked whether Charles River fecal matter was human or animal, which decides what clean-up is needed. Communities invited other officials to hear the findings, and doors kept opening to deeper engagement (case p.7, 2017). That is demand pull, but I cannot verify from the evidence shown that these engagements were paid or renewed, so I treat them as strong interest, not proof of revenue. Why now: this is my opinion. Cities want faster, population-level signals and we have the research base and access to wastewater infrastructure and maps, which the case says requires buy-in. Investor validation: on 2025-06-18 Biobot announced a Series A1 led by Valor Equity Partners, with new investor Atreides Management and Hyperplane participating. Two directors joined the board. The amount and valuation were not disclosed, and I will not quote them. Early skepticism was real: MIT judges did not advance us to the finals and people asked 'why try sewage?' given the existing environmental-testing industry. Our answer is that the wastewater data and city relationships are the differentiator. What I want from this meeting: (1) your hard questions on the path from pilots to recurring contracts, (2) a view on which use case to lead with, opioids or broader infectious disease and environmental work, and (3) a conversation on whether you would back the next stage. I would need to confirm current customer counts, pricing, revenue and retention before making claims on those; I cannot state them from this evidence.

**Cites:** `biobot_fact_opioids_first` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_case_p7_c3` — HBS case 821-045, "Biobot Analytics", p.7; `biobot_case_p10_c3` — HBS case 821-045, "Biobot Analytics", p.10; `biobot_public_funding_and_investors_04` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: Biobot turns wastewater into public-health data for cities. — JEV jev-1.14.0: evidence covers it 83%
- ✅ _supported_: On 2025-06-18 Biobot announced a Series A1 financing led by Valor Equity Partners, with new investor Atreides Management and Hyperplane participating. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Two directors joined Biobot's board alongside the Series A1 announcement. — JEV jev-1.14.0: evidence covers it 89%
- ✅ _supported_: The amount and valuation of the Series A1 were not disclosed, and the speaker will not quote them. — Biobot's announcement did not disclose amount or valuation; Series A1 per Morse.
- ✅ _supported_: MIT judges did not advance Biobot to the finals. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: People asked why Biobot would try sewage given the existing environmental-testing industry. — JEV jev-1.14.0: evidence covers it 87%
- ✅ _supported_: Biobot originated as an MIT research project that planned to start by helping cities track opioid consumption, per case page 1 as of 2017. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: City officials and engineers pulled Biobot toward more uses. — JEV jev-1.14.0: evidence covers it 85%
- ✅ _supported_: Boston's chief engineer asked whether Charles River fecal matter was human or animal, which determines the clean-up needed. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Communities invited other officials to hear Biobot's findings, and doors kept opening to deeper engagement, per case page 7 from 2017. — JEV jev-1.14.0: evidence covers it 94%

<sub>claude-sonnet-5-5 · 1770 in / 1462 out · 11120 ms · $0.01494</sub>

### 4. DEVIL → CEO (round 1)

The weakest point is the 'demand pull' claim: the CEO treats anecdotes from 2016-2017 as evidence of buyer pull, and that evidence does not support the 'recurring contracts' path he is asking us to probe. The Boston chief engineer's question and the forums that officials organized were requests for research findings and one-off analyses, outside the scope of the research project (case p.7, 2017). They are not budgeted purchases. The same case says the opioid focus was chosen to gain admittance to MIT's delta v accelerator, and was backed by one small-city mayor's statement and 'executive support' (case p.11, 2017). That is a narrowing for program entry, not a validated paying market. The CEO's answers also conflict. He cites broad pull toward environmental and infectious-disease uses, yet he asks which use case to lead with, and the evidence shows the opioid choice was driven by accelerator constraints and a single mayor's agenda. In my inference, a request to be 'engaged at another level' by a city engineer does not tell us who holds the budget, whether procurement would allow a subscription, or whether a research collaboration would convert to a contract. The Series A1 announcement (2025-06-18) is investor validation, but it comes with no disclosed amount, valuation, or customer metrics, so it cannot substitute for evidence of retention. The CEO conceded that he cannot state customer counts, pricing, revenue or retention. So the pitch's core claim, that pilots become recurring contracts, currently rests on 2016-2017 anecdotes, and I cannot verify anything more recent from the evidence shown.

**Cites:** `biobot_case_p7_c3` — HBS case 821-045, "Biobot Analytics", p.7; `biobot_case_p11_c3` — HBS case 821-045, "Biobot Analytics", p.11; `biobot_fact_city_engineer_demand` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ❔ _unverifiable_: The Series A1 announcement dated 2025-06-18 is investor validation. — Announcement date and investors are supported; 'validation' is an interpretation not addressed.
- ✅ _supported_: The Series A1 announcement discloses no amount, valuation, or customer metrics. — JEV jev-1.14.0: evidence covers it 94%
- ❔ _unverifiable_: The CEO conceded that he cannot state customer counts, pricing, revenue or retention. — No evidence of a CEO concession; only that public sources lack those metrics.
- ✅ _supported_: The Boston chief engineer's question and the forums officials organized were requests for research findings and one-off analyses outside the scope of the research project (case p.7, 2017). — JEV jev-1.14.0: evidence covers it 87%
- ❔ _unverifiable_: The Boston chief engineer's question and the officials' forums were not budgeted purchases. — Evidence shows requests and forums but never says whether budgets existed.
- ✅ _supported_: The case says the opioid focus was chosen to gain admittance to MIT's delta v accelerator (case p.11, 2017). — JEV jev-1.14.0: evidence covers it 87%
- ✅ _supported_: The case says the opioid focus was backed by one small-city mayor's statement and 'executive support' (case p.11, 2017). — Case p.11 cites a small-city mayor and 'executive support at the highest level'.
- ❔ _unverifiable_: The CEO asks which use case to lead with. — No evidence that the CEO asked this in the meeting.

<sub>claude-sonnet-5-5 · 2564 in / 1516 out · 9669 ms · $0.01639</sub>

### 5. VC ASSISTANT → VC (round 1)

Facts are correct.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Market size (TAM)

### 6. VC → the panel (round 2)

How large is the annual revenue opportunity for Biobot, in dollars, and what is the maths behind it? Please size it with more than one independent method, for example bottom-up by municipal and institutional buyers and top-down from public-health or pharma data budgets. State the source of each input and mark which inputs are assumptions rather than evidence. I would also like to know which buyer segments you count and which you exclude, given that one public source puts a minimum municipal charge at $15,000 per year but warns it cannot be multiplied by cumulative tested sites.

**Cites:** `biobot_public_business_and_market_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: One public source puts a minimum municipal charge for Biobot wastewater testing at $15,000 per year. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: That same public source warns that the $15,000 minimum municipal charge cannot be multiplied by the cumulative number of tested sites to size revenue. — JEV jev-1.14.0: evidence covers it 94%

<sub>claude-sonnet-5-5 · 2849 in / 865 out · 7233 ms · $0.01331</sub>

### 7. MODERATOR → CFO (round 2)

This question will be taken up by the CFO.

<sub>jev:jev-latest · 532 in / 43 out · 104 ms · $0.00002</sub>

### 8. CFO → VC (round 2)

My estimate (not evidence) of the annual revenue opportunity is about $0.5M to $744M depending on method, and I would put the beachhead top-down in a seed deck at about $506M. It rests on case inputs (4 cities, $110B budgets, 20% public health) and only two assumptions, with an explicit share that I can defend, and it is the only method tied to a buyer budget. The share-of-water-market figure ($744.23M) is a sanity check, not a deck anchor, because it assumes 1% of the whole municipal water market and the 53.93% municipal share is derived, not cited by me here. The bottom-up using $15,000 is a floor-price view: 870 cities x $15,000 = $13.05M if every potential city bought one site, which is an upper bound for a minimum-price world, not a forecast. Segments counted: US/Canadian municipal public-health and wastewater buyers (4 beachhead cities, 71 cities over 500,000, 870 cities total per case p.11, 2017). Excluded: states, federal agencies, pharma and universities, because the evidence shows no budgets or prices for them, and I do not count cumulative tested sites. The $15,000 is a reported minimum (Cambridge Day, 2025-11-24), not an average price or a contract count, so I do not multiply it by sites. The $1,000 per-sample price is an assumption from the TAM slide, not evidence. Caveat: I cannot verify from the evidence that any of these cities pay or renew, so all figures are opportunity ceilings, not revenue.

**Maths** (✅ checks out): `Seed-deck pick: $110B x 20% = $22B; $22B x 4.6% = $1.01B; $1.01B x 50% = $506M`

**Assumptions:**

- The 4.6% share of public-health budgets that is addressable to wastewater surveillance is an assumption, not evidence
- 50% capture of that addressable pool is an assumption
- Estimates are opportunity ceilings, not forecasts of revenue

| Estimate | Maths | Check | Assumptions |
|---|---|---|---|
| Beachhead budgets (top-down) | `$110B x 20% = $22B; $22B x 4.6% = $1.01B; $1.01B x 50% = $506M` | ✅ | 4.6% of public-health spend is addressable by wastewater surveillance (assumption); 50% of that addressable pool is captured (assumption); Inputs of $110B and 20% come from the case, 2017, and are founder estimates |
| Share of existing market (top-down) | `$138.00B x 53.93% = $74.42B; $74.42B x 1.00% = $744.23M` | ✅ | Water market forecast of $138B and the 53.93% municipal share are taken from the sizing model; 1% capture of the municipal water market is an assumption; Wastewater surveillance spend is assumed to sit inside the water market, which is unverified |
| Bottom-up at reported minimum price | `870 potential cities x 1 site x $15,000 per year = $13.05M; 71 large cities x $15,000 = $1.07M` | ✅ | Each city buys one site at the $15,000 reported floor (Cambridge Day, 2025-11-24); 870 and 71 city counts are from the case, 2017; Monthly sampling at the assumed $1,000 per sample gives $12,000 per site, below the floor; No multi-site, state, federal, pharma or university buyers are counted |

**Cites:** `biobot_case_p11_c5` — HBS case 821-045, "Biobot Analytics", p.11; `biobot_sizing_share_of_existing_market` — Sizing model (facts + assumptions); `biobot_assume_price_per_sample` — Sizing input - not from the case (assumption or public source); `biobot_public_company_and_offering_05` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: Cumulative tested sites are not counted. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: The $15,000 is a reported minimum price from Cambridge Day dated 2025-11-24, not an average price or contract count. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Seed-deck calculation: $110B x 20% = $22B; $22B x 4.6% = $1.01B; $1.01B x 50% = $506M. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: Top-down beachhead calculation: $110B x 20% = $22B; $22B x 4.6% = $1.01B; $1.01B x 50% = $506M. — Arithmetic verified in code; its inputs are the speaker's own
- ❔ _unverifiable_: The beachhead top-down figure for a seed deck is about $506M. — Evidence gives $110B and $22B inputs but no $506M figure or derivation.
- ✅ _supported_: Share-of-existing-market calculation: $138.00B x 53.93% = $74.42B; $74.42B x 1.00% = $744.23M. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: Bottom-up calculation: 870 potential cities x 1 site x $15,000 per year = $13.05M; 71 large cities x $15,000 = $1.07M. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: The 870 and 71 city counts are from the case, 2017. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: Monthly sampling at the assumed $1,000 per sample gives $12,000 per site, which is below the floor. — 12 samples x $1,000 = $12,000, below $15,000 floor; $1,000 is an assumption.
- ❔ _unverifiable_: The share-of-water-market figure assumes 1% of the whole municipal water market and uses a 53.93% municipal share that is derived rather than cited by the speaker. — 53.93% is derived ($48B/$89B), but the 1% assumption and speaker citation aren't shown.
- ✅ _supported_: The bottom-up figure using $15,000 is a floor-price view in which 870 cities x $15,000 = $13.05M if every potential city bought one site. — 870 x $15,000 = $13.05M; $15,000 is a reported floor.
- ✅ _supported_: Segments counted are US/Canadian municipal public-health and wastewater buyers: 4 beachhead cities, 71 cities over 500,000, and 870 cities total per case p.11 (2017). — JEV jev-1.14.0: evidence covers it 92%

<sub>claude-sonnet-5-5 · 3768 in / 2697 out · 15558 ms · $0.02388</sub>

### 9. DEVIL → CFO (round 2)

The weakest point is the $506M 'seed-deck pick'. The CFO calls it defensible and 'the only method tied to a buyer budget', but its two decisive inputs are invented, and the evidence does not support the way it is built. The 4.6% 'addressable share' and the 50% 'capture' are both flagged as assumptions, and together they create 100% of the number's distance from the $22B budget. 50% capture of a new category inside four cities is an extreme claim for a company whose own evidence shows no confirmed paying or renewing customers; this is my inference. The $22B base is also wrong for the job: it is the founders' own estimate of public-health spend (20% of $110B, case p.11, 2017), and the case says the $110B is the cities' combined municipal budgets and that 20% was a guess by Ghaeli and Matus. The case pairs it with the opioid crisis, but it never shows what share of those budgets could be spent on wastewater monitoring. The only price evidence is the $15,000 per year minimum (Cambridge Day, 2025-11-24), and the CFO's own bottom-up for those same four beachhead cities would be 4 x $15,000 = $60,000. That is about 8,400 times smaller than $506M, and the CFO does not reconcile the gap. A number that high should not be presented as an opportunity in a deck. It also conflicts with the CFO's own estimate 3, where even all 870 cities at the floor price give only $13.05M. The $506M figure is therefore a product of two arbitrary percentages, not a market size, and it cannot be defended as 'tied to a buyer budget'. The pick should be treated as unsupported until a real price and a real adoption rate are shown. I cannot verify any such price or adoption rate from the evidence shown.

**Maths** (✅ checks out): `4 beachhead cities x $15,000 = $60,000; $506M / $60,000 = about 8,433x`

**Assumptions:**

- Each beachhead city buys one site at the reported $15,000 minimum, which is an assumption and not a contract count

**Cites:** `biobot_case_p11_c5` — HBS case 821-045, "Biobot Analytics", p.11; `biobot_fact_beachhead_public_health_spend` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_public_business_and_market_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The case states the 20% figure was a guess by Ghaeli and Matus. — JEV jev-1.14.0: evidence covers it 94%
- ❌ _contradicted_: The only price evidence is the $15,000 per year minimum reported by Cambridge Day on 2025-11-24. — Synthetic scenario records include other price points, e.g. $36,000 standard offer.
- ✅ _supported_: The CFO's bottom-up for the four beachhead cities would be 4 x $15,000 = $60,000. — Arithmetic verified in code; its inputs are the speaker's own
- ❔ _unverifiable_: The $506M figure is about 8,400 times the $60,000 bottom-up figure for the four beachhead cities. — No $506M or $60,000 figure appears in the evidence to check the ratio.
- ✅ _supported_: The CFO's estimate 3 gives only $13.05M even if all 870 cities pay the floor price. — 870 x $15,000 = $13.05M under the floor-price assumption.
- ❔ _unverifiable_: The 4.6% addressable share and the 50% capture rate are both flagged as assumptions in the case. — JEV jev-1.14.0: evidence covers it 14%
- ✅ _supported_: The $22B base is the founders' own estimate of public-health spend, calculated as 20% of $110B. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The case dates the $22B public-health spend estimate to 2017 on page 11. — JEV jev-1.14.0: evidence covers it 91%
- ✅ _supported_: The case states the $110B is the combined municipal budgets of the cities. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 4131 in / 1754 out · 10994 ms · $0.01523</sub>

### 10. VC ASSISTANT → VC (round 2)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 11. VC → the panel (round 3) — _follow-up_

The estimates span roughly $13M (bottom-up at the $15,000 floor, 870 cities) to $506M and $744M (top-down), and the two top-down figures depend on assumed shares that the case does not support. For the 4.6% addressable share and the 50% capture rate, what is the source or reasoning behind each, and which single assumption moves the result most? Separately, what actual price per city and what adoption rate do you expect, and why? Also, what price points beyond the $15,000 minimum do you know of, such as a standard annual offer, and how would they change the bottom-up figure? Finally, which one number would you put in a seed deck, and how does it reconcile with the bottom-up result?

**Cites:** `biobot_case_p11_c5` — HBS case 821-045, "Biobot Analytics", p.11; `biobot_fact_beachhead_public_health_spend` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: The bottom-up estimate is roughly $13M, computed at the $15,000 price floor across 870 cities. — 870 × $15,000 = $13.05M; both inputs are in evidence.
- ❔ _unverifiable_: The top-down estimates are $506M and $744M. — JEV jev-1.14.0: evidence covers it 14%
- ❔ _unverifiable_: The estimates span from roughly $13M (bottom-up) to $744M (top-down). — The $744M top-down figure is not in the evidence; the $13M bottom-up is only derivable.
- ❔ _unverifiable_: The two top-down figures depend on assumed shares, including a 4.6% addressable share and a 50% capture rate. — JEV jev-1.14.0: evidence covers it 13%

<sub>claude-sonnet-5-5 · 5425 in / 1006 out · 9403 ms · $0.01417</sub>

### 12. MODERATOR → CFO (round 3)

This follow-up stays with the CFO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 13. CFO → VC (round 3)

I concede the Devil's point: the $506M is not defensible as a seed-deck number. The 4.6% addressable share has no source in the evidence; I cannot trace it to anything shown. The 50% capture is labelled by the evidence as a TAM-slide judgment call, not evidence (biobot_assume_capture_beachhead, as of 2026-10-22). Capture is the assumption that moves the result most: it halves or doubles the answer linearly, and 50% of a new category in four cities with no verified paying customers is implausible. Together the two percentages produce all of the gap between $22B and $506M. The $22B base is also weak: $110B combined municipal budgets and the 20% public-health share are the founders' own estimate (biobot_case_p11_c5, 2017). The 20% is a guess, and the evidence does not show what share of those budgets could fund wastewater monitoring. I withdraw the claim that this method is 'tied to a buyer budget' in any way that supports $506M. The Devil's reconciliation is right: 4 cities x $15,000 = $60,000, about 8,400x below $506M. I cannot verify an actual adoption rate or a price list from the evidence. The only price evidence I can cite is a synthetic scenario record, not real Biobot data (biobot_dummy_pricing_and_unit_06, 2026-09-30): a $36,000 two-site contract with 48 annual samples, i.e. $18,000 per site per year, above the $15,000 floor. The Cambridge Day $15,000 minimum is not in the evidence block this turn, so I rely on my earlier citation of it and cannot re-verify it. Using the synthetic $36,000 contract raises the bottom-up: 870 cities x $36,000 = $31.32M if every city bought it, and 71 large cities x $36,000 = $2.56M. These are ceilings, not forecasts. At the synthetic cost of $12,920 per contract, contribution is $23,080 (64.1%) before platform allocation. The one number I would put in a seed deck is a bottom-up ceiling of about $31M for 870 cities at the $36,000 synthetic contract (about $13M at the $15,000 floor), with a realistic beachhead of 4 cities x $36,000 = $144,000. The top-down $506M and $744M should be dropped or shown only as non-credible upper bounds. The 1% capture in the water-market method is also an unsupported assumption, though the $744M uses a cited $138B water forecast (biobot_sizing_share_of_existing_market, 2016).

**Maths** (✅ checks out): `$110B x 20% = $22B; $22B x 4.6% x 50% = $506M; 4 cities x $15,000 = $60,000; $506M / $60,000 = about 8,433x; $36,000 / 2 sites = $18,000 per site; 870 x $36,000 = $31.32M; 71 x $36,000 = $2.56M; 4 x $36,000 = $144,000; 48 x $240 + 2 x $700 = $12,920; $36,000 - $12,920 = $23,080; $23,080 / $36,000 = 64.1%`

**Assumptions:**

- Each city buys one $36,000 two-site contract (assumption; the synthetic record is scenario ground truth, not actual Biobot data)
- The $15,000 minimum is from Cambridge Day, 2025-11-24, cited in my earlier answer but not shown in this evidence block
- Adoption: every city in the count is treated as a buyer, a ceiling only, with no adoption rate known
- Counts of 870 and 71 cities are from the case, 2017
- 4.6% addressable share has no source I can show

**Cites:** `biobot_case_p11_c5` — HBS case 821-045, "Biobot Analytics", p.11; `biobot_assume_capture_beachhead` — Sizing input - not from the case (assumption or public source); `biobot_dummy_pricing_and_unit_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_sizing_share_of_existing_market` — Sizing model (facts + assumptions)

- ✅ _supported_: 4 cities x $15,000 = $60,000, which is about 8,400 times below $506M. — Arithmetic verified in code; its inputs are the speaker's own
- ❔ _unverifiable_: $506M divided by $60,000 is about 8,433 times. — $60,000 ARR appears, but $506M does not. The arithmetic itself, 506M/60k ≈ 8,433, is correct.
- ✅ _supported_: The only price evidence cited is a synthetic scenario record, not real Biobot data (biobot_dummy_pricing_and_unit_06, 2026-09-30): a $36,000 two-site contract with 48 annual samples. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: The synthetic $36,000 two-site contract equals $18,000 per site per year, above the $15,000 floor. — $36,000 across two sites is $18,000 per site, above $15,000.
- ✅ _supported_: 870 cities x $36,000 = $31.32M if every city bought the synthetic contract; this is a ceiling, not a forecast. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: 71 large cities x $36,000 = $2.56M if every city bought the synthetic contract; this is a ceiling, not a forecast. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: At the synthetic cost of $12,920 per contract, contribution is $23,080 (64.1%) before platform allocation. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The 4.6% addressable share has no source in the evidence shown. — No 4.6% share appears anywhere in the evidence.
- ❔ _unverifiable_: The $744M water-market figure uses a cited $138B water forecast (biobot_sizing_share_of_existing_market, 2016). — The $138B 2016 forecast is shown, but the $744M calculation is not.
- ✅ _supported_: The counts of 870 and 71 cities are from the case, 2017. — JEV jev-1.14.0: evidence covers it 94%
- ❔ _unverifiable_: The 50% capture is labelled by the evidence (biobot_assume_capture_beachhead, as of 2026-10-22) as a TAM-slide judgment call, not evidence. — The biobot_assume_capture_beachhead chunk is not shown, so its label cannot be checked.
- ❔ _unverifiable_: The two percentages (4.6% and 50%) together produce all of the gap between the $22B base and $506M. — $22B is supported, but $506M and the 4.6%/50% inputs are not shown.
- ✅ _supported_: The $22B base is weak because the $110B combined municipal budgets and the 20% public-health share are the founders' own estimate (biobot_case_p11_c5, 2017). — The $110B and 20% are the founders' own 2017 estimates.

<sub>claude-sonnet-5-5 · 5745 in / 2581 out · 14259 ms · $0.02314</sub>

### 14. DEVIL → CFO (round 3)

The weakest point is the replacement seed-deck number: a roughly $31M ceiling built as 870 cities x one $36,000 contract. It swaps one unsupported assumption for another. The $36,000 is a synthetic scenario record, which the CFO himself says is not actual Biobot data (biobot_dummy_pricing_and_unit_06, 2026-09-30), yet it is presented as the deck anchor. It is also the two-site, 48-sample 'standard' contract, while the entry offer in the evidence is a one-site contract at $15,000 for 12 samples (biobot_dummy_pricing_and_unit_01, 2026-09-30). The CFO's own bottom-up pairs 870 cities, mostly small, with the larger two-site package, which I infer overstates the price for small cities. The 870 count is a 2017 founder target list (biobot_case_p11_c5), not a count of buyers with budgets. The CFO treats every city as a buyer at 100% adoption and then calls it a 'ceiling', but a ceiling that assumes universal adoption is not a market size, and he admits he has no adoption rate. At the entry price, 870 x $15,000 = $13.05M. The $31M therefore rests on a 2.4x price uplift over the entry tier for every city, with no evidence of upsell. The realistic beachhead of $144,000 for 4 cities also assumes all four buy the larger package, though the evidence shows no confirmed buyer in any of them. The 64.1% margin is also weak: it excludes platform allocation, and it uses synthetic unit costs. I cannot verify any real price, adoption rate or cost from the evidence shown.

**Maths** (✅ checks out): `870 x $36,000 = $31.32M; 870 x $15,000 = $13.05M; $36,000 / $15,000 = 2.4x`

**Assumptions:**

- Most of the 870 cities are smaller and would start at the one-site entry tier (inference, not evidence)

**Cites:** `biobot_dummy_pricing_and_unit_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_case_p11_c5` — HBS case 821-045, "Biobot Analytics", p.11

- ✅ _supported_: The replacement seed-deck number is a roughly $31M ceiling built as 870 cities x one $36,000 contract. — 870 × $36,000 = $31.3M; the inputs are in evidence, though the deck number is not.
- ❔ _unverifiable_: The CFO admits he has no adoption rate. — The CFO's admission is not in the evidence.
- ✅ _supported_: At the entry price, 870 x $15,000 = $13.05M. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: The $31M figure rests on a 2.4x price uplift over the entry tier for every city, with no evidence of upsell. — $36k/$15k = 2.4x; no upsell evidence is shown.
- ✅ _supported_: The evidence shows no confirmed buyer in any of the four beachhead cities. — The founders could not identify a buyer in 2017, and no confirmed buyer appears.
- ✅ _supported_: The 64.1% margin excludes platform allocation and uses synthetic unit costs. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: 870 x $36,000 = $31.32M. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: The $36,000 figure is a synthetic scenario record, and the CFO says it is not actual Biobot data (biobot_dummy_pricing_and_unit_06, 2026-09-30). — JEV jev-1.14.0: evidence covers it 94%
- ❔ _unverifiable_: The $36,000 figure is presented as the anchor of the seed deck. — The evidence says nothing about how a seed deck presents $36,000.
- ✅ _supported_: The $36,000 figure corresponds to the two-site, 48-sample 'standard' contract. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The entry offer in the evidence is a one-site contract at $15,000 for 12 samples (biobot_dummy_pricing_and_unit_01, 2026-09-30). — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The CFO's bottom-up pairs 870 cities, mostly small, with the larger two-site package. — Of 870 cities only 71 exceed 500k, so most are small; the pairing is plausible.
- ✅ _supported_: The 870 city count comes from a 2017 founder target list (biobot_case_p11_c5), not a count of buyers with budgets. — JEV jev-1.14.0: evidence covers it 95%
- ❔ _unverifiable_: The CFO treats every city as a buyer at 100% adoption and calls the result a ceiling. — The evidence does not show the CFO assuming 100% adoption or using the word ceiling.

<sub>claude-sonnet-5-5 · 6352 in / 1591 out · 10334 ms · $0.01948</sub>

### 15. VC ASSISTANT → VC (round 3)

Facts are correct.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Customers and who pays

### 16. VC → the panel (round 4)

Turning to customers: the evidence shows a synthetic scenario contract, SIM-MUN-001, that began as a paid pilot in 2024 and renewed in 2025 and 2026 at $36,000 for two sites, and it is labelled as invented, not a real municipality. Setting that scenario aside, who is the first real paying customer you can name, which office holds the budget and signs the cheque, what did the pilot-to-contract conversion look like, and why would that buyer pay now rather than continue with free research collaboration?

**Cites:** `biobot_dummy_commercial_contract_book_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The evidence shows a synthetic scenario contract, SIM-MUN-001, that began as a paid pilot in 2024. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The synthetic scenario contract SIM-MUN-001 renewed in 2025 and 2026 at $36,000 for two sites. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: The synthetic scenario contract SIM-MUN-001 is labelled as invented and not a real municipality. — JEV jev-1.14.0: evidence covers it 98%

<sub>claude-sonnet-5-5 · 7757 in / 539 out · 5526 ms · $0.01807</sub>

### 17. MODERATOR → CEO (round 4)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 538 in / 43 out · 104 ms · $0.00002</sub>

### 18. CEO → VC (round 4)

I cannot name a real paying customer from the evidence shown. The only contract is synthetic (SIM-MUN-001, 2026-09-30), an invented identifier, so it proves nothing about real buyers. Buyer: the 2017 case says the founders did not yet know who in a city would buy (biobot_fact_buyer_unclear). Exhibit 5 lists the end user as the mayor and staff and names NYC, Boston, SF and Washington DC as lead customers, but that is a target segmentation, not a signed customer (biobot_ex5_municipal_governments). Budget: the evidence does not identify which office or budget line pays, and I will not guess. Trigger to buy: the exhibit gives the value proposition of proactive, upstream opioid data to inform resource allocation and evaluate outreach, which is a reason to buy but not a documented trigger event. Conversion, scenario only: among 40 synthetic paid pilots at least 180 days old, 27 converted (67.5% of the cohort, 75.0% of resolved), 9 closed and 4 remain in procurement (biobot_dummy_procurement_and_sales_02). This is scenario ground truth, not real conversion. Why pay rather than keep free research: I cannot show that from the evidence; the answer would be an opinion that recurring, decision-grade data beats one-off research. To confirm, I need the signing office, the budget line, a contract copy, and renewal history for a real city.

**Maths** (✅ checks out): `27 / 40 = 67.5%; 27 / (27 + 9) = 75.0%`

**Cites:** `biobot_fact_buyer_unclear` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_ex5_municipal_governments` — Exhibit 5 (HBS case 821-045, "Biobot Analytics"); `biobot_dummy_commercial_contract_book_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: Exhibit 5 gives the value proposition of proactive, upstream opioid data to inform resource allocation and evaluate outreach. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Among 40 synthetic paid pilots at least 180 days old, 27 converted. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: 27 of 40 synthetic pilots converted is 67.5% of the cohort. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: 27 of 36 resolved synthetic pilots converted is 75.0% of resolved pilots. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Among the synthetic pilots, 9 closed and 4 remain in procurement. — JEV jev-1.14.0: evidence covers it 94%
- ❌ _contradicted_: The only contract in the evidence is SIM-MUN-001, dated 2026-09-30. — Evidence shows 89 contracts and a $14.384M book, plus the MWRA contract and ONDCP contract.
- ✅ _supported_: The contract identifier SIM-MUN-001 is an invented, synthetic identifier. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The 2017 case says the founders did not yet know who in a city would buy. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Exhibit 5 lists the end user as the mayor and staff. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Exhibit 5 names NYC, Boston, SF and Washington DC as lead customers. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The evidence does not identify which office or budget line pays. — No chunk names the paying office or budget line; 2017 sources say buyer unclear.

<sub>claude-sonnet-5-5 · 6190 in / 1405 out · 9538 ms · $0.01827</sub>

### 19. DEVIL → CEO (round 4)

The weakest point is the CEO's closing move: offering the synthetic 67.5% / 75.0% pilot conversion as 'scenario only' while still putting it in the answer to a question about real conversion. It is a trap in two ways. First, the 75.0% of resolved pilots excludes the 4 still in procurement, which is a survivor-style framing. Only the 67.5% full-cohort figure is a clean rate. Second, and more important, the CEO never says that this record contradicts his own admission that he cannot name a buyer, a budget office or a paying customer. A 40-pilot cohort with 27 conversions implies a mature sales process with a known signer, yet the CEO says the buyer is unknown. I infer these two statements cannot both describe the same real company, so the synthetic conversion figure tells the VC nothing about Biobot and risks anchoring the room on an unearned 2/3 conversion rate. The CEO also leans on Exhibit 5's 'lead customer: NYC, Boston, SF, Washington DC', which is a 2017 target list, and on the 'proactive, upstream data' benefit, which is a pitch claim, not a documented trigger. Finally, the 'why pay rather than free research' answer is an unsupported opinion. The answer to the VC's core question (who pays, from which budget, and why now) is empty. I cannot verify any real customer, signer or conversion from the evidence shown.

**Maths** (✅ checks out): `27 / 40 = 67.5%; 27 / (27 + 9) = 75.0%; 4 / 40 = 10% of cohort unresolved`

**Assumptions:**

- A cohort of 40 pilots with a 180-day maturity cut-off implies an established buyer and signer, which is my inference and not evidence

**Cites:** `biobot_fact_buyer_unclear` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_ex5_municipal_governments` — Exhibit 5 (HBS case 821-045, "Biobot Analytics"); `biobot_dummy_procurement_and_sales_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_commercial_contract_book_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: Conversion rate of 27 out of 40 pilots is 67.5%. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Conversion rate of 27 out of 36 resolved pilots (27 plus 9) is 75.0%. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The 4 pilots still in procurement represent 10% of the 40-pilot cohort. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The CEO leans on Exhibit 5's 'lead customer: NYC, Boston, SF, Washington DC', which is a 2017 target list. — Exhibit 5 lists the four cities as 2017 targets, 'beach head' markets, not contracts.

<sub>claude-sonnet-5-5 · 6239 in / 1446 out · 8945 ms · $0.01881</sub>

### 20. VC ASSISTANT → VC (round 4)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 21. VC → the panel (round 5) — _follow-up_

The fact-check says the evidence includes 89 contracts, a $14.384M book, and an MWRA and an ONDCP contract, which conflicts with the answer that only one synthetic contract exists. Which of these contracts are real, paying, and renewing, and what are each one's counterparty, annual value, start and renewal dates, and the budget line that pays? Separately, from first contact to a signed annual contract, how many days did the sales cycle take for the pilots that converted, and what share of revenue comes from federal or grant sources rather than city budgets?

**Cites:** `biobot_dummy_commercial_contract_book_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_case_p12_c4` — HBS case 821-045, "Biobot Analytics", p.12

- ✅ _supported_: The fact-check says the evidence includes 89 contracts. — JEV jev-1.14.0: evidence covers it 86%
- ✅ _supported_: The fact-check says the evidence includes a $14.384M contract book. — Evidence states contract book totals $14,384,000.
- ✅ _supported_: The fact-check says the evidence includes an MWRA contract and an ONDCP contract. — JEV jev-1.14.0: evidence covers it 92%

<sub>claude-sonnet-5-5 · 6761 in / 601 out · 5066 ms · $0.01647</sub>

### 22. MODERATOR → CEO (round 5)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 23. CEO → VC (round 5)

Correction: my earlier claim that only one synthetic contract exists was wrong. The evidence shows a synthetic contract book of 89 contracts and $14,384,000 ARR (biobot_dummy_commercial_contract_book_04, 2026-09-30). It is labelled SYNTHETIC SCENARIO GROUND TRUTH, not actual Biobot data. I cannot say any of these contracts are real, paying or renewing. SIM-MUN-001 is the only contract I saw named, and it is invented. Mix: 28 at $36,000, 41 at $96,000, 16 at $420,000 and 4 at $680,000. The record gives run-rate ARR, not recognized revenue. It names no counterparty, start or renewal date, or budget line. I also see no MWRA or ONDCP contract in the evidence shown, so I cannot verify that claim and I cannot give those fields. Sales cycle (scenario only): the median signed-contract cycle is 163 days for municipal or county buyers and 318 days for state or federal buyers, measured from qualified discovery to signed annual agreement (biobot_dummy_procurement_and_sales_03, 2026-09-30). That is a median, not a per-pilot figure, and I have no real days for converted pilots. Federal or grant share of revenue: the evidence does not split ARR by buyer type, so I do not know it. The higher-value tiers ($420,000 and $680,000) might be state or federal buyers, but that is my inference and the evidence does not show it. The 318-day state/federal cycle suggests those buyers exist in the scenario. This also weakens the Devil's point that my 40-pilot cohort and my 'no buyer known' answer conflict: the 2017 case predates this 2026 scenario, so they describe different periods. The contract book is still synthetic and proves nothing about real Biobot. It also undercuts my CFO colleague's $36,000 anchor, since the book shows larger tiers than $36,000. To confirm I need the real contract ledger with counterparty, ARR, dates, paying office, and grant versus city-budget split.

**Maths** (✅ checks out): `28 x $36,000 = $1,008,000; 41 x $96,000 = $3,936,000; 16 x $420,000 = $6,720,000; 4 x $680,000 = $2,720,000; $1,008,000 + $3,936,000 + $6,720,000 + $2,720,000 = $14,384,000; 28 + 41 + 16 + 4 = 89 contracts`

**Assumptions:**

- The higher-value tiers may be state or federal buyers (inference, not stated in the evidence)
- The synthetic contract book is scenario ground truth and not real Biobot revenue

**Cites:** `biobot_dummy_commercial_contract_book_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_fact_buyer_unclear` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: The evidence shows a synthetic contract book of 89 contracts and $14,384,000 ARR (biobot_dummy_commercial_contract_book_04, 2026-09-30). — JEV jev-1.14.0: evidence covers it 97%
- ❔ _unverifiable_: The 163-day and 318-day figures are medians across signed contracts, not per-pilot figures, and the speaker has no real days for converted pilots. — Medians over signed contracts confirmed; speaker's lack of pilot days not covered.
- ❔ _unverifiable_: The evidence does not split ARR by buyer type, so the federal or grant share of revenue is unknown. — Evidence gives tier counts; federal/grant share of revenue not stated either way.
- ✅ _supported_: 28 x $36,000 = $1,008,000. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: 41 x $96,000 = $3,936,000. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: The synthetic contract book is labelled SYNTHETIC SCENARIO GROUND TRUTH, not actual Biobot data. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: 16 x $420,000 = $6,720,000. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: 4 x $680,000 = $2,720,000. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: $1,008,000 + $3,936,000 + $6,720,000 + $2,720,000 = $14,384,000 total ARR. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: 28 + 41 + 16 + 4 = 89 contracts. — Arithmetic verified in code; its inputs are the speaker's own
- ❔ _unverifiable_: The contract named SIM-MUN-001 is the only contract the speaker saw named, and it is invented. — Evidence says SIM-MUN-001 is invented but not what the speaker saw.
- ✅ _supported_: The contract book mix is 28 contracts at $36,000, 41 at $96,000, 16 at $420,000 and 4 at $680,000. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The contract book record gives run-rate ARR, not recognized revenue. — JEV jev-1.14.0: evidence covers it 97%
- ❔ _unverifiable_: The contract book record names no counterparty, start or renewal date, or budget line. — JEV jev-1.14.0: evidence covers it 9%
- ✅ _supported_: In the scenario, the median signed-contract sales cycle is 163 days for municipal or county buyers and 318 days for state or federal buyers, measured from qualified discovery to signed annual agreement (biobot_dummy_procurement_and_sales_03, 2026-09-30). — JEV jev-1.14.0: evidence covers it 98%

<sub>claude-sonnet-5-5 · 6081 in / 2185 out · 13266 ms · $0.02112</sub>

### 24. DEVIL → CEO (round 5)

The weakest point is the CEO's closing claim that the 2017 case 'predates' the 2026 scenario, so the 'no buyer known' answer and the 40-pilot cohort do not conflict. That is a dodge. The CEO is answering for the company in 2026 and says he cannot name a buyer, office or budget line, yet he still relays a 89-contract, $14.384M book and a 163-day median cycle as if they describe Biobot. The evidence labels both as SYNTHETIC SCENARIO GROUND TRUTH, not actual Biobot data (biobot_dummy_commercial_contract_book_04, 2026-09-30). Citing a 2017 gap to excuse a 2026 gap also fails: the only buyer evidence is from 2017 (biobot_fact_buyer_unclear), and nothing shown replaces it. Second, the CEO speculates that the $420,000 and $680,000 tiers 'might be state or federal buyers' and links that to the 318-day cycle. This is an inference with no support. The book lists tier counts only, with no buyer type, so the speculation hints at federal revenue the evidence does not show. Third, the CEO says the larger tiers 'undercut' the CFO's $36,000 anchor. They do not, since they are equally synthetic and unlinked to any real city. By tier math, 20 contracts at $420,000 or $680,000 make up $9.44M, about 66% of the $14.384M ARR, so the book is concentrated in a few large accounts; I infer one lost large account could swing the total, but the CEO does not note this. The VC's question about real, paying, renewing contracts remains unanswered. I cannot verify any real contract, counterparty or renewal from the evidence shown.

**Maths** (✅ checks out): `16 x $420,000 = $6,720,000; 4 x $680,000 = $2,720,000; $6,720,000 + $2,720,000 = $9,440,000; $9,440,000 / $14,384,000 = 65.6%; 16 + 4 = 20 large contracts of 89`

**Assumptions:**

- Concentration risk is my inference from the tier mix, not a stated fact in the evidence

**Cites:** `biobot_dummy_commercial_contract_book_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_fact_buyer_unclear` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: 16 contracts at $420,000 total $6,720,000. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: 4 contracts at $680,000 total $2,720,000. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: 20 large contracts at $420,000 or $680,000 total $9,440,000. — 16×420,000=6,720,000 plus 4×680,000=2,720,000 gives 9,440,000.
- ✅ _supported_: The 20 large contracts make up about 65.6% of the $14,384,000 ARR. — 9,440,000/14,384,000 = 65.6%.
- ✅ _supported_: The 20 large contracts are 16 plus 4 out of 89 contracts. — JEV jev-1.14.0: evidence covers it 96%
- ❔ _unverifiable_: The CEO does not note this concentration. — JEV jev-1.14.0: evidence covers it 13%
- ❔ _unverifiable_: The VC's question about real, paying, renewing contracts remains unanswered. — Evidence doesn't show the VC's question or whether it was answered.
- ❔ _unverifiable_: The CEO relays an 89-contract, $14.384M book and a 163-day median cycle as if they describe Biobot. — Figures are synthetic; how the CEO framed them is not in evidence.
- ✅ _supported_: The evidence labels the 89-contract, $14.384M book and 163-day median cycle as SYNTHETIC SCENARIO GROUND TRUTH, not actual Biobot data (biobot_dummy_commercial_contract_book_04, 2026-09-30). — JEV jev-1.14.0: evidence covers it 95%
- ❔ _unverifiable_: The only buyer evidence is from 2017 (biobot_fact_buyer_unclear), and nothing shown replaces it. — No valid supporting citation
- ❔ _unverifiable_: The CEO speculates that the $420,000 and $680,000 tiers 'might be state or federal buyers' and links that to the 318-day cycle. — Whether the CEO speculated this way is not in the evidence.
- ❔ _unverifiable_: The book lists tier counts only, with no buyer type. — No valid supporting citation

<sub>claude-sonnet-5-5 · 6037 in / 1771 out · 11040 ms · $0.01928</sub>

### 25. VC ASSISTANT → VC (round 5)

Facts are correct.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Pricing and unit economics

### 26. VC → the panel (round 6)

Moving to pricing and unit economics: for the real business, what is the price per sample, per site and per city contract, what does it cost to serve each unit (lab processing, sampling logistics, analytics), and what gross margin results after platform costs? If you cannot show real figures, say which inputs you would need to verify.


<sub>claude-sonnet-5-5 · 7533 in / 322 out · 4914 ms · $0.01682</sub>

### 27. MODERATOR → CFO (round 6)

This question will be taken up by the CFO.

<sub>jev:jev-latest · 483 in / 43 out · 140 ms · $0.00002</sub>

### 28. CFO → VC (round 6)

I cannot give real prices, costs or margins for Biobot from the evidence shown; every unit figure available is synthetic scenario data (as of 2026-09-30). Scenario network view: 886 sites, 24 scheduled samples per site per year, annualized service cost $6,123,560 against $14,384,000 ARR, giving $8,260,440 gross profit and a 57.4% gross margin (biobot_dummy_pricing_and_unit_05). This is run-rate, not GAAP, and it includes the $400,000 central platform and reporting cost plus $700 per site per year for sampler amortization and maintenance (biobot_dummy_pricing_and_unit_04). It excludes sales, research and corporate overhead, so it is a cost-of-service margin only. This supersedes the 64.1% margin the Devil cited, which excluded platform allocation; the 57.4% figure includes it. The $240 per sample cost appears only inside the biobot_dummy_pricing_and_unit_05 formula; I cannot say what it covers (lab, logistics or analytics) or whether it is real, so I cannot split the three. Per-unit derivation (scenario): sample cost $240 each; site cost $700 per year; platform $400,000 / 886 sites = $451 per site per year; per-site service cost = 24 x $240 + $700 + $451 = $6,911 per year, or about $288 per sample fully loaded. Price per sample, per site and per city contract: I cannot verify these from the evidence in this turn. Earlier turns cited a $36,000 two-site contract and a $15,000 one-site, 12-sample entry offer (biobot_dummy_pricing_and_unit_06 and _01), but those chunks are not displayed here, so treat them as earlier-turn claims. Using them as illustration only: a $36,000 two-site contract with 48 samples costs about 2 x $6,911 = $13,822 at the network average, a margin of about 61.6%. An entry contract at $15,000 for 12 samples at one site would cost 12 x $240 + $700 + $451 = $4,031 on the same inputs, a 73% margin, but this ignores that fixed-site costs are not scaled to sample counts. Inputs I would need to verify: real lab cost per sample, sampling logistics, sampler lifetime, actual price lists, sites per contract, and platform cost scaling beyond 886 sites. The case evidence offers only a 2017 budget context for Boston (Recovery Services Bureau FY18 budget $509,108), which shows the size of a possible buyer budget but not any price paid (biobot_ex7_recovery_services_bureau).

**Maths** (❌ does not add up): `21,264 samples = 886 sites x 24 samples; 21,264 x $240 = $5,103,360; 886 x $700 = $620,200; $5,103,360 + $620,200 + $400,000 = $6,123,560; $14,384,000 - $6,123,560 = $8,260,440; $8,260,440 / $14,384,000 = 57.4%; $400,000 / 886 = $451 per site; 24 x $240 = $5,760; $5,760 + $700 + $451 = $6,911 per site per year; $6,911 / 24 = $288 per sample; 2 x $6,911 = $13,822; ($36,000 - $13,822) / $36,000 = 61.6%; 12 x $240 + $700 + $451 = $4,031; ($15,000 - $4,031) / $15,000 = 73.1%`

**Assumptions:**

- All cost inputs ($240 per sample, $700 per site, $400,000 platform) are synthetic scenario data, not real Biobot costs
- Platform cost of $400,000 is spread evenly per site across 886 sites
- The $36,000 two-site contract (48 samples) and $15,000 one-site (12 samples) prices come from earlier turns, not from displayed evidence, and are used only to illustrate
- The two-site contract uses the average per-site cost, including 24 samples per site, while the $36,000 contract implies 24 per site
- The entry contract's 12 samples cost the same $700 site and $451 platform allocation as a full site, an illustration only
- The $240 per sample is assumed to cover lab, logistics and analytics together; the split is unknown

**Cites:** `biobot_dummy_pricing_and_unit_05` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_ex7_recovery_services_bureau` — Exhibit 7 (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: The 57.4% margin includes the $400,000 central platform and reporting cost. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The 57.4% margin includes $700 per site per year for sampler amortization and maintenance. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The 57.4% margin excludes sales, research and corporate overhead and is therefore a cost-of-service margin only. — JEV jev-1.14.0: evidence covers it 96%
- ❌ _contradicted_: The $240 per sample cost appears only inside the biobot_dummy_pricing_and_unit_05 formula. — $240 also appears in unit_03, unit_06 and other chunks.
- ✅ _supported_: Sample cost is $240 each in the scenario. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: Site cost is $700 per year in the scenario. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Platform cost per site is $451 per year ($400,000 / 886 sites). — JEV jev-1.14.0: evidence covers it 94%
- ❔ _unverifiable_: Per-site service cost is $6,911 per year (24 x $240 + $700 + $451). — $451 platform allocation and $6,911 total not stated; 24×240+700+451=6,911 arithmetic only.
- ✅ _supported_: Every unit figure available for Biobot is synthetic scenario data as of 2026-09-30. — Unit records are labeled synthetic scenario ground truth as of 2026-09-30.
- ❔ _unverifiable_: Fully loaded cost is about $288 per sample. — Evidence gives $240 per sample; $288 not stated.
- ❔ _unverifiable_: Earlier turns cited a $36,000 two-site contract and a $15,000 one-site, 12-sample entry offer. — Both offers exist, but evidence doesn't cover what earlier turns cited.
- ❔ _unverifiable_: The earlier-turn contract chunks are not displayed in this turn. — JEV jev-1.14.0: evidence covers it 14%
- ❔ _unverifiable_: A $36,000 two-site contract with 48 samples costs about $13,822 at the network average, a margin of about 61.6%. — Evidence gives $12,920 and 64.1% before allocation; $13,822 not stated.
- ❔ _unverifiable_: An entry contract at $15,000 for 12 samples at one site would cost $4,031 on the same inputs, a margin of about 73%. — The $4,031 cost and 73% margin are not stated in evidence.
- ✅ _supported_: The case evidence offers a 2017 Boston budget context: Recovery Services Bureau FY18 budget of $509,108. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The scenario network view has 886 sites. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The number of samples in the network is 21,264 (886 sites x 24 samples). — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Sample cost across the network is $5,103,360 (21,264 x $240). — 21,264 × $240 = $5,103,360, consistent with the formula.
- ✅ _supported_: Site cost across the network is $620,200 (886 x $700). — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: Platform allocation per site is $451 ($400,000 / 886). — JEV jev-1.14.0: evidence covers it 91%
- ✅ _supported_: The scenario annualized service cost is $6,123,560. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The scenario annualized ARR is $14,384,000. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The scenario gross profit is $8,260,440. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The scenario gross margin is 57.4%. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The 57.4% margin is run-rate, not GAAP. — JEV jev-1.14.0: evidence covers it 96%
- 🧮 _miscalculated_: 21,264 samples = 886 sites x 24 samples; 21,264 x $240 = $5,103,360; 886 x $700 = $620,200; $5,103,360 + $620,200 + $400,000 = $6,123,560; $14,384,000 - $6,123,560 = $8,260,440; $8,260,440 / $14,384,000 = 57.4%; $400,000 / 886 = $451 per site; 24 x $240 = $5,760; $5,760 + $700 + $451 = $6,911 per site per year; $6,911 / 24 = $288 per sample; 2 x $6,911 = $13,822; ($36,000 - $13,822) / $36,000 = 61.6%; 12 x $240 + $700 + $451 = $4,031; ($15,000 - $4,031) / $15,000 = 73.1% — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 6215 in / 3004 out · 16216 ms · $0.02572</sub>

### 29. DEVIL → CFO (round 6)

The weakest point is the CFO's illustrative contract margins (61.6% on $36,000, 73.1% on the $15,000 entry offer). They rest on prices the CFO admits are not in the displayed evidence, and the arithmetic does not hold up. The $36,000 contract is costed at the network average, $6,911 per site. That average is built on 886 sites generating $14.384M ARR, or about $16,234 ARR per site. A two-site $36,000 contract is $18,000 per site, so it is not clearly representative of the book. The $420,000 and $680,000 tiers imply far more sites or samples per contract than 24 per site. The 57.4% network margin and the 61.6% contract margin therefore cannot both describe the same cost structure without a site count per tier, which the evidence does not give. The $15,000 entry case is worse. It charges only $700 of site cost and $451 of platform cost against 12 samples, yet the CFO concedes fixed site costs do not scale with sample count. Still, he reports 73% as if it were a finding. It is an artifact of dividing the platform cost by site count. The CFO also assumes the $240 covers lab, logistics and analytics together, which is unsupported: the chunk shows only the formula. The CFO's own label says the maths does not add up. I infer the margins are not usable for the VC's question. Separately, the $509,108 Boston budget is a 2017 figure and says nothing about price. I cannot verify any real price, cost or margin from the evidence shown.

**Maths** (✅ checks out): `$14,384,000 / 886 sites = $16,235 ARR per site; $36,000 / 2 sites = $18,000 per site; $15,000 - $4,031 = $10,969 gross profit on the entry case, but 12 samples at one site may not carry a full-site $451 platform allocation`

**Assumptions:**

- The $36,000 and $15,000 prices come from earlier turns, not displayed evidence
- Higher-value tiers likely cover more sites or samples per contract (my inference, not stated in the evidence)

**Cites:** `biobot_dummy_pricing_and_unit_05` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_ex7_recovery_services_bureau` — Exhibit 7 (HBS case 821-045, "Biobot Analytics")

- ❌ _contradicted_: The evidence does not give a site count per tier. — Evidence gives sites per buyer group: 56, 246, 384, 200.
- ❔ _unverifiable_: The $15,000 entry case charges only $700 of site cost and $451 of platform cost against 12 samples. — $700 site cost is stated; $451 platform and entry-case costing aren't.
- ❔ _unverifiable_: The CFO reports 73% for the $15,000 entry case as if it were a finding. — No evidence on how the CFO framed the 73% figure.
- ✅ _supported_: The CFO assumes the $240 covers lab, logistics and analytics together. — JEV jev-1.14.0: evidence covers it 94%
- ❌ _contradicted_: The $509,108 Boston budget is a 2017 figure. — $509,108 is the FY18 budget, not FY17 ($515,747).
- ❔ _unverifiable_: The $15,000 entry case gross profit is $10,969, calculated as $15,000 minus $4,031. — $4,031 cost isn't in evidence; subtraction itself is correct.
- ❌ _contradicted_: The $36,000 contract is costed at the network average of $6,911 per site. — Evidence costs the $36,000 contract at $12,920 using per-sample and per-site inputs.
- ❔ _unverifiable_: The network average cost of $6,911 per site is built on 886 sites generating $14.384M ARR. — 886 sites and $14.384M ARR are given; $6,911 per site isn't stated.
- ✅ _supported_: The network average of $14.384M ARR across 886 sites equals about $16,234 ARR per site. — 14,384,000/886 ≈ 16,234.
- ✅ _supported_: A two-site $36,000 contract equals $18,000 per site. — $36,000 covers two sites, so $18,000 each.

<sub>claude-sonnet-5-5 · 6201 in / 1819 out · 12926 ms · $0.01952</sub>

### 30. VC ASSISTANT → VC (round 6)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 31. VC → the panel (round 7) — _follow-up_

The synthetic record costs the $36,000 two-site contract at $12,920 before platform allocation, a 64.1% contribution margin (biobot_dummy_pricing_and_unit_06), and the network at a 57.4% margin after the $400,000 platform cost (biobot_dummy_pricing_and_unit_05). Which single price or cost input, for example the $240 per sample, the $700 per site, the $400,000 platform cost, or the $15,000 entry price, would most quickly break this margin if it moved against you, and by how much? Separately, what contracted volume of sites would cover the $400,000 platform cost at the contribution margin per site of the entry offer and of the standard two-site contract? Please show the arithmetic and state which inputs are real versus synthetic.

**Cites:** `biobot_dummy_pricing_and_unit_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_05` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The synthetic record costs the $36,000 two-site contract at $12,920 before platform allocation, a 64.1% contribution margin (biobot_dummy_pricing_and_unit_06). — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The network runs at a 57.4% margin after the $400,000 platform cost (biobot_dummy_pricing_and_unit_05). — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 7842 in / 707 out · 8528 ms · $0.01953</sub>

### 32. MODERATOR → CFO (round 7)

This follow-up stays with the CFO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 33. CFO → VC (round 7)

Most fragile input: the $240 per sample cost. It drives 5.10M of the 6.12M network service cost (83%), so a 10% rise (+$24) cuts network margin from 57.4% to about 52.1% (about 5.3 points). The $400,000 platform cost is the least sensitive (2.8% of ARR). The $15,000 entry price is not in displayed evidence, so I cannot test it as a real input. All figures are synthetic scenario data, none real Biobot data. Correction to my previous answer: the 61.6% and 73.1% margins mixed in a platform allocation and rested on prices not shown here. The $36,000 contract is supported by biobot_dummy_pricing_and_unit_06: $12,920 cost, 64.1% contribution before platform. The Devil is right that the $15,000 entry figure and the 73% margin are not usable. The $15,000 offer is from earlier turns, so I treat it as an unverified illustration and mark it as such. Inputs: 886 sites, 24 samples, $240, $700, $400,000 and $14.384M ARR come from biobot_dummy_pricing_and_unit_05; the standard-contract inputs come from _06. Breakeven site volume to cover the $400,000 platform: standard two-site contract, $23,080 contribution per contract (about $11,540 per site), so 18 contracts (about 35 sites). Entry offer (illustrative, unverified): $15,000 less 12 x $240 + $700 = $3,580 cost gives $11,420 per site contribution, so 36 contracts. The entry price and sample count are assumptions, so that result is only illustrative. Sensitivity of the $36,000 contract: the $240 cost at +10% gives 48 x $264 + $1,400 = $14,072, margin 60.9%. Price is the other lever: a 10% cut to $32,400 gives margin ($32,400-$12,920)/$32,400 = 60.1%. The more material sensitivity is mix. Network ARR per site is about $16,235 versus $18,000 per site on the standard contract, so the network average margin is lower than the standard-contract margin.

**Maths** (❌ does not add up): `21,264 samples x $240 = $5,103,360; $5,103,360 / $6,123,560 = 83.3% of service cost; +10% on $240 = $264; 21,264 x $264 = $5,613,696; $5,613,696 + $620,200 + $400,000 = $6,633,896; ($14,384,000 - $6,633,896) / $14,384,000 = 53.9%; $400,000 / $14,384,000 = 2.8%; $23,080 / 2 sites = $11,540 per site; $400,000 / $23,080 = 17.3, rounded up to 18 contracts; 18 x 2 = 36 sites; $15,000 - (12 x $240 + $700) = $15,000 - $3,580 = $11,420; $400,000 / $11,420 = 35.0, rounded up to 36 entry contracts (35.03); 48 x $264 + 2 x $700 = $12,672 + $1,400 = $14,072; ($36,000 - $14,072) / $36,000 = 60.9%; $36,000 x 0.9 = $32,400; ($32,400 - $12,920) / $32,400 = 60.1%`

**Assumptions:**

- All cost inputs are synthetic scenario data, not real Biobot costs
- The $15,000 one-site, 12-sample entry offer comes from earlier turns, not displayed evidence, so its figures are illustrative
- The 10% shock is my chosen test size, not an evidence figure
- Platform cost is treated as fixed at $400,000 regardless of site count
- Entry contract carries one site cost of $700 and no platform allocation in contribution
- Sample cost scales with sample count and is the same across contracts

**Cites:** `biobot_dummy_pricing_and_unit_05` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ❌ _contradicted_: The $240 per sample cost drives $5.10M of the $6.12M network service cost (83%). — 21,264×$240=$5.103M, which is 83.3% of $6.12M; cost matches, so supported actually
- ✅ _supported_: The inputs of 886 sites, 24 samples, $240 per sample, $700 per site, $400,000 platform cost and $14.384M ARR come from biobot_dummy_pricing_and_unit_05. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: The standard-contract inputs come from biobot_dummy_pricing_and_unit_06. — Record 06 gives $36,000, 48 samples, $240, $700 inputs
- ✅ _supported_: The standard two-site contract has $23,080 contribution, about $11,540 per site. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Covering the $400,000 platform cost requires 18 standard contracts, about 35 sites (18 x 2 = 36 sites). — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: The illustrative entry offer yields $11,420 contribution per site after $3,580 cost ($15,000 less 12 x $240 + $700). — 12×240+700=3,580; 15,000−3,580=11,420
- ✅ _supported_: The illustrative entry offer would require 36 contracts to cover the $400,000 platform cost. — 400,000/11,420=35.0 so 36 contracts needed to cover it
- ✅ _supported_: A 10% rise in the $240 per sample cost to $264 reduces the $36,000 contract margin to 60.9% ($14,072 cost). — 48×264+1,400=14,072; margin 60.9%
- ✅ _supported_: A 10% cut to the $36,000 price to $32,400 gives a 60.1% margin. — (32,400−12,920)/32,400=60.1%
- ❌ _contradicted_: A 10% rise in the $240 per sample cost (+$24) cuts network margin from 57.4% to about 52.1%, about 5.3 points. — +$24 adds $510,336 cost; margin ~53.9%, a 3.5-point drop, not 52.1%
- ✅ _supported_: Network ARR per site is about $16,235. — 14,384,000/886=16,235
- ✅ _supported_: 21,264 samples x $240 = $5,103,360 sample cost. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: $5,103,360 divided by $6,123,560 equals 83.3% of service cost. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: A 10% rise to $264 per sample, with $620,200 other service cost and $400,000 platform cost, gives total cost of $6,633,896 and a 53.9% network margin on $14,384,000 ARR. — Cost 6,633,896 and margin 53.9% check out; other cost 620,200
- ✅ _supported_: The $400,000 platform cost is 2.8% of $14,384,000 ARR. — 400,000/14,384,000=2.78%
- ❔ _unverifiable_: The $400,000 platform cost is the least sensitive input at 2.8% of ARR. — Platform is 2.8% of ARR, but least-sensitive ranking not evidenced
- ❌ _contradicted_: The $15,000 entry price is not in displayed evidence and cannot be tested as a real input. — $15,000 appears in evidence as scenario entry offer and public minimum
- ✅ _supported_: All figures in the statement are synthetic scenario data, not real Biobot data. — JEV jev-1.14.0: evidence covers it 94%
- ❔ _unverifiable_: The previously stated 61.6% and 73.1% margins mixed in a platform allocation and rested on prices not shown in the displayed evidence. — Earlier 61.6% and 73.1% figures are not in evidence
- ✅ _supported_: The $36,000 standard contract is supported by biobot_dummy_pricing_and_unit_06, with $12,920 cost and 64.1% contribution before platform. — JEV jev-1.14.0: evidence covers it 97%
- ❌ _contradicted_: The $15,000 offer is an unverified illustration from earlier turns. — Evidence documents $15,000 as scenario tier and public reported minimum
- 🧮 _miscalculated_: 21,264 samples x $240 = $5,103,360; $5,103,360 / $6,123,560 = 83.3% of service cost; +10% on $240 = $264; 21,264 x $264 = $5,613,696; $5,613,696 + $620,200 + $400,000 = $6,633,896; ($14,384,000 - $6,633,896) / $14,384,000 = 53.9%; $400,000 / $14,384,000 = 2.8%; $23,080 / 2 sites = $11,540 per site; $400,000 / $23,080 = 17.3, rounded up to 18 contracts; 18 x 2 = 36 sites; $15,000 - (12 x $240 + $700) = $15,000 - $3,580 = $11,420; $400,000 / $11,420 = 35.0, rounded up to 36 entry contracts (35.03); 48 x $264 + 2 x $700 = $12,672 + $1,400 = $14,072; ($36,000 - $14,072) / $36,000 = 60.9%; $36,000 x 0.9 = $32,400; ($32,400 - $12,920) / $32,400 = 60.1% — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 7172 in / 2707 out · 26946 ms · $0.02621</sub>

### 34. DEVIL → CFO (round 7)

The weakest point is the CFO's headline sensitivity: that a 10% rise in the $240 sample cost cuts network margin from 57.4% to 'about 52.1% (about 5.3 points)'. The CFO's own arithmetic gives 53.9%, a drop of about 3.5 points. The answer states 52.1% and then computes 53.9% in the maths block, so the lead number is wrong and the 'most fragile input' claim is overstated by roughly 50%. The standard-contract sensitivity is also inconsistent: a 10% rise in $240 moves the $36,000 contract margin only from 64.1% to 60.9%, which is 3.2 points, and that is before platform allocation. Second, the breakeven is mislabelled. The CFO says contribution is before platform allocation, so $400,000 / $23,080 = 17.3, rounded up to 18 contracts, is fine. But the entry-offer breakeven of 36 contracts rests on a $15,000 price and 12 samples that the CFO admits are not in displayed evidence, so the 36 figure has no evidentiary footing. Third, the CFO calls price the 'other lever' but tests only a 10% cut, which moves margin 4 points (64.1% to 60.1%), the same order as the cost shock. So the claim that $240 is the most fragile input is not shown against price; I infer price per contract could matter as much, and the evidence shows no price distribution. Fourth, 'mix' is called more material without any calculation of it. The network's $16,235 ARR per site against $18,000 per site is an unquantified gap, and I cannot verify from the evidence how many sites sit in each tier. All inputs are synthetic scenario data, not real Biobot figures.

**Maths** (❌ does not add up): `21,264 x $264 = $5,613,696; $5,613,696 + $620,200 + $400,000 = $6,633,896; ($14,384,000 - $6,633,896) / $14,384,000 = 53.9%; 57.4% - 53.9% = 3.5 points vs CFO's stated 5.3; 64.1% - 60.9% = 3.2 points on the $36,000 contract; 64.1% - 60.1% = 4.0 points for a 10% price cut`

**Assumptions:**

- The 10% shock size is the CFO's chosen test, not an evidence figure
- Price-lever fragility versus cost-lever fragility is my inference, since the evidence gives no price distribution

**Cites:** `biobot_dummy_pricing_and_unit_05` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_06` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The CFO's headline sensitivity states that a 10% rise in the $240 sample cost cuts network margin from 57.4% to about 52.1% (about 5.3 points). — CFO did state it; the numbers themselves are incorrect
- ✅ _supported_: The CFO calls price the other lever but tests only a 10% price cut, which moves margin from 64.1% to 60.1%, a 4.0-point drop of similar order to the cost shock. — 64.1% to 60.1% is 4.0 points
- ❔ _unverifiable_: The network's $16,235 ARR per site compared with $18,000 per site is an unquantified gap. — $16,235 supported, but $18,000 comparison not in evidence
- ✅ _supported_: All inputs are synthetic scenario data and are not real Biobot figures. — JEV jev-1.14.0: evidence covers it 91%
- ✅ _supported_: The CFO's own arithmetic gives a network margin of 53.9% after a 10% rise in the $240 sample cost, a drop of about 3.5 points from 57.4%. — 53.9% vs 57.4% is a 3.5-point drop
- ✅ _supported_: The answer states 52.1% as the lead sensitivity figure but computes 53.9% in the maths block, so the lead number is wrong. — Correct margin is 53.9%, so 52.1% lead figure is wrong
- ✅ _supported_: A 10% rise in the $240 sample cost moves the $36,000 standard-contract margin from 64.1% to 60.9%, a drop of 3.2 points, before platform allocation. — 64.1% to 60.9% is 3.2 points
- ✅ _supported_: The CFO states contribution is before platform allocation. — JEV jev-1.14.0: evidence covers it 89%
- ❌ _contradicted_: $400,000 divided by $23,080 contribution per contract is 17.3, which rounds up to 18 contracts for breakeven, and this calculation is acceptable given contribution is before platform allocation. — 400,000/23,080=17.33, rounds to 18; acceptability claim is a judgment though
- 🧮 _miscalculated_: 21,264 x $264 = $5,613,696; $5,613,696 + $620,200 + $400,000 = $6,633,896; ($14,384,000 - $6,633,896) / $14,384,000 = 53.9%; 57.4% - 53.9% = 3.5 points vs CFO's stated 5.3; 64.1% - 60.9% = 3.2 points on the $36,000 contract; 64.1% - 60.1% = 4.0 points for a 10% price cut — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 7275 in / 1840 out · 18543 ms · $0.02252</sub>

### 35. VC ASSISTANT → VC (round 7)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Technology and defensibility

### 36. VC → the panel (round 8)

Turning to defensibility: what stops a competitor, a university lab, or a city's own water utility from replicating this service, and which specific assets (sampling and lab protocols, data history across sites, validated models, procurement relationships, contracts) would still be hard to copy? Please say which of these exist in real evidence versus only in the synthetic scenario.


<sub>claude-sonnet-5-5 · 9346 in / 333 out · 4492 ms · $0.02055</sub>

### 37. MODERATOR → CTO (round 8)

This question will be taken up by the CTO.

<sub>jev:jev-latest · 493 in / 42 out · 169 ms · $0.00002</sub>

### 38. CTO → VC (round 8)

Defensibility is thin today and I will not oversell it. Real evidence (public source, 2023-08-01): Biobot is assignee of US11714030B2, 'Modular system for upstream wastewater sampling', granted 2023-08-01. That protects a particular claim set on sampling hardware; it does not give exclusivity over wastewater surveillance generally. Real evidence (2017 case, p.12): the founders believed Biobot could differentiate on analytics 'without proprietary technology', so the original thesis was not a technology moat. Replication risk is real and demonstrated: a public source dated 2026-10-22 says state/local public-health labs and university programs can collect, analyze and publish wastewater signals without buying Biobot's full service, and records MWRA ending Biobot's Massachusetts data stream in July 2026. That is a live build-versus-buy precedent. Synthetic scenario only (not actual Biobot data, 2026-09-30): 886 active sites giving longitudinal measurements. The same record states there is no head-to-head evidence that Biobot's forecasts beat another qualified lab, and that site coverage alone is not proof of an exclusive moat. Gaps: I cannot verify from the evidence shown any validated-model accuracy advantage, proprietary lab protocols, procurement relationships, or contract lock-in or renewal terms, so I do not claim them as assets. Most plausibly hard to copy over time, as my opinion: (1) the patented sampler, only if it is deployed at scale and the claims cover alternatives; (2) multi-site longitudinal data, only if it is shown to improve forecasts; (3) a standardized, comparable network across jurisdictions, which a single utility or lab cannot offer. What I would do to close the gaps: run blinded head-to-head accuracy tests against a state lab, document the patent claim scope against workarounds, and measure renewal rates. Until then the honest position is that the moat is a hypothesis, not a proven asset.

**Cites:** `biobot_public_technology_and_intellectual_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_differentiate_on_analytics` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_dummy_capacity_and_product_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The synthetic scenario record states there is no head-to-head evidence that Biobot's forecasts beat another qualified lab. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: The synthetic scenario record states that site coverage alone is not proof of an exclusive moat. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: Biobot is the assignee of US11714030B2, titled 'Modular system for upstream wastewater sampling', which was granted on 2023-08-01. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: US11714030B2 protects a particular claim set on sampling hardware and does not give exclusivity over wastewater surveillance generally. — JEV jev-1.14.0: evidence covers it 90%
- ✅ _supported_: In a 2017 case (p.12), the founders believed Biobot could differentiate on analytics without proprietary technology, so the original thesis was not a technology moat. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: A public source dated 2026-10-22 states that state/local public-health labs and university programs can collect, analyze and publish wastewater signals without buying Biobot's full service. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Records show MWRA ended Biobot's Massachusetts data stream in July 2026. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: As of 2026-09-30, in a synthetic scenario (not actual Biobot data), Biobot has 886 active sites giving longitudinal measurements. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 6896 in / 1756 out · 10617 ms · $0.02616</sub>

### 39. DEVIL → CTO (round 8)

The weakest point is the CTO's 'most plausibly hard to copy' list, especially item (3), a standardized, comparable network that 'a single utility or lab cannot offer'. This is opinion, but nothing in the evidence supports it. The only network evidence is synthetic: 886 active sites, with the same chunk saying site coverage alone is not proof of an exclusive moat (biobot_dummy_capacity_and_product_04). The 2026-10-22 public source (biobot_public_competitors_alternatives_and_06) points the other way. CDC receives data from state/local departments as well as contractors, so a standardized national feed already has non-Biobot sources. MWRA ended Biobot's Massachusetts data stream in July 2026. I infer that a buyer can swap Biobot out of the comparable-network role without losing CDC visibility. The CTO has not shown that cross-jurisdiction comparability is something customers pay for or that it blocks substitution. Item (1) is also overstated. The patent (US11714030B2, granted 2023-08-01) covers a specific claim set on upstream sampling hardware, and the evidence says it does not establish exclusivity across wastewater surveillance (biobot_public_technology_and_intellectual_03). The CTO concedes the 2017 founders' view was no proprietary technology (biobot_fact_differentiate_on_analytics). Nothing shown says the sampler is deployed at scale or that competitors need it, since agencies can collect and analyze samples without it. So all three listed assets are conditional on evidence the CTO admits is missing. Calling the moat 'a hypothesis' is accurate, but ranking the assets implies a hierarchy the evidence does not give. I cannot verify any customer lock-in, renewal or price premium for comparability.

**Assumptions:**

- That CDC's non-Biobot data sources can substitute for a Biobot network is my inference, not stated in the evidence.

**Cites:** `biobot_dummy_capacity_and_product_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_technology_and_intellectual_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_differentiate_on_analytics` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: US patent US11714030B2 was granted on 2023-08-01. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: US11714030B2 covers a specific claim set on upstream sampling hardware. — JEV jev-1.14.0: evidence covers it 85%
- ✅ _supported_: The evidence says US11714030B2 does not establish exclusivity across wastewater surveillance (biobot_public_technology_and_intellectual_03). — JEV jev-1.14.0: evidence covers it 91%
- ✅ _supported_: Agencies can collect and analyze wastewater samples without the Biobot sampler. — JEV jev-1.14.0: evidence covers it 91%
- ❌ _contradicted_: The only network evidence in the record is synthetic and shows 886 active sites. — Public sources also give network evidence (1,000+ locations, 400+ CDC sites); 886 is synthetic.
- ✅ _supported_: The chunk biobot_dummy_capacity_and_product_04 says site coverage alone is not proof of an exclusive moat. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The 2026-10-22 public source biobot_public_competitors_alternatives_and_06 points against Biobot having an exclusive network moat. — Source shows agencies can build their own wastewater data, a build-versus-buy alternative.
- ✅ _supported_: CDC receives wastewater data from state and local health departments as well as from contractors. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: A standardized national wastewater feed already has non-Biobot data sources. — CDC receives data from state/local departments, Verily and WastewaterSCAN, not only Biobot.
- ✅ _supported_: MWRA ended Biobot's Massachusetts data stream in July 2026. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 6828 in / 2736 out · 14420 ms · $0.02115</sub>

### 40. VC ASSISTANT → VC (round 8)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 41. VC → the panel (round 9) — _follow-up_

The CTO concedes the moat is a hypothesis and the record shows MWRA ended Biobot's Massachusetts data stream in July 2026 (biobot_public_competitors_alternatives_and_06). What proof exists today that a customer gets something from Biobot that a state lab or university program cannot provide, such as measured accuracy, lead time, cost per sample or reporting speed? And how many months and how much capital would a well-funded competitor or a large utility need to replicate the sampler, the lab workflow and a comparable multi-site network? Please separate real evidence from synthetic scenario data and state which inputs are your own estimates.

**Cites:** `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_dummy_capacity_and_product_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: According to the cited record (biobot_public_competitors_alternatives_and_06), the Massachusetts Water Resources Authority (MWRA) ended Biobot's Massachusetts data stream in July 2026. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 7321 in / 706 out · 9082 ms · $0.01776</sub>

### 42. MODERATOR → CTO (round 9)

This follow-up stays with the CTO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 43. CTO → VC (round 9)

Short answer: the evidence shown contains no proof that a customer gets something from Biobot that a state lab or university program cannot provide. I cannot verify measured accuracy, lead time, cost per sample or reporting speed advantages. I also cannot give a sourced month or capital figure for replication. Here is how I reached my previous answer and what I correct. (1) Inputs. The patent (US11714030B2, granted 2023-08-01) comes from biobot_public_technology_and_intellectual_03. It covers a specific claim set and does not by itself give exclusivity. The 2017 view that Biobot could differentiate 'without proprietary technology' comes from biobot_fact_differentiate_on_analytics. The substitution evidence comes from biobot_public_competitors_alternatives_and_06 (2026-10-22): agencies can collect, analyze and publish without Biobot's full service, CDC takes data from state/local departments as well as contractors, and MWRA ended Biobot's data stream in July 2026. biobot_public_contracts_customers_and_06 (2026-07) adds that the Boston Globe reported the expiring contract at $400,000/month and the replacement at about one-quarter of that as the state moved to in-house testing. That is contract value, not margin. The synthetic 886 sites and the statement that site coverage is not proof of a moat came from an earlier turn's chunk (biobot_dummy_capacity_and_product_04), which is not displayed now, so I rely on the earlier turn. (2) Correction. The Devil is right that my ranked list of hard-to-copy assets implied a hierarchy the evidence does not give. Item 3, network comparability, is the weakest: CDC already receives non-Biobot feeds, so I withdraw it as a defensible asset. It is only a hypothesis that customers pay for it. The patent and the data history are equally unproven. I treat them as unranked and conditional. (3) The MWRA precedent is the strongest real signal. It suggests price pressure: the replacement was expected to cost about a quarter as much. If the Globe figures are right, one large buyer found in-house testing cheap enough to switch. That is real evidence against a price premium. (4) Replication time and cost. I have no evidence-based figure. My opinion is that a large utility or state lab can replicate the sampling and lab workflow quickly, since MWRA did so within its own contract cycle, while a multi-site network and a validated forecast history take longer. I will not put a month or dollar number on it. The assumption that matters most is that in-house replacement is as good as Biobot's service. The evidence neither confirms nor refutes this, because no head-to-head accuracy data is shown. To close the gap I would run blinded accuracy and turnaround tests against a state lab and measure cost per sample on both sides.

**Maths** (✅ checks out): `$400,000/month x 1/4 = about $100,000/month expected replacement cost; $400,000 - $100,000 = about $300,000/month gap`

**Assumptions:**

- The Boston Globe 'about one-quarter as much' is read as roughly 25% of $400,000 per month
- The in-house program is assumed comparable in scope to the expired Biobot contract, which the evidence does not confirm
- Replication speed is my opinion and not backed by a number in the evidence

**Cites:** `biobot_public_technology_and_intellectual_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_differentiate_on_analytics` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The Boston Globe reported the replacement cost at about one-quarter of the expiring contract value as the state moved to in-house testing. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The $400,000 per month figure is contract value, not margin. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The synthetic 886 sites and the statement that site coverage is not proof of a moat came from an earlier turn's chunk that is not displayed now. — 886 sites and the not-proof-of-moat statement both appear in synthetic ground truth chunks.
- ✅ _supported_: The expected replacement cost is about $100,000 per month. — Replacement about one-quarter of $400,000/month, which is about $100,000.
- ✅ _supported_: The gap between the expiring contract and the expected replacement cost is about $300,000 per month. — $400k minus roughly $100k replacement leaves about $300k per month.
- ✅ _supported_: The patent US11714030B2 was granted 2023-08-01 and covers a specific claim set that does not by itself give exclusivity. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: A 2017 view stated that Biobot could differentiate without proprietary technology. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Agencies can collect, analyze and publish data without Biobot's full service, according to evidence dated 2026-10-22. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: CDC takes data from state and local departments as well as contractors. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: MWRA ended Biobot's data stream in July 2026. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The Boston Globe reported the expiring MWRA contract at $400,000 per month. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 6758 in / 2320 out · 14691 ms · $0.02411</sub>

### 44. DEVIL → CTO (round 9)

The weakest point is the CTO's claim that the MWRA precedent shows 'price pressure' and is 'real evidence against a price premium'. The evidence says only that MWRA ended its Biobot testing in July 2026 as the state moved toward in-house testing. It also says the Boston Globe reported the expiring contract at $400,000/month and the replacement program at about one-quarter as much (biobot_public_contracts_customers_and_06, 2026-07). It does not say the two programs have the same scope, sample count, sites, turnaround or deliverables, and the CTO's own assumptions admit this. A quarter-cost replacement could cover a smaller program, so the $300,000/month 'gap' is not a like-for-like price comparison. I infer it may reflect scope reduction or a state choice to fund less, not a lower Biobot price point. The CTO also says MWRA replicated the sampling and lab workflow 'within its own contract cycle'. The evidence shows a decision and a transition toward in-house testing, not that replication was completed, how long it took, or that it works as well. That claim is opinion presented as precedent. The evidence is a single customer, so it cannot support a general 'no price premium' conclusion. The $100,000 and $300,000 figures rest on reading 'about one-quarter' as exactly 25%, and the source calls the figure contract value, not margin. I cannot verify from the evidence what Biobot's margin on that contract was, so the dollar gap says nothing about Biobot's economics. The CTO's opening that no proof exists of customer benefit is fair, but the follow-on inference about price is stronger than the single reported data point allows.

**Maths** (✅ checks out): `$400,000/month x 25% = $100,000/month; $400,000 - $100,000 = $300,000/month`

**Assumptions:**

- 'About one-quarter' is read as exactly 25% of $400,000/month
- Scope mismatch between the expired Biobot contract and the in-house program is my inference, not stated in the evidence

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The source describes the figure as contract value, not margin. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: $400,000/month x 25% = $100,000/month. — One-quarter of $400,000 is $100,000; arithmetic correct.
- ✅ _supported_: $400,000/month - $100,000/month = $300,000/month. — Subtraction is correct given the reported figures.
- ✅ _supported_: MWRA ended its Biobot testing in July 2026 as the state moved toward in-house testing. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The Boston Globe reported the expiring MWRA Biobot contract at $400,000/month. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The Boston Globe reported the replacement MWRA program at about one-quarter of the expiring contract value. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 6634 in / 1533 out · 11444 ms · $0.01991</sub>

### 45. VC ASSISTANT → VC (round 9)

Facts are correct.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Ideal customer profile and beachhead

### 46. VC → the panel (round 10)

Moving to the beachhead: who exactly is the ideal first customer (type of buyer, size, and the specific pain they feel that they cannot solve with an in-house or state lab), and why is that segment the one to win first given that a large buyer like MWRA moved to in-house testing in July 2026? Please say which parts rest on real evidence, which on synthetic scenario data, and which are your own estimates.

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: MWRA moved to in-house testing in July 2026. — JEV jev-1.14.0: evidence covers it 90%

<sub>claude-sonnet-5-5 · 7216 in / 429 out · 6662 ms · $0.01666</sub>

### 47. MODERATOR → CEO (round 10)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 510 in / 43 out · 160 ms · $0.00002</sub>

### 48. CEO → VC (round 10)

My proposed beachhead is a mid-sized municipality or county/metro public-health agency that has no in-house lab and cannot staff one. I would not lead with state or multi-region agencies. The evidence does not prove this segment is the right one, so treat it as a hypothesis. Evidence by part: (1) REAL, 2026-07: MWRA ended Biobot testing at the end of July 2026 as the state moved to in-house testing. The Globe reported a $400,000/month contract and a replacement at about one-quarter of that (biobot_public_contracts_customers_and_06). That is a material loss, and it shows large, well-resourced buyers can substitute. It is contract value, not margin. I read it as a reason to avoid the large state-level buyer first, which is my opinion. (2) SYNTHETIC, 2026-09-30, not actual Biobot data: the sales database holds 420 reachable accounts, made up of 220 municipalities, 150 county or metro agencies and 50 state or multi-region agencies. These are a screened list, not a published TAM (biobot_dummy_market_sizing_scenario_01). Municipalities and counties are 370 of 420 accounts, about 88%, so they are the most numerous reachable buyers. Nothing shown says how many of them can build in-house, their budgets, or their willingness to pay. (3) REAL, 2017: the founders' beachhead was four cities spending about $22B a year on public health (biobot_fact_beachhead_public_health_spend). That supports city-level buyers as the original target, but it is a budget pool, not proof of paying customers. Why this segment first, as my opinion: it is the largest group of reachable accounts, and it is the one least able to copy the MWRA in-house path. The pain I would target is getting a comparable, regularly reported wastewater signal without building a lab. What I cannot verify: that these buyers feel this pain, their price tolerance, renewal rates, or accuracy versus a state lab. To confirm, I would need win/loss and renewal data by segment, each segment's current lab access, and a pilot priced against the in-house cost.

**Maths** (✅ checks out): `220 municipalities + 150 county/metro agencies = 370 accounts; 370 / 420 = 88.1%`

**Assumptions:**

- Municipalities and county/metro agencies are assumed to lack in-house lab capacity; the evidence does not state this
- Avoiding state-level buyers first is my opinion, based on one MWRA data point
- The pain of a comparable signal without building a lab is hypothesized, not shown in evidence

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_dummy_market_sizing_scenario_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_fact_beachhead_public_health_spend` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: MWRA ended Biobot testing at the end of July 2026 as the state moved to in-house testing. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The sales database holds 420 reachable accounts: 220 municipalities, 150 county or metro agencies, and 50 state or multi-region agencies. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The 420 reachable accounts are a screened list, not a published total addressable market. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Municipalities and county/metro agencies make up 370 of 420 reachable accounts, about 88%. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Municipalities and county/metro agencies are the most numerous reachable buyer group in the sales database. — 220 municipalities exceed 150 county/metro and 50 state accounts; most numerous.
- ✅ _supported_: The founders' original beachhead was four cities spending about $22B a year on public health, as of 2017. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: The Globe reported that the MWRA Biobot contract was $400,000 per month and that its replacement cost about one-quarter of that amount. — JEV jev-1.14.0: evidence covers it 97%
- ❌ _contradicted_: 370 accounts equals 220 municipalities plus 150 county/metro agencies. — 220 + 150 = 370, which matches the arithmetic, but the ref is C10 CEO's claim; sum correct.
- ✅ _supported_: 370 divided by 420 equals 88.1%. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: The MWRA contract value is contract value, not margin. — JEV jev-1.14.0: evidence covers it 92%

<sub>claude-sonnet-5-5 · 6070 in / 2912 out · 16545 ms · $0.02105</sub>

### 49. DEVIL → CEO (round 10)

The weakest point is the CEO's claim that municipalities and counties are the segment 'least able to copy the MWRA in-house path' and feel the pain of getting a signal 'without building a lab'. The CEO's own assumptions admit the evidence does not say this. The 88% figure only shows that these accounts are numerous in a synthetic, sales-screened list (biobot_dummy_market_sizing_scenario_01, 2026-09-30). It says nothing about lab access, budgets or willingness to pay. Account count is not buyer quality, and I infer that small buyers likely pay far less per account than MWRA's reported $400,000/month (biobot_public_contracts_customers_and_06, 2026-07). One MWRA loss at that size could take many small contracts to replace, so the segment logic may trade away revenue per account without a proven gain in retention. The CEO also uses the 2017 four-city $22B spend (biobot_fact_beachhead_public_health_spend) as support for city buyers. That is a total public-health budget, not a wastewater-surveillance budget, and it concerns four large cities, not mid-sized municipalities. The 2017 sizing model's 20% public-health share and 50% capture are estimates, not evidence of demand (biobot_sizing_beachhead_budgets). Finally, the MWRA precedent is one buyer. Treating it as a reason to avoid state buyers, then assuming smaller buyers cannot substitute, applies the same thin evidence in opposite directions. I cannot verify from the evidence shown that any mid-sized buyer has paid for, renewed or lacks alternatives to Biobot. In my inference, the state and university labs that the evidence says can publish signals without Biobot may also serve smaller agencies cheaply or for free, which would undercut the 'cannot build a lab' pain.

**Assumptions:**

- Smaller municipal and county contracts being much smaller than MWRA's is my inference; the evidence shows no per-account prices for these segments
- State and university labs serving smaller agencies cheaply is my inference, not stated in the evidence shown

**Cites:** `biobot_dummy_market_sizing_scenario_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_beachhead_public_health_spend` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_sizing_beachhead_budgets` — Sizing model (facts + assumptions)

- ✅ _supported_: The 2017 four-city $22B figure is a total public-health budget, not a wastewater-surveillance budget. — JEV jev-1.14.0: evidence covers it 83%
- ✅ _supported_: The 2017 four-city $22B figure concerns four large cities, not mid-sized municipalities. — JEV jev-1.14.0: evidence covers it 95%
- ❔ _unverifiable_: The 2017 sizing model's 20% public-health share and 50% capture rate are estimates, not evidence of demand (biobot_sizing_beachhead_budgets). — 20% is labeled an estimate; the 50% capture rate is not in evidence.
- ❔ _unverifiable_: The MWRA precedent is a single buyer. — MWRA is one buyer's loss, but evidence doesn't call it the only precedent.
- ✅ _supported_: The 88% figure shows that municipal and county accounts are numerous in a synthetic, sales-screened list (biobot_dummy_market_sizing_scenario_01, dated 2026-09-30). — 370 of 420 is 88.1%; list is synthetic and sales-screened.
- ✅ _supported_: MWRA's reported spend is $400,000 per month (biobot_public_contracts_customers_and_06, dated 2026-07). — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 5929 in / 1799 out · 10656 ms · $0.01932</sub>

### 50. VC ASSISTANT → VC (round 10)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 51. VC → the panel (round 11) — _follow-up_

The CEO proposed mid-sized municipalities and county or metro agencies as the beachhead, and the only count shown is the synthetic 420-account sales list (biobot_dummy_market_sizing_scenario_01). What evidence shows that any such buyer has actually bought, paid for or renewed Biobot's service, and how many of the 370 municipal and county accounts are real, qualified prospects with no in-house or state lab access and a funded budget line? Please say how you would count them, the source of each input, and which numbers are real, synthetic or your own estimates.

**Cites:** `biobot_dummy_market_sizing_scenario_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ❔ _unverifiable_: The CEO proposed mid-sized municipalities and county or metro agencies as the beachhead for Biobot. — Evidence has no record of the CEO proposing this beachhead; only account and contract data.
- ❔ _unverifiable_: The only count shown in the meeting is a synthetic 420-account sales list (biobot_dummy_market_sizing_scenario_01). — The 420 list exists, but 'only count shown in meeting' is not covered.
- ❌ _contradicted_: The 370 municipal and county accounts figure is a count that is part of the synthetic 420-account list. — Municipal 220 plus county 150 equals 370, so it is a subset of 420; not contradicted.

<sub>claude-sonnet-5-5 · 6694 in / 628 out · 5049 ms · $0.01614</sub>

### 52. MODERATOR → CEO (round 11)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 53. CEO → VC (round 11)

Answer: the evidence shown does not prove that any municipal or county buyer has bought, paid for or renewed Biobot, and it does not let me count real qualified prospects among the 370. The only adoption data is synthetic scenario data (2026-09-30): 89 active paying programs, 21.2% of the 420 accounts, with $14,384,000 contracted ARR (biobot_dummy_market_sizing_scenario_03). It does not split those 89 programs by segment, so I cannot say how many are municipal or county, or whether any renewed. Nothing real shows a paid mid-sized buyer; the real 2026-07 MWRA item is a loss, not a win.

How I got my previous answer: the 420 accounts (220 municipalities, 150 county or metro, 50 state or multi-region) come from biobot_dummy_market_sizing_scenario_01. It is a sales-screened list that excludes accounts already ruled out for procurement, which is not the same as qualified, lab-less, funded prospects. The 2017 $22B came from biobot_fact_beachhead_public_health_spend.

Corrections: the Devil is right that I overreached. (1) The 88% is only account count. It says nothing about lab access, budget or willingness to pay. (2) The $22B is total public-health spend for four large cities, not a wastewater budget and not mid-sized buyers, so I withdraw it as support for this segment. (3) I applied one MWRA data point to avoid state buyers and then assumed smaller buyers cannot substitute. That is inconsistent, so the beachhead is a hypothesis with no evidence behind it.

How I would count real qualified prospects: start with the 370 accounts, then remove each account that (a) has in-house or state/university lab access, (b) has no funded budget line, or (c) is in a procurement cycle that cannot be entered. The survivors are the qualified count. I have no input for any of these filters, so I will not give a number. The arithmetic below shows only what the synthetic list bounds: at most 370 and with no filter applied.

Assumption that matters most: that these buyers lack lab access and have a funded budget. If most can use a state or university lab for free or cheaply, the segment is much smaller than 370. To confirm I need segment-level paid, renewal and churn data for the 89 programs, a lab-access check per account, and budget confirmation.

**Maths** (✅ checks out): `220 municipalities + 150 county/metro = 370 accounts; 370 / 420 = 88.1%; 89 / 420 = 21.2%; $14,384,000 / 89 = about $161,618 average ARR per paying program`

**Assumptions:**

- The 89 paying programs are not split by segment in the evidence, so no municipal or county share can be derived
- Qualified prospects after lab-access and budget filters cannot be counted from the evidence; the filters are my proposed method
- The average ARR per program is a simple mean across all 89 programs and is not specific to municipal or county buyers

**Cites:** `biobot_dummy_market_sizing_scenario_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_market_sizing_scenario_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_fact_beachhead_public_health_spend` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: 370 accounts out of 420 is 88.1% of accounts, the upper bound of the synthetic list with no filter applied. — 370/420 is 88.1%; 'no filter' framing is interpretive but consistent.
- ✅ _supported_: 89 paying programs out of 420 accounts is 21.2%. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: $14,384,000 contracted ARR divided by 89 programs is about $161,618 average ARR per paying program. — JEV jev-1.14.0: evidence covers it 97%
- ❔ _unverifiable_: The only adoption data is synthetic scenario data dated 2026-09-30 showing 89 active paying programs, 21.2% of the 420 accounts, and $14,384,000 contracted ARR (biobot_dummy_market_sizing_scenario_03). — Figures match, but 'only adoption data' is not verifiable; other cohort data exists.
- ✅ _supported_: The real 2026-07 MWRA item is a loss, not a win. — MWRA Biobot testing ended July 2026; described as a material customer loss.
- ✅ _supported_: The 420 accounts (220 municipalities, 150 county or metro, 50 state or multi-region) come from biobot_dummy_market_sizing_scenario_01. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The 420-account list is sales-screened and excludes accounts already ruled out for procurement, which is not the same as qualified, lab-less, funded prospects. — List is sales-screened, excluding accounts ruled out for procurement; no lab or funding filter.
- ✅ _supported_: The 2017 $22B figure came from biobot_fact_beachhead_public_health_spend. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 6385 in / 2224 out · 13631 ms · $0.02251</sub>

### 54. DEVIL → CEO (round 11)

The weakest point is the CEO's use of the synthetic 89 paying programs and $14,384,000 ARR as 'the only adoption data' and the derived average of about $161,618 per program. That average is a mean across an unsplit mix, so it cannot describe municipal or county buyers, and it invites a false read of beachhead revenue. In my inference, a few large state or multi-region contracts could dominate the total and pull the mean far above what a mid-sized municipality pays. The CEO admits the split is missing but still computes the figure. The 420-account list is described as 'qualified' in biobot_dummy_market_sizing_scenario_03 but 'sales-screened, not ruled out for procurement' in scenario_01, and the CEO relies on the second wording only to reject the first. The evidence also says the $14,384,000 is 33.2% of a $43,320,000 'planning opportunity' while 89 programs are 21.2% of accounts, so contract mix is uneven by the source's own words (scenario_03). That supports my inference that the mean is unrepresentative. Also, the filter method (lab access, budget line, procurement cycle) is untested, and the CEO admits no inputs exist, so it is a checklist, not a count. Nothing real in the evidence shows any municipal or county buyer paid or renewed, so the beachhead remains an untested guess rather than a segment. Finally, the 2017 $511M sizing (biobot_sizing_beachhead_budgets) rests on a 50% capture assumption and is not evidence of demand.

**Maths** (✅ checks out): `$14,384,000 / 89 = about $161,618 average ARR per program; 21.2% of accounts vs 33.2% of planning opportunity as ARR share shows unequal contract sizes`

**Assumptions:**

- A few large state or multi-region contracts could dominate the ARR total; this is my inference, not stated in the evidence
- Municipal contracts being smaller than the mean is my inference

**Cites:** `biobot_dummy_market_sizing_scenario_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_market_sizing_scenario_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_sizing_beachhead_budgets` — Sizing model (facts + assumptions)

- ❔ _unverifiable_: The CEO uses the synthetic 89 paying programs and $14,384,000 ARR as 'the only adoption data'. — Numbers match, but the CEO's 'only adoption data' phrasing is not evidenced.
- ✅ _supported_: The 89 paying programs are 21.2% of accounts according to scenario_03. — JEV jev-1.14.0: evidence covers it 99%
- ❔ _unverifiable_: The filter method (lab access, budget line, procurement cycle) is untested. — No evidence on whether the filter method was tested.
- ❔ _unverifiable_: The CEO admits no inputs exist for the filter method, so it is a checklist rather than a count. — No evidence of a CEO admission about filter inputs.
- ✅ _supported_: Average ARR per program is about $161,618, calculated as $14,384,000 divided by 89. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The CEO derives an average of about $161,618 ARR per program from $14,384,000 divided by 89 programs. — JEV jev-1.14.0: evidence covers it 96%
- ❌ _contradicted_: The CEO admits the split of programs by buyer type is missing but still computes the average figure. — Evidence gives program split by buyer type, so it is not missing.
- ✅ _supported_: The 420-account list is described as 'qualified' in biobot_dummy_market_sizing_scenario_03 and as 'sales-screened, not ruled out for procurement' in scenario_01. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The $14,384,000 is 33.2% of a $43,320,000 'planning opportunity' according to scenario_03. — JEV jev-1.14.0: evidence covers it 98%

<sub>claude-sonnet-5-5 · 6280 in / 1608 out · 10382 ms · $0.01934</sub>

### 55. VC ASSISTANT → VC (round 11)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Go-to-market and scaling

### 56. VC → the panel (round 12)

The synthetic contract book (2026-09-30) lists 28 municipal programs at an average $36,000, 41 county or metro at $96,000, 16 state at $420,000 and 4 federal or multi-region at $680,000 (biobot_dummy_commercial_contract_book_02). Earlier turns said the 89 programs were not split by segment, so which is right, and what does this split show? If it holds, how would you win the next municipal and county accounts and scale through procurement, and what is the sales cycle, cost to acquire each account and renewal evidence behind that? Please mark each input as real, synthetic or your own estimate.

**Cites:** `biobot_dummy_commercial_contract_book_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_market_sizing_scenario_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The synthetic contract book dated 2026-09-30 lists 28 municipal programs at an average of $36,000. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The synthetic contract book dated 2026-09-30 lists 41 county or metro programs at an average of $96,000. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The synthetic contract book dated 2026-09-30 lists 16 state programs at an average of $420,000. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The synthetic contract book dated 2026-09-30 lists 4 federal or multi-region programs at an average of $680,000. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The synthetic contract book (biobot_dummy_commercial_contract_book_02) is a synthetic dataset, not real company records. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The segment counts in the synthetic contract book sum to 89 programs (28 + 41 + 16 + 4). — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: Estimated total contract value from the synthetic book's segment averages is $1,008,000 for municipal programs (28 x $36,000), $3,936,000 for county or metro programs (41 x $96,000), $6,720,000 for state programs (16 x $420,000) and $2,720,000 for federal or multi-region programs (4 x $680,000), totaling $14,384,000. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 7529 in / 1131 out · 14222 ms · $0.01839</sub>

### 57. MODERATOR → CEO (round 12)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 582 in / 43 out · 159 ms · $0.00002</sub>

### 58. CEO → VC (round 12)

The contract book is right and my earlier statement was wrong: I said the 89 programs were not split by segment, but biobot_dummy_commercial_contract_book_01 (synthetic, 2026-09-30) splits them into 28 municipal, 41 county or metro, 16 state and 4 federal or multi-region. I withdraw that claim and my $161,618 average, which hid the mix. All of this is synthetic scenario data, not actual Biobot data, and the Devil's inference that large contracts dominate the mean is confirmed. Municipal and county programs are 69 of 89, or 77.5%, but only about $5.0M of $14.0M, or 36%. State and federal programs are 20 programs, 22.5%, and about $9.4M, or 65%. The book totals $14,383,000, close to the $14,384,000 stated ARR; I treat the $1,000 gap as rounding and cannot verify it. Revenue is therefore concentrated in the buyers who take 318 days to sign and who can substitute in-house, as MWRA did in real 2026-07 reporting. What the split does not show: renewals, churn, margin, acquisition cost or lab access by segment. The evidence is silent on cost to acquire an account and on renewal rates, so I give no number. The sales-cycle medians are synthetic and measured from qualified discovery to signed annual agreement. The sequence of first customers is my plan, an opinion. 1) Counties and metro agencies first: they average $96,000 versus $36,000 for municipalities, with a 163-day cycle. Target about 5.5 months from qualified discovery to signature. 2) Municipalities second, ideally as a paid pilot, which the source treats as a separate stage and not proof of renewal. Municipal deals at $36,000 are about 38% of a county deal, so I would reach them through county or regional bundles rather than one by one, where the same procurement can cover several cities. That bundling is a hypothesis. 3) State and federal buyers last, with a roughly 318-day cycle (about 10.5 months), and only after the in-house risk is understood. The procurement path: qualify budget and lab access in discovery, run a paid pilot, convert to an annual agreement, and track renewal as the proof point. To confirm, I need acquisition cost per account, pilot-to-annual conversion, renewal and churn by segment, and in-house substitution rates.

**Maths** (✅ checks out): `28 + 41 = 69 municipal and county programs; 69 / 89 = 77.5%; 28 x $36,000 = $1,008,000; 41 x $96,000 = $3,936,000; $1,008,000 + $3,936,000 = $4,944,000; 16 x $420,000 = $6,720,000; 4 x $680,000 = $2,720,000; $6,720,000 + $2,720,000 = $9,440,000; $4,944,000 + $9,440,000 = $14,384,000; $4,944,000 / $14,384,000 = 34.4%; $9,440,000 / $14,384,000 = 65.6%; 16 + 4 = 20 programs; 20 / 89 = 22.5%; $36,000 / $96,000 = 37.5%; 163 days / 30.4 = 5.4 months; 318 days / 30.4 = 10.5 months`

**Assumptions:**

- Contract-book average values are applied to every program in the segment, so segment totals are approximations
- Rounding explains the match between the $14,384,000 book total and stated ARR
- The first-customer sequence (county, then municipal, then state) is my opinion
- Bundling municipalities through county or regional procurement is an untested hypothesis
- Programs in the book are assumed to include any renewals; the evidence gives no renewal data

**Cites:** `biobot_dummy_commercial_contract_book_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_commercial_contract_book_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ❌ _contradicted_: The contract book totals $14,383,000, close to the stated ARR of $14,384,000. — Book totals exactly $14,384,000, not $14,383,000.
- ✅ _supported_: Revenue is concentrated in buyers who take 318 days to sign and who can substitute in-house, as MWRA did in real 2026-07 reporting. — 318-day state/federal cycle, large revenue share, MWRA in-house move in 2026-07 all supported.
- ❌ _contradicted_: The split does not show renewals, churn, margin, acquisition cost or lab access by segment. — Evidence includes renewal and margin data, though not by segment; segment claim unclear.
- ✅ _supported_: The sales-cycle medians in the data are synthetic and measured from qualified discovery to signed annual agreement. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Counties and metro agencies average $96,000 per program versus $36,000 for municipalities. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: County and metro programs have a 163-day sales cycle. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Municipal deals at $36,000 are about 38% of a county deal, calculated as $36,000 / $96,000 = 37.5%. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The contract book biobot_dummy_commercial_contract_book_01 (synthetic, dated 2026-09-30) splits 89 programs into 28 municipal, 41 county or metro, 16 state and 4 federal or multi-region. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: All of the scenario data is synthetic and not actual Biobot data. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Municipal and county programs total 69 of 89 programs, which is 77.5%. — JEV jev-1.14.0: evidence covers it 97%
- ❌ _contradicted_: Municipal and county programs account for about $5.0M of the $14.0M book, about 36%. — Municipal+county = 28×36k+41×96k = $4.944M of $14.384M, about 34.4%, not $5.0M/$14.0M/36%.
- ✅ _supported_: State and federal programs total 20 programs, which is 22.5% of 89. — 16+4=20 of 89 = 22.5%.
- ❌ _contradicted_: State and federal programs account for about $9.4M of the book, about 65%. — State+federal = 6.72M+2.72M = $9.44M, about 65.6% of $14.384M; roughly consistent, but the $14.0M base differs.

<sub>claude-sonnet-5-5 · 6700 in / 2708 out · 15352 ms · $0.02511</sub>

### 59. DEVIL → CEO (round 12)

The weakest point is the CEO's county-first plan, which rests on one synthetic cycle figure that does not fit the evidence. The CEO says counties are a '163-day cycle' and targets about 5.5 months. The source gives 163 days as one median for municipal OR county buyers combined (biobot_dummy_procurement_and_sales_03, 2026-09-30). Nothing shown gives a county-only cycle, so the claim that counties are faster than municipalities is unsupported. The CEO also never reaches a unit-economics conclusion. With no acquisition cost, no renewal data and no margin, the plan cannot show a $96,000 county deal is worth winning. A 5.4-month median also means half of deals take longer. I infer this strains cash if each deal needs a paid pilot first, since a pilot is a separate stage and not proof of renewal. The bundling idea, where one regional procurement covers several $36,000 municipal deals, is an untested hypothesis with no evidence of any such procurement path. I infer it may stretch the cycle beyond 163 days, because a multi-party deal needs more approvals. The 36% versus 65% revenue split also cuts against the plan. Winning counties first targets the segment that brings in about a third of ARR, and it delays the buyers who hold two-thirds. Finally, there is a small arithmetic inconsistency. The CEO's text says the book totals $14,383,000 and calls the $1,000 gap rounding. The CEO's own maths gives exactly $14,384,000, so no gap exists and the 'rounding' remark is wrong. The text also states 36% and 65% while the maths shows 34.4% and 65.6%. The prose percentages do not match the CEO's own calculation.

**Maths** (❌ does not add up): `28 x $36,000 + 41 x $96,000 = $1,008,000 + $3,936,000 = $4,944,000; $4,944,000 / $14,384,000 = 34.4%; 16 x $420,000 + 4 x $680,000 = $9,440,000; $4,944,000 + $9,440,000 = $14,384,000`

**Assumptions:**

- Multi-party bundled procurement lengthening the cycle beyond 163 days is my inference, not stated in the evidence
- Paid pilots straining cash before annual signature is my inference; no pilot price or cost is shown

**Cites:** `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_commercial_contract_book_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ❔ _unverifiable_: There is no evidence in the meeting of any such bundled procurement path. — The 'bundled procurement path' is undefined in the evidence.
- ❔ _unverifiable_: The revenue split is 36% versus 65% in the CEO's prose. — JEV jev-1.14.0: evidence covers it 10%
- ❔ _unverifiable_: The CEO's text states the book totals $14,383,000 and calls a $1,000 gap rounding. — Evidence lacks the CEO's meeting text; it only gives $14,384,000.
- ✅ _supported_: The CEO's own arithmetic gives exactly $14,384,000, so no gap exists and the 'rounding' remark is wrong. — The arithmetic sums exactly to $14,384,000, so no gap exists.
- 🧮 _miscalculated_: 28 x $36,000 + 41 x $96,000 = $4,944,000. — Arithmetic does not match the stated result
- ✅ _supported_: $4,944,000 / $14,384,000 = 34.4%. — Arithmetic verified in code; its inputs are the speaker's own
- 🧮 _miscalculated_: 16 x $420,000 + 4 x $680,000 = $9,440,000. — Arithmetic does not match the stated result
- ✅ _supported_: $4,944,000 + $9,440,000 = $14,384,000. — Arithmetic verified in code; its inputs are the speaker's own
- ❔ _unverifiable_: The prose percentages of 36% and 65% do not match the CEO's own calculation of 34.4% and 65.6%. — JEV jev-1.14.0: evidence covers it 14%
- ✅ _supported_: The source gives 163 days as one median for municipal or county buyers combined (biobot_dummy_procurement_and_sales_03, dated 2026-09-30). — JEV jev-1.14.0: evidence covers it 98%
- ❌ _contradicted_: Nothing shown in the meeting gives a county-only sales cycle. — Municipal-or-county median cycle of 163 days is given, though not county-only.
- ❔ _unverifiable_: The CEO never reaches a unit-economics conclusion in the meeting. — JEV jev-1.14.0: evidence covers it 10%
- ❔ _unverifiable_: The plan shows no acquisition cost, renewal data or margin for a $96,000 county deal. — Network margin exists, but no county-deal CAC, renewal or margin shown; plan contents unknown.
- ✅ _supported_: A 5.4-month median cycle means half of deals take longer than 5.4 months. — 163 days is about 5.4 months; median means half take longer.
- 🧮 _miscalculated_: 28 x $36,000 + 41 x $96,000 = $1,008,000 + $3,936,000 = $4,944,000; $4,944,000 / $14,384,000 = 34.4%; 16 x $420,000 + 4 x $680,000 = $9,440,000; $4,944,000 + $9,440,000 = $14,384,000 — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 6815 in / 1930 out · 11994 ms · $0.02137</sub>

### 60. VC ASSISTANT → VC (round 12)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 61. VC → the panel (round 13) — _follow-up_

Take the county-first plan step by step. The synthetic evidence gives one median of 163 days for municipal or county buyers combined, not a county-only figure (biobot_dummy_procurement_and_sales_03). What is the timeline for the first county wins, in months, from first discovery calls to a signed annual agreement and a first renewal decision? For each step (discovery, paid pilot, annual agreement), how many salespeople, pilot staff and dollars does it need, and what is the source of each input (real, synthetic, or your own estimate)? And if the first county says no or stalls past 163 days, what is your fallback, and how long can the business fund the wait given the current annualized operating burn of about $4,039,560 (biobot_dummy_historical_efficiency_and_05)?

**Cites:** `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_historical_efficiency_and_05` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The synthetic evidence gives one median of 163 days for municipal or county buyers combined. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The current annualized operating burn is about $4,039,560. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The $4,039,560 annualized operating burn figure is taken from the source biobot_dummy_historical_efficiency_and_05. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 8164 in / 929 out · 7133 ms · $0.01991</sub>

### 62. MODERATOR → CEO (round 13)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 63. CEO → VC (round 13)

Answer: I cannot give a county-only timeline, staffing or cost per step, because the evidence is silent on them. I concede the Devil's points. (1) The 163 days is one synthetic median for municipal OR county buyers combined (biobot_dummy_procurement_and_sales_03, 2026-09-30), so I withdraw my claim that counties are faster than municipalities or that the target is a county-specific 5.5 months. (2) My arithmetic was right and the prose was wrong: the segment totals sum to exactly $14,384,000, so there is no $1,000 gap and no rounding. Municipal and county are 34.4% of ARR, not 36%, and state and federal are 65.6%, not 65%. (3) Winning counties first targets about a third of ARR and delays the buyers holding two-thirds. I still think counties are the lower-risk entry because they are less able to substitute, but that is an opinion, and the evidence does not show it. Timeline from what the evidence supports: the combined median is 163 days from qualified discovery to signed annual agreement, about 5.4 months, so half of deals take longer. The pilot is a separate paid stage: a typical synthetic 90-day municipal pilot costs $9,000 for two sites and 12 samples, credited against the first annual subscription if the buyer converts within 90 days (biobot_dummy_procurement_and_sales_01). I do not know whether the 90-day pilot sits inside or before the 163 days. If it sits inside, discovery to signature stays at about 5.4 months. If the pilot comes first, the clock could run to roughly 8.4 months (an assumption; I cannot verify the order). A first renewal decision then comes 12 months after the annual agreement starts, so about 17 to 20 months from first discovery. That is my arithmetic, not data, and no renewal evidence exists. Salespeople, pilot staff and dollars per step: no evidence, so I give no number and will not invent one. Fallback if the first county stalls past 163 days: run several county and municipal discovery tracks in parallel, so no single buyer decides the timeline, and use the $9,000 credited pilot as a low-cost entry. That is my plan, not evidence. Runway: the historical efficiency chunk (biobot_dummy_historical_efficiency_and_05) cited by the VC puts annualized operating burn at about $4,039,560, but that chunk is not in the evidence shown to me, so I rely on the VC's figure. On the VC's number, burn is about $336,630 per month, so a 5.4-month wait costs about $1.8M and a 10.5-month state cycle costs about $3.5M. I do not know the cash balance, so I cannot state how long we can fund a wait. Most important assumption: that the pilot converts and renews at county buyers. To confirm I need county-only cycle data, pilot-to-annual conversion, renewal and churn by segment, acquisition cost, sales and pilot headcount, and cash on hand.

**Maths** (✅ checks out): `163 days / 30.4 = 5.4 months; 90 days / 30.4 = 3.0 months; 5.4 + 3.0 = 8.4 months (only if the pilot precedes the cycle); 8.4 + 12 = 20.4 months to first renewal (upper case); 5.4 + 12 = 17.4 months (lower case); $4,039,560 / 12 = $336,630 per month; $336,630 x 5.4 = $1.82M; 318 days / 30.4 = 10.5 months; $336,630 x 10.5 = $3.53M; 28 x $36,000 + 41 x $96,000 = $4,944,000; $4,944,000 / $14,384,000 = 34.4%; $9,440,000 / $14,384,000 = 65.6%`

**Assumptions:**

- Whether the 90-day pilot falls inside or before the 163-day cycle is unknown; the 8.4-month case assumes it comes before
- First renewal decision occurs 12 months after the annual agreement starts
- Annualized burn of about $4,039,560 is taken from the VC's citation; the chunk is not shown to me
- Burn stays flat over the wait
- Parallel county and municipal tracks as a fallback is my plan and untested
- Counties being less able to substitute than states is my opinion
- No data on headcount, acquisition cost, conversion or cash on hand, so none is given

**Cites:** `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ❌ _contradicted_: The 163-day median is one synthetic median for municipal and county buyers combined (biobot_dummy_procurement_and_sales_03, 2026-09-30). — 163 days is the municipal/county median; state/federal is separate at 318, not combined.
- ❌ _contradicted_: No renewal evidence exists in the evidence shown. — Evidence shows renewal cohorts with 89.5%-92.3% renewal rates.
- ✅ _supported_: Historical efficiency chunk biobot_dummy_historical_efficiency_and_05, cited by the VC, puts annualized operating burn at about $4,039,560; that chunk is not in the evidence shown to the speaker. — Chunk states burn about $4,039,560; the speaker-visibility part cannot be checked.
- ✅ _supported_: Monthly burn is about $336,630 on the VC's annualized figure of $4,039,560. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: A 5.4-month wait costs about $1.8M in burn. — $336,630/month × 5.4 ≈ $1.82M.
- ✅ _supported_: A 10.5-month state cycle costs about $3.5M in burn. — $336,630 × 10.5 ≈ $3.53M; 318 days ≈ 10.5 months.
- ❔ _unverifiable_: The segment totals sum to exactly $14,384,000, with no $1,000 gap and no rounding issue. — Total ARR $14,384,000 stated; segment sum not shown, but 28×36k+41×96k+16×420k+4×680k not fully in evidence.
- ❔ _unverifiable_: Municipal and county buyers account for 34.4% of ARR and state and federal buyers account for 65.6% of ARR. — JEV jev-1.14.0: evidence covers it 14%
- ❔ _unverifiable_: Winning counties first targets about one third of ARR and delays buyers holding about two thirds of ARR. — JEV jev-1.14.0: evidence covers it 9%
- ❌ _contradicted_: The combined median time from qualified discovery to signed annual agreement is 163 days, about 5.4 months, and half of deals take longer. — 163 days applies only to municipal/county buyers, not a combined median.
- ✅ _supported_: A typical synthetic 90-day municipal pilot costs $9,000 for two sites and 12 samples, credited against the first annual subscription if the buyer converts within 90 days (biobot_dummy_procurement_and_sales_01). — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 7688 in / 2534 out · 15775 ms · $0.02843</sub>

### 64. DEVIL → CEO (round 13)

The weakest point is the burn-based runway math: the CEO prices the wait at $336,630 per month x 5.4 months = $1.8M, but that cost is not incremental. Burn is incurred whether or not a county signs, so it measures nothing about how long the business can fund a stalled deal. Without cash on hand, the figures cannot answer the VC's question, and they are presented as a cost of waiting. They also ignore revenue: the synthetic book shows $14,384,000 contracted ARR (biobot_dummy_commercial_contract_book_02, as cited by the VC and CEO), so net burn could differ greatly from the $4,039,560 operating burn. I cannot verify from the evidence shown whether that burn is gross or net of revenue, and the CEO takes it unseen from the VC's citation. The 8.4-month upper case is also shaky. The pilot spec says fees are credited if the buyer converts within 90 days, which implies the pilot leads into the annual agreement, but the CEO leaves the order unresolved and then stacks 12 months on top to get 17 to 20 months to first renewal. That range rests on an assumed 12-month term and unknown ordering, so it is not evidence. Notably, a $9,000 credited pilot is 25% of a $36,000 municipal deal and about 9% of a $96,000 county deal, but no pilot cost to Biobot is known, so the claimed low-cost entry has no margin. The fallback of parallel county and municipal tracks also needs more salespeople, and the CEO gives no headcount or cost. In my inference, it raises burn while the cycle wait is unchanged, which cuts against the runway the CEO cannot state. The plan therefore gives no number for how long the company survives a stalled first county.

**Maths** (✅ checks out): `$9,000 / $36,000 = 25%; $9,000 / $96,000 = 9.4%`

**Assumptions:**

- Operating burn may be gross or net of the $14,384,000 ARR; I cannot verify which from the evidence shown
- Parallel tracks raising burn is my inference; no headcount or cost is shown

**Cites:** `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The CEO prices the wait at $336,630 per month x 5.4 months = $1.8M. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: The pilot spec says fees are credited if the buyer converts within 90 days. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: A $9,000 credited pilot is 25% of a $36,000 municipal deal. — JEV jev-1.14.0: evidence covers it 91%
- ✅ _supported_: A $9,000 credited pilot is about 9.4% of a $96,000 county deal. — JEV jev-1.14.0: evidence covers it 95%
- ❔ _unverifiable_: The CEO gives no headcount or cost for the parallel tracks. — JEV jev-1.14.0: evidence covers it 15%
- ❔ _unverifiable_: The CEO presents the burn-based figures as a cost of waiting. — Evidence does not show what the CEO said or how it was framed.
- ✅ _supported_: The synthetic book shows $14,384,000 contracted ARR (biobot_dummy_commercial_contract_book_02), as cited by the VC and CEO. — JEV jev-1.14.0: evidence covers it 98%

<sub>claude-sonnet-5-5 · 7547 in / 1762 out · 12194 ms · $0.02215</sub>

### 65. VC ASSISTANT → VC (round 13)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Competition and alternatives

### 66. VC → the panel (round 14)

Now that cash on hand and burn are visible, I want to move to alternatives. Who else solves this problem for a municipal or county buyer today, whether commercial labs, state or university labs, in-house utility programs or other vendors, and why would a buyer pick Biobot over each? Please say which of your answers are real evidence, synthetic scenario data or your own opinion, and what the buyer pays for the alternative versus Biobot's $36,000 municipal and $96,000 county averages.

**Cites:** `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: Cash on hand and burn rate are now visible in the meeting. — Cash ($12.8M) and burn ($336,630/month) figures are in the evidence.
- ✅ _supported_: Biobot's average price for municipal buyers is $36,000. — 28 municipal programs average $36,000 in the scenario.
- ✅ _supported_: Biobot's average price for county buyers is $96,000. — 41 county or metro programs at $96,000 each.

<sub>claude-sonnet-5-5 · 8593 in / 569 out · 6165 ms · $0.01966</sub>

### 67. MODERATOR → CEO (round 14)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 521 in / 43 out · 72 ms · $0.00002</sub>

### 68. CEO → VC (round 14)

From the evidence shown I can name two competitors or alternatives, and I cannot give a price for either. (1) Verily Health / Sightline: a direct national wastewater testing and analytics competitor (public source, 2026-10-14). Its 2026 CDC award continues work begun in 2023 and combines lab testing, analytics, pathogen sequencing and possible metagenomics. CDC names Verily as the current national testing contractor and Biobot as the previous one. That is a loss of a national contract and the strongest threat I can cite. The source does not say Verily sells to municipal or county buyers, or offers substance-use or pharma products. (2) WastewaterSCAN: a Stanford/Emory academic network with philanthropic funding and Verily as lab partner (public source, 2026-10-22). It says it operates at nearly 150 sites and measures more than a dozen infectious diseases, with a public dashboard, and CDC receives its data. It is an alternative data source, not a separate commercial competitor. I cannot tell from the excerpt whether a municipality pays for it. Not in the evidence shown: commercial labs, state or university labs and in-house utility programs. Earlier turns cited MWRA going in-house in 2026-07 reporting, but that chunk is not displayed here, so I treat it as unverified. I know no price for any alternative, so I cannot compare against Biobot's synthetic averages of $36,000 municipal and $96,000 county (biobot_dummy_commercial_contract_book_02, synthetic, 2026-09-30). Why a buyer picks Biobot: the evidence does not prove a reason. The only evidence of buyer preference is 89 synthetic paying programs (biobot_dummy_commercial_contract_book_01), and the synthetic data does not say why they chose us, renewed, or rejected alternatives. My opinion is that a local buyer wants a contracted analyst to interpret results for its own decisions, which a free public dashboard may not give. That is a hypothesis to test in buyer interviews, not a fact. A free dashboard also caps what a buyer will pay for raw data, which is a risk to the $36,000 price. To confirm I need alternative prices, win/loss reasons, and renewal data.

**Assumptions:**

- Buyers value local interpretation and decision support beyond a free dashboard: my opinion, untested
- A free public dashboard may cap willingness to pay for raw data: my inference
- I know no price for any alternative, so no price comparison is made

**Cites:** `biobot_public_competitors_alternatives_and_02` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_05` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_dummy_commercial_contract_book_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_commercial_contract_book_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: WastewaterSCAN operates at nearly 150 sites and measures more than a dozen infectious diseases, with a public dashboard. — Nearly 150 sites, over a dozen diseases, public dashboard all stated.
- ✅ _supported_: CDC receives WastewaterSCAN data. — Evidence says CDC receives its data.
- ❌ _contradicted_: Commercial labs, state or university labs, and in-house utility programs are not in the evidence shown. — State/local labs and university programs appear in the evidence as alternatives.
- ✅ _supported_: Biobot's synthetic averages are $36,000 for municipal and $96,000 for county buyers. — JEV jev-1.14.0: evidence covers it 96%
- ❔ _unverifiable_: The only evidence of buyer preference is 89 synthetic paying programs in biobot_dummy_commercial_contract_book_01. — Evidence includes other synthetic and public customer records, so 'only' is not established.
- ✅ _supported_: The synthetic data does not say why those programs chose Biobot, renewed, or rejected alternatives. — Synthetic records give counts and renewal funding, not reasons for choosing Biobot.
- ✅ _supported_: Verily Health / Sightline is a direct national wastewater testing and analytics competitor, according to a public source dated 2026-10-14. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: Verily Health / Sightline's 2026 CDC award continues work begun in 2023 and combines lab testing, analytics, pathogen sequencing and possible metagenomics. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: CDC names Verily as the current national wastewater testing contractor and Biobot as the previous one. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The cited source does not say Verily sells to municipal or county buyers, or offers substance-use or pharma products. — Source says it does not prove Verily offers substance-use or pharma products.
- ✅ _supported_: WastewaterSCAN is a Stanford/Emory academic network with philanthropic funding and Verily as lab partner, according to a public source dated 2026-10-22. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 7317 in / 2188 out · 11359 ms · $0.02327</sub>

### 69. DEVIL → CEO (round 14)

The weakest point is the CEO's claim that the 'strongest threat I can cite' is Verily and that WastewaterSCAN is 'not a separate commercial competitor', combined with an answer that names only two alternatives and so leaves the VC's actual question about municipal and county buyers unanswered. Both cited sources describe national players: Verily's CDC award and WastewaterSCAN's network of nearly 150 sites (biobot_public_competitors_alternatives_and_02, 2026-10-14; biobot_public_competitors_alternatives_and_05, 2026-10-22). Neither says anything about what a municipal or county buyer pays or chooses. The CEO then leans on the Verily/CDC loss as the lead threat to a $36,000 municipal and $96,000 county book, but a national CDC contract loss does not show that local buyers will switch. I cannot verify that link from the evidence. The CEO also says WastewaterSCAN is 'an alternative data source', yet then asserts that a free dashboard 'caps what a buyer will pay'. That is the CEO's own inference, and the source gives no price and does not say whether any municipality pays or declines to pay because of it. It is stated as a risk without evidence, and the CEO treats it as the bigger pricing danger without testing it. Verily is also WastewaterSCAN's lab partner, per the source. The CEO treats the two as separate threats, but the same lab capacity may sit behind both a paid national contract and a free local dashboard. I infer this could let Verily offer local buyers a cheaper or free route, and the CEO gives no reason Biobot's $36,000 holds against that. Finally, the 'why a buyer picks Biobot' reason, a contracted analyst interpreting local results, is the CEO's untested opinion with no win/loss, renewal or interview evidence behind it. The pitch therefore has no evidenced differentiation at the local level, which is where 69 of the 89 programs sit.

**Assumptions:**

- Verily sharing lab capacity with WastewaterSCAN could let it offer local buyers a cheaper or free route: my inference, not stated in the evidence
- The sources do not show whether municipal or county buyers pay for any alternative; I cannot verify either way

**Cites:** `biobot_public_competitors_alternatives_and_02` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_05` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The CEO describes WastewaterSCAN as 'an alternative data source'. — Source uses the phrase, but the CEO's wording is not shown.
- ❔ _unverifiable_: The CEO asserts that a free dashboard 'caps what a buyer will pay'. — JEV jev-1.14.0: evidence covers it 10%
- ❔ _unverifiable_: The source gives no price and does not say whether any municipality pays or declines to pay because of the free dashboard. — The 'free dashboard' source is not in the evidence shown.
- ✅ _supported_: Verily is WastewaterSCAN's lab partner, per the source. — JEV jev-1.14.0: evidence covers it 90%
- ❔ _unverifiable_: The CEO treats Verily and WastewaterSCAN as separate threats. — The CEO's statement is not shown; the source ties Verily to WastewaterSCAN.
- ❔ _unverifiable_: The CEO's answer names only two alternatives in response to the VC's question about municipal and county buyers. — The CEO's actual answer is not in the evidence.
- ✅ _supported_: There is no win/loss, renewal or interview evidence supporting the contracted analyst differentiation. — Evidence has renewal rates only, with no win/loss or interview data on analyst differentiation.
- ✅ _supported_: 69 of the 89 programs are local programs. — 28 municipal plus 41 county or metro equals 69 of 89.
- ✅ _supported_: Sources do not show whether municipal or county buyers pay for any alternative. — No evidence shows municipal or county payments for alternatives.
- ✅ _supported_: Verily received a CDC award. — Verily's 2026 CDC award is stated.
- ✅ _supported_: WastewaterSCAN operates a network of nearly 150 sites. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Both cited sources (biobot_public_competitors_alternatives_and_02 dated 2026-10-14 and biobot_public_competitors_alternatives_and_05 dated 2026-10-22) describe national players. — JEV jev-1.14.0: evidence covers it 92%
- ❌ _contradicted_: The cited sources do not say what a municipal or county buyer pays or chooses. — Evidence gives municipal and county prices and funding sources.
- ❔ _unverifiable_: The CEO leans on the Verily/CDC loss as the lead threat to a $36,000 municipal and $96,000 county book. — The CEO's framing is not in the evidence.

<sub>claude-sonnet-5-5 · 7517 in / 1873 out · 12230 ms · $0.02258</sub>

### 70. VC ASSISTANT → VC (round 14)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 71. VC → the panel (round 15) — _follow-up_

The CEO said commercial labs, state or university labs and in-house programs are not in the evidence shown, but the evidence lists state/local public-health laboratories and university programs as a demonstrated build-versus-buy alternative, and records the end of Biobot's Massachusetts data stream at MWRA in July 2026 (biobot_public_competitors_alternatives_and_06, 2026-10-22). Where do you lose or fail to renew deals today, and to whom: Verily, in-house or state labs, free networks, or no purchase at all? For each loss, what reason did the buyer give, and is that reason from real records, the synthetic packet, or your own opinion? Also, what share of the 69 municipal and county programs could replace you with a free or in-house option, and what evidence supports that share?

**Cites:** `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_02` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_05` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The evidence shown lists state/local public-health laboratories and university programs as a demonstrated build-versus-buy alternative. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: The evidence records the end of Biobot's Massachusetts data stream at MWRA in July 2026. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The source of these evidence records is biobot_public_competitors_alternatives_and_06, dated 2026-10-22. — Chunk 06 is dated as of 2026-10-22 as stated.

<sub>claude-sonnet-5-5 · 8405 in / 739 out · 6243 ms · $0.02047</sub>

### 72. MODERATOR → CEO (round 15)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 73. CEO → VC (round 15)

I correct my earlier answer: I said commercial labs, state or university labs and in-house programs were not in the evidence shown, but they are. Chunk biobot_public_competitors_alternatives_and_06 (public source, 2026-10-22) lists state/local public-health laboratories and university programs as a demonstrated build-versus-buy alternative, and records via MWRA the end of Biobot's Massachusetts data stream in July 2026. biobot_public_business_and_market_03 (2024-05-01, with later content on the 2026 MWRA end date) confirms in-house substitution and procurement renewal risk. The Devil is also right that Verily's CDC award (biobot_public_competitors_alternatives_and_02, 2026-10-14) is a national contract loss and does not show local buyers switching. Where we lose today: the evidence shows two concrete losses. (1) The CDC national testing contract went to Verily, with Biobot named as the previous contractor. (2) The Massachusetts data stream at MWRA ended in July 2026; the source records the end, and I cannot verify the buyer's stated reason or whether it moved in-house, to a state lab or to a free network. I cannot verify any loss or non-renewal among the 69 municipal and county programs, and no win/loss or renewal records are shown. I do not know whether any buyer chose no purchase at all. Source of each reason: none is a buyer-stated reason from records. Any reason I offer is my opinion. My opinion is that the main local risk is in-house or state-lab substitution, since the evidence calls it demonstrated, with free networks like WastewaterSCAN a secondary risk (a data source, not evidenced as a payment substitute). Share of the 69 programs that could replace us with a free or in-house option: I cannot give a number, because the evidence shows the alternative exists but not how many buyers have the capacity or intent to use it. Giving a percentage would be invented. Most important assumption: that the MWRA exit is a signal of wider local in-house risk. It is one case, and could be unrepresentative. To confirm I need win/loss and churn records by segment with stated reasons, buyer lab capacity, alternative prices, and renewal dates for the 69 programs.

**Assumptions:**

- The MWRA exit is one data point and may not represent the other municipal and county buyers: my caution, not evidence
- In-house or state-lab substitution being the main local risk is my opinion
- I cannot verify the reason MWRA ended the stream or where its data now comes from

**Cites:** `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_02` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_05` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: No source records a buyer-stated reason for losses or non-renewals. — Sources note ends and substitution but no buyer-stated reasons.
- ✅ _supported_: Chunk biobot_public_competitors_alternatives_and_06 (public source, dated 2026-10-22) lists state/local public-health laboratories and university programs as a demonstrated build-versus-buy alternative. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Chunk biobot_public_competitors_alternatives_and_06 records via MWRA that Biobot's Massachusetts data stream ended in July 2026. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Chunk biobot_public_business_and_market_03 (dated 2024-05-01, with later content on the 2026 MWRA end date) confirms in-house substitution and procurement renewal risk. — JEV jev-1.14.0: evidence covers it 87%
- ✅ _supported_: The Verily CDC award (biobot_public_competitors_alternatives_and_02, dated 2026-10-14) is a national contract loss and does not show local buyers switching. — CDC award is national; evidence shows no local buyer switching tied to it.
- ✅ _supported_: The CDC national testing contract went to Verily, with Biobot named as the previous contractor. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The Massachusetts data stream at MWRA ended in July 2026. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 7149 in / 1858 out · 11052 ms · $0.02275</sub>

### 74. DEVIL → CEO (round 15)

The weakest point is the CEO's claim that the evidence 'shows two concrete losses' and that the MWRA exit is 'one data point'. Chunk biobot_public_competitors_alternatives_and_06 only says MWRA 'records the end of Biobot's Massachusetts data stream in July 2026', and biobot_public_business_and_market_03 says the MWRA 'confirms' it ended. Neither says MWRA was a paying municipal or county customer, that it was a loss rather than a planned or grant-funded end, or that it substituted in-house. The CEO then builds 'in-house or state-lab substitution is the main local risk' on that one unexplained event. That ranking is opinion presented as the lead conclusion, and the CEO never tests it against the other candidates it lists: Verily, free networks or no purchase. The 'demonstrated' label in chunk 06 describes a general build-versus-buy option, not that any of Biobot's 69 municipal and county programs has switched or could. Separately, the CEO calls the CDC loss a 'national contract loss' that does not show local buyers switching, yet still counts it as one of the two 'concrete losses' in answer to a question about where municipal and county deals are lost. That inflates the loss record. Chunk 03 itself warns that these events show risk 'not the company's complete 2026 customer situation', so the CEO's 'two concrete losses' overstates what the evidence supports. In my inference, the honest count of evidenced local losses is zero to one, with the reason unknown. A related gap: the CEO gives no share for the 69 programs, which is fair, but also offers no bounded range from the evidence, such as how many programs sit in states with a public-health lab. The VC's question on replaceable share stays unanswered and the risk cannot be sized.

**Assumptions:**

- MWRA's end of the data stream may be a planned, grant-related or non-commercial end rather than a lost paying contract: my inference, not stated in the evidence
- The evidenced count of local losses is zero to one: my reading of the cited chunks
- I cannot verify from the evidence shown whether MWRA was a paying customer

**Cites:** `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: Chunk biobot_public_business_and_market_03 warns that these events show risk 'not the company's complete 2026 customer situation'. — JEV jev-1.14.0: evidence covers it 90%
- ✅ _supported_: Chunk biobot_public_competitors_alternatives_and_06 says MWRA 'records the end of Biobot's Massachusetts data stream in July 2026'. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Chunk biobot_public_business_and_market_03 says the MWRA 'confirms' that its Biobot data stream ended. — JEV jev-1.14.0: evidence covers it 95%
- ❌ _contradicted_: Neither cited chunk says MWRA was a paying municipal or county customer, that its end was a loss rather than a planned or grant-funded end, or that it substituted in-house. — Cited chunks mention in-house substitution; MWRA described as testing ended as state moved in-house.
- ❔ _unverifiable_: The CEO does not test the in-house substitution ranking against the other candidates it lists: Verily and free networks or no purchase. — JEV jev-1.14.0: evidence covers it 14%
- ✅ _supported_: The 'demonstrated' label in chunk 06 describes a general build-versus-buy option, not evidence that any of Biobot's 69 municipal and county programs has switched or could switch. — Chunk 06 describes general alternative; no municipal switching shown among 69 programs.
- ❔ _unverifiable_: The CEO calls the CDC loss a 'national contract loss' that does not show local buyers switching, yet counts it as one of the two 'concrete losses' in answer to a question about municipal and county deal losses. — Evidence does not contain the CEO's answer or question context.

<sub>claude-sonnet-5-5 · 6975 in / 1762 out · 12409 ms · $0.02134</sub>

### 75. VC ASSISTANT → VC (round 15)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 76. VC → the panel (round 16) — _follow-up_

The evidence now has a dated public report that the MWRA said its Biobot testing ended at the end of July 2026 as the state moved toward in-house testing, with an expiring contract reported at $400,000/month and a replacement expected to cost about one-quarter as much (biobot_public_contracts_customers_and_06, 2026-07). Given that, how much of the $14,384,000 contracted ARR in the synthetic book, if any, reflects Massachusetts or similar state-level in-house substitution exposure? Is the reported $400,000/month contract in or outside that synthetic book, and which of your figures here are real records, synthetic data or opinion?

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The evidence includes a dated public report (2026-07) that the MWRA said its Biobot wastewater testing ended at the end of July 2026 as the state moved toward in-house testing. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The publicly reported Biobot contract with the MWRA was expiring and reported at $400,000 per month. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: The synthetic venture book has $14,384,000 in contracted ARR. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 7767 in / 797 out · 5792 ms · $0.01865</sub>

### 77. MODERATOR → CEO (round 16)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 78. CEO → VC (round 16)

I cannot say how much of the $14,384,000 synthetic contracted ARR reflects Massachusetts or state-level in-house exposure, and I cannot say whether the reported $400,000/month MWRA contract is inside or outside that book. The evidence shown gives no state-level breakdown of the synthetic book and no customer list, so any share would be invented. Correction first: the Devil is right that I overstated 'two concrete losses'. The MWRA end is now directly evidenced as a Biobot customer loss with a stated cause: biobot_public_contracts_customers_and_06 (public source, 2026-07) says the testing ended end of July 2026 as the state moved toward in-house testing, and the Boston Globe reported the contract at $400,000/month with a replacement at about one-quarter of that cost. That settles my earlier doubts about the reason and about in-house substitution, but only for MWRA. The CDC loss is national and is not a local-buyer loss. The evidenced local loss count is one, MWRA, and I cannot verify any among the 69 municipal and county programs. How I reached this: the synthetic ARR and the $36,000 and $96,000 averages come from the contract book cited earlier (biobot_dummy_commercial_contract_book_02, synthetic, 2026-09-30), and I cannot see that chunk here. The MWRA figures are real public reporting. The two datasets are different in kind: the public figure is contract value, not net revenue or margin, per the chunk itself. Scale check using only the reported figures: $400,000/month x 12 = $4.8M/year, which would be about 33% of $14,384,000 if it were inside the book. But that is an arithmetic comparison, not a finding, because the book's $36,000 and $96,000 averages for local programs look nothing like a $4.8M annualized state contract, which suggests it is probably outside the synthetic book. That is my inference, not a fact. Replacement at about one-quarter: $400,000 x 25% = $100,000/month, so the state still spends something on testing, but I cannot verify that any of it goes to Biobot. Most important assumption: that the real MWRA contract and the synthetic book are separate. If MWRA were in the book, one-third of ARR would be gone; if outside, the synthetic ARR overstates nothing but the real company has already lost a much larger contract than any single local program. To confirm I need the synthetic book's customer list by state and contract type, whether MWRA is included, expiry dates, and which other states have in-house plans. Opinion: other state-level buyers carry the same substitution risk, but I cannot size it from the evidence.

**Maths** (✅ checks out): `$400,000/month x 12 months = $4,800,000/year; $4,800,000 / $14,384,000 = 33.4%; $400,000/month x 25% = $100,000/month`

**Assumptions:**

- The reported $400,000/month MWRA contract is likely outside the synthetic book, given the $36,000 and $96,000 averages: my inference, not stated
- The 33% comparison applies only if MWRA were inside the synthetic book, which I cannot verify
- Other state-level buyers may face similar in-house substitution: my opinion, not evidenced
- I cannot see the synthetic contract book chunk displayed here, so its contents are taken from earlier turns

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The CDC loss is national and is not a local-buyer loss. — CDC loss is a national contract (Verily replaced Biobot); MWRA is the separate state loss.
- ✅ _supported_: The evidenced local-buyer loss count is one (MWRA). — Only MWRA is evidenced as a lost local/state buyer; no others appear.
- ✅ _supported_: The synthetic ARR figure and the $36,000 and $96,000 averages come from biobot_dummy_commercial_contract_book_02 (synthetic, 2026-09-30). — JEV jev-1.14.0: evidence covers it 94%
- ❔ _unverifiable_: The speaker cannot see the cited contract book chunk in the current display. — Evidence does not say what the speaker can see in the display.
- ✅ _supported_: The MWRA figures are real public reporting. — MWRA notice and Boston Globe reporting are public sources.
- ✅ _supported_: The public MWRA figure is contract value, not net revenue or margin. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: Annualizing $400,000/month gives $4.8M per year. — $400,000 x 12 = $4.8M.
- ❔ _unverifiable_: $4.8M per year would be about 33.4% of $14,384,000. — No valid supporting citation
- ✅ _supported_: Replacement at one-quarter of $400,000 is $100,000/month. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: The evidence shown gives no state-level breakdown of the synthetic book and no customer list. — Book gives counts by tier, no state breakdown; only an invented ID, no real customer list.
- ✅ _supported_: The MWRA loss is evidenced as a Biobot customer loss with a stated cause in biobot_public_contracts_customers_and_06 (public source, 2026-07). — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: The public source says the MWRA testing ended at the end of July 2026 as the state moved toward in-house testing. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The Boston Globe reported the MWRA contract at $400,000/month with a replacement at about one-quarter of that cost. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The MWRA contract was reported at $400,000 per month. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The MWRA replacement cost is reported as about one-quarter of the original contract cost. — JEV jev-1.14.0: evidence covers it 94%

<sub>claude-sonnet-5-5 · 6708 in / 2462 out · 14427 ms · $0.02386</sub>

### 79. DEVIL → CEO (round 16)

The weakest point is the CEO's inference that the $400,000/month MWRA contract is 'probably outside the synthetic book' because the $36,000 and $96,000 averages look nothing like $4.8M. The synthetic book is not made only of those two tiers. Chunk biobot_dummy_commercial_contract_book_04 (synthetic, 2026-09-30) shows four tiers: 28 x $36,000, 41 x $96,000, 16 x $420,000 and 4 x $680,000. The CEO compared MWRA only with the two local averages and ignored the 20 programs at $420,000 and $680,000. Even those are far below $4.8M, so MWRA still would not fit as a single line. But the book shows no customer names or states, and a $4.8M state contract could be split across lines or be outside the book. The CEO's reasoning therefore supports neither inclusion nor exclusion, and the 'probably outside' inference rests on an incomplete comparison. The 33.4% figure is also misleading as a scale check. It divides a real annualized contract value by a synthetic ARR with no stated link, and the source says the monthly figure is contract value, not net revenue. The CEO also says 'if outside, the synthetic ARR overstates nothing', which is wrong in a different way. If MWRA's $4.8M was real Biobot revenue and sits outside the book, the book is not the company's whole contracted base and cannot answer the VC's exposure question. Separately, the CEO states 'the state still spends something' on replacement testing as $100,000/month. The source says only 'about one-quarter as much', so that is an expected cost and not a confirmed spend. In my inference, the 20 large-tier programs ($420,000 and $680,000), worth 16 x $420,000 + 4 x $680,000 = $9,440,000, are the likeliest place for state-level substitution exposure. That is 65.6% of ARR, which the CEO never examined. I cannot verify from the evidence who those buyers are.

**Maths** (✅ checks out): `16 x $420,000 = $6,720,000; 4 x $680,000 = $2,720,000; $6,720,000 + $2,720,000 = $9,440,000; $9,440,000 / $14,384,000 = 65.6%`

**Assumptions:**

- The 20 large-tier programs are the likeliest place for state-level substitution exposure: my inference, not stated in the evidence
- The book may or may not contain MWRA; I cannot verify either way since it lists no customers or states

**Cites:** `biobot_dummy_commercial_contract_book_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The CEO states that 'the state still spends something' on replacement testing at $100,000/month. — Replacement is about one-quarter of $400k/month, which implies $100k, but the CEO's statement isn't shown.
- ✅ _supported_: The source says only 'about one-quarter as much' for replacement testing, so $100,000/month is an expected cost and not a confirmed spend. — JEV jev-1.14.0: evidence covers it 91%
- 🧮 _miscalculated_: The 20 large-tier programs ($420,000 and $680,000) are worth 16 x $420,000 + 4 x $680,000 = $9,440,000. — Arithmetic does not match the stated result
- ✅ _supported_: The 20 large-tier programs' $9,440,000 is 65.6% of the $14,384,000 ARR, which the CEO never examined. — 16x420k + 4x680k = 9.44M; 9.44/14.384 = 65.6%. CEO behavior unverified.
- ✅ _supported_: The synthetic book is not made only of the two tiers compared by the CEO. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: Chunk biobot_dummy_commercial_contract_book_04 (synthetic, dated 2026-09-30) shows four tiers: 28 programs at $36,000, 41 at $96,000, 16 at $420,000 and 4 at $680,000. — JEV jev-1.14.0: evidence covers it 97%
- ❔ _unverifiable_: The CEO compared MWRA only with the two local averages and ignored the 20 programs at $420,000 and $680,000. — Tier counts match 16+4=20, but the CEO's comparison is not in the evidence.
- ✅ _supported_: Even the $420,000 and $680,000 tiers are far below $4.8M, so MWRA would not fit as a single line in the book. — Max tier is $680,000, far below $4.8M; no program exceeds $680,000.
- ✅ _supported_: The book shows no customer names or states, so a $4.8M state contract could be split across lines or be outside the book. — Book lists only counts by tier, with no names or states.
- ✅ _supported_: The source says the monthly figure is contract value, not net revenue. — JEV jev-1.14.0: evidence covers it 92%

<sub>claude-sonnet-5-5 · 6755 in / 1879 out · 11204 ms · $0.02159</sub>

### 80. VC ASSISTANT → VC (round 16)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 81. VC → the panel (round 17) — _follow-up_

The synthetic contract book lists 16 state programs at $420,000 each and 4 federal or multi-region programs at $680,000 each, and the book shows no customer names or states (biobot_dummy_commercial_contract_book_02). A public report says the MWRA contract ended in July 2026 as the state moved toward in-house testing (biobot_public_contracts_customers_and_06, 2026-07). Do any of the 16 state programs have in-house testing plans, a procurement re-bid, or an expiry date in the next 12 months? If you cannot name those programs, which records would you need to size that exposure, and which of your answers are real records, synthetic data or opinion?

**Cites:** `biobot_dummy_commercial_contract_book_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The synthetic contract book lists 16 state programs. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The synthetic contract book lists each of the 16 state programs at $420,000. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The synthetic contract book lists 4 federal or multi-region programs. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The synthetic contract book lists each of the 4 federal or multi-region programs at $680,000. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The synthetic contract book (biobot_dummy_commercial_contract_book_02) shows no customer names or states. — Book chunks give counts by tier; only invented ID SIM-MUN-001, no names or states.
- ✅ _supported_: A public report (biobot_public_contracts_customers_and_06, dated 2026-07) says the MWRA contract ended in July 2026. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: A public report (biobot_public_contracts_customers_and_06, dated 2026-07) says the state moved toward in-house testing. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 7504 in / 905 out · 5999 ms · $0.01827</sub>

### 82. MODERATOR → CEO (round 17)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 83. CEO → VC (round 17)

I cannot name any of the 16 state programs, and I cannot say whether any has an in-house plan, a re-bid or an expiry in the next 12 months. The evidence shows no customer names, states, expiry dates or procurement status. Corrections first: the Devil is right that my 'probably outside the book' inference compared MWRA only with the $36,000 and $96,000 tiers and ignored the $420,000 and $680,000 tiers. Even those are far below $4.8M, so the book neither shows nor rules out MWRA. I also should not have said 'the synthetic ARR overstates nothing': if MWRA was real Biobot revenue outside the book, the book is not the whole contracted base and cannot answer the exposure question. The $100,000/month is the reported expected replacement cost ('about one-quarter'), not a confirmed spend. How I got here: the tiers come from the synthetic book (biobot_dummy_commercial_contract_book_02, 2026-09-30: 16 state programs at $420,000, 4 federal or multi-region at $680,000; the Devil cited a fuller four-tier count in chunk 04, which I cannot see). The MWRA facts are real public reporting (biobot_public_contracts_customers_and_06, 2026-07): testing ended end of July 2026 as the state moved toward in-house testing, reported at $400,000/month, and the source says that is contract value, not net revenue or margin. Data kinds: tiers and the $14,384,000 total are synthetic; MWRA is real reporting; any view of which states are at risk is opinion. Opinion: the 16 state programs are the likeliest place for similar substitution risk, since MWRA is a state-level buyer, but one case does not size it. Maximum possible ceiling, not a forecast: 16 x $420,000 = $6,720,000, which is 46.7% of $14,384,000, if every state program were lost, which I do not claim. Records I need: the contract book with customer, state, contract type, start and expiry dates, and renewal or re-bid terms; each state's lab capacity and in-house or budget announcements; whether MWRA is in the book; win/loss and churn records with buyer-stated reasons. Most important assumption: that state programs resemble MWRA in substitution risk. It is untested, and could be too high or too low.

**Maths** (✅ checks out): `16 programs x $420,000 = $6,720,000; $6,720,000 / $14,384,000 = 46.7%; $400,000/month x 12 months = $4,800,000/year`

**Assumptions:**

- State programs may face MWRA-like in-house substitution risk: my opinion, not evidenced
- The 46.7% is a ceiling if all 16 state programs were lost, not a forecast
- The synthetic book is scenario data and I cannot verify whether MWRA is in it
- The tier counts of 28, 41, 16 and 4 are taken from the synthetic book; the 'four tiers' chunk 04 is cited by the Devil and I cannot see it

**Cites:** `biobot_dummy_commercial_contract_book_02` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The Devil cited a fuller four-tier count in chunk 04, which the speaker cannot see. — Evidence says nothing about who cited chunk 04 or visibility.
- ✅ _supported_: MWRA testing ended at the end of July 2026 as the state moved toward in-house testing. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: MWRA testing was reported at $400,000/month in public reporting (biobot_public_contracts_customers_and_06, 2026-07). — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The public source states the $400,000/month is contract value, not net revenue or margin. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The tiers and the $14,384,000 total are synthetic data. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: MWRA reporting is real public reporting. — MWRA's own page and notice are public sources confirming the testing ended.
- ✅ _supported_: The maximum possible ceiling is 16 x $420,000 = $6,720,000. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: $6,720,000 is 46.7% of $14,384,000. — 6,720,000/14,384,000 = 46.7%; 16×420k = 6.72M.
- ✅ _supported_: $400,000/month x 12 months = $4,800,000/year. — $400,000 × 12 = $4.8M; arithmetic holds.
- ✅ _supported_: The tier counts of 28, 41, 16 and 4 are taken from the synthetic book. — JEV jev-1.14.0: evidence covers it 97%
- ❌ _contradicted_: The evidence shown contains no customer names, states, expiry dates or procurement status. — Evidence names MWRA, Massachusetts, end-July 2026 expiry, and renewal funding status.
- ✅ _supported_: The $100,000/month is the reported expected replacement cost ('about one-quarter'), not a confirmed spend. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: The tiers come from the synthetic book biobot_dummy_commercial_contract_book_02 dated 2026-09-30, which lists 16 state programs at $420,000 and 4 federal or multi-region programs at $680,000. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 7108 in / 2337 out · 14639 ms · $0.02370</sub>

### 84. DEVIL → CEO (round 17)

The weakest point is the CEO's 'maximum possible ceiling' of 16 x $420,000 = $6,720,000 (46.7% of ARR), which is not a ceiling. It assumes the only exposure sits in the 16 state programs, but the CEO's own evidence undermines that. The 4 federal or multi-region programs at $680,000 are also exposed to procurement loss: the CDC contract went to Verily (biobot_public_competitors_alternatives_and_02, 2026-10-14), and biobot_public_business_and_market_03 flags government contract concentration. Adding them gives 16 x $420,000 + 4 x $680,000 = $9,440,000, or 65.6% of the synthetic ARR, so the stated ceiling understates the maximum by $2,720,000. The label also hides a deeper problem: the CEO has no evidence that the 16 programs are state-level at all. The only source for that is the VC's description of the book, and the book lists no names or states. The 'state programs resemble MWRA' premise is also doubtful on the CEO's own numbers. MWRA at $4.8M a year is over 11 times a $420,000 tier, so one MWRA-style loss is not comparable to one line of the book. The 46.7% figure therefore reads as a bounded risk number when it is an arithmetic sum of tier counts with no link to any expiry, re-bid or in-house plan. In my inference, the honest statement is that exposure cannot be bounded from the evidence shown, and the ceiling should be the whole book, $14,384,000, until contract terms are seen. Separately, the CEO relies on tier counts from a chunk it admits it cannot see, so the 16 and 4 counts are second-hand to it. I cannot verify from the evidence shown which programs are state, federal or multi-region.

**Maths** (✅ checks out): `16 x $420,000 = $6,720,000; 4 x $680,000 = $2,720,000; $6,720,000 + $2,720,000 = $9,440,000; $9,440,000 / $14,384,000 = 65.6%; $4,800,000 / $420,000 = 11.4x`

**Assumptions:**

- The 4 federal or multi-region programs face procurement loss risk comparable to the CDC case: my inference, not stated in the evidence
- The tier counts of 16 and 4 come from the VC's description and earlier turns, not from a chunk displayed here
- The whole-book ceiling of $14,384,000 is a logical bound, not a forecast

**Cites:** `biobot_public_competitors_alternatives_and_02` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❌ _contradicted_: The only source describing the 16 programs as state-level is the VC's description of the book. — Scenario contract book chunks themselves describe 16 state programs.
- ✅ _supported_: The book lists no program names or states. — Book shows only tier counts and values, no program names or states.
- ✅ _supported_: MWRA is at $4.8M a year. — $400,000/month annualizes to $4.8M; though the contract expired.
- ✅ _supported_: MWRA at $4.8M a year is 11.4 times a $420,000 tier. — 4.8M/420k = 11.43; arithmetic holds.
- ❔ _unverifiable_: The CEO's ceiling assumes the only exposure sits in the 16 state programs. — CEO's ceiling assumption is not in the evidence.
- ✅ _supported_: The CDC contract went to Verily (biobot_public_competitors_alternatives_and_02, 2026-10-14). — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: biobot_public_business_and_market_03 flags government contract concentration. — JEV jev-1.14.0: evidence covers it 90%
- 🧮 _miscalculated_: Adding the 4 federal or multi-region programs gives 16 x $420,000 + 4 x $680,000 = $9,440,000. — Arithmetic does not match the stated result
- ✅ _supported_: $9,440,000 is 65.6% of the synthetic ARR of $14,384,000. — 9,440,000/14,384,000 ≈ 65.6%; ARR confirmed $14,384,000.
- ❔ _unverifiable_: The stated ceiling understates the maximum by $2,720,000. — $2.72M matches top-four ARR, but the stated ceiling isn't shown.

<sub>claude-sonnet-5-5 · 7037 in / 1938 out · 11344 ms · $0.02230</sub>

### 85. VC ASSISTANT → VC (round 17)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Founders' background and journey

### 86. VC → the panel (round 18)

Competition is closed enough, so I turn to the team. Please tell me about the founders' backgrounds: how they met, how Biobot started as an MIT spin-out, and the key moments of the journey so far. Which of these are documented records and which are your own recollection or opinion?


<sub>claude-sonnet-5-5 · 7567 in / 294 out · 4195 ms · $0.01678</sub>

### 87. MODERATOR → CEO (round 18)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 481 in / 43 out · 157 ms · $0.00002</sub>

### 88. CEO → VC (round 18)

Documented story, from the evidence shown: Mariana Matus and Newsha Ghaeli met in Ghaeli's first week at MIT, when both were charged with making the Underworlds project real (biobot_case_p3_c2, case as of 2017). Ghaeli arrived to a recently funded Underworlds proposal from Carlo Ratti. She recalled it 'blew my mind'. She trained in architecture in Canada, which counts as an engineering degree, and had never worked in biology. She managed the six-department effort and handled the 'urban' side: mappings, implementation and government contacts. Matus is described as the scientist with 'a giant vision' (biobot_case_p13_c3). I cannot verify from the evidence shown her degree, her exact role, or any dates for her background. Before Biobot, Matus had faced a similar commit-or-not decision with a smart toilet (biobot_case_p13_c3). They tried to recruit a roboticist post-doc from Underworlds, who declined as a father of two young children. Matus wanted to be careful with early hires and said they needed a mechanical engineer or business help, but felt they were a good team (biobot_case_p12_c1). They also faced a trade-off between hardware development, which could yield proprietary IP, and a more flexible lab-based approach (biobot_case_p12_c1). Public sources say the founders had worked together on MIT wastewater research before forming the company, used MIT DesignX, Sandbox, the MIT $100K competition and delta v, and that Biobot was on the 2017 delta v roster, pitched opioid monitoring at Demo Day and joined YC W18 (biobot_public_history_and_operating_02, as of 2018). Team dynamic, in their own words: Ghaeli wanted to lean into building relationships with mayors and executives. Matus said they think differently but react alike on big things, with a 'low ego' approach of 'if it doesn't work, there is always another job' (biobot_case_p13_c3). Those quotes are recollections by the founders, recorded in the case, not independent records. Not in the evidence shown: the incorporation date, founding equity, who holds the CEO title, later hires, and 2018-2026 milestones beyond YC. I will not fill those in. Opinion: the complementary pairing of science and city-government relationships suits a municipal-facing business, but the evidence does not show that it has delivered results. Also, the evidence shows no sign of a technical or commercial leader beyond the two founders, so key-person dependence is a risk I cannot size.

**Assumptions:**

- The complementary-skills fit with municipal sales is my opinion, not evidenced
- Key-person dependence on the two founders is my opinion; I cannot size it from the evidence

**Cites:** `biobot_case_p3_c2` — HBS case 821-045, "Biobot Analytics", p.3; `biobot_case_p13_c3` — HBS case 821-045, "Biobot Analytics", p.13; `biobot_case_p12_c1` — HBS case 821-045, "Biobot Analytics", p.12; `biobot_public_history_and_operating_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: Mariana Matus and Newsha Ghaeli met in Ghaeli's first week at MIT, when both were charged with making the Underworlds project real (biobot_case_p3_c2, case as of 2017). — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Before Biobot, Matus faced a similar commit-or-not decision with a smart toilet (biobot_case_p13_c3). — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: The founders tried to recruit a roboticist post-doc from Underworlds, who declined as a father of two young children. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Matus wanted to be careful with early hires and said they needed a mechanical engineer or business help, but felt they were a good team (biobot_case_p12_c1). — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: The founders faced a trade-off between hardware development, which could yield proprietary IP, and a more flexible lab-based approach (biobot_case_p12_c1). — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Public sources say the founders had worked together on MIT wastewater research before forming the company. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: Public sources say Biobot used MIT DesignX, Sandbox, the MIT $100K competition and delta v. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: Public sources say Biobot was on the 2017 delta v roster, pitched opioid monitoring at Demo Day and joined YC W18 (biobot_public_history_and_operating_02, as of 2018). — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Ghaeli wanted to lean into building relationships with mayors and executives. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Matus said they think differently but react alike on big things, with a 'low ego' approach of 'if it doesn't work, there is always another job' (biobot_case_p13_c3). — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: Ghaeli arrived at MIT to a recently funded Underworlds proposal from Carlo Ratti. — Ratti welcomed her on day one with the recently funded Underworlds proposal.
- ✅ _supported_: The evidence shown does not include the incorporation date, founding equity, who holds the CEO title, later hires, or 2018-2026 milestones beyond YC. — None of the shown chunks cover incorporation date, equity, CEO title, later hires, or post-YC milestones.
- ✅ _supported_: The evidence shows no sign of a technical or commercial leader beyond the two founders. — Evidence shows only the two founders; the roboticist declined and a hiring gap is noted.
- ✅ _supported_: Ghaeli recalled the Underworlds proposal as 'blew my mind'. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Ghaeli trained in architecture in Canada. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Ghaeli had never worked in biology. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Ghaeli managed the six-department Underworlds effort and handled the 'urban' side: mappings, implementation and government contacts. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: Matus is described as the scientist with 'a giant vision' (biobot_case_p13_c3). — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 7277 in / 2256 out · 13361 ms · $0.02362</sub>

### 89. DEVIL → CEO (round 18)

The weakest point is the CEO's claim that the founders' 'low ego' quote, the 'think differently but react alike' dynamic, and the Matus/Ghaeli role split are supported by biobot_case_p13_c3 and that 'Ghaeli wanted to lean into building relationships with mayors', because that chunk is not in the evidence displayed. Only biobot_case_p12_c1 and biobot_case_p3_c2 are shown. The CEO also cites biobot_public_history_and_operating_02 for MIT DesignX, Sandbox, the $100K competition, delta v, Demo Day and YC W18, and that chunk is not displayed either. I cannot verify those quotes or facts from the evidence shown, so the CEO is presenting second-hand material as documented record. The displayed chunks support only a narrow story: Ghaeli met Matus in her first week at MIT, they were charged with making Underworlds real, a roboticist post-doc declined to join, and Matus said they needed a mechanical engineer or business help but were a good team. They do not show that Matus is 'the scientist with a giant vision', that she faced a smart-toilet decision, or that the pair complement each other. The CEO also states 'Ghaeli trained in architecture in Canada'. The chunk quotes her saying an architecture degree in Canada is an engineering degree, but that is her own recollection, not a checked record of her degree. The CEO's 'complementary pairing' opinion rests on this unverified attribution of roles. The CEO is also internally inconsistent: it says it will not fill gaps, yet it asserts the later 2017-2018 timeline and the founders' motives as fact. In my inference, the spin-out narrative beyond what chunks p3_c2 and p12_c1 show should be treated as unconfirmed until the missing chunks are displayed.

**Assumptions:**

- The chunks biobot_case_p13_c3 and biobot_public_history_and_operating_02 may exist in the full evidence set but are not displayed to me: my inference, not a claim they do not exist

**Cites:** `biobot_case_p3_c2` — HBS case 821-045, "Biobot Analytics", p.3; `biobot_case_p12_c1` — HBS case 821-045, "Biobot Analytics", p.12

- ✅ _supported_: Matus and Ghaeli were charged with making Underworlds real. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: A roboticist post-doc declined to join the venture. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Matus said they needed a mechanical engineer or business help but were a good team. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The CEO states that Ghaeli trained in architecture in Canada. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: In the chunk, Ghaeli says an architecture degree in Canada is an engineering degree; this is her own recollection, not a checked record of her degree. — It is Ghaeli's recalled statement, with no independent verification of the degree.
- ❔ _unverifiable_: Chunk biobot_case_p13_c3 is not in the evidence displayed to the auditor. — No valid supporting citation
- ❌ _contradicted_: Only biobot_case_p12_c1 and biobot_case_p3_c2 are shown in the displayed evidence. — Several other chunks are displayed besides those two.
- ✅ _supported_: Chunk biobot_public_history_and_operating_02 is not displayed in the evidence shown to the auditor. — The chunk is not among the displayed evidence for this claim.
- ✅ _supported_: Ghaeli met Matus in her first week at MIT. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 7156 in / 1797 out · 11186 ms · $0.02120</sub>

### 90. VC ASSISTANT → VC (round 18)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 91. VC → the panel (round 19) — _follow-up_

The case chunks shown record that the founders wanted a mechanical engineer or someone for the business side, and that a roboticist post-doc declined to join (biobot_case_p12_c1, as of 2017). That is a 2017 record. Today, what are the biggest gaps in the team, and who is the next hire you would make? For each gap, which is a documented record, which is current internal knowledge, and which is your opinion? Also, what would the hire change in the next 12 months, for example in sales, lab operations, or procurement and renewals?

**Cites:** `biobot_case_p12_c1` — HBS case 821-045, "Biobot Analytics", p.12

- ✅ _supported_: The case chunks shown record that the founders wanted a mechanical engineer or someone for the business side. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: The case chunks shown record that a roboticist post-doc declined to join the team (biobot_case_p12_c1, as of 2017). — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The recorded information about the founders' hiring wants and the declined roboticist post-doc is a 2017 record. — JEV jev-1.14.0: evidence covers it 87%

<sub>claude-sonnet-5-5 · 7619 in / 535 out · 7557 ms · $0.01775</sub>

### 92. MODERATOR → CEO (round 19)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 93. CEO → VC (round 19)

Correction first: the Devil is right. In my last answer I cited biobot_case_p13_c3 and biobot_public_history_and_operating_02, and neither is displayed to me now. I should not have presented the 'giant vision' and 'low ego' quotes, the smart-toilet decision, the Ghaeli-and-mayors role split, or the delta v / Demo Day / YC W18 timeline as documented. Treat them as unconfirmed here. What the displayed chunks support: Ghaeli met Matus in her first week at MIT and both were charged with making Underworlds real (biobot_case_p3_c2, 2017). Ghaeli said an architecture degree in Canada is an engineering degree. That is her own recollection, not a checked record of her degree. A roboticist post-doc declined to join (biobot_case_p12_c1, 2017). Matus said they needed a mechanical engineer or someone on the business side but were a good team (biobot_case_p12_c1; biobot_fact_hiring_gap, 2017). The complementary-pairing view is opinion and rests on no displayed evidence.

Answer to your question. Documented gaps (2017 record only): a mechanical engineer and a business-side partner. I cannot verify from the evidence whether either gap was filled, who is on the team today, or who holds which role. Current internal knowledge: I have none in the evidence shown. I cannot report current headcount, open roles or hiring plans, and I will not invent them. Opinion, for a municipal and state buyer base: the next hire I would want is a senior commercial leader owning procurement, renewals and re-bids. The reason is that the MWRA contract ended in July 2026 (biobot_public_contracts_customers_and_06, 2026-07). The CDC contract reportedly went to Verily (biobot_public_competitors_alternatives_and_02, 2026-10-14). Together these point to renewal and procurement risk. These are the only reasons for ranking it first, and they are not an internal diagnosis. A lab-operations or mechanical hire is the second candidate, given the 2017 record. I would rank it second without evidence on today's lab constraints.

What the hire would change in 12 months is a forecast, not a record. A renewals owner could track expiry dates, in-house testing plans and re-bid terms for each program, which I cannot see today. I cannot quantify any revenue effect. To confirm: the current org chart, open roles, who owns renewals now, and churn and win/loss records with buyer-stated reasons.

Most important assumption: that the 2017 gaps and the 2026 renewal events point to a commercial and procurement gap today. That is untested. The gap may already be filled, or the real gap may be elsewhere.

**Assumptions:**

- A senior commercial leader for procurement and renewals as the next hire is my opinion, not evidenced
- The 2017 gaps may or may not persist in 2026; I cannot verify the current team
- The 12-month effect of the hire is a forecast with no supporting numbers
- MWRA and CDC outcomes are treated as signals of renewal risk, not as proof of a team gap

**Cites:** `biobot_case_p3_c2` — HBS case 821-045, "Biobot Analytics", p.3; `biobot_case_p12_c1` — HBS case 821-045, "Biobot Analytics", p.12; `biobot_fact_hiring_gap` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ❔ _unverifiable_: The speaker corrected a previous answer that cited biobot_case_p13_c3 and biobot_public_history_and_operating_02, which are not displayed. — JEV jev-1.14.0: evidence covers it 12%
- ✅ _supported_: The MWRA contract ended in July 2026 (biobot_public_contracts_customers_and_06, 2026-07). — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The CDC contract reportedly went to Verily (biobot_public_competitors_alternatives_and_02, 2026-10-14). — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: Ghaeli met Matus in her first week at MIT and both were charged with making Underworlds real, according to biobot_case_p3_c2 (2017). — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: Ghaeli said an architecture degree in Canada is an engineering degree; this is her own recollection and not a checked record of her degree. — Ghaeli's recalled quote says an architecture degree in Canada is an engineering degree.
- ✅ _supported_: A roboticist post-doc declined to join, according to biobot_case_p12_c1 (2017). — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Matus said they needed a mechanical engineer or someone on the business side but were a good team (biobot_case_p12_c1; biobot_fact_hiring_gap, 2017). — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Documented team gaps in the 2017 record are a mechanical engineer and a business-side partner. — JEV jev-1.14.0: evidence covers it 94%

<sub>claude-sonnet-5-5 · 6956 in / 2340 out · 13872 ms · $0.02426</sub>

### 94. DEVIL → CEO (round 19)

The weakest point is the CEO's pick of a senior commercial leader for procurement and renewals as the next hire. It rests on two events, MWRA ending in July 2026 and the CDC contract reportedly going to Verily, and neither shows a team gap. The MWRA source, as the CEO described it earlier, says the state moved toward in-house testing. That is a buyer's make-versus-buy decision, and a renewals owner may not have been able to prevent it. The CDC loss is a competitive procurement outcome, and the evidence does not say whether Biobot's sales team lost on price, product or process. So the CEO infers a commercial-staffing gap from outcomes with different causes, and the CEO admits it cannot see churn or win/loss reasons. The one documented gap is the 2017 record: a mechanical engineer or someone on the business side. Matus said 'either', so even in 2017 the business-side hire was an alternative to a mechanical hire, not a priority. Moving from that 2017 quote to a 2026 commercial hire is a nine-year extrapolation. The CEO also ranks lab or mechanical hiring second 'without evidence on today's lab constraints', which means the ranking has no evidence on either side. In my inference, the honest answer is that the next hire cannot be ranked from the displayed evidence. The CEO's claim that a renewals owner could track expiry dates and re-bid terms is also weak, because that tracking needs the contract records the CEO says it does not have. If those records exist, a hire is not needed to start. If they do not, the problem is systems and data, not a senior title. I cannot verify current headcount or who owns renewals, so the 'gap' may already be filled.

**Assumptions:**

- The MWRA and CDC outcomes may have causes unrelated to Biobot's commercial staffing: my inference, not stated in the evidence
- The contract-tracking problem may be a records or systems issue rather than a headcount issue: my inference

**Cites:** `biobot_case_p12_c1` — HBS case 821-045, "Biobot Analytics", p.12

- ✅ _supported_: The one documented gap is the 2017 record calling for a mechanical engineer or someone on the business side. — JEV jev-1.14.0: evidence covers it 93%
- ❔ _unverifiable_: The CEO ranks lab or mechanical hiring second 'without evidence on today's lab constraints'. — No evidence on a CEO ranking of lab/mechanical hiring or current lab constraints.
- ✅ _supported_: The CEO's choice of a senior commercial leader for procurement and renewals rests on two events: MWRA ending in July 2026 and the CDC contract reportedly going to Verily. — Evidence confirms both events as procurement risks; the CEO's hiring rationale is not shown.

<sub>claude-sonnet-5-5 · 7064 in / 1645 out · 13226 ms · $0.02050</sub>

### 95. VC ASSISTANT → VC (round 19)

Facts are correct.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Vision and long-term ambition

### 96. VC → the panel (round 20)

Turning to vision: if Biobot works, where is the company in ten years, and what is the end-state product? Is it still wastewater surveillance sold to government agencies, or something broader? Given that agencies can run their own wastewater testing (biobot_public_competitors_alternatives_and_06, 2026-10-22), what stays defensible at that scale? Please separate documented records, current internal plans, and your own opinion.

**Cites:** `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The speaker states that government agencies can run their own wastewater testing, citing biobot_public_competitors_alternatives_and_06 dated 2026-10-22. — JEV jev-1.14.0: evidence covers it 88%

<sub>claude-sonnet-5-5 · 6953 in / 457 out · 6012 ms · $0.01616</sub>

### 97. MODERATOR → CEO (round 20)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 514 in / 43 out · 117 ms · $0.00002</sub>

### 98. CEO → VC (round 20)

Opinion, not record: in ten years I want Biobot to be a population-health data layer for cities and agencies, not only a wastewater testing vendor. Wastewater surveillance would stay the sensor. The product would be a decision system on top of it: trend signals, early warnings, and measures of whether interventions worked. Buyers would include governments first, and possibly other institutions later. Nothing in the displayed evidence says Biobot is pursuing this. I cannot report internal plans, a roadmap or an R&D budget, and I will not invent them.

Documented records. (1) In 2017 the founders were turning sewage into information about 'population health' and wanted to turn data in urine and stool into a business (biobot_case_p1_c1, 2017). In the same chunk, they had been told the idea was too broad and had no defensible intellectual property. (2) Ghaeli cited urbanization, GovTech and the push toward data-driven public-health decisions as market trends (biobot_case_p9_c4, 2017). (3) Agencies can collect, analyze and publish wastewater signals without buying Biobot's complete service. MWRA recorded the end of Biobot's Massachusetts data stream in July 2026 (biobot_public_competitors_alternatives_and_06, 2026-10-22).

What stays defensible. The evidence points the other way for raw testing: the 2026 record shows that build-versus-buy is a real alternative, and the 2017 case says critics saw no defensible IP. I therefore do not claim testing alone is defensible. My opinion is that any moat has to come from things I cannot verify here: a longitudinal data set across many jurisdictions, comparable benchmarks that a single agency cannot produce, analytics that agencies adopt in their workflows, and service so cheap or reliable that in-house testing is not worth it. Each is a hypothesis. I cannot verify that Biobot holds any of them, for example the size or exclusivity of its data, or the terms of its data rights.

Steps from today, as a plan I would propose rather than a record: (1) Stop treating testing as the product, and sell analytics and benchmarking that work even when an agency runs its own lab. (2) Test whether agencies that insource still pay for analytics, using the MWRA case as the first test. (3) Show that the data improves decisions, with buyer-stated outcomes. (4) Only then widen to other buyers. I would confirm before committing: the data rights in contracts, the customer mix, and win/loss reasons, none of which I can see.

**Assumptions:**

- The ten-year vision of a population-health data layer is my opinion, not an internal plan or a documented record
- Defensibility would rest on longitudinal data, benchmarks and analytics, none of which is evidenced for Biobot
- Agencies that insource testing might still pay for analytics; this is untested
- I have no evidence of current internal plans, roadmap or R&D spending

**Cites:** `biobot_case_p1_c1` — HBS case 821-045, "Biobot Analytics", p.1; `biobot_case_p9_c4` — HBS case 821-045, "Biobot Analytics", p.9; `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: MWRA recorded the end of Biobot's Massachusetts data stream in July 2026. — JEV jev-1.14.0: evidence covers it 97%
- ❌ _contradicted_: The displayed evidence contains nothing showing Biobot is pursuing a population-health data layer strategy. — Evidence shows population-health framing, data-driven public health, and data-selling vision.
- ✅ _supported_: In 2017 Biobot's founders were turning sewage into information about population health and wanted to turn urine and stool data into a business. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: In 2017, the founders were told their idea was too broad and had no defensible intellectual property. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Ghaeli cited urbanization, GovTech and the push toward data-driven public-health decisions as market trends in 2017. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Agencies can collect, analyze and publish wastewater signals without buying Biobot's complete service. — JEV jev-1.14.0: evidence covers it 95%

<sub>claude-sonnet-5-5 · 6821 in / 2102 out · 14586 ms · $0.02297</sub>

### 99. DEVIL → CEO (round 20)

The weakest point is the CEO's proposed pivot to 'sell analytics and benchmarking that work even when an agency runs its own lab', tested first on MWRA. The displayed evidence shows only that MWRA recorded the end of Biobot's Massachusetts data stream in July 2026 (biobot_public_competitors_alternatives_and_06, 2026-10-22). It does not say why MWRA left, or whether it was offered or would pay for analytics. Calling MWRA 'the first test' of insourcing-plus-analytics is an inference. The same chunk says CDC receives data from state/local departments as well as contractors, and that agencies can 'collect, analyze, and publish' signals themselves. 'Analyze' and 'publish' are the analytics layer, so that chunk points to agencies doing it in-house, not buying it. The CEO's own cited evidence undercuts the claim that analytics is defensible once testing is insourced. The 2017 chunk (biobot_case_p1_c1) says critics saw 'no defensible intellectual property and limited scalability'. That is a critic's view, and it applies as much to a software analytics layer as to testing. The CEO then lists four moat hypotheses (longitudinal data, benchmarks, workflow adoption, cost) and admits it cannot verify that Biobot holds any of them. A ten-year vision whose defensibility rests on zero verified assets, and whose first test case is a lost customer, is unsupported. The 'population health' framing in biobot_case_p1_c1 is a 2017 founder aspiration, not evidence of a data-layer strategy. In my inference, an agency that has built its own lab is the buyer least likely to pay for a vendor's analytics on data it now owns. Benchmarks need comparable data from many jurisdictions, and insourcing agencies would stop supplying that data to Biobot. The CEO's own step 2 therefore risks shrinking the data set that step 1 depends on. The answer gives no evidence on pricing, willingness to pay, or data-sharing rights, so it does not show any durable position at ten years.

**Assumptions:**

- An insourcing agency is less likely to buy vendor analytics on data it owns: my inference, not evidenced
- Insourcing agencies may stop supplying data to Biobot benchmarks: my inference
- MWRA's reason for ending the data stream is not stated in the displayed evidence

**Cites:** `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_case_p1_c1` — HBS case 821-045, "Biobot Analytics", p.1

- ✅ _supported_: The displayed evidence shows that MWRA recorded the end of Biobot's Massachusetts data stream in July 2026 (biobot_public_competitors_alternatives_and_06, dated 2026-10-22). — JEV jev-1.14.0: evidence covers it 95%
- ❔ _unverifiable_: The CEO lists four moat hypotheses: longitudinal data, benchmarks, workflow adoption, and cost. — Evidence does not show the CEO's stated moat hypotheses; it only gives scenario data.
- ❔ _unverifiable_: The CEO admits it cannot verify that Biobot holds any of the four moat hypotheses. — Evidence shows no proof of moat but not what the CEO admitted.
- ❌ _contradicted_: The answer gives no evidence on pricing, willingness to pay, or data-sharing rights. — Evidence includes pricing floor and willingness-to-pay uncertainty; data-sharing rights unaddressed.
- ❌ _contradicted_: The displayed evidence does not say why MWRA ended the Biobot data stream. — Evidence says state moved toward in-house testing, giving a reason for the end.
- ❌ _contradicted_: MWRA's reason for ending the data stream is not stated in the displayed evidence. — Evidence states the state moved toward in-house testing.
- ✅ _supported_: The displayed evidence does not say whether MWRA was offered analytics or would pay for analytics. — No evidence of analytics being offered to or purchased by MWRA.
- ✅ _supported_: The chunk biobot_public_competitors_alternatives_and_06 says CDC receives data from state and local departments as well as contractors. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The chunk biobot_public_competitors_alternatives_and_06 says agencies can 'collect, analyze, and publish' signals themselves. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The 2017 chunk biobot_case_p1_c1 quotes critics saying Biobot had 'no defensible intellectual property and limited scalability'. — JEV jev-1.14.0: evidence covers it 94%

<sub>claude-sonnet-5-5 · 6849 in / 2021 out · 13435 ms · $0.02163</sub>

### 100. VC ASSISTANT → VC (round 20)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 101. VC → the panel (round 21) — _follow-up_

Of the four steps you proposed (sell analytics that work when an agency runs its own lab, test whether insourcing agencies still pay, prove the data improves decisions, then widen to other buyers), which is the riskiest, and what specific, observable result would show it is achievable? Please say which inputs come from documented records, which from current internal knowledge, and which are opinion. Also, an agency that moves testing in-house may stop sending samples or data to Biobot. How would the benchmarking and longitudinal-data steps work in that case?

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The four proposed steps are: sell analytics that work when an agency runs its own lab, test whether insourcing agencies still pay, prove the data improves decisions, and widen to other buyers. — Evidence does not mention these four proposed steps in the meeting answer.

<sub>claude-sonnet-5-5 · 7128 in / 535 out · 5346 ms · $0.01697</sub>

### 102. MODERATOR → CEO (round 21)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 103. CEO → VC (round 21)

Correction first: the Devil is right on three points. (1) MWRA as the 'first test' is my inference. The evidence I can see does not say why MWRA left or whether it was offered analytics, and the chunk I cited for MWRA (biobot_public_competitors_alternatives_and_06) is not displayed to me now, so I cannot re-verify it here. (2) That chunk, as the Devil describes it, says agencies can 'collect, analyze, and publish' signals themselves, which undercuts my claim that analytics is defensible after insourcing. (3) My step 2 could shrink the data that step 1 depends on. I withdraw 'MWRA as first test' and treat the analytics pivot as a hypothesis, not a plan.

Riskiest step: step 2, whether insourcing agencies still pay. It decides whether the pivot has any buyer. Step 3 (proving better decisions) is hard but only matters if step 2 holds. Step 1 is a build I cannot size.

Observable result that would show step 2 is achievable (my opinion, not a record): at least one agency that runs its own lab signs a paid, multi-period analytics agreement, with the buyer stating in writing why it pays and what it uses the output for. Pass and fail thresholds (number of agencies, price, term) I do not know and will not invent. They need to be set against actual pricing and cost data.

Inputs by type. Documented: in 2017 critics said the idea had no defensible IP and limited scalability, and the founders framed their work as 'population health' (biobot_case_p1_c1, 2017); the 2017 trend claims on data-driven government (biobot_case_p9_c4, 2017); Biobot publicly describes public-sector, pharma and life-science use cases and modeled weekly county/ZIP data, but the paying customer base is not disclosed (biobot_public_company_and_offering_03, 2026-10-21); public sources do not establish revenue, retention, active paying customers or backlog (biobot_public_research_findings_and_03, 2026-10-22). Current internal knowledge: none in the evidence; I cannot report pricing, data rights or willingness to pay. Opinion: everything else, including the four steps.

Data after insourcing. Honest answer: benchmarking and longitudinal data would mostly not work from an agency's own samples, because a lab-owning agency may stop sending data. Possible routes, all unverified: (a) contracts that grant data-sharing or de-identified benchmarking rights as a condition of analytics service; (b) building benchmarks from public agency dashboards, which agencies publish themselves per the Devil's reading of the chunk, but this gives weaker, non-standardized data; (c) keeping other sources of samples, such as the modeled national network (biobot_public_company_and_offering_03), as the benchmark base. I cannot verify that Biobot holds any of these rights or sources. If none is available, the data-layer moat fails and the vision should be narrowed.

Most important assumption: that an agency owning its lab will still pay a vendor for analytics on data it owns. It is untested, and the Devil's inference that it is the buyer least likely to pay may be right. What I would confirm: why MWRA ended, contract data rights, pricing, and win/loss reasons.

**Assumptions:**

- Step 2 (insourcing agencies still paying) is the riskiest step: my opinion
- A paid multi-period analytics agreement from a lab-owning agency, with buyer-stated reasons, would be the observable signal: my opinion, no thresholds known
- Contracts may or may not grant data-sharing or benchmarking rights: unverified
- MWRA's reason for leaving is not stated in the evidence displayed to me
- The Devil's description of the competitors chunk is taken as stated, since that chunk is not displayed to me

**Cites:** `biobot_case_p1_c1` — HBS case 821-045, "Biobot Analytics", p.1; `biobot_case_p9_c4` — HBS case 821-045, "Biobot Analytics", p.9; `biobot_public_company_and_offering_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_research_findings_and_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: In 2017 critics said the idea had no defensible IP and limited scalability, and the founders framed their work as 'population health'. — JEV jev-1.14.0: evidence covers it 89%
- ✅ _supported_: Biobot publicly describes public-sector, pharma and life-science use cases and modeled weekly county/ZIP data, but its paying customer base is not disclosed. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Public sources do not establish Biobot's revenue, retention, active paying customers or backlog. — JEV jev-1.14.0: evidence covers it 89%
- ❔ _unverifiable_: The speaker has no current internal knowledge of pricing, data rights or willingness to pay. — Evidence says nothing about what the speaker knows internally; only public pricing gaps.
- ❔ _unverifiable_: The speaker cannot verify that Biobot holds any data-sharing rights or sample sources needed for these routes. — Evidence doesn't address the speaker's verification ability; agencies control sample access.
- ✅ _supported_: The evidence displayed to the speaker does not say why MWRA left or whether MWRA was offered analytics. — Displayed chunks say MWRA moved in-house but give no reasons or analytics offer.
- ❔ _unverifiable_: The speaker cannot re-verify the cited chunk biobot_public_competitors_alternatives_and_06 because it is not displayed to them. — Claim concerns the speaker's display; the chunk appears in evidence here, but display unknown.
- ✅ _supported_: The Devil describes the chunk biobot_public_competitors_alternatives_and_06 as saying agencies can collect, analyze, and publish signals themselves. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 7164 in / 2743 out · 16181 ms · $0.02644</sub>

### 104. DEVIL → CEO (round 21)

The weakest point is the CEO's fallback route (c): keep 'the modeled national network' as the benchmark base. The cited chunk (biobot_public_company_and_offering_03, 2026-10-21) only says Biobot samples 'hundreds of US sites' and offers 'modeled weekly county/ZIP data'. It does not say who owns those sites, whether the samples are Biobot's own or come from customers, or whether the network is still collecting. If those sites include agency customers who later insource, as MWRA did, the base shrinks in the same way the CEO is trying to avoid. The route is circular. In my inference, modeled data is also not a longitudinal measurement, so it cannot prove that an analytics layer improves decisions (step 3), or serve as a comparable benchmark for agencies that run their own labs. 'Modeled' suggests extrapolation from a subset, and the evidence gives no accuracy or validation figures. A benchmark made of modeled numbers cannot show an agency that its own measured data is out of line, because the reference carries its own uncertainty. The CEO's other routes, (a) contractual data rights and (b) public dashboards, are admitted to be unverified or weaker and non-standardized. Every route therefore fails or depends on facts the CEO cannot see. The answer still says that if none is available 'the vision should be narrowed', which concedes that the moat has no verified basis. The 'observable result' of one paid analytics agreement from a lab-owning agency also has no threshold and does not test benchmarking at all. One buyer paying does not show that a cross-jurisdiction data layer exists. I cannot verify the network's composition, ownership or data rights from the evidence shown.

**Assumptions:**

- Modeled data is a weaker benchmark than measured data and cannot validate decision improvement: my inference, not evidenced
- Sites in the sampled network may include agency customers who could insource: my inference, not stated in the evidence
- The evidence does not say who owns the sampling sites or the data rights

**Cites:** `biobot_public_company_and_offering_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_competitors_alternatives_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The cited chunk biobot_public_company_and_offering_03 (dated 2026-10-21) says Biobot samples 'hundreds of US sites'. — JEV jev-1.14.0: evidence covers it 96%
- ❔ _unverifiable_: The CEO's routes (a) contractual data rights and (b) public dashboards are admitted to be unverified or weaker and non-standardized. — JEV jev-1.14.0: evidence covers it 11%
- ❔ _unverifiable_: The answer states that if none of the routes is available, 'the vision should be narrowed', which concedes that the moat has no verified basis. — Evidence doesn't contain the answer quote; characterization as conceding moat is interpretive.
- ✅ _supported_: The cited chunk biobot_public_company_and_offering_03 (dated 2026-10-21) says Biobot offers 'modeled weekly county/ZIP data'. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 7109 in / 1605 out · 9835 ms · $0.02091</sub>

### 105. VC ASSISTANT → VC (round 21)

Facts are correct.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: The raise: amount, terms and investors

### 106. VC → the panel (round 22)

Moving to the raise: how much are you raising now, on what terms or valuation, from which investors, and how many months of runway will it fund? Please separate documented records from current internal plans and opinion, and say if you cannot see these figures.

**Cites:** `biobot_public_research_findings_and_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)


<sub>claude-sonnet-5-5 · 7531 in / 254 out · 4313 ms · $0.01673</sub>

### 107. MODERATOR → CFO (round 22)

This question will be taken up by the CFO.

<sub>jev:jev-latest · 471 in / 43 out · 98 ms · $0.00002</sub>

### 108. CFO → VC (round 22)

I cannot see a current raise: no amount, instrument, valuation, terms or target investors appear in the evidence, so I will not state them. What I can say: (1) Runway, from a synthetic scenario record dated 2026-09-30 (scenario ground truth, not actual Biobot data): unrestricted cash $12.8M, operating cash burn about $336,630 per month, so roughly 38 months at constant burn, before working-capital changes and not a forecast (biobot_dummy_pricing_and_unit_08). On that scenario, no raise is needed to survive about three years; any raise would be for growth or to buy extra runway, and I do not know which. (2) Funding history, public and unreconciled: Biobot's own announcements establish $20M in 2021 and an undisclosed 2025 round. Third-party databases show a roughly $13.4M-$13.45M round dated 2025-06-06, labelled Series A, Series B or Series A1 depending on the source, and totals of about $40.17M to $41.4M. A secondary site gives about $126.8M valuation as of 2025-06-06 with no filing behind it; treat all of these as estimates (biobot_public_funding_and_investors_05). (3) Investors: a screenshot-only lead says a 2018 $2.5M seed with 16 investors, and Government Technology confirms the amount and Ekistic as lead but not the investor count. The lead is unverified, so I do not rely on it (biobot_public_screenshot_only_funding_leads_08). I cannot verify the current cap table, the investors in the 2025 round, or whether grants are in the totals. Opinion: given the open questions on MWRA and data rights, I would not size a raise until win/loss, retention and contract data are confirmed, and I would tie the amount to milestones, not to a target runway.

**Maths** (✅ checks out): `$12,800,000 cash / $336,630 per month = about 38.0 months`

**Assumptions:**

- Burn stays constant at the scenario level, with no working-capital changes
- The synthetic 2026 packet is scenario ground truth, not actual Biobot data
- Any new raise would be additional to the $12.8M of cash; the evidence does not say whether it is already included

**Cites:** `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_funding_and_investors_05` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_screenshot_only_funding_leads_08` — Unverified lead (Biobot public data and source ledger)

- ✅ _supported_: Third-party databases show a round of roughly $13.4M to $13.45M dated 2025-06-06. — JEV jev-1.14.0: evidence covers it 89%
- ✅ _supported_: Third-party databases label the 2025-06-06 round as Series A, Series B or Series A1 depending on the source. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Third-party databases show total funding of about $40.17M to $41.4M. — Screenshots show $41.4M; CB Insights shows about $40.17M.
- ✅ _supported_: A secondary site gives a valuation of about $126.8M as of 2025-06-06, with no filing behind it. — JEV jev-1.14.0: evidence covers it 89%
- ✅ _supported_: A screenshot-only lead says Biobot had a 2018 $2.5M seed round with 16 investors. — $2.5M 2018 seed supported; the 16-investor screenshot lead isn't shown in evidence.
- ✅ _supported_: Government Technology confirms the $2.5M amount and Ekistic as lead investor for the 2018 seed, but not the investor count. — Report gives $2.5M and Ekistic lead; no investor count mentioned.
- ✅ _supported_: Cash of $12.8M divided by burn of $336,630 per month equals about 38.0 months. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: A synthetic scenario record dated 2026-09-30 reports unrestricted cash of $12.8M. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: A synthetic scenario record dated 2026-09-30 reports operating cash burn of about $336,630 per month. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: At constant burn, the scenario's cash would last roughly 38 months, before working-capital changes, and this is not a forecast. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: Biobot's own announcements establish a $20M funding round in 2021. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: Biobot's own announcements establish an undisclosed funding round in 2025. — JEV jev-1.14.0: evidence covers it 94%

<sub>claude-sonnet-5-5 · 6746 in / 1867 out · 10789 ms · $0.02633</sub>

### 109. DEVIL → CFO (round 22)

The weakest point is the CFO's claim that 'no raise is needed to survive about three years'. It takes the synthetic 2026-09-30 record (biobot_dummy_pricing_and_unit_08) as constant burn, but that record's $336,630 monthly burn depends on annualized gross profit of $8,260,440 holding. The CFO has just conceded that MWRA ended its Biobot data stream in July 2026 and that retention, win/loss and contract data are unconfirmed. A lost customer is a direct threat to the gross profit that makes burn only $336,630. In my inference, if revenue falls, burn rises and the 38 months shrinks. The 38-month figure also excludes working-capital changes, and the evidence does not say whether the $12.8M is net of the 2025 round proceeds or what restricted cash or debt exists. The CFO also cannot say whether a raise has already closed. The runway claim is therefore a best case. It cannot support a statement about whether a raise is needed. Sensitivity: if gross profit fell by a quarter, burn would rise by about $2.07M a year and runway would drop to about 25 months. The conclusion is inconsistent with the CFO's own advice not to size a raise until retention is confirmed, because that advice admits the revenue base is unverified. Separately, Growjo's $3.5M revenue estimate (biobot_public_business_and_market_01, 2026-10-22) is unverified and conflicts with the synthetic gross profit of $8.26M plus $6.12M service cost, which implies about $14.4M revenue. The two sources differ by about four times, and the CFO did not reconcile them. I treat Growjo as weak evidence, but the gap shows how fragile the base is.

**Maths** (❌ does not add up): `$8,260,440 x 25% = $2,065,110 lost gross profit per year; $4,039,560 + $2,065,110 = $6,104,670 burn per year = $508,723 per month; $12,800,000 / $508,723 = about 25.2 months; $8,260,440 + $6,123,560 = $14,384,000 implied revenue; $14,384,000 / $3,500,000 = about 4.1x`

**Assumptions:**

- A 25% fall in gross profit is an illustrative stress case, not evidenced
- Service cost stays at $6,123,560 regardless of gross profit changes in the stress case, so cost is not scaled down
- Implied revenue equals gross profit plus service cost, as the synthetic record appears to define it
- MWRA's loss reducing revenue is my inference; its size is not shown in the evidence

**Cites:** `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_business_and_market_01` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The CFO claimed that no raise is needed to survive about three years. — Runway of 38 months is shown, but the CFO's statement isn't in evidence.
- ✅ _supported_: The 38-month runway figure excludes working-capital changes. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The evidence does not say whether the $12.8M cash figure is net of the 2025 round proceeds or what restricted cash or debt exists. — Cash stated as unrestricted; no source covers 2025 proceeds, restricted cash or debt.
- ✅ _supported_: If gross profit fell by a quarter, burn would rise by about $2.07M a year. — 25% of $8,260,440 is about $2.065M, which raises burn by that amount.
- ✅ _supported_: If gross profit fell by a quarter, runway would drop to about 25 months. — Burn $6.10M/yr; $12.8M divided by $508.7K a month is about 25.2 months.
- ✅ _supported_: Growjo estimates revenue of $3.5M (biobot_public_business_and_market_01, dated 2026-10-22). — JEV jev-1.14.0: evidence covers it 90%
- ✅ _supported_: The Growjo $3.5M revenue estimate is unverified. — JEV jev-1.14.0: evidence covers it 91%
- ✅ _supported_: The synthetic gross profit of $8.26M plus $6.12M service cost implies about $14.4M revenue. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: A 25% fall in gross profit ($8,260,440 x 25%) equals $2,065,110 lost gross profit per year. — 8,260,440 × 0.25 = 2,065,110.
- ✅ _supported_: Annual burn of $4,039,560 plus $2,065,110 equals $6,104,670, which is $508,723 per month. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: $12,800,000 divided by $508,723 per month is about 25.2 months of runway. — 4,039,560+2,065,110=6,104,670/12=508,723; 12.8M/508,723≈25.16.
- ✅ _supported_: Implied revenue of $8,260,440 plus $6,123,560 equals $14,384,000. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: Implied revenue of $14,384,000 divided by Growjo's $3,500,000 is about 4.1x. — 14,384,000/3,500,000=4.11; Growjo revenue is $3.5M.
- ✅ _supported_: The synthetic 2026-09-30 record shows monthly burn of $336,630. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The synthetic 2026-09-30 record shows annualized gross profit of $8,260,440. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The $336,630 monthly burn depends on annualized gross profit of $8,260,440 holding. — JEV jev-1.14.0: evidence covers it 96%
- ❔ _unverifiable_: The CFO conceded that MWRA ended its Biobot data stream in July 2026. — MWRA ended in July 2026 is supported; the CFO conceding it is not shown.
- ❔ _unverifiable_: The CFO conceded that retention, win/loss and contract data are unconfirmed. — JEV jev-1.14.0: evidence covers it 8%
- 🧮 _miscalculated_: $8,260,440 x 25% = $2,065,110 lost gross profit per year; $4,039,560 + $2,065,110 = $6,104,670 burn per year = $508,723 per month; $12,800,000 / $508,723 = about 25.2 months; $8,260,440 + $6,123,560 = $14,384,000 implied revenue; $14,384,000 / $3,500,000 = about 4.1x — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 6880 in / 2261 out · 13579 ms · $0.02219</sub>

### 110. VC ASSISTANT → VC (round 22)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 111. VC → the panel (round 23) — _follow-up_

The synthetic 2026-09-30 packet shows annualized gross profit of $8,260,440 and cash burn of about $336,630 a month, against FY2024 gross profit of $2,300,000 and an operating loss of $7,000,000. What drove the improvement in burn between those two dates, and how much of it depends on revenue that could leave, as MWRA did? If a raise were sized from that base, what would you fund with a smaller amount, what would you cut, and which milestone would be the first to slip?

**Cites:** `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_historical_efficiency_and_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The synthetic 2026-09-30 packet shows annualized gross profit of $8,260,440. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The synthetic 2026-09-30 packet shows cash burn of about $336,630 a month. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: FY2024 gross profit was $2,300,000. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: FY2024 operating loss was $7,000,000. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: Annualized gross profit of $8,260,440 is larger than FY2024 gross profit of $2,300,000. — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: A monthly cash burn of about $336,630 corresponds to roughly $4,039,560 of annual cash burn. — JEV jev-1.14.0: evidence covers it 94%

<sub>claude-sonnet-5-5 · 7418 in / 714 out · 6305 ms · $0.01764</sub>

### 112. MODERATOR → CFO (round 23)

This follow-up stays with the CFO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 113. CFO → VC (round 23)

Drivers of the burn change, from the two synthetic records (scenario ground truth, not actual Biobot data): FY2024 (2024-12-31) gross profit $2.30M = $6.35M revenue - $4.05M service cost; opex $9.30M; operating loss $7.00M. 2026-09-30 annualized: gross profit $8,260,440 after service cost $6,123,560; opex $12.3M; burn about $4,039,560 a year. Burn fell by about $2.96M a year because gross profit rose by about $5.96M while opex rose by $3.0M. The records do not say why gross profit rose (price, volume, mix, new customers), so I cannot attribute the improvement and will not guess. Dependence on revenue that could leave: I cannot see revenue by customer, so I cannot size MWRA or any one customer. Only the Devil's stress case is available: a 25% fall in gross profit lifts burn to about $6.10M a year and cuts runway to about 25 months. That case is illustrative, not evidenced. Corrections to my last answer: 'no raise is needed to survive about three years' was too strong. The 38 months is a best case at constant burn, before working capital, and it assumes the revenue base holds. I also did not reconcile Growjo's $3.5M revenue estimate (biobot_public_business_and_market_01, 2026-10-22, unverified, weak) with the synthetic implied revenue of about $14.4M. I rely on the synthetic record because it is the scenario ground truth, and the 4x gap is a reason to confirm actual revenue before any raise. If a raise were sized from this base: I cannot see any raise plan, so this is opinion. I would fund (1) retention and win/loss work, contract data-rights review and the paid-analytics test with a lab-owning agency, and (2) the existing core service. I would cut or defer the portion of the $12.3M of sales, research and corporate expense not tied to those tests; the evidence gives no breakdown, so I cannot say which lines or how much. The first milestone to slip would be the paid analytics agreement from a lab-owning agency, because it is the riskiest, depends on a buyer's decision and has no threshold or pricing data behind it. The most important assumption is that gross profit holds near $8.26M; if it does not, burn and runway change quickly.

**Maths** (❌ does not add up): `FY2024 burn proxy: $2,300,000 - $9,300,000 = -$7,000,000; 2026 annualized: $8,260,440 - $12,300,000 = -$4,039,560; improvement = $7,000,000 - $4,039,560 = $2,960,440 per year; gross profit change = $8,260,440 - $2,300,000 = $5,960,440; opex change = $12,300,000 - $9,300,000 = $3,000,000; $5,960,440 - $3,000,000 = $2,960,440; base runway = $12,800,000 / $336,630 = about 38.0 months; stress: $8,260,440 x 25% = $2,065,110; $4,039,560 + $2,065,110 = $6,104,670 per year = $508,723 per month; $12,800,000 / $508,723 = about 25.2 months; implied revenue = $8,260,440 + $6,123,560 = $14,384,000`

**Assumptions:**

- The synthetic packet is scenario ground truth, not actual Biobot data
- The FY2024 operating loss is compared with the 2026 annualized burn as like for like; both are before working-capital changes and financing
- A 25% gross profit fall is an illustrative stress case, not evidenced
- Service cost stays fixed in the stress case
- The evidence does not say what drove the gross profit rise or whether MWRA's revenue is in it
- Which opex lines are discretionary is unknown
- Any raise would be on top of the $12.8M; the evidence does not say whether 2025 proceeds are in it

**Cites:** `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_historical_efficiency_and_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_public_funding_and_investors_05` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: FY2024 gross profit was $2.30M, equal to $6.35M revenue minus $4.05M service cost (synthetic scenario record, not actual Biobot data). — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: Base runway of about 38.0 months ($12.8M divided by $336,630 per month) is a best case at constant burn, before working capital, and assumes the revenue base holds. — JEV jev-1.14.0: evidence covers it 89%
- ✅ _supported_: Growjo's $3.5M revenue estimate (unverified, weak source) has not been reconciled with the synthetic implied revenue of about $14.4M, a gap of roughly 4x. — Growjo $3.5M unverified; implied $14.38M; ratio ~4.1x; no reconciliation shown.
- ✅ _supported_: Implied synthetic revenue of about $14,384,000 equals annualized gross profit plus service cost ($8,260,440 + $6,123,560). — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: FY2024 opex was $9.30M and the FY2024 operating loss was $7.00M (synthetic scenario record). — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The 2026-09-30 annualized gross profit was $8,260,440 after service cost of $6,123,560 (synthetic scenario record). — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The 2026-09-30 annualized opex was $12.3M and annualized burn was about $4,039,560 a year (synthetic scenario record). — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: Annual burn fell by about $2.96M because gross profit rose by about $5.96M while opex rose by $3.0M. — GP 2.3→8.26M (+5.96M); opex 9.3→12.3M (+3.0M); burn 7.0→4.04M (−2.96M).
- ✅ _supported_: The records do not say why gross profit rose (price, volume, mix or new customers), so the improvement cannot be attributed. — Records give totals only, no price/volume/mix decomposition.
- ✅ _supported_: Revenue by customer is not visible in the evidence, so MWRA or any single customer's share of revenue cannot be sized. — No revenue-by-customer figures; MWRA contract value only reported, not revenue share.
- ❔ _unverifiable_: The stress case burn of $6,104,670 a year equals $508,723 per month, and $12.8M divided by that gives about 25.2 months of runway. — Stress case burn of $6,104,670 not in evidence; only base 4.04M.
- 🧮 _miscalculated_: FY2024 burn proxy: $2,300,000 - $9,300,000 = -$7,000,000; 2026 annualized: $8,260,440 - $12,300,000 = -$4,039,560; improvement = $7,000,000 - $4,039,560 = $2,960,440 per year; gross profit change = $8,260,440 - $2,300,000 = $5,960,440; opex change = $12,300,000 - $9,300,000 = $3,000,000; $5,960,440 - $3,000,000 = $2,960,440; base runway = $12,800,000 / $336,630 = about 38.0 months; stress: $8,260,440 x 25% = $2,065,110; $4,039,560 + $2,065,110 = $6,104,670 per year = $508,723 per month; $12,800,000 / $508,723 = about 25.2 months; implied revenue = $8,260,440 + $6,123,560 = $14,384,000 — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 7296 in / 2574 out · 15240 ms · $0.02663</sub>

### 114. DEVIL → CFO (round 23)

The weakest point is the CFO's attribution of the burn improvement: the answer says gross profit rose $5.96M and opex rose $3.0M, then treats FY2024 and the 2026 annualized figures as like for like. They are not comparable. FY2024 is an audited-style full-year actual in the scenario record, while the 2026 figure is an annualization of a single date (2026-09-30) and the evidence does not say what period it annualizes. If it extrapolates a strong quarter or month, the $8.26M can be a run-rate peak, not a level that holds. In my inference, a 3.6x jump in gross profit ($2.30M to $8.26M) in under two years is exactly the kind of change that needs a driver, and the CFO admits she cannot name one (price, volume, mix, new customers). Without a driver, she cannot say whether the revenue is recurring, which is the answer to the VC's question about revenue that could leave. Her response to that question is a 25% stress case borrowed from the Devil, which is arbitrary and not evidenced. The cost side is also shaky: she holds service cost fixed at $6.12M in the stress case, but service cost rose only from $4.05M to $6.12M while gross profit rose 3.6x. That implies revenue about $14.4M versus $6.35M, so the revenue base more than doubled, which makes dependence on a few large customers plausible (my inference, no customer data shown). Also, the 'does NOT add up' flag matters: the improvement figure compares a loss before financing with a burn before working capital, so the $2.96M is a proxy, not a measured cash change. Finally, 'fund retention and win/loss work' is not a sized use of funds: she gives no cost for it, so the question of what a smaller raise would fund is unanswered.

**Maths** (❌ does not add up): `$8,260,440 / $2,300,000 = about 3.6x gross profit growth; $8,260,440 + $6,123,560 = $14,384,000 implied 2026 revenue; $14,384,000 / $6,350,000 = about 2.3x revenue growth; $6,123,560 / $4,050,000 = about 1.5x service cost growth`

**Assumptions:**

- The 2026 annualized figure may rest on a short period; the evidence does not state the base period (my inference)
- Large revenue growth could mean customer concentration; no customer-level data is shown (my inference)
- Implied revenue equals gross profit plus service cost, as the synthetic record appears to define it

**Cites:** `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_historical_efficiency_and_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ❔ _unverifiable_: The CFO's response to the VC's question about revenue that could leave uses a 25% stress case borrowed from the Devil. — JEV jev-1.14.0: evidence covers it 7%
- ✅ _supported_: The CFO holds service cost fixed at $6.12M in the stress case. — $6,123,560 service cost used in burn; stress case itself not described.
- ✅ _supported_: Service cost rose from $4.05M to $6.12M. — FY2024 $4.05M; annualized 2026 $6,123,560.
- ✅ _supported_: Implied 2026 revenue is about $14.4M versus $6.35M previously, meaning the revenue base more than doubled. — $14.384M vs $6.35M is 2.27x, more than doubled.
- ✅ _supported_: The CFO's answer states gross profit rose $5.96M and opex rose $3.0M. — Figures match computed differences; whether CFO stated them is not directly shown.
- ✅ _supported_: Gross profit growth is about 3.6x, calculated as $8,260,440 / $2,300,000. — 8,260,440/2,300,000 = 3.59.
- ✅ _supported_: Implied 2026 revenue is $14,384,000, calculated as $8,260,440 + $6,123,560. — 8,260,440 + 6,123,560 = 14,384,000, matching contracted ARR.
- ✅ _supported_: Revenue growth is about 2.3x, calculated as $14,384,000 / $6,350,000. — 14.384/6.35 = 2.27x, about 2.3x.
- ✅ _supported_: Service cost growth is about 1.5x, calculated as $6,123,560 / $4,050,000. — 6,123,560/4,050,000 = 1.51x.
- ✅ _supported_: The CFO's answer treats FY2024 and the 2026 annualized figures as like for like. — FY2024 actual vs annualized 2026 run-rate compared directly; different time bases.
- ❔ _unverifiable_: FY2024 is an audited-style full-year actual in the scenario record. — Full-year amounts shown, but nothing mentions audit status.
- ✅ _supported_: The 2026 figure is an annualization of a single date, 2026-09-30. — 2026 figures annualized as of 2026-09-30 snapshot.
- ❌ _contradicted_: The evidence does not say what period the 2026 figure annualizes. — Annualization basis given: 886 sites, 24 samples/yr, as of 2026-09-30.
- ✅ _supported_: Gross profit rose from $2.30M to $8.26M, a 3.6x jump, in under two years. — 2.30M to 8.26M is 3.59x, end-2024 to Sept 2026.
- 🧮 _miscalculated_: $8,260,440 / $2,300,000 = about 3.6x gross profit growth; $8,260,440 + $6,123,560 = $14,384,000 implied 2026 revenue; $14,384,000 / $6,350,000 = about 2.3x revenue growth; $6,123,560 / $4,050,000 = about 1.5x service cost growth — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 7188 in / 2119 out · 12818 ms · $0.02251</sub>

### 115. VC ASSISTANT → VC (round 23)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Use of funds and milestones

### 116. VC → the panel (round 24)

Setting aside any specific raise amount, what would new capital be spent on, split by category such as sales, research, sampling and lab operations, and corporate? For each category, say which dated milestone it must reach before the next round, and what the observable pass or fail evidence for each would be. Please separate documented records from internal plans and opinion, and say if you cannot see these plans.

**Cites:** `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)


<sub>claude-sonnet-5-5 · 8113 in / 378 out · 4043 ms · $0.01833</sub>

### 117. MODERATOR → CFO (round 24)

This question will be taken up by the CFO.

<sub>jev:jev-latest · 496 in / 43 out · 131 ms · $0.00002</sub>

### 118. CFO → VC (round 24)

I cannot see any documented spend plan, budget split or dated milestones. The evidence has only one cost split: annualized sales, research and corporate opex of $12,300,000 as a single line, plus $6,123,560 service cost (biobot_dummy_pricing_and_unit_08, synthetic, 2026-09-30). It does not break opex into sales, research, sampling/lab or corporate. So the amounts below are my opinion and illustrative, not a plan, and I will not state a raise size. Framing (opinion): no raise is sized here. I would fund only the tests that decide whether to raise at all, and I would size a raise from them. Proposed test budgets (all assumptions, to be replaced with real quotes): (1) Retention and win/loss, with a contract data-rights review, in the corporate category: $150k, milestone 2027-03-31 (assumed date). Pass: every customer whose contract ended or was due in the cohort is coded renew or lost with a reason, and the data-rights status of each contract is documented. Fail: revenue by customer or contract terms cannot be produced. This also addresses the 33-renewal cohort evidence: 24 renewed with a dedicated budget line and 9 on time-limited grant money (biobot_dummy_procurement_and_sales_04, synthetic, 2026-09-30), so the pass test should also report how much of the 9 grant-funded renewals renewed again. (2) Sales, paid-analytics test with lab-owning agencies: $400k, milestone 2027-06-30 (assumed). Pass: at least 1 signed paid agreement (my threshold, not evidenced) with a stated price and term. Fail: none signed, or only free pilots, since the evidence says a free pilot is not a paying program (biobot_dummy_historical_operating_trajectory_01). (3) Research, validation of the modeled network against measured data: $250k, milestone 2027-06-30 (assumed). Pass: published error figures against measured sites with an agreed tolerance. Fail: no validation figures can be produced. (4) Sampling and lab operations and the core service: no new capital, funded by gross profit as already modeled; sizing is impossible without a breakdown. Total of the three tests is $800k, equal to about 2.4 months of the current burn. This is not a recommended raise, only the test cost. Documented versus not: only the cash, burn and opex lines are documented. All category amounts, dates and pass thresholds are my assumptions. I cannot verify actual spend by category, team size, or whether any of this is already underway. Any raise sized later should follow these results and should be checked against the unreconciled gross profit base, since I cannot reconcile 2026 run-rate gross profit to a driver.

**Maths** (✅ checks out): `$150,000 + $400,000 + $250,000 = $800,000; $800,000 / $336,630 per month = about 2.4 months of burn`

**Assumptions:**

- All category amounts are my illustrative estimates, not from evidence
- Milestone dates 2027-03-31 and 2027-06-30 are assumed
- The pass thresholds, including one signed paid agreement, are my own and not evidenced
- The synthetic packet is scenario ground truth, not actual Biobot data
- Sampling and lab operations are funded from existing gross profit; the evidence gives no breakdown
- Burn of $336,630 per month holds

**Cites:** `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_historical_operating_trajectory_01` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The evidence shows a 33-renewal cohort: 24 renewed with a dedicated budget line and 9 on time-limited grant money (biobot_dummy_procurement_and_sales_04, synthetic, 2026-09-30). — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The evidence states that a free pilot is not a paying program (biobot_dummy_historical_operating_trajectory_01). — JEV jev-1.14.0: evidence covers it 97%
- ❌ _contradicted_: The evidence contains only one cost split: annualized sales, research and corporate opex of $12,300,000 as a single line. — Evidence also gives FY2024 and FY2025 opex figures, so it is not the only split.
- ❔ _unverifiable_: The total of the three tests is $800,000. — JEV jev-1.14.0: evidence covers it 11%
- ✅ _supported_: $800,000 equals about 2.4 months of the current burn at $336,630 per month. — 800,000/336,630 ≈ 2.38 months using the stated monthly burn.
- ❔ _unverifiable_: Only the cash, burn and opex lines are documented in the evidence. — No valid supporting citation
- ✅ _supported_: The evidence contains a $6,123,560 service cost (biobot_dummy_pricing_and_unit_08, synthetic, 2026-09-30). — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The evidence does not break opex into sales, research, sampling/lab or corporate categories. — Opex is only given as combined sales, research and corporate lump sums.

<sub>claude-sonnet-5-5 · 7138 in / 2647 out · 15547 ms · $0.02504</sub>

### 119. DEVIL → CFO (round 24)

The weakest point is the CFO's $800k test budget and its claim that it equals 'about 2.4 months of the current burn'. None of the $150k, $400k or $250k figures has evidence behind it; she admits they are illustrative, yet the total is presented as the cost of deciding whether to raise. That total is a number with no costing basis. It is also framed against the wrong base. Her own earlier concession was that the 38-month burn is a best case and is not stress-tested, so '2.4 months of burn' makes the tests look cheap against a figure she has already called unreliable. In my inference, a credible paid-analytics test with a lab-owning agency needs sales cycle time, pilot support, and lab and analytic staff, and $400k for a single signed agreement is likely understated. The evidence shows municipal and county buyers rely on appropriations and state buyers on grant mixes (biobot_dummy_procurement_and_sales_04), so a buyer's budget cycle may not fit a 2027-06-30 milestone, a date she also assumed. The pass threshold of one signed agreement is her own and is weak: a single contract would not show repeatable demand, and it would pass the test even if the 9 grant-funded renewals were not renewed again. Her item (4) assumes zero new capital for sampling and lab operations, but she admits she cannot see the breakdown of the $6.12M service cost or the $12.3M opex, so she cannot know the tests would not draw on existing lab capacity. The $800k therefore has no support as a cost and the milestone logic is not sized, so it cannot answer the VC's request for a spend split.

**Maths** (✅ checks out): `$150,000 + $400,000 + $250,000 = $800,000; $800,000 / $336,630 per month = about 2.4 months`

**Assumptions:**

- The CFO's category amounts, dates and thresholds are her own assumptions, not evidenced
- A paid-analytics test likely costs more than $400k (my inference, no costing evidence shown)
- Buyer budget cycles may not fit the 2027-06-30 milestone (my inference)

**Cites:** `biobot_dummy_procurement_and_sales_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_pricing_and_unit_08` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The CFO's $800k test budget is claimed to equal about 2.4 months of the current burn. — The claim is that CFO asserted 2.4 months; arithmetic matches burn.
- ✅ _supported_: The evidence shows municipal and county buyers rely on appropriations and state buyers rely on grant mixes. — JEV jev-1.14.0: evidence covers it 94%
- ❔ _unverifiable_: The CFO's item (4) assumes zero new capital for sampling and lab operations. — JEV jev-1.14.0: evidence covers it 11%
- ❔ _unverifiable_: The CFO admits she cannot see the breakdown of the $6.12M service cost or the $12.3M opex. — JEV jev-1.14.0: evidence covers it 9%
- ✅ _supported_: $150,000 + $400,000 + $250,000 = $800,000. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: $800,000 / $336,630 per month is about 2.4 months. — 800,000 / 336,630 = 2.38, about 2.4 months.
- ❔ _unverifiable_: The $800k total is presented as the cost of deciding whether to raise capital. — JEV jev-1.14.0: evidence covers it 11%

<sub>claude-sonnet-5-5 · 7131 in / 1709 out · 12369 ms · $0.02120</sub>

### 120. VC ASSISTANT → VC (round 24)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 121. VC → the panel (round 25) — _follow-up_

Which single milestone do you expect the next investor to judge you on, and how likely is it to be hit by its date? Please state the metric, the threshold and the date. Given a 163-day median municipal or county sales cycle and 318 days for state or federal buyers, say whether that date leaves enough time to sign and show renewal, and which part of your answer comes from documented records versus your own estimate.

**Cites:** `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: The median municipal or county sales cycle is 163 days. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The median state or federal sales cycle is 318 days. — JEV jev-1.14.0: evidence covers it 97%

<sub>claude-sonnet-5-5 · 7617 in / 461 out · 5912 ms · $0.01762</sub>

### 122. MODERATOR → CFO (round 25)

This follow-up stays with the CFO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 123. CFO → VC (round 25)

The single milestone I would expect the next investor to judge us on (my opinion, not documented): a signed paid annual agreement from a lab-owning agency, plus retention evidence, by 2027-06-30. Metric: signed paid annual agreements with a stated price and term. Threshold: at least 2 (corrected from my earlier 1; the Devil is right that one contract does not show repeatable demand), with the 9 grant-funded renewals from the due cohort checked for second renewal. Likelihood: low for 2 signed by 2027-06-30 given the cycle below; I give no percentage because the evidence has no win-rate or pipeline data. Documented versus estimate: the cycle lengths are documented; the threshold, date, count and likelihood are mine. Timing: the median cycle is 163 days for municipal or county buyers and 318 days for state or federal buyers, measured from qualified discovery to signed annual agreement, and a pilot is a separate paid stage, not proof of renewal (biobot_dummy_procurement_and_sales_03, synthetic, 2026-09-30). From 2026-10-31 to 2027-06-30 is 242 days. That fits a 163-day municipal median only if discovery is already qualified and the buyer is typical, with 79 days of slack. It does not fit the 318-day state or federal median, which would close around 2027-09-14. And a first annual renewal cannot be shown by 2027-06-30, since a 163-day signing from now ends about 2027-04-12 and a renewal would fall about a year later. So the date leaves enough time to sign at most one municipal or county deal, and not enough to show renewal. Honest correction: a deal signed by 2027-06-30 is evidence of sales, not retention. For renewal evidence I can only use the existing cohort: 24 of 33 renewed with a dedicated budget line and 9 on time-limited grant money (biobot_dummy_procurement_and_sales_04, synthetic, 2026-09-30). That is 33 renewals; I cannot see the non-renewals, so I cannot state a renewal rate. Also, the earlier $400k test cost and the 2.4-month comparison were unsupported illustrations and rested on a burn figure I already called a best case; I withdraw them as costing. The most important assumption is that discovery is already qualified for any target agency; if not, the clock starts later and the date is missed. I cannot verify any pipeline, whether the lab-owning agencies are in discovery, or whether a raise has already closed.

**Maths** (❌ does not add up): `2026-10-31 to 2027-06-30 = 30 (Nov) + 31 (Dec) + 31 (Jan) + 28 (Feb) + 31 (Mar) + 30 (Apr) + 31 (May) + 30 (Jun) = 242 days; 242 - 163 = 79 days slack (municipal or county); 163 - 242 = -79 means signing fits; 318 - 242 = 76 days late (state or federal); 24 + 9 = 33 renewed contracts`

**Assumptions:**

- Clock starts 2026-10-31 with discovery already qualified; the evidence does not say any agency is in discovery
- Median cycle is applied to a single deal; actual deals vary around the median
- Threshold of 2 signed paid agreements, the date 2027-06-30 and the low likelihood are my opinion, not evidenced
- A lab-owning agency buys on the municipal or county cycle; this is unknown and could be the state or federal cycle
- Annual agreement means renewal is due about 12 months after signing
- Synthetic packet is scenario ground truth, not actual Biobot data

**Cites:** `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ✅ _supported_: A 318-day state or federal cycle starting 2026-10-31 would close around 2027-09-14. — 2026-10-31 plus 318 days is 2027-09-15, so about 2027-09-14.
- ✅ _supported_: A 163-day signing from 2026-10-31 ends about 2027-04-12, so a first annual renewal would fall about a year later and cannot be shown by 2027-06-30. — 163 days from 2026-10-31 is 2027-04-12; renewal a year later is after June 2027.
- ✅ _supported_: The existing cohort contains 24 renewals with a dedicated budget line and 9 renewals on time-limited grant money, totaling 33 renewals. — JEV jev-1.14.0: evidence covers it 98%
- ❌ _contradicted_: The renewal count of 33 covers only renewals; non-renewals are not visible, so a renewal rate cannot be stated. — Evidence shows 35 due, 33 renewed, 2 churned: 94.3% renewal.
- ❌ _contradicted_: The speaker gives no percentage likelihood because the evidence contains no win-rate or pipeline data. — Evidence does contain pipeline data: 54 opportunities, $7.6M unweighted, $2.9M weighted.
- ✅ _supported_: The median sales cycle is 163 days for municipal or county buyers and 318 days for state or federal buyers, measured from qualified discovery to signed annual agreement. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: A pilot is a separate paid stage and is not proof of renewal. — JEV jev-1.14.0: evidence covers it 96%
- ❔ _unverifiable_: From 2026-10-31 to 2027-06-30 is 242 days. — JEV jev-1.14.0: evidence covers it 15%
- ✅ _supported_: The 242-day window leaves 79 days of slack against the 163-day municipal or county median. — 242 minus 163 equals 79; municipal median is 163.
- ✅ _supported_: The 242-day window is 76 days short of the 318-day state or federal median. — 318 minus 242 equals 76; median confirmed.
- 🧮 _miscalculated_: 2026-10-31 to 2027-06-30 = 30 (Nov) + 31 (Dec) + 31 (Jan) + 28 (Feb) + 31 (Mar) + 30 (Apr) + 31 (May) + 30 (Jun) = 242 days; 242 - 163 = 79 days slack (municipal or county); 163 - 242 = -79 means signing fits; 318 - 242 = 76 days late (state or federal); 24 + 9 = 33 renewed contracts — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 7673 in / 2410 out · 14994 ms · $0.02650</sub>

### 124. DEVIL → CFO (round 25)

The weakest point is the CFO's timing arithmetic and the conclusion built on it: she says 242 days 'fits a 163-day municipal median' with 79 days of slack, then says the date 'leaves enough time to sign at most one municipal or county deal'. That conflicts with her own threshold of at least 2 signed agreements. A median is the midpoint, so about half of deals take longer than 163 days. Even in her best case of qualified discovery and a municipal buyer, the chance that any single deal closes inside 242 days is not 'fits'. She gives no distribution, and the evidence shows only medians (biobot_dummy_procurement_and_sales_03, 2026-09-30). The 79 days of slack is therefore not a safety margin. The clock start is also invented: the 2026-10-31 start assumes discovery is already qualified, and she concedes she cannot see any pipeline. If no agency is qualified today, a 163-day cycle from, say, a 2027-01 qualification lands around 2027-06-12 at the median and leaves almost no margin. For state or federal buyers the date is missed. She also treats a lab-owning agency as municipal or county without evidence; as an inference, such an agency is plausibly state-level, which would put it on the 318-day cycle and make the milestone impossible. Her 'low likelihood' conclusion is therefore probably right, but the reasoning that one municipal deal fits is unsupported. A milestone that the documented cycle data makes close to unattainable is not a credible answer to the VC's request for the single milestone the next investor will judge. She also drops retention from the milestone ('evidence of sales, not retention') while still naming retention evidence in the metric, so the metric and threshold do not match. Finally, she cites the 33-renewal cohort (24 dedicated budget, 9 grant) but cannot state a renewal rate, while the FY2022 cohort in the evidence gives 10 of 11 renewed, or 90.9% (biobot_dummy_historical_contract_cohorts_04, 2022-12-31). That is a documented retention figure she did not use. It is four years old, so it cannot stand in for the 2026 rate, but retention evidence exists that her milestone ignores.

**Maths** (✅ checks out): `2026-10-31 to 2027-06-30 = 242 days; 242 - 163 = 79 days of slack at the median; 318 - 242 = 76 days late for state or federal; 10 / 11 = 90.9% FY2022 renewal`

**Assumptions:**

- A lab-owning agency may be a state-level buyer on the 318-day cycle (my inference, not evidenced)
- Roughly half of deals take longer than the median (statistical property of a median; no distribution shown)
- Discovery may not yet be qualified for any target agency; the evidence shows no pipeline

**Cites:** `biobot_dummy_procurement_and_sales_03` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_procurement_and_sales_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data); `biobot_dummy_historical_contract_cohorts_04` — Synthetic Biobot boardroom operating packet (synthetic scenario, not actual company data)

- ❔ _unverifiable_: The CFO's milestone drops retention ('evidence of sales, not retention') while the metric still names retention evidence, so the metric and threshold do not match. — Evidence does not include the CFO's milestone or metric wording.
- ✅ _supported_: The FY2022 cohort in the evidence shows 10 of 11 renewed, a 90.9% renewal rate (biobot_dummy_historical_contract_cohorts_04, 2022-12-31). — JEV jev-1.14.0: evidence covers it 98%
- ✅ _supported_: The municipal median procurement cycle is 163 days, and the state or federal cycle is 318 days (documented in the evidence). — JEV jev-1.14.0: evidence covers it 97%
- ❔ _unverifiable_: The span from 2026-10-31 to 2027-06-30 is 242 days. — JEV jev-1.14.0: evidence covers it 11%
- ✅ _supported_: 242 minus 163 equals 79 days of slack at the median. — Arithmetic verified in code; its inputs are the speaker's own
- ✅ _supported_: 318 minus 242 equals 76 days late for a state or federal buyer. — Arithmetic verified in code; its inputs are the speaker's own
- ❔ _unverifiable_: The CFO's stated threshold is at least 2 signed agreements. — JEV jev-1.14.0: evidence covers it 9%
- ❔ _unverifiable_: A median is the midpoint of a distribution, so about half of deals take longer than the median cycle. — Evidence states medians but not the statistical definition; general knowledge only.
- ✅ _supported_: The CFO gives no distribution of procurement cycle times, and the evidence shows only medians. — Only median cycle figures appear; no distribution given.
- ✅ _supported_: The 2026-10-31 clock start assumes discovery is already qualified. — Cycle is measured from qualified discovery, so the start implies qualification.

<sub>claude-sonnet-5-5 · 7604 in / 2136 out · 13776 ms · $0.02441</sub>

### 125. VC ASSISTANT → VC (round 25)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

## Topic: Key risks

### 126. VC → the panel (round 26)

What is the single biggest risk to this business, whether access to sewer sites and samples, data privacy, regulation, team or funding, and what is the observable early-warning signal for it, the threshold that would tell you it is materializing, and what you would do then? Please separate what documented records show from your own estimate, and say if you cannot see the underlying data.


<sub>claude-sonnet-5-5 · 8494 in / 337 out · 4787 ms · $0.01880</sub>

### 127. MODERATOR → CEO (round 26)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 491 in / 43 out · 150 ms · $0.00002</sub>

### 128. CEO → VC (round 26)

Biggest risk (my opinion): revenue durability, meaning that buyers who pay with time-limited or appropriated money stop paying. This is the 'who pays' risk Ratti flagged in 2017 and it is still open. Documented: Ratti warned the founders 'you don't know who's going to pay for this data yet' (biobot_fact_ratti_who_pays, 2017), and selling to governments with tight budgets and burdensome purchasing was seen as hard (biobot_fact_access_and_procurement, 2017). The only 2026 renewal evidence I have is that 24 of 33 renewed on a dedicated budget line and 9 on time-limited grant money (biobot_dummy_procurement_and_sales_04, synthetic, 2026-09-30, cited from earlier turns, not displayed here). I rank this above site access and privacy because the evidence shows access is a relationship problem the team has handled before (biobot_case_p6_c2, 2017) and the privacy page states community data do not identify households (biobot_public_business_and_market_05, 2026-10-22). I cannot verify the 2026 access, regulatory or team position from the evidence shown. Early-warning signal (my proposal, not documented): the share of expiring contracts that renew, split by dedicated budget versus grant funding. Threshold (my estimate): any grant-funded cohort that fails to renew a second time, or any cohort where the number of contracts coded 'lost' is not reported. A single large customer leaving would be a second trigger, but I cannot see revenue by customer. I also cannot state a renewal rate, because the non-renewals are not visible to me. Action if it materializes: (1) code every lost or at-risk contract with a reason; (2) move grant-funded customers onto appropriated budget lines, which is what the 24 renewals show works; (3) hold hiring and spend until the gross profit driver is explained, because the $2.30M to $8.26M gross profit jump is still unreconciled to a driver. I would not claim the risk is controlled until revenue by customer and contract terms can be produced.

**Assumptions:**

- Ranking revenue durability as the biggest risk is my opinion
- The renewal-cohort split and the gross profit figures come from earlier turns, not from the chunks displayed now
- The warning signal and threshold are my own proposals, not evidenced
- I cannot see revenue by customer, so concentration is unverified

**Cites:** `biobot_fact_ratti_who_pays` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_fact_access_and_procurement` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_public_business_and_market_05` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_case_p6_c2` — HBS case 821-045, "Biobot Analytics", p.6

- ❌ _contradicted_: The gross profit figure rose from $2.30M to $8.26M, and the driver of this jump is still unreconciled. — Figures match, but driver is reconciled: ARR/site-based run-rate vs FY2024 reported; 'unreconciled' unsupported.
- ✅ _supported_: Ratti warned the founders in 2017 that 'you don't know who's going to pay for this data yet'. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: In 2017, selling to governments with tight budgets and burdensome purchasing was seen as hard. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The only 2026 renewal evidence the speaker has is that 24 of 33 contracts renewed on a dedicated budget line and 9 renewed on time-limited grant money. — JEV jev-1.14.0: evidence covers it 92%
- ❔ _unverifiable_: The 2026 renewal evidence is from a synthetic source cited from earlier turns and not displayed in the current chunks. — Claim is about what chunks display; evidence doesn't address provenance of turns.
- ✅ _supported_: The privacy page states that community data do not identify households. — JEV jev-1.14.0: evidence covers it 96%

<sub>claude-sonnet-5-5 · 6833 in / 1901 out · 12615 ms · $0.02182</sub>

### 129. DEVIL → CEO (round 26)

The weakest point is the CEO's ranking of revenue durability as the biggest risk, because her own evidence points to a more concrete, already-materialized risk that she ignores: loss of anchor government customers. The displayed 2024-05-01 public source (biobot_public_business_and_market_03) says the CDC program was material in size, that a later Verily award is recorded, and that the MWRA confirms the Massachusetts Biobot data stream ended in July 2026. That is documented evidence of procurement loss and in-house substitution, and she never cites it. She instead leans on 2017 quotes (who pays, tight budgets), which are nine years old, and on a 24-of-33 renewal split that she admits is not displayed in this evidence block and that cannot show non-renewals. Her warning signal is also weak. 'Any grant-funded cohort that fails to renew a second time' fires only after a loss, so it is lagging rather than early. 'Any cohort where lost contracts are not reported' measures disclosure, not the business. She cannot see revenue by customer, so she cannot rank this risk above concentration or substitution, yet she waves concentration away as 'unverified'. Her argument that access is handled rests on a 2017 chunk (biobot_case_p6_c2) that is not displayed here, and I cannot verify it. Her action plan, moving grant customers onto appropriated lines, assumes buyers will convert. The 24 renewals show some did, but nothing shows the 9 can or will. The public caveat (biobot_public_research_findings_and_03) says public sources do not establish retention or active customer counts, so the MWRA and CDC facts are limited. Even so, they are the only dated, current signals in the evidence, and they undercut her claim that access and customer risk are controlled. In my inference, the biggest risk is substitution and loss of large public customers, and her signal would detect it too late.

**Assumptions:**

- Customer loss and in-house substitution being the larger risk is my inference, not evidenced
- The CEO's cited chunks biobot_case_p6_c2 and the renewal cohort are not displayed, so I cannot verify them
- The public source does not show the full 2026 customer picture, as it states itself

**Cites:** `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_research_findings_and_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_ratti_who_pays` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_fact_access_and_procurement` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ❔ _unverifiable_: The CEO ranks revenue durability as the biggest risk. — JEV jev-1.14.0: evidence covers it 9%
- ✅ _supported_: The 24 renewals show some grant customers converted, but nothing shows the remaining 9 can or will convert. — 24 dedicated-budget, 9 grant-funded renewals; no evidence on their conversion.
- ✅ _supported_: The public caveat biobot_public_research_findings_and_03 says public sources do not establish retention or active customer counts. — JEV jev-1.14.0: evidence covers it 91%
- ✅ _supported_: The displayed public source biobot_public_business_and_market_03 (dated 2024-05-01) says the CDC program was material in size. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: The displayed public source biobot_public_business_and_market_03 records a later Verily award. — JEV jev-1.14.0: evidence covers it 91%
- ✅ _supported_: The displayed public source biobot_public_business_and_market_03 says the MWRA confirms the Massachusetts Biobot data stream ended in July 2026. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The CEO relies on 2017 quotes about who pays and tight budgets, which are nine years old. — Both quotes dated 2017, about nine years before 2026.

<sub>claude-sonnet-5-5 · 6508 in / 1747 out · 11633 ms · $0.02056</sub>

### 130. VC ASSISTANT → VC (round 26)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 131. VC → the panel (round 27) — _follow-up_

The Devil points to the Massachusetts loss as a materialized risk: the MWRA says its Biobot testing ended at the end of July 2026 as the state moved in-house, and the Boston Globe reported the expiring contract was worth $400,000/month with a replacement expected at about one-quarter of that cost. If substitution of large public customers is the risk, what specific mitigation would you fund, what does it cost, who owns it, and what leading signal, observable before a contract expires, would show it is failing? Please separate documented records from your own estimates, and say whether you can see how much of current revenue sits with customers that could build in-house, since the public sources do not establish current revenue or active customers.

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_research_findings_and_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The MWRA says its Biobot testing ended at the end of July 2026 as the state moved testing in-house. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The Boston Globe reported that the expiring Massachusetts Biobot contract was worth $400,000 per month. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The Boston Globe reported that a replacement for the expiring Massachusetts Biobot contract was expected to cost about one-quarter of the original contract value, roughly $100,000 per month. — JEV jev-1.14.0: evidence covers it 92%

<sub>claude-sonnet-5-5 · 7350 in / 1158 out · 7511 ms · $0.01807</sub>

### 132. MODERATOR → CEO (round 27)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 133. CEO → VC (round 27)

I concede the Devil's point: the Massachusetts loss is documented, current and larger evidence than my 2017 quotes, so substitution of large public customers is at least co-equal with revenue durability, and my lagging signal was too late. How I got my earlier answer: the 'who pays' and tight-budget risks came from 2017 chunks (biobot_fact_ratti_who_pays, biobot_fact_access_and_procurement); I did not use the 2026 MWRA evidence, which was an omission. The documented facts: MWRA says Biobot testing ended at the end of July 2026 as the state moved in-house; the Boston Globe (2026-07-04) reported the expiring contract at $400,000/month with a replacement expected at about one-quarter of that cost; the figure is contract value, not net revenue or margin (biobot_public_contracts_customers_and_06). The 2024-05-01 source records CDC program materiality and a later Verily award, and says these are not the full 2026 customer picture (biobot_public_business_and_market_03). Can I see how much current revenue sits with customers able to build in-house? No. I cannot verify revenue by customer, active customers or contract end dates from the evidence shown. Mitigation I would propose (my opinion, not documented): (1) Owner CEO, with the CFO supplying data: within 30 days produce a customer-by-customer table of revenue, contract end date and in-house capability, before funding anything. (2) Owner head of sales: for each large public customer, offer a lower-priced tier or analytics layer (modelling, cross-site comparison, reporting) so that the customer keeps buying even if it moves sampling in-house. The Globe's one-quarter replacement cost suggests price pressure, not only loss. I cannot cost this: the evidence has no cost data, so I give no dollar figure and would need the sales and engineering time estimate before asking for money. Leading signal observable before expiry (my proposal): a large customer's contract is within 12 months of end with no renewal discussion, an RFP or procurement notice for in-house or competing testing, or a customer hiring lab staff or buying equipment. Threshold: any customer above an agreed share of revenue with a lab-build signal or no renewal meeting 6 months before expiry triggers escalation to me. The share cutoff is not evidenced, and I would set it once revenue by customer is known. Most important assumption: that the MWRA case generalises to other public customers. Nothing in the evidence shows that. Quantifying the Massachusetts case: the reported replacement at one-quarter cost implies roughly $100,000/month of retained spend versus $400,000/month, i.e. a gap of about $300,000/month in program value, but this is the Globe's figure for the state's replacement, not Biobot revenue.

**Maths** (✅ checks out): `$400,000/month x 1/4 = $100,000/month; $400,000 - $100,000 = $300,000/month`

**Assumptions:**

- The Globe's one-quarter replacement cost applies to the whole program, and it is not Biobot revenue or margin
- The MWRA case may not generalise to other public customers
- The lower-priced tier or analytics layer is my proposal, with no evidence buyers would pay for it
- Signal thresholds such as 6 and 12 months are my own proposals, not evidenced
- I cannot see revenue by customer, so the share of revenue exposed to in-house build is unknown

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_ratti_who_pays` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_fact_access_and_procurement` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ❌ _contradicted_: The evidence has no cost data, so the speaker gives no dollar figure for the mitigation. — Evidence does contain cost data: $400,000/month contract, replacement about one-quarter the cost.
- ❌ _contradicted_: The reported replacement at one-quarter cost implies roughly $100,000 per month of retained spend. — Replacement is the state's in-house program; Biobot's testing ended, so spend isn't retained.
- ✅ _supported_: The gap between the $400,000 per month contract and the roughly $100,000 per month replacement is about $300,000 per month in program value. — JEV jev-1.14.0: evidence covers it 93%
- ✅ _supported_: MWRA says Biobot testing ended at the end of July 2026 as the state moved testing in-house. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The Boston Globe reported on 2026-07-04 that the expiring Biobot contract was $400,000 per month, with a replacement expected at about one-quarter of that cost. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The $400,000 per month figure is contract value, not Biobot net revenue or margin. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The 2024-05-01 source records CDC program materiality and a later Verily award. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The 2024-05-01 source states that CDC and Verily are not the full 2026 customer picture. — Source says these show risk, not the company's complete 2026 customer situation.

<sub>claude-sonnet-5-5 · 6853 in / 2493 out · 15807 ms · $0.02453</sub>

### 134. DEVIL → CEO (round 27)

The weakest point is the CEO's mitigation: a lower-priced tier or analytics layer sold to customers who are leaving to build in-house. Her own quantification undercuts it. She says the Globe's one-quarter replacement cost 'suggests price pressure', then proposes cutting price. If the buyer's in-house replacement costs about $100,000/month against $400,000/month, a cheaper Biobot tier has to compete with that number. She gives no price, cost or evidence that any buyer would pay for analytics, and she admits she cannot cost it, so the VC has no figure to fund. The mitigation is an untested idea, not a plan. Her leading signals also fail the VC's test. 'Within 12 months of end with no renewal discussion' fires for nearly every contract, since contracts are always within 12 months of expiry at some point, so it does not discriminate. The 'share of revenue' cutoff is left undefined, so the trigger cannot fire. The 6-month meeting threshold likely comes too late: in my inference, a state that moves in-house needs lead time for procurement, staffing and lab capacity, and the MWRA decision was probably made well before the July 2026 end date. The $300,000/month gap is also a weak input. The Globe figure covers the state's replacement program, not Biobot revenue or margin (biobot_public_contracts_customers_and_06, 2026-07), and she assumes the one-quarter ratio applies to the whole program. A gap in contract value is not a loss of Biobot revenue, and the evidence does not show how much the state would have kept buying. She also concedes she cannot see revenue by customer, so she cannot say whether Massachusetts is an outlier or typical. Finally, the 30-day customer table delays any spend, and the first signal is only a guess at 'in-house capability' that nothing shows Biobot can observe. The public source also states it does not establish the full 2026 customer picture (biobot_public_business_and_market_03, 2024-05-01).

**Maths** (✅ checks out): `$400,000/month x 1/4 = $100,000/month; $400,000 - $100,000 = $300,000/month contract-value gap, not Biobot revenue`

**Assumptions:**

- The MWRA decision was probably made well before July 2026 (my inference, not evidenced)
- A contract is always within 12 months of expiry at some point, so the 12-month signal would fire for every contract

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The public source states it does not establish the full 2026 customer picture. — JEV jev-1.14.0: evidence covers it 90%
- ✅ _supported_: $400,000/month x 1/4 = $100,000/month. — $400,000 times one-quarter equals $100,000 per month; arithmetic matches reported figures.
- ✅ _supported_: $400,000 - $100,000 = $300,000/month contract-value gap, which is not Biobot revenue. — Arithmetic verified in code; its inputs are the speaker's own

<sub>claude-sonnet-5-5 · 6831 in / 1773 out · 12672 ms · $0.02135</sub>

### 135. VC ASSISTANT → VC (round 27)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 136. VC → the panel (round 28) — _follow-up_

The CEO could not cost the substitution mitigation and could not see revenue by customer. Setting aside any figure you cannot support, what is the smallest piece of evidence you could produce, and from which records, to show how much current revenue sits with customers who could move testing in-house, and how much of that revenue the Massachusetts loss has already removed? Please separate documented records from your estimates.

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)


<sub>claude-sonnet-5-5 · 7610 in / 454 out · 4871 ms · $0.01735</sub>

### 137. MODERATOR → CEO (round 28)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 138. CEO → VC (round 28)

Smallest evidence I could produce: one customer-level revenue ledger, built from billing/invoice records and signed contracts, with one row per customer and these columns: trailing-12-month recognised revenue, contract start and end dates, funding source, and a flag for in-house testing capability or procurement. From it, two numbers: (a) revenue from customers flagged as able to build in-house divided by total revenue; (b) Massachusetts revenue removed, meaning the MWRA contract's trailing-12-month Biobot revenue and its share of the total. I cannot see any of this from the evidence shown, so I give no figure and will not estimate the exposure. How I got my earlier answer and what I correct: (1) Documented inputs: MWRA says Biobot testing ended at the end of July 2026; the Boston Globe (2026-07-04) put the expiring contract at $400,000/month with a replacement at about one-quarter of that cost; the figure is contract value, not net revenue or margin (biobot_public_contracts_customers_and_06). The 2024-05-01 source records CDC materiality and a later Verily award and says it does not give the full 2026 customer picture (biobot_public_business_and_market_03). The 2017 chunks on who pays and tight budgets are old and secondary (biobot_fact_ratti_who_pays, biobot_fact_access_and_procurement). (2) Corrections: the Devil is right that my $300,000/month gap is a gap in the state's program value, not lost Biobot revenue, so I withdraw it as a measure of Biobot's loss. The Globe's ratio is for the state's replacement, so applying it to the whole program was an assumption. My 12-month no-renewal-discussion signal was non-discriminating, since every contract passes through that window. My 6-month threshold was probably too late. Better leading signals, as proposals: a customer's RFP or procurement notice for testing, lab staffing or equipment purchases, or a falling sample volume, with the trigger set once the ledger gives revenue shares. The MWRA record shows the move was announced as a shift to in-house testing, so I would also ask for the date it first told Biobot. (3) Most important assumption: that billing and contract records can be tied to customers and that in-house capability can be judged from public or customer-stated information. If Biobot cannot observe capability, only the revenue-at-risk upper bound (all public-sector revenue) can be reported, not a real exposure. I cannot verify whether those records exist in the evidence shown.

**Assumptions:**

- Billing and contract records can be tied to individual customers and contract end dates (not verified)
- In-house capability can be assessed from customer disclosures or public information (my proposal, not evidenced)
- The share-of-revenue trigger cutoff is to be set after the ledger exists; no cutoff is evidenced
- The Globe's one-quarter replacement cost applies to the state's program, not to Biobot revenue or margin

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_ratti_who_pays` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_fact_access_and_procurement` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: MWRA says Biobot testing ended at the end of July 2026. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The Boston Globe, dated 2026-07-04, put the expiring MWRA contract at $400,000 per month, with a replacement at about one-quarter of that cost. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The $400,000 per month figure is contract value, not net revenue or margin. — JEV jev-1.14.0: evidence covers it 95%
- ✅ _supported_: The 2024-05-01 source records CDC materiality and a later Verily award and does not give the full 2026 customer picture. — JEV jev-1.14.0: evidence covers it 93%

<sub>claude-sonnet-5-5 · 6703 in / 2119 out · 13266 ms · $0.02296</sub>

### 139. DEVIL → CEO (round 28)

The weakest point is the CEO's proposed ledger, because it cannot answer the VC's question. The question was how much revenue sits with customers who could move testing in-house. Her central column is a 'flag for in-house capability' that she admits Biobot may not be able to observe. If it cannot, her fallback is 'all public-sector revenue' as an upper bound, which is nearly the whole book if the customers are mostly public. That bound would say almost nothing about exposure. The ledger would also be backward-looking: trailing-12-month revenue says nothing about which customers are about to leave. Capability is also the wrong test. The MWRA record shows a state moving in-house, and the Globe's one-quarter replacement cost points to a price-driven decision rather than a capability gap. In my inference, many customers could build in-house if the savings were large, so a capability flag would mark almost everyone or no one. A price-gap flag would fit the evidence better, but she does not propose it. Her second number, revenue removed by Massachusetts, is also only a historical share and not an exposure metric. She does not say whether the MWRA contract was Biobot's largest, and she cannot see it. She withdrew the $300,000/month figure but kept the same unverified link between the Globe's $400,000/month contract value and Biobot revenue. The evidence says that figure is contract value, not revenue or margin (biobot_public_contracts_customers_and_06, 2026-07). The 2024-05-01 source gives no 2026 customer picture (biobot_public_business_and_market_03). She also never says who verifies the flags, or whether customers would disclose their plans before they leave. I cannot see whether the billing and contract records exist, and neither can she.

**Assumptions:**

- That many public customers could build in-house if the price gap were large is my inference, not evidenced
- That most of Biobot's customers are public-sector is my inference; the evidence does not give the current customer mix

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The CEO does not propose a price-gap flag. — JEV jev-1.14.0: evidence covers it 9%
- ❔ _unverifiable_: The CEO withdrew the $300,000/month figure. — No evidence of a $300,000/month figure or any CEO withdrawal.
- ❔ _unverifiable_: The CEO kept the same unverified link between the Globe's $400,000/month contract value and Biobot revenue. — Evidence says $400k/month is contract value, not revenue; CEO's behavior not covered.
- ✅ _supported_: The evidence indicates the Globe's $400,000/month figure is contract value, not revenue or margin. — JEV jev-1.14.0: evidence covers it 94%
- ❌ _contradicted_: The 2024-05-01 source (biobot_public_business_and_market_03) gives no 2026 customer picture. — The chunk itself cites MWRA's July 2026 ending, though it says it is not the complete 2026 picture.
- ❔ _unverifiable_: The CEO's central column is a 'flag for in-house capability' that she admits Biobot may not be able to observe. — Evidence covers in-house substitution risk but nothing about the CEO's column or admissions.
- ✅ _supported_: The MWRA record shows a state moving testing in-house. — JEV jev-1.14.0: evidence covers it 95%

<sub>claude-sonnet-5-5 · 6287 in / 1678 out · 10725 ms · $0.01909</sub>

### 140. VC ASSISTANT → VC (round 28)

Facts are incorrect.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 141. VC → the panel (round 29) — _follow-up_

The Devil argues that a capability flag is the wrong test and a price-gap flag would fit the Massachusetts evidence better. Could you show, from billing records, contract terms, or any customer-stated pricing, how many current customers pay a per-sample or monthly price that a cheaper in-house alternative could undercut by a similar margin, and what date MWRA first told Biobot it would move in-house? Please separate documented records from estimates, and say if you cannot see these.

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_research_findings_and_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)


<sub>claude-sonnet-5-5 · 7267 in / 465 out · 4956 ms · $0.01703</sub>

### 142. MODERATOR → CEO (round 29)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 143. CEO → VC (round 29)

No, I cannot show either from the evidence shown. I cannot see billing records, contract terms, per-sample or monthly prices, or customer-stated pricing, so I cannot count how many customers pay a price a cheaper in-house option could undercut, and I will not estimate it. I also cannot see the date MWRA first told Biobot it would move in-house. The only dated records say that MWRA testing ended at the end of July 2026 as the state moved toward in-house testing, and that the Boston Globe (2026-07-04) reported a $400,000/month contract with a replacement expected at about one-quarter of that cost; this is contract value, not net revenue or margin (biobot_public_contracts_customers_and_06). The 2024-05-01 source records CDC materiality and a later Verily award and says it does not give the full 2026 customer picture (biobot_public_business_and_market_03). How I got my previous answer: I took only these dated records and the 2017 chunks (biobot_fact_ratti_who_pays, biobot_fact_access_and_procurement), which are old and secondary. Corrections: the Devil is right that a capability flag is the weaker test, since the Globe's one-quarter ratio points to price rather than capability. I also concede that my ledger's trailing-12-month revenue is backward-looking, that the MWRA contract is a single case, and that the $400,000 figure still is not Biobot revenue. A better ledger, as my proposal, would add per customer: price per sample or per month from the contract, the customer's own stated or procured alternative price, the contract end date, and the date the customer first signalled a change. From it I would compute the share of revenue where the price gap is at or above the roughly 75% implied by the Globe's ratio. That 75% is the state's replacement program, not a Biobot-specific gap. Records needed: invoices and signed contracts for prices, MWRA correspondence for the notice date, and procurement notices or customer statements for alternative prices. Most important assumption: that customers disclose or publish their alternative costs, because if they do not, the price-gap share cannot be measured and only an upper bound is possible.

**Maths** (not machine-checkable): `$400,000/month x 1/4 = $100,000/month; ($400,000 - $100,000) / $400,000 = 75% price gap`

**Assumptions:**

- The Globe's one-quarter replacement cost applies to the state's program, not to Biobot revenue or margin
- A 75% price-gap threshold is borrowed from the MWRA case and may not generalise to other customers
- Billing records and contracts can be tied to per-customer prices and end dates (not verified)
- Customers disclose or publish their alternative costs (my proposal, not evidenced)

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_ratti_who_pays` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_fact_access_and_procurement` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: $400,000/month x 1/4 = $100,000/month. — Contract $400,000/month; replacement about one-quarter the cost gives $100,000/month.
- ✅ _supported_: ($400,000 - $100,000) / $400,000 = 75% price gap. — Replacement at one-quarter cost implies a 75% gap; arithmetic is correct.
- ✅ _supported_: The 75% price gap is the state's replacement program ratio, not a Biobot-specific gap. — Ratio comes from the state's expiring contract versus replacement program, not Biobot pricing generally.
- ✅ _supported_: MWRA testing with Biobot ended at the end of July 2026 as the state moved toward in-house testing. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The Boston Globe reported on 2026-07-04 a $400,000/month MWRA contract with a replacement expected at about one-quarter of that cost. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The $400,000/month figure is contract value, not Biobot net revenue or margin. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The 2024-05-01 source records CDC materiality and a later Verily award and does not give the full 2026 customer picture. — JEV jev-1.14.0: evidence covers it 92%

<sub>claude-sonnet-5-5 · 6532 in / 2017 out · 11405 ms · $0.02185</sub>

### 144. DEVIL → CEO (round 29)

The weakest point is the CEO's 75% price-gap threshold. She computes it as ($400,000 - $100,000) / $400,000 and proposes flagging customers whose gap is at or above it. The two inputs are not like-for-like. The $400,000/month is Biobot's expiring contract value. The 'one-quarter' is the Globe's expectation for the state's replacement program (biobot_public_contracts_customers_and_06, 2026-07). The evidence does not say the replacement covers the same scope, sample volume, frequency or sites. My inference is that an in-house program may do less, so 75% could be a scope reduction and not a price gap for the same service. She also says the figure may not generalise, yet makes it the test. That creates two failures. First, it is a single observation from one state, so it is a poor cutoff for other customers. Second, a threshold that high would miss customers who leave at a 30% or 50% saving. The MWRA case shows only the outcome (Biobot lost), not the saving that would trigger a switch. She also says nothing about whether the replacement cost includes labs, staff and capital, or only running costs. A 'cost' of $100,000/month may not be comparable with Biobot's price. The proposed ledger needs customers' alternative prices, which she concedes customers may never disclose. If they do not, the flag cannot be built and the measure fails. So the proposal rests on a number that was never validated and on data she cannot show exists. It also cannot answer the VC's question about current customers' exposure.

**Maths** (not machine-checkable): `($400,000 - $100,000) / $400,000 = 75% implied gap, but only if the replacement has the same scope as the expiring contract`

**Assumptions:**

- That the replacement program may cover less scope than the Biobot contract is my inference, not evidenced
- That customers might switch at savings well below 75% is my inference, not evidenced

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The proposed ledger requires customers' alternative prices, which the CEO concedes customers may never disclose. — JEV jev-1.14.0: evidence covers it 13%
- ✅ _supported_: Implied gap of 75% holds only if the replacement has the same scope as the expiring contract. — Evidence gives no scope comparison, so the cost gap is like-for-like only if scope matches.
- ❔ _unverifiable_: The CEO computes the 75% price gap as ($400,000 - $100,000) / $400,000. — Evidence gives the one-quarter ratio; the CEO's specific computation isn't in evidence.
- ❔ _unverifiable_: The CEO proposes flagging customers whose price gap is at or above 75%. — JEV jev-1.14.0: evidence covers it 7%
- ✅ _supported_: The $400,000 per month figure is Biobot's expiring contract value. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The $100,000 per month figure is the Globe's expectation for the state's replacement program, per biobot_public_contracts_customers_and_06 dated 2026-07. — JEV jev-1.14.0: evidence covers it 92%
- ✅ _supported_: The evidence does not say whether the replacement program covers the same scope, sample volume, frequency or sites as the Biobot contract. — Evidence gives only contract value and approximate replacement cost; no scope or volume details.

<sub>claude-sonnet-5-5 · 6302 in / 1551 out · 12206 ms · $0.01872</sub>

### 145. VC ASSISTANT → VC (round 29)

Facts are correct.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 146. VC → the panel (round 30) — _follow-up_

Setting aside the 75% figure, what documented evidence could you show, from signed contracts or procurement documents, of what scope (sites, sample frequency, sample volume) the expiring Massachusetts contract covered versus what the in-house replacement program will cover, and does that record exist? Please separate documented records from estimates, and say if you cannot see it.

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22)


<sub>claude-sonnet-5-5 · 6982 in / 385 out · 4612 ms · $0.01600</sub>

### 147. MODERATOR → CEO (round 30)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 148. CEO → VC (round 30)

No, I cannot show that record from the evidence shown, and I cannot verify whether it exists. I have no signed contract, procurement document or MWRA scope statement giving sites, sample frequency or sample volume for the expiring Massachusetts contract or the in-house replacement, so I will not estimate the scope difference. Documented records, all in biobot_public_contracts_customers_and_06 (2026-07): MWRA says Biobot testing ended at the end of July 2026 as the state moved toward in-house testing; the Boston Globe (2026-07-04) reported the expiring contract at $400,000/month and the replacement expected to cost about one-quarter as much; the source states this is contract value, not net revenue or margin. biobot_public_business_and_market_03 (2024-05-01) records CDC materiality and a later Verily award, and says it does not give the full 2026 customer picture. The 2017 chunks (biobot_fact_ratti_who_pays, biobot_fact_access_and_procurement) are old and secondary. How I got my previous answer: I took the $400,000 and one-quarter figures and computed a 75% gap. Corrections: the Devil is right that this was not like-for-like. The evidence does not say the replacement covers the same scope, so 75% may be partly a scope reduction and not a price gap, and I withdraw it as a threshold and as a measure of price sensitivity. It is a single observation, so it also cannot say at what saving other customers would switch. What I would obtain, as a proposal: the signed MWRA contract and its scope-of-work or task orders for sites, sampling frequency and volumes; the state's procurement or budget documents for the in-house program; and Biobot's own invoices for volumes delivered. Only then could I compare like-for-like unit prices. Most important assumption: that the replacement is comparable in scope to the Biobot service, which the evidence does not establish. If it covers less, the Massachusetts case says little about price competitiveness, and I would instead log the loss as a procurement and substitution risk with an unknown cause.

**Maths** (not machine-checkable): `($400,000/month - $100,000/month) / $400,000/month = 75% implied gap, valid only if scope is identical, which is not evidenced`

**Assumptions:**

- The replacement program may cover less scope than the Biobot contract (not evidenced either way)
- Scope-of-work documents and invoices exist and can be obtained (not verified)
- The Globe's one-quarter replacement cost applies to the state's program, not to Biobot revenue or margin

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_fact_ratti_who_pays` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_fact_access_and_procurement` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ❔ _unverifiable_: The previous answer computed a 75% gap from the $400,000 and one-quarter figures. — Evidence gives $400k/month and one-quarter, but nothing about a previous answer's computation.
- ❔ _unverifiable_: The implied gap of 75% is ($400,000/month - $100,000/month) / $400,000/month. — Evidence implies $100k/month (one-quarter of $400k) but doesn't show this formula was used.
- ✅ _supported_: The implied gap is 75%, valid only if scope is identical, which is not evidenced. — 75% follows from one-quarter; scope identity isn't evidenced, only the replacement's approximate cost.
- ✅ _supported_: MWRA says Biobot testing ended at the end of July 2026 as the state moved toward in-house testing. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The Boston Globe (2026-07-04) reported the expiring Biobot contract at $400,000 per month. — JEV jev-1.14.0: evidence covers it 97%
- ✅ _supported_: The Boston Globe (2026-07-04) reported the replacement expected to cost about one-quarter of the expiring contract value. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The source states the $400,000 per month figure is contract value, not net revenue or margin. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The biobot_public_business_and_market_03 record (2024-05-01) records CDC materiality and a later Verily award. — JEV jev-1.14.0: evidence covers it 94%
- ✅ _supported_: The biobot_public_business_and_market_03 record states it does not give the full 2026 customer picture. — JEV jev-1.14.0: evidence covers it 92%

<sub>claude-sonnet-5-5 · 6013 in / 2123 out · 11627 ms · $0.02049</sub>

### 149. DEVIL → CEO (round 30)

The weakest point is the CEO's closing fallback: if the replacement covers less scope, she would 'log the loss as a procurement and substitution risk with an unknown cause'. That is a non-answer, and it is also inconsistent with her own corrections. The VC asked what the record shows about scope, and she correctly says she cannot see it. But she then proposes to resolve the question by obtaining documents (signed contract, task orders, state budget papers, Biobot invoices) without saying who would release them, or that they would settle the matter. My inference is that a state's budget papers give a top-line program cost, not a like-for-like unit-price comparison, so the proposed documents may not yield what she says they will. Her 'unknown cause' label also hides a point the evidence already supports. The MWRA record says the move was to in-house testing, and the Globe's expected cost is about one-quarter (biobot_public_contracts_customers_and_06, 2026-07). Whatever the scope, the customer chose to build and run its own program. So the substitution risk is documented in the evidence, not unknown; only the price-sensitivity part is unknown. She also leaves open a possibility that hurts her: a reduced-scope replacement could mean the customer found most of the Biobot service unnecessary, which would be a value problem and not just a substitution problem. She never considers this. Last, her plan only measures one lost contract. The 2024-05-01 source flags government contract concentration (biobot_public_business_and_market_03), yet she gives no step to size how many other customers sit in the same position, so the answer does not address the VC's underlying exposure concern. I cannot verify from the evidence whether any of these records exist.

**Assumptions:**

- State budget or procurement documents give program-level cost rather than unit-price comparisons (my inference, not evidenced)
- A reduced-scope replacement could signal low perceived value of the Biobot service (my inference, not evidenced)

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The 2024-05-01 source flags government contract concentration. — JEV jev-1.14.0: evidence covers it 92%
- ❔ _unverifiable_: The CEO proposed resolving the scope question by obtaining the signed contract, task orders, state budget papers, and Biobot invoices. — JEV jev-1.14.0: evidence covers it 8%
- ✅ _supported_: The MWRA record says the move was to in-house testing. — JEV jev-1.14.0: evidence covers it 96%
- ✅ _supported_: The Globe's expected cost for the MWRA program is about one-quarter of the prior level. — JEV jev-1.14.0: evidence covers it 93%

<sub>claude-sonnet-5-5 · 5747 in / 1562 out · 10049 ms · $0.01795</sub>

### 150. VC ASSISTANT → VC (round 30)

Facts are correct.

<sub>jev:jev-latest · 0 in / 0 out · 0 ms · $0.00000</sub>

### 151. VC → the panel (round 31)

I have covered every topic, and the evidence leaves the customer base and the Massachusetts scope unverified. I cannot size the exposure, so I will weigh that gap in my decision and I am ready to decide.

**Cites:** `biobot_public_contracts_customers_and_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

<sub>claude-sonnet-5-5 · 6659 in / 363 out · 3621 ms · $0.01512</sub>

## Fact-checks

| Claim | Speaker | Type | Verdict | Confidence | Checker | Evidence | Note |
|---|---|---|---|---:|---|---|---|
| The speaker asks the pitching founder to present what Biobot does, who it serves and why they buy, and why this is the right moment for the business. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks the pitching founder to state what they want from this meeting. | VC | opinion | · not_checked |  |  |  |  |
| Biobot turns wastewater into public-health data for cities. | CEO | fact | ✅ supported | 0.81 | jev:jev-latest | `biobot_case_p7_c3`, `biobot_public_technology_and_intellectual_01`, `biobot_caption_p17_c1`, `biobot_case_p6_c1` | JEV jev-1.14.0: evidence covers it 83% |
| On 2025-06-18 Biobot announced a Series A1 financing led by Valor Equity Partners, with new investor Atreides Management and Hyperplane participating. | CEO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_funding_and_investors_04`, `biobot_public_check_of_the_07`, `biobot_public_funding_and_investors_05` | JEV jev-1.14.0: evidence covers it 97% |
| Two directors joined Biobot's board alongside the Series A1 announcement. | CEO | fact | ✅ supported | 0.82 | jev:jev-latest | `biobot_public_funding_and_investors_04` | JEV jev-1.14.0: evidence covers it 89% |
| The amount and valuation of the Series A1 were not disclosed, and the speaker will not quote them. | CEO | fact | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_public_funding_and_investors_04` | Biobot's announcement did not disclose amount or valuation; Series A1 per Morse. |
| MIT judges did not advance Biobot to the finals. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_case_p10_c3`, `biobot_fact_mit_100k_pitch` | JEV jev-1.14.0: evidence covers it 96% |
| People asked why Biobot would try sewage given the existing environmental-testing industry. | CEO | fact | ✅ supported | 0.82 | jev:jev-latest | `biobot_case_p10_c3`, `biobot_fact_existing_testing_industry` | JEV jev-1.14.0: evidence covers it 87% |
| The wastewater data and city relationships are Biobot's differentiator. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker wants hard questions on the path from pilots to recurring contracts, a view on whether to lead with opioids or broader infectious disease and environmental work, and a conversation on whether the investor would back the next stage. | CEO | opinion | · not_checked |  |  |  |  |
| Current customer counts, pricing, revenue and retention have not been established from the evidence and must be confirmed before any claims are made on them. | CEO | opinion | · not_checked |  |  |  |  |
| Biobot originated as an MIT research project that planned to start by helping cities track opioid consumption, per case page 1 as of 2017. | CEO | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_fact_opioids_first`, `biobot_public_history_and_operating_02`, `biobot_case_p1_c3`, `biobot_public_check_of_the_01` | JEV jev-1.14.0: evidence covers it 95% |
| City officials and engineers pulled Biobot toward more uses. | CEO | fact | ✅ supported | 0.81 | jev:jev-latest | `biobot_case_p7_c3` | JEV jev-1.14.0: evidence covers it 85% |
| Boston's chief engineer asked whether Charles River fecal matter was human or animal, which determines the clean-up needed. | CEO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_case_p7_c3`, `biobot_fact_city_engineer_demand`, `biobot_case_p11_c1` | JEV jev-1.14.0: evidence covers it 95% |
| Communities invited other officials to hear Biobot's findings, and doors kept opening to deeper engagement, per case page 7 from 2017. | CEO | fact | ✅ supported | 0.81 | jev:jev-latest | `biobot_case_p7_c3` | JEV jev-1.14.0: evidence covers it 94% |
| The speaker cannot verify from the evidence shown that these city engagements were paid or renewed. | CEO | opinion | · not_checked |  |  |  |  |
| The city engagements should be treated as strong interest rather than proof of revenue. | CEO | opinion | · not_checked |  |  |  |  |
| Cities want faster, population-level public-health signals. | CEO | opinion | · not_checked |  |  |  |  |
| Biobot has the research base, access to wastewater infrastructure and maps, and city buy-in needed for this market, which the case says is required. | CEO | opinion | · not_checked |  |  |  |  |
| The CEO treats anecdotes from 2016-2017 as evidence of buyer pull. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's answers conflict regarding use-case pull. | DEVIL | opinion | · not_checked |  |  |  |  |
| The evidence shows the opioid choice was driven by accelerator constraints and a single mayor's agenda. | DEVIL | opinion | · not_checked |  |  |  |  |
| A request to be 'engaged at another level' by a city engineer does not establish who holds the budget. | DEVIL | opinion | · not_checked |  |  |  |  |
| A request to be 'engaged at another level' by a city engineer does not establish whether procurement would allow a subscription. | DEVIL | opinion | · not_checked |  |  |  |  |
| A request to be 'engaged at another level' by a city engineer does not establish whether a research collaboration would convert to a contract. | DEVIL | opinion | · not_checked |  |  |  |  |
| The Series A1 announcement dated 2025-06-18 is investor validation. | DEVIL | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_public_funding_and_investors_04` | Announcement date and investors are supported; 'validation' is an interpretation not addressed. |
| The Series A1 announcement discloses no amount, valuation, or customer metrics. | DEVIL | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_funding_and_investors_05`, `biobot_public_funding_and_investors_04` | JEV jev-1.14.0: evidence covers it 94% |
| The Series A1 announcement cannot substitute for evidence of customer retention. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO conceded that he cannot state customer counts, pricing, revenue or retention. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_06` | No evidence of a CEO concession; only that public sources lack those metrics. |
| The pitch's core claim, that pilots become recurring contracts, currently rests on 2016-2017 anecdotes. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 2016-2017 anecdotes do not support the 'recurring contracts' path the CEO asks the reviewer to probe. | DEVIL | opinion | · not_checked |  |  |  |  |
| The reviewer cannot verify anything more recent than 2016-2017 from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The Boston chief engineer's question and the forums officials organized were requests for research findings and one-off analyses outside the scope of the research project (case p.7, 2017). | DEVIL | fact | ✅ supported | 0.80 | jev:jev-latest | `biobot_case_p7_c3`, `biobot_fact_city_engineer_demand` | JEV jev-1.14.0: evidence covers it 87% |
| The Boston chief engineer's question and the officials' forums were not budgeted purchases. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_case_p7_c3`, `biobot_fact_buyer_unclear` | Evidence shows requests and forums but never says whether budgets existed. |
| The case says the opioid focus was chosen to gain admittance to MIT's delta v accelerator (case p.11, 2017). | DEVIL | fact | ✅ supported | 0.80 | jev:jev-latest | `biobot_case_p11_c3`, `biobot_public_check_of_the_01`, `biobot_public_history_and_operating_02` | JEV jev-1.14.0: evidence covers it 87% |
| The case says the opioid focus was backed by one small-city mayor's statement and 'executive support' (case p.11, 2017). | DEVIL | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_case_p11_c3`, `biobot_fact_mayor_opioids` | Case p.11 cites a small-city mayor and 'executive support at the highest level'. |
| The opioid focus was a narrowing for program entry, not a validated paying market. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO cites broad pull toward environmental and infectious-disease uses. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO asks which use case to lead with. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_case_p11_c3` | No evidence that the CEO asked this in the meeting. |
| One public source puts a minimum municipal charge for Biobot wastewater testing at $15,000 per year. | VC | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_business_and_market_02`, `biobot_public_company_and_offering_05`, `biobot_dummy_pricing_and_unit_01` | JEV jev-1.14.0: evidence covers it 94% |
| That same public source warns that the $15,000 minimum municipal charge cannot be multiplied by the cumulative number of tested sites to size revenue. | VC | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_business_and_market_02` | JEV jev-1.14.0: evidence covers it 94% |
| The speaker asks how large Biobot's annual revenue opportunity is in dollars and what calculation supports it. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for sizing with more than one independent method, such as bottom-up by municipal and institutional buyers and top-down from public-health or pharma data budgets. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for the source of each input and for inputs that are assumptions to be marked separately from evidence. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks which buyer segments are counted and which are excluded in the revenue sizing. | VC | opinion | · not_checked |  |  |  |  |
| The estimate of annual revenue opportunity is about $0.5M to $744M depending on method. | CFO | opinion | · not_checked |  |  |  |  |
| Cumulative tested sites are not counted. | CFO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_dummy_historical_operating_trajectory_01`, `biobot_public_business_and_market_02`, `biobot_dummy_historical_operating_trajectory_04`, `biobot_public_history_and_operating_07`, `biobot_dummy_historical_contract_cohorts_02`, `biobot_dummy_historical_operating_trajectory_05` | JEV jev-1.14.0: evidence covers it 93% |
| The $15,000 is a reported minimum price from Cambridge Day dated 2025-11-24, not an average price or contract count. | CFO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_company_and_offering_05`, `biobot_public_business_and_market_02`, `biobot_dummy_pricing_and_unit_01` | JEV jev-1.14.0: evidence covers it 95% |
| The $1,000 per-sample price is an assumption from the TAM slide and not evidence. | CFO | forecast | · not_checked |  |  |  |  |
| The speaker cannot verify from the evidence that any of these cities pay or renew. | CFO | opinion | · not_checked |  |  |  |  |
| All figures are opportunity ceilings, not revenue. | CFO | opinion | · not_checked |  |  |  |  |
| Seed-deck calculation: $110B x 20% = $22B; $22B x 4.6% = $1.01B; $1.01B x 50% = $506M. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The 4.6% share of public-health budgets addressable to wastewater surveillance is an assumption, not evidence. | CFO | forecast | · not_checked |  |  |  |  |
| 50% capture of the addressable pool is an assumption. | CFO | forecast | · not_checked |  |  |  |  |
| The estimates are opportunity ceilings, not forecasts of revenue. | CFO | opinion | · not_checked |  |  |  |  |
| Top-down beachhead calculation: $110B x 20% = $22B; $22B x 4.6% = $1.01B; $1.01B x 50% = $506M. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The beachhead top-down figure for a seed deck is about $506M. | CFO | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_fact_beachhead_public_health_spend`, `biobot_fact_beachhead_municipal_budgets` | Evidence gives $110B and $22B inputs but no $506M figure or derivation. |
| 4.6% of public-health spend is addressable by wastewater surveillance (assumption). | CFO | forecast | · not_checked |  |  |  |  |
| 50% of the addressable pool is captured (assumption). | CFO | forecast | · not_checked |  |  |  |  |
| The $110B and 20% inputs come from the case, dated 2017, and are founder estimates. | CFO | opinion | · not_checked |  |  |  |  |
| Share-of-existing-market calculation: $138.00B x 53.93% = $74.42B; $74.42B x 1.00% = $744.23M. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The $138B water market forecast and 53.93% municipal share are taken from the sizing model. | CFO | opinion | · not_checked |  |  |  |  |
| 1% capture of the municipal water market is an assumption. | CFO | forecast | · not_checked |  |  |  |  |
| Wastewater surveillance spend is assumed to sit inside the water market, which is unverified. | CFO | opinion | · not_checked |  |  |  |  |
| Bottom-up calculation: 870 potential cities x 1 site x $15,000 per year = $13.05M; 71 large cities x $15,000 = $1.07M. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| Each city buys one site at the $15,000 reported floor (Cambridge Day, 2025-11-24). | CFO | forecast | · not_checked |  |  |  |  |
| The 870 and 71 city counts are from the case, 2017. | CFO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_cities_over_500k`, `biobot_fact_cities_including_smaller` | JEV jev-1.14.0: evidence covers it 94% |
| The top-down method rests on case inputs (4 cities, $110B budgets, 20% public health) and two assumptions, and is the only method tied to a buyer budget. | CFO | opinion | · not_checked |  |  |  |  |
| Monthly sampling at the assumed $1,000 per sample gives $12,000 per site, which is below the floor. | CFO | number | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_fact_samples_per_year`, `biobot_public_company_and_offering_05` | 12 samples x $1,000 = $12,000, below $15,000 floor; $1,000 is an assumption. |
| No multi-site, state, federal, pharma or university buyers are counted. | CFO | opinion | · not_checked |  |  |  |  |
| The share-of-water-market figure of $744.23M is a sanity check, not a deck anchor. | CFO | opinion | · not_checked |  |  |  |  |
| The share-of-water-market figure assumes 1% of the whole municipal water market and uses a 53.93% municipal share that is derived rather than cited by the speaker. | CFO | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_fact_municipal_share_of_water_market` | 53.93% is derived ($48B/$89B), but the 1% assumption and speaker citation aren't shown. |
| The bottom-up figure using $15,000 is a floor-price view in which 870 cities x $15,000 = $13.05M if every potential city bought one site. | CFO | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller`, `biobot_public_company_and_offering_05` | 870 x $15,000 = $13.05M; $15,000 is a reported floor. |
| The $13.05M bottom-up figure is an upper bound for a minimum-price world and not a forecast. | CFO | opinion | · not_checked |  |  |  |  |
| Segments counted are US/Canadian municipal public-health and wastewater buyers: 4 beachhead cities, 71 cities over 500,000, and 870 cities total per case p.11 (2017). | CFO | fact | ✅ supported | 0.85 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_cities_including_smaller`, `biobot_fact_cities_over_500k` | JEV jev-1.14.0: evidence covers it 92% |
| States, federal agencies, pharma and universities are excluded because the evidence shows no budgets or prices for them. | CFO | opinion | · not_checked |  |  |  |  |
| The CFO calls the $506M 'seed-deck pick' defensible and 'the only method tied to a buyer budget'. | DEVIL | opinion | · not_checked |  |  |  |  |
| The case states the 20% figure was a guess by Ghaeli and Matus. | DEVIL | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_public_health_share_estimate` | JEV jev-1.14.0: evidence covers it 94% |
| The case pairs the $22B base with the opioid crisis but does not show what share of those budgets could be spent on wastewater monitoring. | DEVIL | opinion | · not_checked |  |  |  |  |
| The only price evidence is the $15,000 per year minimum reported by Cambridge Day on 2025-11-24. | DEVIL | fact | ❌ contradicted | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01` | Synthetic scenario records include other price points, e.g. $36,000 standard offer. |
| The CFO's bottom-up for the four beachhead cities would be 4 x $15,000 = $60,000. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The $506M figure is about 8,400 times the $60,000 bottom-up figure for the four beachhead cities. | DEVIL | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_beachhead_public_health_spend` | No $506M or $60,000 figure appears in the evidence to check the ratio. |
| The CFO does not reconcile the gap between the $506M figure and the $60,000 bottom-up figure. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's estimate 3 gives only $13.05M even if all 870 cities pay the floor price. | DEVIL | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller`, `biobot_public_company_and_offering_05` | 870 x $15,000 = $13.05M under the floor-price assumption. |
| The $506M figure conflicts with the CFO's own estimate 3. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $506M figure should not be presented as an opportunity in a deck. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $506M figure is a product of two arbitrary percentages rather than a market size. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $506M seed-deck pick's two decisive inputs, the 4.6% addressable share and the 50% capture rate, are invented. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $506M figure cannot be defended as tied to a buyer budget. | DEVIL | opinion | · not_checked |  |  |  |  |
| The seed-deck pick should be treated as unsupported until a real price and a real adoption rate are shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify any real price or adoption rate from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| Each beachhead city buys one site at the reported $15,000 minimum. | DEVIL | forecast | · not_checked |  |  |  |  |
| The 4.6% addressable share and the 50% capture rate are both flagged as assumptions in the case. | DEVIL | fact | ❔ unverifiable | 0.79 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 14% |
| The 4.6% addressable share and the 50% capture rate together account for 100% of the number's distance from the $22B budget. | DEVIL | opinion | · not_checked |  |  |  |  |
| 50% capture of a new category inside four cities is an extreme claim. | DEVIL | opinion | · not_checked |  |  |  |  |
| The company's own evidence shows no confirmed paying or renewing customers. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $22B base is the founders' own estimate of public-health spend, calculated as 20% of $110B. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_beachhead_public_health_spend`, `biobot_fact_public_health_share_estimate` | JEV jev-1.14.0: evidence covers it 96% |
| The case dates the $22B public-health spend estimate to 2017 on page 11. | DEVIL | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_beachhead_public_health_spend` | JEV jev-1.14.0: evidence covers it 91% |
| The case states the $110B is the combined municipal budgets of the cities. | DEVIL | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_beachhead_municipal_budgets` | JEV jev-1.14.0: evidence covers it 97% |
| The bottom-up estimate is roughly $13M, computed at the $15,000 price floor across 870 cities. | VC | number | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller`, `biobot_public_business_and_market_02` | 870 × $15,000 = $13.05M; both inputs are in evidence. |
| The top-down estimates are $506M and $744M. | VC | number | ❔ unverifiable | 0.79 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 14% |
| The estimates span from roughly $13M (bottom-up) to $744M (top-down). | VC | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_case_p11_c5` | The $744M top-down figure is not in the evidence; the $13M bottom-up is only derivable. |
| The two top-down figures depend on assumed shares, including a 4.6% addressable share and a 50% capture rate. | VC | fact | ❔ unverifiable | 0.80 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 13% |
| The case material does not support the assumed 4.6% addressable share and 50% capture rate. | VC | opinion | · not_checked |  |  |  |  |
| The 4.6% addressable share and the 50% capture rate are assumptions rather than sourced figures. | VC | opinion | · not_checked |  |  |  |  |
| The $506M figure is not defensible as a seed-deck number. | CFO | opinion | · not_checked |  |  |  |  |
| 4 cities x $15,000 = $60,000, which is about 8,400 times below $506M. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| $506M divided by $60,000 is about 8,433 times. | CFO | number | ❔ unverifiable | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_operating_trajectory_02` | $60,000 ARR appears, but $506M does not. The arithmetic itself, 506M/60k ≈ 8,433, is correct. |
| The speaker cannot verify an actual adoption rate or a price list from the evidence. | CFO | opinion | · not_checked |  |  |  |  |
| The only price evidence cited is a synthetic scenario record, not real Biobot data (biobot_dummy_pricing_and_unit_06, 2026-09-30): a $36,000 two-site contract with 48 annual samples. | CFO | fact | ✅ supported | 0.78 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06`, `biobot_dummy_pricing_and_unit_02` | JEV jev-1.14.0: evidence covers it 92% |
| The synthetic $36,000 two-site contract equals $18,000 per site per year, above the $15,000 floor. | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_02`, `biobot_dummy_pricing_and_unit_01` | $36,000 across two sites is $18,000 per site, above $15,000. |
| The Cambridge Day $15,000 minimum is not in the evidence block this turn and cannot be re-verified by the speaker. | CFO | opinion | · not_checked |  |  |  |  |
| 870 cities x $36,000 = $31.32M if every city bought the synthetic contract; this is a ceiling, not a forecast. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| 71 large cities x $36,000 = $2.56M if every city bought the synthetic contract; this is a ceiling, not a forecast. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| At the synthetic cost of $12,920 per contract, contribution is $23,080 (64.1%) before platform allocation. | CFO | number | ✅ supported | 0.91 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06` | JEV jev-1.14.0: evidence covers it 97% |
| A bottom-up ceiling of about $31M for 870 cities at the $36,000 synthetic contract (about $13M at the $15,000 floor) is the one number that should go in a seed deck. | CFO | forecast | · not_checked |  |  |  |  |
| The 4.6% addressable share has no source in the evidence shown. | CFO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_case_p11_c5` | No 4.6% share appears anywhere in the evidence. |
| A realistic beachhead of 4 cities x $36,000 = $144,000. | CFO | forecast | · not_checked |  |  |  |  |
| The top-down $506M and $744M figures should be dropped or shown only as non-credible upper bounds. | CFO | opinion | · not_checked |  |  |  |  |
| The 1% capture in the water-market method is an unsupported assumption. | CFO | opinion | · not_checked |  |  |  |  |
| The $744M water-market figure uses a cited $138B water forecast (biobot_sizing_share_of_existing_market, 2016). | CFO | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_water_market_2022_forecast` | The $138B 2016 forecast is shown, but the $744M calculation is not. |
| Each city buys one $36,000 two-site contract. | CFO | forecast | · not_checked |  |  |  |  |
| Every city in the count is treated as a buyer, a ceiling only, with no adoption rate known. | CFO | forecast | · not_checked |  |  |  |  |
| The counts of 870 and 71 cities are from the case, 2017. | CFO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_cities_over_500k`, `biobot_fact_cities_including_smaller` | JEV jev-1.14.0: evidence covers it 94% |
| The 50% capture is labelled by the evidence (biobot_assume_capture_beachhead, as of 2026-10-22) as a TAM-slide judgment call, not evidence. | CFO | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) |  | The biobot_assume_capture_beachhead chunk is not shown, so its label cannot be checked. |
| Capture is the assumption that moves the result most, halving or doubling the answer linearly. | CFO | opinion | · not_checked |  |  |  |  |
| 50% capture of a new category in four cities with no verified paying customers is implausible. | CFO | opinion | · not_checked |  |  |  |  |
| The two percentages (4.6% and 50%) together produce all of the gap between the $22B base and $506M. | CFO | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_fact_beachhead_public_health_spend` | $22B is supported, but $506M and the 4.6%/50% inputs are not shown. |
| The $22B base is weak because the $110B combined municipal budgets and the 20% public-health share are the founders' own estimate (biobot_case_p11_c5, 2017). | CFO | fact | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_case_p11_c5`, `biobot_fact_public_health_share_estimate`, `biobot_fact_beachhead_public_health_spend` | The $110B and 20% are the founders' own 2017 estimates. |
| The 20% public-health share is a guess, and the evidence does not show what share of municipal budgets could fund wastewater monitoring. | CFO | opinion | · not_checked |  |  |  |  |
| The claim that the method is 'tied to a buyer budget' is withdrawn insofar as it supports $506M. | CFO | opinion | · not_checked |  |  |  |  |
| The replacement seed-deck number is a roughly $31M ceiling built as 870 cities x one $36,000 contract. | DEVIL | number | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller` | 870 × $36,000 = $31.3M; the inputs are in evidence, though the deck number is not. |
| A ceiling that assumes universal adoption is not a market size. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO admits he has no adoption rate. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | The CFO's admission is not in the evidence. |
| At the entry price, 870 x $15,000 = $13.05M. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The $31M figure rests on a 2.4x price uplift over the entry tier for every city, with no evidence of upsell. | DEVIL | number | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01` | $36k/$15k = 2.4x; no upsell evidence is shown. |
| The realistic beachhead of $144,000 for 4 cities assumes all four buy the larger package. | DEVIL | opinion | · not_checked |  |  |  |  |
| The evidence shows no confirmed buyer in any of the four beachhead cities. | DEVIL | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_buyer_unclear`, `biobot_case_p10_c4` | The founders could not identify a buyer in 2017, and no confirmed buyer appears. |
| The 64.1% margin excludes platform allocation and uses synthetic unit costs. | DEVIL | fact | ✅ supported | 0.87 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06` | JEV jev-1.14.0: evidence covers it 93% |
| The 64.1% margin is weak. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify any real price, adoption rate or cost from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| 870 x $36,000 = $31.32M. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The $36,000 figure is a synthetic scenario record, and the CFO says it is not actual Biobot data (biobot_dummy_pricing_and_unit_06, 2026-09-30). | DEVIL | fact | ✅ supported | 0.81 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06`, `biobot_dummy_pricing_and_unit_02` | JEV jev-1.14.0: evidence covers it 94% |
| Most of the 870 cities are smaller and would start at the one-site entry tier. | DEVIL | forecast | · not_checked |  |  |  |  |
| The $36,000 figure is presented as the anchor of the seed deck. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | The evidence says nothing about how a seed deck presents $36,000. |
| The $36,000 figure corresponds to the two-site, 48-sample 'standard' contract. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06`, `biobot_dummy_pricing_and_unit_02`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 96% |
| The entry offer in the evidence is a one-site contract at $15,000 for 12 samples (biobot_dummy_pricing_and_unit_01, 2026-09-30). | DEVIL | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_dummy_pricing_and_unit_01` | JEV jev-1.14.0: evidence covers it 96% |
| The CFO's bottom-up pairs 870 cities, mostly small, with the larger two-site package. | DEVIL | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller`, `biobot_fact_cities_over_500k` | Of 870 cities only 71 exceed 500k, so most are small; the pairing is plausible. |
| The CFO's bottom-up pricing overstates the price for small cities. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 870 city count comes from a 2017 founder target list (biobot_case_p11_c5), not a count of buyers with budgets. | DEVIL | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_cities_including_smaller` | JEV jev-1.14.0: evidence covers it 95% |
| The CFO treats every city as a buyer at 100% adoption and calls the result a ceiling. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) |  | The evidence does not show the CFO assuming 100% adoption or using the word ceiling. |
| The evidence shows a synthetic scenario contract, SIM-MUN-001, that began as a paid pilot in 2024. | VC | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_06` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic scenario contract SIM-MUN-001 renewed in 2025 and 2026 at $36,000 for two sites. | VC | fact | ✅ supported | 0.84 | jev:jev-latest | `biobot_dummy_commercial_contract_book_06`, `biobot_dummy_pricing_and_unit_06` | JEV jev-1.14.0: evidence covers it 95% |
| The synthetic scenario contract SIM-MUN-001 is labelled as invented and not a real municipality. | VC | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_commercial_contract_book_06` | JEV jev-1.14.0: evidence covers it 98% |
| The speaker asks who the first real paying customer is, which office holds the budget and signs the cheque, what the pilot-to-contract conversion looked like, and why that buyer would pay now rather than continue free research collaboration. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks the audience to set aside the synthetic scenario contract SIM-MUN-001 when identifying real customers. | VC | opinion | · not_checked |  |  |  |  |
| The model cannot name a real paying customer from the evidence shown. | CEO | opinion | · not_checked |  |  |  |  |
| Exhibit 5 gives the value proposition of proactive, upstream opioid data to inform resource allocation and evaluate outreach. | CEO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_ex5_municipal_governments`, `biobot_ex5_state_governments`, `biobot_case_p11_c4` | JEV jev-1.14.0: evidence covers it 93% |
| The value proposition in Exhibit 5 is a reason to buy but not a documented trigger event. | CEO | opinion | · not_checked |  |  |  |  |
| Among 40 synthetic paid pilots at least 180 days old, 27 converted. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_procurement_and_sales_02` | JEV jev-1.14.0: evidence covers it 96% |
| 27 of 40 synthetic pilots converted is 67.5% of the cohort. | CEO | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_procurement_and_sales_02` | JEV jev-1.14.0: evidence covers it 97% |
| 27 of 36 resolved synthetic pilots converted is 75.0% of resolved pilots. | CEO | number | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_procurement_and_sales_02` | JEV jev-1.14.0: evidence covers it 96% |
| Among the synthetic pilots, 9 closed and 4 remain in procurement. | CEO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_dummy_procurement_and_sales_02` | JEV jev-1.14.0: evidence covers it 94% |
| The synthetic pilot conversion figures are scenario ground truth, not real conversion. | CEO | opinion | · not_checked |  |  |  |  |
| The model cannot show from the evidence why a city would pay rather than keep free research. | CEO | opinion | · not_checked |  |  |  |  |
| Recurring, decision-grade data beats one-off research. | CEO | opinion | · not_checked |  |  |  |  |
| Confirming a real customer requires the signing office, the budget line, a contract copy, and renewal history for a real city. | CEO | forecast | · not_checked |  |  |  |  |
| The only contract in the evidence is SIM-MUN-001, dated 2026-09-30. | CEO | fact | ❌ contradicted | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01`, `biobot_dummy_commercial_contract_book_04` | Evidence shows 89 contracts and a $14.384M book, plus the MWRA contract and ONDCP contract. |
| The contract identifier SIM-MUN-001 is an invented, synthetic identifier. | CEO | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_dummy_commercial_contract_book_06` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic contract SIM-MUN-001 proves nothing about real buyers. | CEO | opinion | · not_checked |  |  |  |  |
| The 2017 case says the founders did not yet know who in a city would buy. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_fact_buyer_unclear`, `biobot_fact_ratti_who_pays`, `biobot_case_p12_c4`, `biobot_case_p10_c4` | JEV jev-1.14.0: evidence covers it 96% |
| Exhibit 5 lists the end user as the mayor and staff. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_ex5_municipal_governments` | JEV jev-1.14.0: evidence covers it 97% |
| Exhibit 5 names NYC, Boston, SF and Washington DC as lead customers. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_ex5_municipal_governments`, `biobot_ex5_public_health_departments`, `biobot_case_p11_c5` | JEV jev-1.14.0: evidence covers it 96% |
| The NYC, Boston, SF and Washington DC entries in Exhibit 5 are a target segmentation, not signed customers. | CEO | opinion | · not_checked |  |  |  |  |
| The evidence does not identify which office or budget line pays. | CEO | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_buyer_unclear`, `biobot_fact_ratti_who_pays`, `biobot_dummy_commercial_contract_book_06` | No chunk names the paying office or budget line; 2017 sources say buyer unclear. |
| The CEO's closing move offered the synthetic 67.5% / 75.0% pilot conversion as 'scenario only' while still including it in the answer to a question about real conversion. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's answer to 'why pay rather than free research' is an unsupported opinion. | DEVIL | opinion | · not_checked |  |  |  |  |
| The answer to the VC's core question (who pays, from which budget, and why now) is empty. | DEVIL | opinion | · not_checked |  |  |  |  |
| No real customer, signer or conversion can be verified from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| Conversion rate of 27 out of 40 pilots is 67.5%. | DEVIL | number | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_procurement_and_sales_02` | JEV jev-1.14.0: evidence covers it 96% |
| Conversion rate of 27 out of 36 resolved pilots (27 plus 9) is 75.0%. | DEVIL | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_procurement_and_sales_02` | JEV jev-1.14.0: evidence covers it 97% |
| The 4 pilots still in procurement represent 10% of the 40-pilot cohort. | DEVIL | number | ✅ supported | 0.91 | jev:jev-latest | `biobot_dummy_procurement_and_sales_02` | JEV jev-1.14.0: evidence covers it 97% |
| A cohort of 40 pilots with a 180-day maturity cut-off implies an established buyer and signer. | DEVIL | forecast | · not_checked |  |  |  |  |
| The 75.0% figure of resolved pilots excludes the 4 pilots still in procurement, which is a survivor-style framing. | DEVIL | opinion | · not_checked |  |  |  |  |
| Only the 67.5% full-cohort figure is a clean conversion rate. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO never says that the synthetic conversion record contradicts his admission that he cannot name a buyer, a budget office or a paying customer. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO stated that he cannot name a buyer, a budget office or a paying customer. | DEVIL | opinion | · not_checked |  |  |  |  |
| A 40-pilot cohort with 27 conversions implies a mature sales process with a known signer. | DEVIL | opinion | · not_checked |  |  |  |  |
| The synthetic conversion figure tells the VC nothing about Biobot and risks anchoring the room on an unearned 2/3 conversion rate. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO leans on Exhibit 5's 'lead customer: NYC, Boston, SF, Washington DC', which is a 2017 target list. | DEVIL | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_ex5_municipal_governments`, `biobot_case_p11_c5` | Exhibit 5 lists the four cities as 2017 targets, 'beach head' markets, not contracts. |
| The 'proactive, upstream data' benefit is a pitch claim, not a documented trigger. | DEVIL | opinion | · not_checked |  |  |  |  |
| The fact-check says the evidence includes 89 contracts. | VC | fact | ✅ supported | 0.78 | jev:jev-latest | `biobot_dummy_revenue_and_retention_04`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 86% |
| The fact-check says the evidence includes a $14.384M contract book. | VC | fact | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_04` | Evidence states contract book totals $14,384,000. |
| The fact-check says the evidence includes an MWRA contract and an ONDCP contract. | VC | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_05`, `biobot_public_contracts_customers_and_06`, `biobot_public_check_of_the_08` | JEV jev-1.14.0: evidence covers it 92% |
| The fact-check conflicts with the answer that only one synthetic contract exists. | VC | opinion | · not_checked |  |  |  |  |
| Which of the listed contracts are real, paying, and renewing, and what each one's counterparty, annual value, start and renewal dates, and paying budget line are, is an open question posed in the meeting. | VC | opinion | · not_checked |  |  |  |  |
| The meeting asks how many days the sales cycle took, from first contact to a signed annual contract, for the pilots that converted. | VC | opinion | · not_checked |  |  |  |  |
| The meeting asks what share of revenue comes from federal or grant sources rather than city budgets. | VC | opinion | · not_checked |  |  |  |  |
| The evidence shows a synthetic contract book of 89 contracts and $14,384,000 ARR (biobot_dummy_commercial_contract_book_04, 2026-09-30). | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_revenue_and_retention_05`, `biobot_dummy_commercial_contract_book_01`, `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 97% |
| The 163-day and 318-day figures are medians across signed contracts, not per-pilot figures, and the speaker has no real days for converted pilots. | CEO | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | Medians over signed contracts confirmed; speaker's lack of pilot days not covered. |
| The evidence does not split ARR by buyer type, so the federal or grant share of revenue is unknown. | CEO | fact | ❔ unverifiable | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_04`, `biobot_public_business_and_market_06` | Evidence gives tier counts; federal/grant share of revenue not stated either way. |
| The higher-value tiers ($420,000 and $680,000) may be state or federal buyers; this is an inference not shown in the evidence. | CEO | opinion | · not_checked |  |  |  |  |
| The 318-day state/federal sales cycle suggests state or federal buyers exist in the scenario. | CEO | opinion | · not_checked |  |  |  |  |
| The 2017 case predates the 2026 scenario, so the 40-pilot cohort and the 'no buyer known' answer describe different periods and do not conflict. | CEO | opinion | · not_checked |  |  |  |  |
| The synthetic contract book is still synthetic and proves nothing about real Biobot. | CEO | opinion | · not_checked |  |  |  |  |
| The contract book shows larger tiers than $36,000, which undercuts the CFO colleague's $36,000 anchor. | CEO | opinion | · not_checked |  |  |  |  |
| Confirming requires the real contract ledger with counterparty, ARR, dates, paying office, and grant versus city-budget split. | CEO | forecast | · not_checked |  |  |  |  |
| 28 x $36,000 = $1,008,000. | CEO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| 41 x $96,000 = $3,936,000. | CEO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The synthetic contract book is labelled SYNTHETIC SCENARIO GROUND TRUTH, not actual Biobot data. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 96% |
| 16 x $420,000 = $6,720,000. | CEO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| 4 x $680,000 = $2,720,000. | CEO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| $1,008,000 + $3,936,000 + $6,720,000 + $2,720,000 = $14,384,000 total ARR. | CEO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| 28 + 41 + 16 + 4 = 89 contracts. | CEO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The higher-value tiers may be state or federal buyers. | CEO | forecast | · not_checked |  |  |  |  |
| The synthetic contract book is scenario ground truth and not real Biobot revenue. | CEO | forecast | · not_checked |  |  |  |  |
| The speaker cannot say any of the synthetic contracts are real, paying or renewing. | CEO | opinion | · not_checked |  |  |  |  |
| The contract named SIM-MUN-001 is the only contract the speaker saw named, and it is invented. | CEO | fact | ❔ unverifiable | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_06` | Evidence says SIM-MUN-001 is invented but not what the speaker saw. |
| The contract book mix is 28 contracts at $36,000, 41 at $96,000, 16 at $420,000 and 4 at $680,000. | CEO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 98% |
| The contract book record gives run-rate ARR, not recognized revenue. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_revenue_and_retention_05` | JEV jev-1.14.0: evidence covers it 97% |
| The contract book record names no counterparty, start or renewal date, or budget line. | CEO | fact | ❔ unverifiable | 0.76 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 9% |
| The speaker sees no MWRA or ONDCP contract in the evidence shown and cannot verify that claim or supply those fields. | CEO | opinion | · not_checked |  |  |  |  |
| In the scenario, the median signed-contract sales cycle is 163 days for municipal or county buyers and 318 days for state or federal buyers, measured from qualified discovery to signed annual agreement (biobot_dummy_procurement_and_sales_03, 2026-09-30). | CEO | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03` | JEV jev-1.14.0: evidence covers it 98% |
| The CEO's closing claim that the 2017 case predates the 2026 scenario, so the 'no buyer known' answer and the 40-pilot cohort do not conflict, is a dodge. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speculation hints at federal revenue the evidence does not show. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO says the larger tiers 'undercut' the CFO's $36,000 anchor. | DEVIL | opinion | · not_checked |  |  |  |  |
| The larger tiers do not undercut the CFO's $36,000 anchor, since they are equally synthetic and unlinked to any real city. | DEVIL | opinion | · not_checked |  |  |  |  |
| 16 contracts at $420,000 total $6,720,000. | DEVIL | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 97% |
| 4 contracts at $680,000 total $2,720,000. | DEVIL | number | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_revenue_and_retention_07` | JEV jev-1.14.0: evidence covers it 96% |
| 20 large contracts at $420,000 or $680,000 total $9,440,000. | DEVIL | number | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_04` | 16×420,000=6,720,000 plus 4×680,000=2,720,000 gives 9,440,000. |
| The 20 large contracts make up about 65.6% of the $14,384,000 ARR. | DEVIL | number | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_04` | 9,440,000/14,384,000 = 65.6%. |
| The 20 large contracts are 16 plus 4 out of 89 contracts. | DEVIL | number | ✅ supported | 0.90 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 96% |
| The book is concentrated in a few large accounts, so one lost large account could swing the total. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO does not note this concentration. | DEVIL | fact | ❔ unverifiable | 0.80 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 13% |
| The CEO is answering for the company in 2026 and says he cannot name a buyer, office or budget line. | DEVIL | opinion | · not_checked |  |  |  |  |
| The VC's question about real, paying, renewing contracts remains unanswered. | DEVIL | fact | ❔ unverifiable | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_06`, `biobot_dummy_commercial_contract_book_01` | Evidence doesn't show the VC's question or whether it was answered. |
| Concentration risk is an inference from the tier mix, not a stated fact in the evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| No real contract, counterparty or renewal can be verified from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO relays an 89-contract, $14.384M book and a 163-day median cycle as if they describe Biobot. | DEVIL | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_01`, `biobot_dummy_procurement_and_sales_03` | Figures are synthetic; how the CEO framed them is not in evidence. |
| The evidence labels the 89-contract, $14.384M book and 163-day median cycle as SYNTHETIC SCENARIO GROUND TRUTH, not actual Biobot data (biobot_dummy_commercial_contract_book_04, 2026-09-30). | DEVIL | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 95% |
| The only buyer evidence is from 2017 (biobot_fact_buyer_unclear), and nothing shown replaces it. | DEVIL | fact | ❔ unverifiable | 0.00 | claude-sonnet-5-5 (escalated) |  | No valid supporting citation |
| Citing a 2017 gap to excuse a 2026 gap fails. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO speculates that the $420,000 and $680,000 tiers 'might be state or federal buyers' and links that to the 318-day cycle. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_procurement_and_sales_03` | Whether the CEO speculated this way is not in the evidence. |
| The speculation that the larger tiers are state or federal buyers is an inference with no support. | DEVIL | opinion | · not_checked |  |  |  |  |
| The book lists tier counts only, with no buyer type. | DEVIL | fact | ❔ unverifiable | 0.00 | claude-sonnet-5-5 (escalated) |  | No valid supporting citation |
| The speaker asks what the price per sample, per site and per city contract is for the real business. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks what it costs to serve each unit, including lab processing, sampling logistics and analytics. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks what gross margin results after platform costs. | VC | opinion | · not_checked |  |  |  |  |
| The speaker states that if real figures cannot be shown, the presenter should say which inputs would need to be verified. | VC | opinion | · not_checked |  |  |  |  |
| The speaker cannot give real prices, costs or margins for Biobot from the evidence shown. | CFO | opinion | · not_checked |  |  |  |  |
| The 57.4% margin includes the $400,000 central platform and reporting cost. | CFO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_04`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 97% |
| The 57.4% margin includes $700 per site per year for sampler amortization and maintenance. | CFO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_pricing_and_unit_04`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 96% |
| The 57.4% margin excludes sales, research and corporate overhead and is therefore a cost-of-service margin only. | CFO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_dummy_pricing_and_unit_04`, `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_historical_efficiency_and_04`, `biobot_dummy_pricing_and_unit_08` | JEV jev-1.14.0: evidence covers it 96% |
| The 57.4% figure supersedes the 64.1% margin the Devil cited, which excluded platform allocation. | CFO | opinion | · not_checked |  |  |  |  |
| The $240 per sample cost appears only inside the biobot_dummy_pricing_and_unit_05 formula. | CFO | fact | ❌ contradicted | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_03`, `biobot_dummy_pricing_and_unit_06` | $240 also appears in unit_03, unit_06 and other chunks. |
| The speaker cannot say what the $240 per sample cost covers or whether it is real, so cannot split it into lab, logistics and analytics. | CFO | opinion | · not_checked |  |  |  |  |
| Sample cost is $240 each in the scenario. | CFO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_03`, `biobot_dummy_pricing_and_unit_06`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 98% |
| Site cost is $700 per year in the scenario. | CFO | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_04`, `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_06` | JEV jev-1.14.0: evidence covers it 97% |
| Platform cost per site is $451 per year ($400,000 / 886 sites). | CFO | number | ✅ supported | 0.86 | jev:jev-latest | `biobot_dummy_pricing_and_unit_04`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 94% |
| Per-site service cost is $6,911 per year (24 x $240 + $700 + $451). | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | $451 platform allocation and $6,911 total not stated; 24×240+700+451=6,911 arithmetic only. |
| Every unit figure available for Biobot is synthetic scenario data as of 2026-09-30. | CFO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_07` | Unit records are labeled synthetic scenario ground truth as of 2026-09-30. |
| Fully loaded cost is about $288 per sample. | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_03`, `biobot_dummy_pricing_and_unit_05` | Evidence gives $240 per sample; $288 not stated. |
| The speaker cannot verify price per sample, per site or per city contract from the evidence in this turn. | CFO | opinion | · not_checked |  |  |  |  |
| Earlier turns cited a $36,000 two-site contract and a $15,000 one-site, 12-sample entry offer. | CFO | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01`, `biobot_dummy_pricing_and_unit_02` | Both offers exist, but evidence doesn't cover what earlier turns cited. |
| The earlier-turn contract chunks are not displayed in this turn. | CFO | fact | ❔ unverifiable | 0.77 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 14% |
| A $36,000 two-site contract with 48 samples costs about $13,822 at the network average, a margin of about 61.6%. | CFO | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_06` | Evidence gives $12,920 and 64.1% before allocation; $13,822 not stated. |
| An entry contract at $15,000 for 12 samples at one site would cost $4,031 on the same inputs, a margin of about 73%. | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01`, `biobot_dummy_pricing_and_unit_05` | The $4,031 cost and 73% margin are not stated in evidence. |
| The entry contract margin calculation ignores that fixed-site costs do not scale to sample counts. | CFO | opinion | · not_checked |  |  |  |  |
| Inputs needed for verification include real lab cost per sample, sampling logistics, sampler lifetime, actual price lists, sites per contract, and platform cost scaling beyond 886 sites. | CFO | opinion | · not_checked |  |  |  |  |
| The case evidence offers a 2017 Boston budget context: Recovery Services Bureau FY18 budget of $509,108. | CFO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_ex7_recovery_services_bureau` | JEV jev-1.14.0: evidence covers it 97% |
| The Recovery Services Bureau budget shows the size of a possible buyer budget but not any price paid. | CFO | opinion | · not_checked |  |  |  |  |
| The scenario network view has 886 sites. | CFO | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_capacity_and_product_01`, `biobot_dummy_commercial_contract_book_03`, `biobot_dummy_capacity_and_product_04` | JEV jev-1.14.0: evidence covers it 97% |
| The number of samples in the network is 21,264 (886 sites x 24 samples). | CFO | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 97% |
| Sample cost across the network is $5,103,360 (21,264 x $240). | CFO | number | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | 21,264 × $240 = $5,103,360, consistent with the formula. |
| Site cost across the network is $620,200 (886 x $700). | CFO | number | ✅ supported | 0.85 | jev:jev-latest | `biobot_dummy_pricing_and_unit_04`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 92% |
| Platform allocation per site is $451 ($400,000 / 886). | CFO | number | ✅ supported | 0.76 | jev:jev-latest | `biobot_dummy_pricing_and_unit_04`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 91% |
| All cost inputs ($240 per sample, $700 per site, $400,000 platform) are synthetic scenario data, not real Biobot costs. | CFO | forecast | · not_checked |  |  |  |  |
| The $400,000 platform cost is spread evenly per site across 886 sites. | CFO | forecast | · not_checked |  |  |  |  |
| The $240 per sample cost is assumed to cover lab, logistics and analytics together, with the split unknown. | CFO | forecast | · not_checked |  |  |  |  |
| The scenario assumes 24 scheduled samples per site per year. | CFO | forecast | · not_checked |  |  |  |  |
| The scenario annualized service cost is $6,123,560. | CFO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_08` | JEV jev-1.14.0: evidence covers it 98% |
| The scenario annualized ARR is $14,384,000. | CFO | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_revenue_and_retention_05`, `biobot_dummy_revenue_and_retention_07`, `biobot_dummy_market_sizing_scenario_03` | JEV jev-1.14.0: evidence covers it 97% |
| The scenario gross profit is $8,260,440. | CFO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_08` | JEV jev-1.14.0: evidence covers it 98% |
| The scenario gross margin is 57.4%. | CFO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 98% |
| The 57.4% margin is run-rate, not GAAP. | CFO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 96% |
| 21,264 samples = 886 sites x 24 samples; 21,264 x $240 = $5,103,360; 886 x $700 = $620,200; $5,103,360 + $620,200 + $400,000 = $6,123,560; $14,384,000 - $6,123,560 = $8,260,440; $8,260,440 / $14,384,000 = 57.4%; $400,000 / 886 = $451 per site; 24 x $240 = $5,760; $5,760 + $700 + $451 = $6,911 per site per year; $6,911 / 24 = $288 per sample; 2 x $6,911 = $13,822; ($36,000 - $13,822) / $36,000 = 61.6%; 12 x $240 + $700 + $451 = $4,031; ($15,000 - $4,031) / $15,000 = 73.1% | CFO | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The CFO's illustrative contract margins of 61.6% on a $36,000 contract and 73.1% on a $15,000 entry offer are the weakest point of the discussion. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 57.4% network margin and the 61.6% contract margin cannot both describe the same cost structure without a site count per tier. | DEVIL | opinion | · not_checked |  |  |  |  |
| The evidence does not give a site count per tier. | DEVIL | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_03` | Evidence gives sites per buyer group: 56, 246, 384, 200. |
| The $15,000 entry case charges only $700 of site cost and $451 of platform cost against 12 samples. | DEVIL | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_04` | $700 site cost is stated; $451 platform and entry-case costing aren't. |
| The CFO concedes that fixed site costs do not scale with sample count. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO reports 73% for the $15,000 entry case as if it were a finding. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | No evidence on how the CFO framed the 73% figure. |
| The 73% entry-case margin is an artifact of dividing the platform cost by site count. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO assumes the $240 covers lab, logistics and analytics together. | DEVIL | fact | ✅ supported | 0.83 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_03` | JEV jev-1.14.0: evidence covers it 94% |
| The assumption that the $240 covers lab, logistics and analytics together is unsupported, because the chunk shows only the formula. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's own label says the maths does not add up. | DEVIL | opinion | · not_checked |  |  |  |  |
| The illustrative margins are not usable for the VC's question. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's illustrative margins rest on prices the CFO admits are not in the displayed evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $509,108 Boston budget is a 2017 figure. | DEVIL | fact | ❌ contradicted | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_ex7_recovery_services_bureau` | $509,108 is the FY18 budget, not FY17 ($515,747). |
| The $509,108 Boston budget says nothing about price. | DEVIL | opinion | · not_checked |  |  |  |  |
| No real price, cost or margin can be verified from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $15,000 entry case gross profit is $10,969, calculated as $15,000 minus $4,031. | DEVIL | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01` | $4,031 cost isn't in evidence; subtraction itself is correct. |
| 12 samples at one site may not carry a full-site $451 platform allocation. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $36,000 and $15,000 prices come from earlier turns, not displayed evidence. | DEVIL | forecast | · not_checked |  |  |  |  |
| Higher-value tiers likely cover more sites or samples per contract. | DEVIL | forecast | · not_checked |  |  |  |  |
| The arithmetic behind the CFO's illustrative contract margins does not hold up. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $36,000 contract is costed at the network average of $6,911 per site. | DEVIL | fact | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_06` | Evidence costs the $36,000 contract at $12,920 using per-sample and per-site inputs. |
| The network average cost of $6,911 per site is built on 886 sites generating $14.384M ARR. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | 886 sites and $14.384M ARR are given; $6,911 per site isn't stated. |
| The network average of $14.384M ARR across 886 sites equals about $16,234 ARR per site. | DEVIL | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | 14,384,000/886 ≈ 16,234. |
| A two-site $36,000 contract equals $18,000 per site. | DEVIL | number | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_02` | $36,000 covers two sites, so $18,000 each. |
| A two-site $36,000 contract is not clearly representative of the book. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $420,000 and $680,000 tiers imply far more sites or samples per contract than 24 per site. | DEVIL | opinion | · not_checked |  |  |  |  |
| The synthetic record costs the $36,000 two-site contract at $12,920 before platform allocation, a 64.1% contribution margin (biobot_dummy_pricing_and_unit_06). | VC | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06` | JEV jev-1.14.0: evidence covers it 98% |
| The network runs at a 57.4% margin after the $400,000 platform cost (biobot_dummy_pricing_and_unit_05). | VC | number | ✅ supported | 0.93 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 96% |
| The question asks which single price or cost input, such as the $240 per sample, the $700 per site, the $400,000 platform cost, or the $15,000 entry price, would most quickly break the margin if it moved against the speaker, and by how much. | VC | opinion | · not_checked |  |  |  |  |
| The question asks what contracted volume of sites would cover the $400,000 platform cost at the contribution margin per site of the entry offer and of the standard two-site contract. | VC | opinion | · not_checked |  |  |  |  |
| The question asks the speaker to state which inputs are real versus synthetic. | VC | opinion | · not_checked |  |  |  |  |
| The $240 per sample cost drives $5.10M of the $6.12M network service cost (83%). | CFO | number | ❌ contradicted | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | 21,264×$240=$5.103M, which is 83.3% of $6.12M; cost matches, so supported actually |
| The inputs of 886 sites, 24 samples, $240 per sample, $700 per site, $400,000 platform cost and $14.384M ARR come from biobot_dummy_pricing_and_unit_05. | CFO | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_04`, `biobot_dummy_pricing_and_unit_03`, `biobot_dummy_pricing_and_unit_07` | JEV jev-1.14.0: evidence covers it 95% |
| The standard-contract inputs come from biobot_dummy_pricing_and_unit_06. | CFO | fact | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_06` | Record 06 gives $36,000, 48 samples, $240, $700 inputs |
| The standard two-site contract has $23,080 contribution, about $11,540 per site. | CFO | number | ✅ supported | 0.80 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06` | JEV jev-1.14.0: evidence covers it 93% |
| Covering the $400,000 platform cost requires 18 standard contracts, about 35 sites (18 x 2 = 36 sites). | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The illustrative entry offer yields $11,420 contribution per site after $3,580 cost ($15,000 less 12 x $240 + $700). | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01`, `biobot_dummy_pricing_and_unit_04` | 12×240+700=3,580; 15,000−3,580=11,420 |
| The illustrative entry offer would require 36 contracts to cover the $400,000 platform cost. | CFO | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01`, `biobot_dummy_pricing_and_unit_04` | 400,000/11,420=35.0 so 36 contracts needed to cover it |
| The entry price and sample count are assumptions, so the entry breakeven result is only illustrative. | CFO | forecast | · not_checked |  |  |  |  |
| A 10% rise in the $240 per sample cost to $264 reduces the $36,000 contract margin to 60.9% ($14,072 cost). | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_06` | 48×264+1,400=14,072; margin 60.9% |
| A 10% cut to the $36,000 price to $32,400 gives a 60.1% margin. | CFO | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_06` | (32,400−12,920)/32,400=60.1% |
| Price is the other sensitivity lever for the standard contract. | CFO | opinion | · not_checked |  |  |  |  |
| A 10% rise in the $240 per sample cost (+$24) cuts network margin from 57.4% to about 52.1%, about 5.3 points. | CFO | number | ❌ contradicted | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | +$24 adds $510,336 cost; margin ~53.9%, a 3.5-point drop, not 52.1% |
| Mix is the more material sensitivity: network ARR per site is about $16,235 versus $18,000 per site on the standard contract, so the network average margin is lower than the standard-contract margin. | CFO | opinion | · not_checked |  |  |  |  |
| Network ARR per site is about $16,235. | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | 14,384,000/886=16,235 |
| 21,264 samples x $240 = $5,103,360 sample cost. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| $5,103,360 divided by $6,123,560 equals 83.3% of service cost. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| A 10% rise to $264 per sample, with $620,200 other service cost and $400,000 platform cost, gives total cost of $6,633,896 and a 53.9% network margin on $14,384,000 ARR. | CFO | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | Cost 6,633,896 and margin 53.9% check out; other cost 620,200 |
| The $400,000 platform cost is 2.8% of $14,384,000 ARR. | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_04` | 400,000/14,384,000=2.78% |
| The platform cost is treated as fixed at $400,000 regardless of site count. | CFO | forecast | · not_checked |  |  |  |  |
| The 10% shock is a chosen test size, not an evidence figure. | CFO | opinion | · not_checked |  |  |  |  |
| The entry contract carries one site cost of $700 and no platform allocation in contribution. | CFO | forecast | · not_checked |  |  |  |  |
| Sample cost scales with sample count and is the same across contracts. | CFO | forecast | · not_checked |  |  |  |  |
| The $400,000 platform cost is the least sensitive input at 2.8% of ARR. | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | Platform is 2.8% of ARR, but least-sensitive ranking not evidenced |
| The $15,000 entry price is not in displayed evidence and cannot be tested as a real input. | CFO | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01`, `biobot_public_company_and_offering_05` | $15,000 appears in evidence as scenario entry offer and public minimum |
| All figures in the statement are synthetic scenario data, not real Biobot data. | CFO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_06`, `biobot_dummy_pricing_and_unit_01` | JEV jev-1.14.0: evidence covers it 94% |
| The previously stated 61.6% and 73.1% margins mixed in a platform allocation and rested on prices not shown in the displayed evidence. | CFO | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | Earlier 61.6% and 73.1% figures are not in evidence |
| The $36,000 standard contract is supported by biobot_dummy_pricing_and_unit_06, with $12,920 cost and 64.1% contribution before platform. | CFO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06`, `biobot_dummy_pricing_and_unit_02` | JEV jev-1.14.0: evidence covers it 97% |
| The $15,000 entry figure and the 73% margin are not usable. | CFO | opinion | · not_checked |  |  |  |  |
| The $15,000 offer is an unverified illustration from earlier turns. | CFO | fact | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_01` | Evidence documents $15,000 as scenario tier and public reported minimum |
| 21,264 samples x $240 = $5,103,360; $5,103,360 / $6,123,560 = 83.3% of service cost; +10% on $240 = $264; 21,264 x $264 = $5,613,696; $5,613,696 + $620,200 + $400,000 = $6,633,896; ($14,384,000 - $6,633,896) / $14,384,000 = 53.9%; $400,000 / $14,384,000 = 2.8%; $23,080 / 2 sites = $11,540 per site; $400,000 / $23,080 = 17.3, rounded up to 18 contracts; 18 x 2 = 36 sites; $15,000 - (12 x $240 + $700) = $15,000 - $3,580 = $11,420; $400,000 / $11,420 = 35.0, rounded up to 36 entry contracts (35.03); 48 x $264 + 2 x $700 = $12,672 + $1,400 = $14,072; ($36,000 - $14,072) / $36,000 = 60.9%; $36,000 x 0.9 = $32,400; ($32,400 - $12,920) / $32,400 = 60.1% | CFO | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The CFO's headline sensitivity states that a 10% rise in the $240 sample cost cuts network margin from 57.4% to about 52.1% (about 5.3 points). | DEVIL | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | CFO did state it; the numbers themselves are incorrect |
| The CFO calls price the other lever but tests only a 10% price cut, which moves margin from 64.1% to 60.1%, a 4.0-point drop of similar order to the cost shock. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_06` | 64.1% to 60.1% is 4.0 points |
| The claim that $240 sample cost is the most fragile input is not shown against price sensitivity. | DEVIL | opinion | · not_checked |  |  |  |  |
| Price per contract could matter as much as sample cost, and the evidence shows no price distribution. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CFO calls product mix more material without any calculation of mix effects. | DEVIL | opinion | · not_checked |  |  |  |  |
| The network's $16,235 ARR per site compared with $18,000 per site is an unquantified gap. | DEVIL | number | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | $16,235 supported, but $18,000 comparison not in evidence |
| The number of sites in each tier cannot be verified from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| All inputs are synthetic scenario data and are not real Biobot figures. | DEVIL | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 91% |
| The 10% shock size is the CFO's chosen test rather than an evidence figure. | DEVIL | forecast | · not_checked |  |  |  |  |
| Price-lever fragility versus cost-lever fragility is an inference, since the evidence gives no price distribution. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's own arithmetic gives a network margin of 53.9% after a 10% rise in the $240 sample cost, a drop of about 3.5 points from 57.4%. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | 53.9% vs 57.4% is a 3.5-point drop |
| The answer states 52.1% as the lead sensitivity figure but computes 53.9% in the maths block, so the lead number is wrong. | DEVIL | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | Correct margin is 53.9%, so 52.1% lead figure is wrong |
| The claim that $240 is the most fragile input is overstated by roughly 50% relative to the 3.5-point computed drop versus the stated 5.3 points. | DEVIL | opinion | · not_checked |  |  |  |  |
| A 10% rise in the $240 sample cost moves the $36,000 standard-contract margin from 64.1% to 60.9%, a drop of 3.2 points, before platform allocation. | DEVIL | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_06` | 64.1% to 60.9% is 3.2 points |
| The standard-contract sensitivity is inconsistent with the headline network sensitivity claim. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO states contribution is before platform allocation. | DEVIL | fact | ✅ supported | 0.78 | jev:jev-latest | `biobot_dummy_pricing_and_unit_06` | JEV jev-1.14.0: evidence covers it 89% |
| $400,000 divided by $23,080 contribution per contract is 17.3, which rounds up to 18 contracts for breakeven, and this calculation is acceptable given contribution is before platform allocation. | DEVIL | number | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_06`, `biobot_dummy_pricing_and_unit_04` | 400,000/23,080=17.33, rounds to 18; acceptability claim is a judgment though |
| The entry-offer breakeven of 36 contracts rests on a $15,000 price and 12 samples that the CFO admits are not in the displayed evidence, so the 36 figure has no evidentiary footing. | DEVIL | opinion | · not_checked |  |  |  |  |
| 21,264 x $264 = $5,613,696; $5,613,696 + $620,200 + $400,000 = $6,633,896; ($14,384,000 - $6,633,896) / $14,384,000 = 53.9%; 57.4% - 53.9% = 3.5 points vs CFO's stated 5.3; 64.1% - 60.9% = 3.2 points on the $36,000 contract; 64.1% - 60.1% = 4.0 points for a 10% price cut | DEVIL | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The statement asks what would prevent a competitor, a university lab, or a city's water utility from replicating the service. | VC | opinion | · not_checked |  |  |  |  |
| The statement asks which specific assets (sampling and lab protocols, data history across sites, validated models, procurement relationships, contracts) would remain hard to copy. | VC | opinion | · not_checked |  |  |  |  |
| The statement asks the auditor to distinguish which of these assets exist in real evidence versus only in the synthetic scenario. | VC | opinion | · not_checked |  |  |  |  |
| Defensibility is thin today. | CTO | opinion | · not_checked |  |  |  |  |
| The synthetic scenario record states there is no head-to-head evidence that Biobot's forecasts beat another qualified lab. | CTO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_dummy_capacity_and_product_04` | JEV jev-1.14.0: evidence covers it 95% |
| The synthetic scenario record states that site coverage alone is not proof of an exclusive moat. | CTO | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_dummy_capacity_and_product_04` | JEV jev-1.14.0: evidence covers it 94% |
| The evidence shown does not allow verification of any validated-model accuracy advantage, proprietary lab protocols, procurement relationships, or contract lock-in or renewal terms. | CTO | opinion | · not_checked |  |  |  |  |
| Biobot's validated-model accuracy advantage, proprietary lab protocols, procurement relationships, and contract lock-in or renewal terms are not claimed as assets. | CTO | opinion | · not_checked |  |  |  |  |
| The patented sampler is one of the most plausibly hard-to-copy assets over time, but only if it is deployed at scale and the claims cover alternatives. | CTO | opinion | · not_checked |  |  |  |  |
| Multi-site longitudinal data is one of the most plausibly hard-to-copy assets over time, but only if it is shown to improve forecasts. | CTO | opinion | · not_checked |  |  |  |  |
| A standardized, comparable network across jurisdictions is one of the most plausibly hard-to-copy assets over time, because a single utility or lab cannot offer it. | CTO | opinion | · not_checked |  |  |  |  |
| Gaps can be closed by running blinded head-to-head accuracy tests against a state lab, documenting the patent claim scope against workarounds, and measuring renewal rates. | CTO | forecast | · not_checked |  |  |  |  |
| The moat is currently a hypothesis, not a proven asset. | CTO | opinion | · not_checked |  |  |  |  |
| Biobot is the assignee of US11714030B2, titled 'Modular system for upstream wastewater sampling', which was granted on 2023-08-01. | CTO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_technology_and_intellectual_03` | JEV jev-1.14.0: evidence covers it 96% |
| US11714030B2 protects a particular claim set on sampling hardware and does not give exclusivity over wastewater surveillance generally. | CTO | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_public_technology_and_intellectual_03` | JEV jev-1.14.0: evidence covers it 90% |
| In a 2017 case (p.12), the founders believed Biobot could differentiate on analytics without proprietary technology, so the original thesis was not a technology moat. | CTO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_fact_differentiate_on_analytics`, `biobot_case_p12_c2`, `biobot_fact_alm_stay_at_mit` | JEV jev-1.14.0: evidence covers it 95% |
| Replication risk is real and demonstrated. | CTO | opinion | · not_checked |  |  |  |  |
| A public source dated 2026-10-22 states that state/local public-health labs and university programs can collect, analyze and publish wastewater signals without buying Biobot's full service. | CTO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 93% |
| Records show MWRA ended Biobot's Massachusetts data stream in July 2026. | CTO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 96% |
| A live build-versus-buy precedent exists for Biobot's wastewater surveillance business. | CTO | opinion | · not_checked |  |  |  |  |
| As of 2026-09-30, in a synthetic scenario (not actual Biobot data), Biobot has 886 active sites giving longitudinal measurements. | CTO | number | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_capacity_and_product_04`, `biobot_dummy_capacity_and_product_01`, `biobot_dummy_commercial_contract_book_03` | JEV jev-1.14.0: evidence covers it 96% |
| The weakest point is the CTO's 'most plausibly hard to copy' list. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CTO has not shown that cross-jurisdiction comparability is something customers pay for or that it blocks substitution. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CTO's item (1) is overstated. | DEVIL | opinion | · not_checked |  |  |  |  |
| US patent US11714030B2 was granted on 2023-08-01. | DEVIL | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_technology_and_intellectual_03` | JEV jev-1.14.0: evidence covers it 97% |
| US11714030B2 covers a specific claim set on upstream sampling hardware. | DEVIL | fact | ✅ supported | 0.77 | jev:jev-latest | `biobot_public_technology_and_intellectual_03` | JEV jev-1.14.0: evidence covers it 85% |
| The evidence says US11714030B2 does not establish exclusivity across wastewater surveillance (biobot_public_technology_and_intellectual_03). | DEVIL | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_dummy_capacity_and_product_04`, `biobot_public_competitors_alternatives_and_06`, `biobot_public_technology_and_intellectual_03`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 91% |
| The CTO conceded that the 2017 founders' view was that the company had no proprietary technology (biobot_fact_differentiate_on_analytics). | DEVIL | opinion | · not_checked |  |  |  |  |
| Nothing shown establishes that the sampler is deployed at scale or that competitors need it. | DEVIL | opinion | · not_checked |  |  |  |  |
| Agencies can collect and analyze wastewater samples without the Biobot sampler. | DEVIL | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 91% |
| All three assets the CTO lists are conditional on evidence the CTO admits is missing. | DEVIL | opinion | · not_checked |  |  |  |  |
| Calling the moat 'a hypothesis' is accurate. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CTO's item (3), a standardized, comparable network that 'a single utility or lab cannot offer', is an opinion, and nothing in the evidence supports it. | DEVIL | opinion | · not_checked |  |  |  |  |
| Ranking the three assets implies a hierarchy the evidence does not give. | DEVIL | opinion | · not_checked |  |  |  |  |
| No customer lock-in, renewal or price premium for comparability can be verified from the evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| CDC's non-Biobot data sources can substitute for a Biobot network. | DEVIL | forecast | · not_checked |  |  |  |  |
| The only network evidence in the record is synthetic and shows 886 active sites. | DEVIL | number | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_capacity_and_product_04` | Public sources also give network evidence (1,000+ locations, 400+ CDC sites); 886 is synthetic. |
| The chunk biobot_dummy_capacity_and_product_04 says site coverage alone is not proof of an exclusive moat. | DEVIL | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_dummy_capacity_and_product_04` | JEV jev-1.14.0: evidence covers it 94% |
| The 2026-10-22 public source biobot_public_competitors_alternatives_and_06 points against Biobot having an exclusive network moat. | DEVIL | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_06` | Source shows agencies can build their own wastewater data, a build-versus-buy alternative. |
| CDC receives wastewater data from state and local health departments as well as from contractors. | DEVIL | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 93% |
| A standardized national wastewater feed already has non-Biobot data sources. | DEVIL | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_06`, `biobot_public_competitors_alternatives_and_05` | CDC receives data from state/local departments, Verily and WastewaterSCAN, not only Biobot. |
| MWRA ended Biobot's Massachusetts data stream in July 2026. | DEVIL | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 96% |
| A buyer can swap Biobot out of the comparable-network role without losing CDC visibility. | DEVIL | forecast | · not_checked |  |  |  |  |
| In the meeting, the CTO conceded that the moat is a hypothesis. | VC | opinion | · not_checked |  |  |  |  |
| According to the cited record (biobot_public_competitors_alternatives_and_06), the Massachusetts Water Resources Authority (MWRA) ended Biobot's Massachusetts data stream in July 2026. | VC | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03` | JEV jev-1.14.0: evidence covers it 97% |
| The evidence shown contains no proof that a customer gets something from Biobot that a state lab or university program cannot provide. | CTO | opinion | · not_checked |  |  |  |  |
| The Boston Globe reported the replacement cost at about one-quarter of the expiring contract value as the state moved to in-house testing. | CTO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The $400,000 per month figure is contract value, not margin. | CTO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 94% |
| The synthetic 886 sites and the statement that site coverage is not proof of a moat came from an earlier turn's chunk that is not displayed now. | CTO | fact | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_03`, `biobot_dummy_capacity_and_product_04` | 886 sites and the not-proof-of-moat statement both appear in synthetic ground truth chunks. |
| Network comparability is the weakest hard-to-copy asset, and it is withdrawn as a defensible asset because CDC already receives non-Biobot feeds. | CTO | opinion | · not_checked |  |  |  |  |
| The patent and the data history are equally unproven and are treated as unranked and conditional. | CTO | opinion | · not_checked |  |  |  |  |
| The MWRA precedent suggests price pressure and is real evidence against a price premium. | CTO | opinion | · not_checked |  |  |  |  |
| A large utility or state lab can replicate the sampling and lab workflow quickly, since MWRA did so within its own contract cycle. | CTO | opinion | · not_checked |  |  |  |  |
| A multi-site network and a validated forecast history take longer to replicate than the sampling and lab workflow. | CTO | opinion | · not_checked |  |  |  |  |
| The assumption that in-house replacement is as good as Biobot's service is the assumption that matters most and is neither confirmed nor refuted by the evidence shown. | CTO | opinion | · not_checked |  |  |  |  |
| Blinded accuracy and turnaround tests against a state lab and cost-per-sample measurement on both sides would close the evidence gap. | CTO | forecast | · not_checked |  |  |  |  |
| The speaker cannot verify Biobot's measured accuracy, lead time, cost per sample or reporting speed advantages from the evidence shown. | CTO | opinion | · not_checked |  |  |  |  |
| The expected replacement cost is about $100,000 per month. | CTO | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Replacement about one-quarter of $400,000/month, which is about $100,000. |
| The gap between the expiring contract and the expected replacement cost is about $300,000 per month. | CTO | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | $400k minus roughly $100k replacement leaves about $300k per month. |
| The Boston Globe 'about one-quarter as much' is read as roughly 25% of $400,000 per month. | CTO | forecast | · not_checked |  |  |  |  |
| The in-house program is comparable in scope to the expired Biobot contract, which the evidence does not confirm. | CTO | forecast | · not_checked |  |  |  |  |
| Replication speed is the speaker's opinion and is not backed by a number in the evidence. | CTO | opinion | · not_checked |  |  |  |  |
| The speaker cannot give a sourced month or capital figure for replication of Biobot's capabilities. | CTO | opinion | · not_checked |  |  |  |  |
| The patent US11714030B2 was granted 2023-08-01 and covers a specific claim set that does not by itself give exclusivity. | CTO | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_public_technology_and_intellectual_03` | JEV jev-1.14.0: evidence covers it 94% |
| A 2017 view stated that Biobot could differentiate without proprietary technology. | CTO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_fact_differentiate_on_analytics`, `biobot_case_p12_c2` | JEV jev-1.14.0: evidence covers it 95% |
| Agencies can collect, analyze and publish data without Biobot's full service, according to evidence dated 2026-10-22. | CTO | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 92% |
| CDC takes data from state and local departments as well as contractors. | CTO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_public_measurements_and_04` | JEV jev-1.14.0: evidence covers it 92% |
| MWRA ended Biobot's data stream in July 2026. | CTO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03` | JEV jev-1.14.0: evidence covers it 97% |
| The Boston Globe reported the expiring MWRA contract at $400,000 per month. | CTO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The CTO claims the MWRA precedent shows 'price pressure' and is 'real evidence against a price premium'. | DEVIL | opinion | · not_checked |  |  |  |  |
| The evidence shows a decision and a transition toward in-house testing, but does not show that replication was completed, how long it took, or that it works as well. | DEVIL | opinion | · not_checked |  |  |  |  |
| A single customer precedent cannot support a general 'no price premium' conclusion. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $100,000/month and $300,000/month figures depend on reading 'about one-quarter' as exactly 25%. | DEVIL | opinion | · not_checked |  |  |  |  |
| The source describes the figure as contract value, not margin. | DEVIL | fact | ✅ supported | 0.86 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_02`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 92% |
| The evidence does not show Biobot's margin on the MWRA contract, so the dollar gap says nothing about Biobot's economics. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CTO's opening claim that no proof exists of customer benefit is fair. | DEVIL | opinion | · not_checked |  |  |  |  |
| The follow-on inference about price is stronger than the single reported data point allows. | DEVIL | opinion | · not_checked |  |  |  |  |
| $400,000/month x 25% = $100,000/month. | DEVIL | number | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | One-quarter of $400,000 is $100,000; arithmetic correct. |
| $400,000/month - $100,000/month = $300,000/month. | DEVIL | number | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Subtraction is correct given the reported figures. |
| MWRA ended its Biobot testing in July 2026 as the state moved toward in-house testing. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 96% |
| The Boston Globe reported the expiring MWRA Biobot contract at $400,000/month. | DEVIL | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 97% |
| The Boston Globe reported the replacement MWRA program at about one-quarter of the expiring contract value. | DEVIL | number | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The evidence does not establish that the expired Biobot program and the replacement program have the same scope, sample count, sites, turnaround or deliverables. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CTO's own assumptions admit that the two programs are not shown to be comparable in scope. | DEVIL | opinion | · not_checked |  |  |  |  |
| A quarter-cost replacement could cover a smaller program, so the $300,000/month gap is not a like-for-like price comparison. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $300,000/month gap may reflect scope reduction or a state choice to fund less rather than a lower Biobot price point. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CTO says MWRA replicated the sampling and lab workflow within its own contract cycle. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker asks who exactly the ideal first customer is for the beachhead, including buyer type, size, and the specific pain they feel that they cannot solve with an in-house or state lab. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks why the segment should be won first given that a large buyer like MWRA moved to in-house testing in July 2026. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks the audience to separate which parts of the answer rest on real evidence, which rest on synthetic scenario data, and which are the speaker's own estimates. | VC | opinion | · not_checked |  |  |  |  |
| MWRA moved to in-house testing in July 2026. | VC | fact | ✅ supported | 0.80 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 90% |
| MWRA ended Biobot testing at the end of July 2026 as the state moved to in-house testing. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The sales database holds 420 reachable accounts: 220 municipalities, 150 county or metro agencies, and 50 state or multi-region agencies. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_01` | JEV jev-1.14.0: evidence covers it 97% |
| The 420 reachable accounts are a screened list, not a published total addressable market. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_01`, `biobot_dummy_market_sizing_scenario_02`, `biobot_dummy_market_sizing_scenario_03` | JEV jev-1.14.0: evidence covers it 96% |
| Municipalities and county/metro agencies make up 370 of 420 reachable accounts, about 88%. | CEO | number | ✅ supported | 0.89 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_01` | JEV jev-1.14.0: evidence covers it 95% |
| Municipalities and county/metro agencies are the most numerous reachable buyer group in the sales database. | CEO | fact | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_01` | 220 municipalities exceed 150 county/metro and 50 state accounts; most numerous. |
| The founders' original beachhead was four cities spending about $22B a year on public health, as of 2017. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_fact_beachhead_public_health_spend`, `biobot_fact_public_health_share_estimate`, `biobot_case_p11_c5` | JEV jev-1.14.0: evidence covers it 95% |
| The four-city, $22B public-health budget pool supports city-level buyers as the original target. | CEO | opinion | · not_checked |  |  |  |  |
| A public-health budget pool is not proof of paying customers. | CEO | opinion | · not_checked |  |  |  |  |
| The municipality and county/metro segment is the one least able to copy the MWRA in-house path. | CEO | opinion | · not_checked |  |  |  |  |
| The pain to target is getting a comparable, regularly reported wastewater signal without building a lab. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify that these buyers feel the wastewater-signal pain, their price tolerance, their renewal rates, or their accuracy versus a state lab. | CEO | opinion | · not_checked |  |  |  |  |
| The Globe reported that the MWRA Biobot contract was $400,000 per month and that its replacement cost about one-quarter of that amount. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| Nothing shown says how many municipal or county buyers can build in-house, their budgets, or their willingness to pay. | CEO | opinion | · not_checked |  |  |  |  |
| Confirming the segment requires win/loss and renewal data by segment, each segment's current lab access, and a pilot priced against the in-house cost. | CEO | forecast | · not_checked |  |  |  |  |
| 370 accounts equals 220 municipalities plus 150 county/metro agencies. | CEO | number | ❌ contradicted | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_01` | 220 + 150 = 370, which matches the arithmetic, but the ref is C10 CEO's claim; sum correct. |
| 370 divided by 420 equals 88.1%. | CEO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| Municipalities and county/metro agencies are assumed to lack in-house lab capacity; the evidence does not state this. | CEO | opinion | · not_checked |  |  |  |  |
| The MWRA contract value is contract value, not margin. | CEO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 92% |
| The MWRA loss is a material loss. | CEO | opinion | · not_checked |  |  |  |  |
| The MWRA loss shows that large, well-resourced buyers can substitute Biobot testing. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker's proposed beachhead is a mid-sized municipality or county/metro public-health agency that has no in-house lab and cannot staff one. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker would not lead with state or multi-region agencies. | CEO | opinion | · not_checked |  |  |  |  |
| Avoiding the large state-level buyer first is the speaker's opinion, based on one MWRA data point. | CEO | opinion | · not_checked |  |  |  |  |
| The evidence does not prove that the municipality and county/metro segment is the right beachhead, so it should be treated as a hypothesis. | CEO | opinion | · not_checked |  |  |  |  |
| The CEO claims municipalities and counties are the segment least able to copy the MWRA in-house path and feel the pain of getting a signal without building a lab. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 2017 four-city $22B figure is a total public-health budget, not a wastewater-surveillance budget. | DEVIL | fact | ✅ supported | 0.76 | jev:jev-latest | `biobot_fact_beachhead_public_health_spend` | JEV jev-1.14.0: evidence covers it 83% |
| The 2017 four-city $22B figure concerns four large cities, not mid-sized municipalities. | DEVIL | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_fact_beachhead_public_health_spend`, `biobot_fact_beachhead_municipal_budgets`, `biobot_case_p11_c5` | JEV jev-1.14.0: evidence covers it 95% |
| The 2017 sizing model's 20% public-health share and 50% capture rate are estimates, not evidence of demand (biobot_sizing_beachhead_budgets). | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_public_health_share_estimate` | 20% is labeled an estimate; the 50% capture rate is not in evidence. |
| The MWRA precedent is a single buyer. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03` | MWRA is one buyer's loss, but evidence doesn't call it the only precedent. |
| Treating the MWRA precedent as a reason to avoid state buyers, then assuming smaller buyers cannot substitute, applies the same thin evidence in opposite directions. | DEVIL | opinion | · not_checked |  |  |  |  |
| I cannot verify from the evidence shown that any mid-sized buyer has paid for, renewed or lacks alternatives to Biobot. | DEVIL | opinion | · not_checked |  |  |  |  |
| State and university labs that the evidence says can publish signals without Biobot may also serve smaller agencies cheaply or for free, which would undercut the 'cannot build a lab' pain. | DEVIL | opinion | · not_checked |  |  |  |  |
| Smaller municipal and county contracts being much smaller than MWRA's is the speaker's inference; the evidence shows no per-account prices for these segments. | DEVIL | forecast | · not_checked |  |  |  |  |
| State and university labs serving smaller agencies cheaply is the speaker's inference, not stated in the evidence shown. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO's own assumptions admit the evidence does not support the claim that municipalities and counties are least able to copy the MWRA in-house path. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 88% figure shows that municipal and county accounts are numerous in a synthetic, sales-screened list (biobot_dummy_market_sizing_scenario_01, dated 2026-09-30). | DEVIL | fact | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_01` | 370 of 420 is 88.1%; list is synthetic and sales-screened. |
| The 88% figure says nothing about lab access, budgets or willingness to pay of these buyers. | DEVIL | opinion | · not_checked |  |  |  |  |
| Account count is not a measure of buyer quality. | DEVIL | opinion | · not_checked |  |  |  |  |
| MWRA's reported spend is $400,000 per month (biobot_public_contracts_customers_and_06, dated 2026-07). | DEVIL | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| Small municipal and county buyers likely pay far less per account than MWRA's reported $400,000 per month. | DEVIL | forecast | · not_checked |  |  |  |  |
| Replacing one MWRA-sized loss could take many small contracts, so the segment logic may trade away revenue per account without a proven gain in retention. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO uses the 2017 four-city $22B spend (biobot_fact_beachhead_public_health_spend) as support for city buyers. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO proposed mid-sized municipalities and county or metro agencies as the beachhead for Biobot. | VC | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_01` | Evidence has no record of the CEO proposing this beachhead; only account and contract data. |
| The only count shown in the meeting is a synthetic 420-account sales list (biobot_dummy_market_sizing_scenario_01). | VC | fact | ❔ unverifiable | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_01` | The 420 list exists, but 'only count shown in meeting' is not covered. |
| No evidence was shown in this meeting that any municipal or county buyer has bought, paid for or renewed Biobot's service. | VC | opinion | · not_checked |  |  |  |  |
| The meeting did not show how many of the 370 municipal and county accounts are real, qualified prospects with no in-house or state lab access and a funded budget line. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks how to count qualified prospects, their input sources, and which numbers are real, synthetic or estimates. | VC | opinion | · not_checked |  |  |  |  |
| The 370 municipal and county accounts figure is a count that is part of the synthetic 420-account list. | VC | number | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_01` | Municipal 220 plus county 150 equals 370, so it is a subset of 420; not contradicted. |
| Accounts in the synthetic list cannot be treated as evidence of real buyers, paid customers or renewals. | VC | opinion | · not_checked |  |  |  |  |
| The evidence shown does not prove that any municipal or county buyer has bought, paid for or renewed Biobot. | CEO | opinion | · not_checked |  |  |  |  |
| The $22B figure is total public-health spend for four large cities, not a wastewater budget and not mid-sized buyers, so it is withdrawn as support for this segment. | CEO | opinion | · not_checked |  |  |  |  |
| The 88% figure is only an account count and says nothing about lab access, budget or willingness to pay. | CEO | opinion | · not_checked |  |  |  |  |
| Applying one MWRA data point to avoid state buyers while assuming smaller buyers cannot substitute is inconsistent, so the beachhead is a hypothesis with no evidence behind it. | CEO | opinion | · not_checked |  |  |  |  |
| Qualified prospects can be counted by starting with the 370 accounts and removing those with in-house or state/university lab access, no funded budget line, or procurement cycles that cannot be entered. | CEO | opinion | · not_checked |  |  |  |  |
| No input exists for the lab-access, budget or procurement filters, so no qualified prospect number is given. | CEO | opinion | · not_checked |  |  |  |  |
| 370 accounts out of 420 is 88.1% of accounts, the upper bound of the synthetic list with no filter applied. | CEO | number | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_01` | 370/420 is 88.1%; 'no filter' framing is interpretive but consistent. |
| 89 paying programs out of 420 accounts is 21.2%. | CEO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_03` | JEV jev-1.14.0: evidence covers it 98% |
| $14,384,000 contracted ARR divided by 89 programs is about $161,618 average ARR per paying program. | CEO | number | ✅ supported | 0.82 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_03`, `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 97% |
| The 89 paying programs are not split by segment in the evidence, so no municipal or county share can be derived. | CEO | forecast | · not_checked |  |  |  |  |
| Qualified prospects after lab-access and budget filters cannot be counted from the evidence; the filters are a proposed method. | CEO | forecast | · not_checked |  |  |  |  |
| The evidence shown does not allow counting real qualified prospects among the 370 accounts. | CEO | opinion | · not_checked |  |  |  |  |
| The average ARR per program is a simple mean across all 89 programs and is not specific to municipal or county buyers. | CEO | forecast | · not_checked |  |  |  |  |
| The most important assumption is that these buyers lack lab access and have a funded budget; if most can use a state or university lab for free or cheaply, the segment is much smaller than 370. | CEO | forecast | · not_checked |  |  |  |  |
| The only adoption data is synthetic scenario data dated 2026-09-30 showing 89 active paying programs, 21.2% of the 420 accounts, and $14,384,000 contracted ARR (biobot_dummy_market_sizing_scenario_03). | CEO | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_03` | Figures match, but 'only adoption data' is not verifiable; other cohort data exists. |
| The 89 active paying programs are not split by segment in the evidence, so it cannot be said how many are municipal or county or whether any renewed. | CEO | opinion | · not_checked |  |  |  |  |
| Nothing real shows a paid mid-sized buyer. | CEO | opinion | · not_checked |  |  |  |  |
| The real 2026-07 MWRA item is a loss, not a win. | CEO | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | MWRA Biobot testing ended July 2026; described as a material customer loss. |
| The 420 accounts (220 municipalities, 150 county or metro, 50 state or multi-region) come from biobot_dummy_market_sizing_scenario_01. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_01`, `biobot_dummy_market_sizing_scenario_02` | JEV jev-1.14.0: evidence covers it 98% |
| The 420-account list is sales-screened and excludes accounts already ruled out for procurement, which is not the same as qualified, lab-less, funded prospects. | CEO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_01` | List is sales-screened, excluding accounts ruled out for procurement; no lab or funding filter. |
| The 2017 $22B figure came from biobot_fact_beachhead_public_health_spend. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_fact_beachhead_public_health_spend` | JEV jev-1.14.0: evidence covers it 96% |
| The CEO uses the synthetic 89 paying programs and $14,384,000 ARR as 'the only adoption data'. | DEVIL | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_03` | Numbers match, but the CEO's 'only adoption data' phrasing is not evidenced. |
| The 89 paying programs are 21.2% of accounts according to scenario_03. | DEVIL | number | ✅ supported | 0.99 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_03` | JEV jev-1.14.0: evidence covers it 99% |
| The gap between 21.2% of accounts and 33.2% of planning opportunity ARR indicates unequal contract sizes in the source's own terms, supporting the inference that the mean is unrepresentative. | DEVIL | opinion | · not_checked |  |  |  |  |
| The filter method (lab access, budget line, procurement cycle) is untested. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | No evidence on whether the filter method was tested. |
| The CEO admits no inputs exist for the filter method, so it is a checklist rather than a count. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | No evidence of a CEO admission about filter inputs. |
| Nothing in the evidence shows any municipal or county buyer paid or renewed, so the beachhead remains an untested guess rather than a segment. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 2017 $511M sizing in biobot_sizing_beachhead_budgets rests on a 50% capture assumption and is not evidence of demand. | DEVIL | opinion | · not_checked |  |  |  |  |
| Average ARR per program is about $161,618, calculated as $14,384,000 divided by 89. | DEVIL | number | ✅ supported | 0.76 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_03`, `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 96% |
| Municipal contracts being smaller than the mean is an inference of the speaker, not stated in the evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO derives an average of about $161,618 ARR per program from $14,384,000 divided by 89 programs. | DEVIL | number | ✅ supported | 0.83 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_03`, `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 96% |
| The derived average of about $161,618 per program is a mean across an unsplit mix of buyer types and cannot describe municipal or county buyers. | DEVIL | opinion | · not_checked |  |  |  |  |
| The average of about $161,618 per program invites a false read of beachhead revenue. | DEVIL | opinion | · not_checked |  |  |  |  |
| A few large state or multi-region contracts could dominate the ARR total and pull the mean far above what a mid-sized municipality pays. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO admits the split of programs by buyer type is missing but still computes the average figure. | DEVIL | fact | ❌ contradicted | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02` | Evidence gives program split by buyer type, so it is not missing. |
| The 420-account list is described as 'qualified' in biobot_dummy_market_sizing_scenario_03 and as 'sales-screened, not ruled out for procurement' in scenario_01. | DEVIL | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_01`, `biobot_dummy_market_sizing_scenario_03` | JEV jev-1.14.0: evidence covers it 97% |
| The CEO relies on the scenario_01 wording to reject the scenario_03 'qualified' wording. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $14,384,000 is 33.2% of a $43,320,000 'planning opportunity' according to scenario_03. | DEVIL | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_market_sizing_scenario_03` | JEV jev-1.14.0: evidence covers it 98% |
| The synthetic contract book dated 2026-09-30 lists 28 municipal programs at an average of $36,000. | VC | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_pricing_and_unit_06` | JEV jev-1.14.0: evidence covers it 98% |
| The synthetic segment split indicates that larger segments (state and federal) carry much higher average contract values than municipal and county segments. | VC | opinion | · not_checked |  |  |  |  |
| A municipal and county go-to-market strategy can win accounts through procurement, but the sales cycle, customer acquisition cost and renewal evidence are not established by the synthetic contract book and must be marked as estimates or sourced separately. | VC | opinion | · not_checked |  |  |  |  |
| I cannot verify the sales cycle, cost to acquire each account, or renewal rates for municipal and county accounts from the evidence shown. | VC | opinion | · not_checked |  |  |  |  |
| If the synthetic segment split holds, the company should prioritise municipal and county accounts through procurement channels as a forecast path to scale, while treating the synthetic book as an illustrative input rather than real evidence. | VC | forecast | · not_checked |  |  |  |  |
| The synthetic contract book dated 2026-09-30 lists 41 county or metro programs at an average of $96,000. | VC | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic contract book dated 2026-09-30 lists 16 state programs at an average of $420,000. | VC | fact | ✅ supported | 0.85 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 96% |
| The synthetic contract book dated 2026-09-30 lists 4 federal or multi-region programs at an average of $680,000. | VC | fact | ✅ supported | 0.83 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic contract book (biobot_dummy_commercial_contract_book_02) is a synthetic dataset, not real company records. | VC | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_market_sizing_scenario_03`, `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_06`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 97% |
| The segment counts in the synthetic contract book sum to 89 programs (28 + 41 + 16 + 4). | VC | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_market_sizing_scenario_03`, `biobot_dummy_revenue_and_retention_04`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 98% |
| Earlier turns in this meeting stated that the 89 programs were not split by segment. | VC | opinion | · not_checked |  |  |  |  |
| The synthetic contract book's segment split conflicts with the earlier statement that the 89 programs were not split by segment, and the speaker has not resolved which is correct. | VC | opinion | · not_checked |  |  |  |  |
| Estimated total contract value from the synthetic book's segment averages is $1,008,000 for municipal programs (28 x $36,000), $3,936,000 for county or metro programs (41 x $96,000), $6,720,000 for state programs (16 x $420,000) and $2,720,000 for federal or multi-region programs (4 x $680,000), totaling $14,384,000. | VC | number | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_market_sizing_scenario_03`, `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_revenue_and_retention_07` | JEV jev-1.14.0: evidence covers it 97% |
| The speaker withdraws the earlier claim that the 89 programs were not split by segment. | CEO | opinion | · not_checked |  |  |  |  |
| The contract book totals $14,383,000, close to the stated ARR of $14,384,000. | CEO | number | ❌ contradicted | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_04` | Book totals exactly $14,384,000, not $14,383,000. |
| The $1,000 gap between the contract book total and stated ARR is treated as rounding, which the speaker cannot verify. | CEO | opinion | · not_checked |  |  |  |  |
| Revenue is concentrated in buyers who take 318 days to sign and who can substitute in-house, as MWRA did in real 2026-07 reporting. | CEO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03`, `biobot_public_contracts_customers_and_06`, `biobot_dummy_commercial_contract_book_02` | 318-day state/federal cycle, large revenue share, MWRA in-house move in 2026-07 all supported. |
| The split does not show renewals, churn, margin, acquisition cost or lab access by segment. | CEO | fact | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_historical_contract_cohorts_04` | Evidence includes renewal and margin data, though not by segment; segment claim unclear. |
| The evidence is silent on cost to acquire an account and on renewal rates, so the speaker gives no number for either. | CEO | opinion | · not_checked |  |  |  |  |
| The sales-cycle medians in the data are synthetic and measured from qualified discovery to signed annual agreement. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_historical_efficiency_and_02`, `biobot_dummy_historical_efficiency_and_01`, `biobot_dummy_historical_efficiency_and_04` | JEV jev-1.14.0: evidence covers it 97% |
| Counties and metro agencies average $96,000 per program versus $36,000 for municipalities. | CEO | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 97% |
| County and metro programs have a 163-day sales cycle. | CEO | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_historical_efficiency_and_05`, `biobot_dummy_historical_efficiency_and_02` | JEV jev-1.14.0: evidence covers it 97% |
| The speaker targets about 5.5 months from qualified discovery to signature for counties and metro agencies. | CEO | forecast | · not_checked |  |  |  |  |
| Municipal deals at $36,000 are about 38% of a county deal, calculated as $36,000 / $96,000 = 37.5%. | CEO | number | ✅ supported | 0.80 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 96% |
| The speaker withdraws the earlier claim of a $161,618 average contract value, which the speaker says hid the segment mix. | CEO | opinion | · not_checked |  |  |  |  |
| Municipalities should be reached through county or regional bundles, where one procurement may cover several cities. | CEO | opinion | · not_checked |  |  |  |  |
| Bundling municipalities through county or regional procurement is an untested hypothesis. | CEO | forecast | · not_checked |  |  |  |  |
| Municipal buyers should ideally first be approached with a paid pilot, which the source treats as a separate stage and not proof of renewal. | CEO | opinion | · not_checked |  |  |  |  |
| State and federal buyers should be approached last, with a roughly 318-day cycle (about 10.5 months), and only after the in-house risk is understood. | CEO | opinion | · not_checked |  |  |  |  |
| The recommended procurement path is: qualify budget and lab access in discovery, run a paid pilot, convert to an annual agreement, and track renewal as the proof point. | CEO | opinion | · not_checked |  |  |  |  |
| Confirming the plan requires acquisition cost per account, pilot-to-annual conversion, renewal and churn by segment, and in-house substitution rates. | CEO | opinion | · not_checked |  |  |  |  |
| The first-customer sequence of counties, then municipalities, then state and federal buyers is the speaker's opinion. | CEO | opinion | · not_checked |  |  |  |  |
| Contract-book average values are applied to every program in a segment, so segment totals are approximations. | CEO | forecast | · not_checked |  |  |  |  |
| Rounding explains the match between the $14,384,000 book total and stated ARR. | CEO | opinion | · not_checked |  |  |  |  |
| Programs in the book are assumed to include any renewals, since the evidence gives no renewal data. | CEO | opinion | · not_checked |  |  |  |  |
| The contract book biobot_dummy_commercial_contract_book_01 (synthetic, dated 2026-09-30) splits 89 programs into 28 municipal, 41 county or metro, 16 state and 4 federal or multi-region. | CEO | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_commercial_contract_book_01`, `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 98% |
| All of the scenario data is synthetic and not actual Biobot data. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_commercial_contract_book_01`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_capacity_and_product_04`, `biobot_dummy_capacity_and_product_03`, `biobot_dummy_market_sizing_scenario_01` | JEV jev-1.14.0: evidence covers it 95% |
| The Devil's inference that large contracts dominate the mean is confirmed by the segment split. | CEO | opinion | · not_checked |  |  |  |  |
| Municipal and county programs total 69 of 89 programs, which is 77.5%. | CEO | number | ✅ supported | 0.89 | jev:jev-latest | `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 97% |
| Municipal and county programs account for about $5.0M of the $14.0M book, about 36%. | CEO | number | ❌ contradicted | 0.93 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | Municipal+county = 28×36k+41×96k = $4.944M of $14.384M, about 34.4%, not $5.0M/$14.0M/36%. |
| State and federal programs total 20 programs, which is 22.5% of 89. | CEO | number | ✅ supported | 0.97 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_01` | 16+4=20 of 89 = 22.5%. |
| State and federal programs account for about $9.4M of the book, about 65%. | CEO | number | ❌ contradicted | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02` | State+federal = 6.72M+2.72M = $9.44M, about 65.6% of $14.384M; roughly consistent, but the $14.0M base differs. |
| The weakest point is the CEO's county-first plan, which rests on one synthetic cycle figure that does not fit the evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| Counties needing a paid pilot first would strain cash. | DEVIL | forecast | · not_checked |  |  |  |  |
| A paid pilot is a separate stage and is not proof of renewal. | DEVIL | opinion | · not_checked |  |  |  |  |
| The bundling idea, where one regional procurement covers several $36,000 municipal deals, is an untested hypothesis. | DEVIL | opinion | · not_checked |  |  |  |  |
| There is no evidence in the meeting of any such bundled procurement path. | DEVIL | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) |  | The 'bundled procurement path' is undefined in the evidence. |
| Bundled multi-party procurement may stretch the cycle beyond 163 days because multi-party deals need more approvals. | DEVIL | forecast | · not_checked |  |  |  |  |
| The revenue split is 36% versus 65% in the CEO's prose. | DEVIL | fact | ❔ unverifiable | 0.82 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 10% |
| Winning counties first targets the segment that brings in about a third of ARR and delays the buyers who hold two-thirds. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's text states the book totals $14,383,000 and calls a $1,000 gap rounding. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_04` | Evidence lacks the CEO's meeting text; it only gives $14,384,000. |
| The CEO's own arithmetic gives exactly $14,384,000, so no gap exists and the 'rounding' remark is wrong. | DEVIL | number | ✅ supported | 0.92 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02` | The arithmetic sums exactly to $14,384,000, so no gap exists. |
| 28 x $36,000 + 41 x $96,000 = $4,944,000. | DEVIL | number | 🧮 miscalculated | 1.00 | python |  | Arithmetic does not match the stated result |
| The CEO says counties are a '163-day cycle' and targets about 5.5 months. | DEVIL | opinion | · not_checked |  |  |  |  |
| $4,944,000 / $14,384,000 = 34.4%. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| 16 x $420,000 + 4 x $680,000 = $9,440,000. | DEVIL | number | 🧮 miscalculated | 1.00 | python |  | Arithmetic does not match the stated result |
| $4,944,000 + $9,440,000 = $14,384,000. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The prose percentages of 36% and 65% do not match the CEO's own calculation of 34.4% and 65.6%. | DEVIL | number | ❔ unverifiable | 0.78 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 14% |
| Multi-party bundled procurement lengthening the cycle beyond 163 days is the speaker's inference and is not stated in the evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| Paid pilots straining cash before annual signature is the speaker's inference, since no pilot price or cost is shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The source gives 163 days as one median for municipal or county buyers combined (biobot_dummy_procurement_and_sales_03, dated 2026-09-30). | DEVIL | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_historical_efficiency_and_05`, `biobot_dummy_historical_efficiency_and_02` | JEV jev-1.14.0: evidence covers it 98% |
| Nothing shown in the meeting gives a county-only sales cycle. | DEVIL | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_historical_efficiency_and_05` | Municipal-or-county median cycle of 163 days is given, though not county-only. |
| The claim that counties are faster than municipalities is unsupported by the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO never reaches a unit-economics conclusion in the meeting. | DEVIL | fact | ❔ unverifiable | 0.85 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 10% |
| The plan shows no acquisition cost, renewal data or margin for a $96,000 county deal. | DEVIL | fact | ❔ unverifiable | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | Network margin exists, but no county-deal CAC, renewal or margin shown; plan contents unknown. |
| The plan cannot show that a $96,000 county deal is worth winning. | DEVIL | opinion | · not_checked |  |  |  |  |
| A 5.4-month median cycle means half of deals take longer than 5.4 months. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | 163 days is about 5.4 months; median means half take longer. |
| 28 x $36,000 + 41 x $96,000 = $1,008,000 + $3,936,000 = $4,944,000; $4,944,000 / $14,384,000 = 34.4%; 16 x $420,000 + 4 x $680,000 = $9,440,000; $4,944,000 + $9,440,000 = $14,384,000 | DEVIL | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The synthetic evidence gives one median of 163 days for municipal or county buyers combined. | VC | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_historical_efficiency_and_05`, `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_historical_efficiency_and_02`, `biobot_dummy_historical_efficiency_and_01` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic evidence does not give a separate county-only sales cycle figure. | VC | opinion | · not_checked |  |  |  |  |
| The current annualized operating burn is about $4,039,560. | VC | number | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_historical_efficiency_and_05`, `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 96% |
| The $4,039,560 annualized operating burn figure is taken from the source biobot_dummy_historical_efficiency_and_05. | VC | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_historical_efficiency_and_05`, `biobot_dummy_pricing_and_unit_08` | JEV jev-1.14.0: evidence covers it 97% |
| The county-first plan should be taken step by step, beginning with discovery calls and moving through a paid pilot to an annual agreement and a first renewal decision. | VC | opinion | · not_checked |  |  |  |  |
| The 163-day median is one synthetic median for municipal and county buyers combined (biobot_dummy_procurement_and_sales_03, 2026-09-30). | CEO | fact | ❌ contradicted | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | 163 days is the municipal/county median; state/federal is separate at 318, not combined. |
| If the pilot falls before the cycle, discovery to signature could take roughly 8.4 months. | CEO | forecast | · not_checked |  |  |  |  |
| A first renewal decision comes 12 months after the annual agreement starts, so first renewal falls about 17 to 20 months after first discovery. | CEO | forecast | · not_checked |  |  |  |  |
| No renewal evidence exists in the evidence shown. | CEO | fact | ❌ contradicted | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_contract_cohorts_04`, `biobot_dummy_historical_contract_cohorts_05`, `biobot_dummy_historical_contract_cohorts_06` | Evidence shows renewal cohorts with 89.5%-92.3% renewal rates. |
| No evidence on salespeople, pilot staff, or dollars per step is available, so no number is given for them. | CEO | opinion | · not_checked |  |  |  |  |
| Historical efficiency chunk biobot_dummy_historical_efficiency_and_05, cited by the VC, puts annualized operating burn at about $4,039,560; that chunk is not in the evidence shown to the speaker. | CEO | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_05` | Chunk states burn about $4,039,560; the speaker-visibility part cannot be checked. |
| Monthly burn is about $336,630 on the VC's annualized figure of $4,039,560. | CEO | number | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 96% |
| A 5.4-month wait costs about $1.8M in burn. | CEO | number | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | $336,630/month × 5.4 ≈ $1.82M. |
| A 10.5-month state cycle costs about $3.5M in burn. | CEO | number | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_procurement_and_sales_03` | $336,630 × 10.5 ≈ $3.53M; 318 days ≈ 10.5 months. |
| The speaker does not know the cash balance, so cannot state how long the company can fund a wait. | CEO | opinion | · not_checked |  |  |  |  |
| The most important assumption is that the pilot converts and renews at county buyers. | CEO | forecast | · not_checked |  |  |  |  |
| The speaker withdraws the claim that counties are faster than municipalities or that the target is a county-specific 5.5-month timeline. | CEO | opinion | · not_checked |  |  |  |  |
| Fallback if the first county stalls past 163 days: run parallel county and municipal discovery tracks and use the $9,000 credited pilot as a low-cost entry; this plan is untested. | CEO | forecast | · not_checked |  |  |  |  |
| The segment totals sum to exactly $14,384,000, with no $1,000 gap and no rounding issue. | CEO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_revenue_and_retention_07` | Total ARR $14,384,000 stated; segment sum not shown, but 28×36k+41×96k+16×420k+4×680k not fully in evidence. |
| Municipal and county buyers account for 34.4% of ARR and state and federal buyers account for 65.6% of ARR. | CEO | number | ❔ unverifiable | 0.77 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 14% |
| Winning counties first targets about one third of ARR and delays buyers holding about two thirds of ARR. | CEO | number | ❔ unverifiable | 0.88 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 9% |
| Counties are the lower-risk entry point because they are less able to substitute than states. | CEO | opinion | · not_checked |  |  |  |  |
| The combined median time from qualified discovery to signed annual agreement is 163 days, about 5.4 months, and half of deals take longer. | CEO | number | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | 163 days applies only to municipal/county buyers, not a combined median. |
| A typical synthetic 90-day municipal pilot costs $9,000 for two sites and 12 samples, credited against the first annual subscription if the buyer converts within 90 days (biobot_dummy_procurement_and_sales_01). | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_procurement_and_sales_01` | JEV jev-1.14.0: evidence covers it 97% |
| The speaker does not know whether the 90-day pilot falls inside or before the 163-day cycle. | CEO | opinion | · not_checked |  |  |  |  |
| The CEO prices the wait at $336,630 per month x 5.4 months = $1.8M. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The 8.4-month upper case is shaky. | DEVIL | opinion | · not_checked |  |  |  |  |
| The pilot spec says fees are credited if the buyer converts within 90 days. | DEVIL | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_dummy_procurement_and_sales_01` | JEV jev-1.14.0: evidence covers it 96% |
| The pilot spec's conversion credit implies the pilot leads into the annual agreement. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO leaves the order of pilot and annual agreement unresolved and stacks 12 months on top to get 17 to 20 months to first renewal. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 17 to 20 month range rests on an assumed 12-month term and unknown ordering, so it is not evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| A $9,000 credited pilot is 25% of a $36,000 municipal deal. | DEVIL | number | ✅ supported | 0.80 | jev:jev-latest | `biobot_dummy_commercial_contract_book_06` | JEV jev-1.14.0: evidence covers it 91% |
| A $9,000 credited pilot is about 9.4% of a $96,000 county deal. | DEVIL | number | ✅ supported | 0.82 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02` | JEV jev-1.14.0: evidence covers it 95% |
| No pilot cost to Biobot is known, so the claimed low-cost entry has no margin. | DEVIL | opinion | · not_checked |  |  |  |  |
| The fallback of parallel county and municipal tracks requires more salespeople. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO gives no headcount or cost for the parallel tracks. | DEVIL | fact | ❔ unverifiable | 0.78 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 15% |
| The burn-based runway cost of waiting is not incremental, because burn is incurred whether or not a county signs. | DEVIL | opinion | · not_checked |  |  |  |  |
| Parallel tracks raise burn while the cycle wait is unchanged, which cuts against the runway the CEO cannot state. | DEVIL | forecast | · not_checked |  |  |  |  |
| The plan gives no number for how long the company survives a stalled first county. | DEVIL | opinion | · not_checked |  |  |  |  |
| Operating burn may be gross or net of the $14,384,000 ARR; I cannot verify which from the evidence shown. | DEVIL | forecast | · not_checked |  |  |  |  |
| Parallel tracks raising burn is an inference; no headcount or cost is shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| Burn-based figures measure nothing about how long the business can fund a stalled deal. | DEVIL | opinion | · not_checked |  |  |  |  |
| Without cash on hand, the figures cannot answer the VC's question. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO presents the burn-based figures as a cost of waiting. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) |  | Evidence does not show what the CEO said or how it was framed. |
| The synthetic book shows $14,384,000 contracted ARR (biobot_dummy_commercial_contract_book_02), as cited by the VC and CEO. | DEVIL | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_revenue_and_retention_05`, `biobot_dummy_pricing_and_unit_07` | JEV jev-1.14.0: evidence covers it 98% |
| Net burn could differ greatly from the $4,039,560 operating burn because revenue is ignored. | DEVIL | opinion | · not_checked |  |  |  |  |
| I cannot verify from the evidence shown whether the $4,039,560 operating burn is gross or net of revenue. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO takes the burn figure unseen from the VC's citation. | DEVIL | opinion | · not_checked |  |  |  |  |
| Cash on hand and burn rate are now visible in the meeting. | VC | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | Cash ($12.8M) and burn ($336,630/month) figures are in the evidence. |
| The speaker wants to move the discussion to alternatives to Biobot. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks who else solves the problem for municipal or county buyers, including commercial labs, state or university labs, in-house utility programs, and other vendors. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks why a buyer would pick Biobot over each alternative. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks which answers are real evidence, synthetic scenario data, or the responder's own opinion. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks what the buyer pays for each alternative compared with Biobot's $36,000 municipal average and $96,000 county average. | VC | opinion | · not_checked |  |  |  |  |
| Biobot's average price for municipal buyers is $36,000. | VC | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02` | 28 municipal programs average $36,000 in the scenario. |
| Biobot's average price for county buyers is $96,000. | VC | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02` | 41 county or metro programs at $96,000 each. |
| From the evidence shown, the speaker can name two competitors or alternatives. | CEO | opinion | · not_checked |  |  |  |  |
| WastewaterSCAN operates at nearly 150 sites and measures more than a dozen infectious diseases, with a public dashboard. | CEO | fact | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_05` | Nearly 150 sites, over a dozen diseases, public dashboard all stated. |
| CDC receives WastewaterSCAN data. | CEO | fact | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_05` | Evidence says CDC receives its data. |
| WastewaterSCAN is an alternative data source rather than a separate commercial competitor. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot tell from the excerpt whether a municipality pays for WastewaterSCAN. | CEO | opinion | · not_checked |  |  |  |  |
| Commercial labs, state or university labs, and in-house utility programs are not in the evidence shown. | CEO | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_06` | State/local labs and university programs appear in the evidence as alternatives. |
| Earlier turns cited MWRA going in-house in 2026-07, but that chunk is not displayed and is treated as unverified. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker knows no price for any alternative, so cannot compare against Biobot's synthetic averages of $36,000 municipal and $96,000 county. | CEO | opinion | · not_checked |  |  |  |  |
| Biobot's synthetic averages are $36,000 for municipal and $96,000 for county buyers. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_pricing_and_unit_02`, `biobot_dummy_market_sizing_scenario_02` | JEV jev-1.14.0: evidence covers it 96% |
| The evidence does not prove why a buyer picks Biobot. | CEO | opinion | · not_checked |  |  |  |  |
| The only evidence of buyer preference is 89 synthetic paying programs in biobot_dummy_commercial_contract_book_01. | CEO | fact | ❔ unverifiable | 0.50 | claude-sonnet-5-5 (escalated) |  | Evidence includes other synthetic and public customer records, so 'only' is not established. |
| The speaker cannot give a price for either named competitor or alternative. | CEO | opinion | · not_checked |  |  |  |  |
| The synthetic data does not say why those programs chose Biobot, renewed, or rejected alternatives. | CEO | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_01` | Synthetic records give counts and renewal funding, not reasons for choosing Biobot. |
| A local buyer wants a contracted analyst to interpret results for its own decisions, which a free public dashboard may not give. | CEO | opinion | · not_checked |  |  |  |  |
| A local buyer's preference for a contracted analyst is a hypothesis to test in buyer interviews, not a fact. | CEO | opinion | · not_checked |  |  |  |  |
| A free dashboard caps what a buyer will pay for raw data, which is a risk to the $36,000 price. | CEO | forecast | · not_checked |  |  |  |  |
| To confirm, the speaker needs alternative prices, win/loss reasons, and renewal data. | CEO | opinion | · not_checked |  |  |  |  |
| Buyers value local interpretation and decision support beyond a free dashboard. | CEO | opinion | · not_checked |  |  |  |  |
| A free public dashboard may cap willingness to pay for raw data. | CEO | forecast | · not_checked |  |  |  |  |
| No price comparison is made because no alternative price is known. | CEO | opinion | · not_checked |  |  |  |  |
| Verily Health / Sightline is a direct national wastewater testing and analytics competitor, according to a public source dated 2026-10-14. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_competitors_alternatives_and_02`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 94% |
| Verily Health / Sightline's 2026 CDC award continues work begun in 2023 and combines lab testing, analytics, pathogen sequencing and possible metagenomics. | CEO | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_public_competitors_alternatives_and_02` | JEV jev-1.14.0: evidence covers it 92% |
| CDC names Verily as the current national wastewater testing contractor and Biobot as the previous one. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_competitors_alternatives_and_02`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 96% |
| Verily losing a national contract to Biobot constitutes a loss of a national contract for Biobot. | CEO | opinion | · not_checked |  |  |  |  |
| Verily is the strongest threat the speaker can cite. | CEO | opinion | · not_checked |  |  |  |  |
| The cited source does not say Verily sells to municipal or county buyers, or offers substance-use or pharma products. | CEO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_02` | Source says it does not prove Verily offers substance-use or pharma products. |
| WastewaterSCAN is a Stanford/Emory academic network with philanthropic funding and Verily as lab partner, according to a public source dated 2026-10-22. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_competitors_alternatives_and_05` | JEV jev-1.14.0: evidence covers it 96% |
| The CEO claims the 'strongest threat I can cite' is Verily and that WastewaterSCAN is 'not a separate commercial competitor'. | DEVIL | opinion | · not_checked |  |  |  |  |
| The link between the Verily/CDC loss and local buyer switching cannot be verified from the evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO describes WastewaterSCAN as 'an alternative data source'. | DEVIL | fact | ❔ unverifiable | 0.75 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_05` | Source uses the phrase, but the CEO's wording is not shown. |
| The CEO asserts that a free dashboard 'caps what a buyer will pay'. | DEVIL | fact | ❔ unverifiable | 0.85 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 10% |
| The claim that a free dashboard caps what a buyer will pay is the CEO's own inference without supporting evidence from the source. | DEVIL | opinion | · not_checked |  |  |  |  |
| The source gives no price and does not say whether any municipality pays or declines to pay because of the free dashboard. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) |  | The 'free dashboard' source is not in the evidence shown. |
| The CEO treats the free dashboard as the bigger pricing danger without testing it. | DEVIL | opinion | · not_checked |  |  |  |  |
| Verily is WastewaterSCAN's lab partner, per the source. | DEVIL | fact | ✅ supported | 0.85 | jev:jev-latest | `biobot_public_competitors_alternatives_and_05` | JEV jev-1.14.0: evidence covers it 90% |
| The CEO treats Verily and WastewaterSCAN as separate threats. | DEVIL | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_05` | The CEO's statement is not shown; the source ties Verily to WastewaterSCAN. |
| Verily sharing lab capacity with WastewaterSCAN could let it offer local buyers a cheaper or free route. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO gives no reason why Biobot's $36,000 price holds against a cheaper or free route from Verily. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's answer names only two alternatives in response to the VC's question about municipal and county buyers. | DEVIL | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) |  | The CEO's actual answer is not in the evidence. |
| The 'why a buyer picks Biobot' reason, a contracted analyst interpreting local results, is the CEO's untested opinion. | DEVIL | opinion | · not_checked |  |  |  |  |
| There is no win/loss, renewal or interview evidence supporting the contracted analyst differentiation. | DEVIL | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_contract_cohorts_04` | Evidence has renewal rates only, with no win/loss or interview data on analyst differentiation. |
| The pitch has no evidenced differentiation at the local level. | DEVIL | opinion | · not_checked |  |  |  |  |
| 69 of the 89 programs are local programs. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_01` | 28 municipal plus 41 county or metro equals 69 of 89. |
| Sources do not show whether municipal or county buyers pay for any alternative. | DEVIL | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_02`, `biobot_public_competitors_alternatives_and_05` | No evidence shows municipal or county payments for alternatives. |
| The CEO's answer leaves the VC's question about municipal and county buyers unanswered. | DEVIL | opinion | · not_checked |  |  |  |  |
| Verily received a CDC award. | DEVIL | fact | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_02` | Verily's 2026 CDC award is stated. |
| WastewaterSCAN operates a network of nearly 150 sites. | DEVIL | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_competitors_alternatives_and_05` | JEV jev-1.14.0: evidence covers it 95% |
| Both cited sources (biobot_public_competitors_alternatives_and_02 dated 2026-10-14 and biobot_public_competitors_alternatives_and_05 dated 2026-10-22) describe national players. | DEVIL | fact | ✅ supported | 0.84 | jev:jev-latest | `biobot_public_competitors_alternatives_and_02`, `biobot_public_competitors_alternatives_and_05` | JEV jev-1.14.0: evidence covers it 92% |
| The cited sources do not say what a municipal or county buyer pays or chooses. | DEVIL | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_04` | Evidence gives municipal and county prices and funding sources. |
| The CEO leans on the Verily/CDC loss as the lead threat to a $36,000 municipal and $96,000 county book. | DEVIL | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) |  | The CEO's framing is not in the evidence. |
| A national CDC contract loss does not show that local buyers will switch. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO said commercial labs, state or university labs and in-house programs are not in the evidence shown. | VC | opinion | · not_checked |  |  |  |  |
| The evidence shown lists state/local public-health laboratories and university programs as a demonstrated build-versus-buy alternative. | VC | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 92% |
| The evidence records the end of Biobot's Massachusetts data stream at MWRA in July 2026. | VC | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 96% |
| The source of these evidence records is biobot_public_competitors_alternatives_and_06, dated 2026-10-22. | VC | fact | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_06` | Chunk 06 is dated as of 2026-10-22 as stated. |
| The statement does not identify any specific lost or non-renewed deal, buyer, or buyer-stated reason. | VC | opinion | · not_checked |  |  |  |  |
| The statement's question of whether the company loses deals to Verily, in-house or state labs, free networks, or no purchase cannot be answered with a reliable share or buyer-level detail from the statement alone. | VC | opinion | · not_checked |  |  |  |  |
| The statement does not provide a figure for what share of the 69 municipal and county programs could replace the company with a free or in-house option. | VC | opinion | · not_checked |  |  |  |  |
| The speaker corrects an earlier answer that said commercial labs, state or university labs and in-house programs were not in the evidence shown. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker does not know whether any buyer chose no purchase at all. | CEO | opinion | · not_checked |  |  |  |  |
| No source records a buyer-stated reason for losses or non-renewals. | CEO | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_06` | Sources note ends and substitution but no buyer-stated reasons. |
| The main local risk is in-house or state-lab substitution, since the evidence calls it demonstrated. | CEO | opinion | · not_checked |  |  |  |  |
| Free networks like WastewaterSCAN are a secondary risk as a data source, not evidenced as a payment substitute. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot give a share of the 69 programs that could replace Biobot with a free or in-house option, because the evidence does not show buyer capacity or intent. | CEO | opinion | · not_checked |  |  |  |  |
| The MWRA exit is a signal of wider local in-house risk. | CEO | forecast | · not_checked |  |  |  |  |
| The MWRA exit is one data point and may not represent other municipal and county buyers. | CEO | opinion | · not_checked |  |  |  |  |
| Confirming the risk requires win/loss and churn records by segment with stated reasons, buyer lab capacity, alternative prices, and renewal dates for the 69 programs. | CEO | opinion | · not_checked |  |  |  |  |
| Chunk biobot_public_competitors_alternatives_and_06 (public source, dated 2026-10-22) lists state/local public-health laboratories and university programs as a demonstrated build-versus-buy alternative. | CEO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 93% |
| Chunk biobot_public_competitors_alternatives_and_06 records via MWRA that Biobot's Massachusetts data stream ended in July 2026. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| Chunk biobot_public_business_and_market_03 (dated 2024-05-01, with later content on the 2026 MWRA end date) confirms in-house substitution and procurement renewal risk. | CEO | fact | ✅ supported | 0.82 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_06`, `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 87% |
| The Verily CDC award (biobot_public_competitors_alternatives_and_02, dated 2026-10-14) is a national contract loss and does not show local buyers switching. | CEO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_02`, `biobot_public_business_and_market_03` | CDC award is national; evidence shows no local buyer switching tied to it. |
| The CDC national testing contract went to Verily, with Biobot named as the previous contractor. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_02`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 96% |
| The Massachusetts data stream at MWRA ended in July 2026. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 97% |
| The speaker cannot verify the buyer's stated reason for the MWRA ending or whether the data moved in-house, to a state lab, or to a free network. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify any loss or non-renewal among the 69 municipal and county programs, and no win/loss or renewal records are shown. | CEO | opinion | · not_checked |  |  |  |  |
| The CEO claims the evidence 'shows two concrete losses' and that the MWRA exit is 'one data point'. | DEVIL | opinion | · not_checked |  |  |  |  |
| Counting the CDC loss among the two concrete local losses inflates the loss record. | DEVIL | opinion | · not_checked |  |  |  |  |
| Chunk biobot_public_business_and_market_03 warns that these events show risk 'not the company's complete 2026 customer situation'. | DEVIL | fact | ✅ supported | 0.85 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 90% |
| The CEO's 'two concrete losses' overstates what the evidence supports. | DEVIL | opinion | · not_checked |  |  |  |  |
| The evidenced count of local municipal or county losses is zero to one, with the reason for the MWRA end unknown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO gives no share figure for the 69 programs and offers no bounded range from the evidence, such as how many programs sit in states with a public-health lab. | DEVIL | opinion | · not_checked |  |  |  |  |
| The VC's question on replaceable share remains unanswered and the risk cannot be sized from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| MWRA's end of the data stream may be a planned, grant-related or non-commercial end rather than a lost paying contract. | DEVIL | forecast | · not_checked |  |  |  |  |
| The evidenced count of local losses is zero to one. | DEVIL | opinion | · not_checked |  |  |  |  |
| It cannot be verified from the evidence shown whether MWRA was a paying customer. | DEVIL | opinion | · not_checked |  |  |  |  |
| Chunk biobot_public_competitors_alternatives_and_06 says MWRA 'records the end of Biobot's Massachusetts data stream in July 2026'. | DEVIL | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| Chunk biobot_public_business_and_market_03 says the MWRA 'confirms' that its Biobot data stream ended. | DEVIL | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_06`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03` | JEV jev-1.14.0: evidence covers it 95% |
| Neither cited chunk says MWRA was a paying municipal or county customer, that its end was a loss rather than a planned or grant-funded end, or that it substituted in-house. | DEVIL | fact | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06` | Cited chunks mention in-house substitution; MWRA described as testing ended as state moved in-house. |
| The CEO builds the claim 'in-house or state-lab substitution is the main local risk' on the single MWRA event. | DEVIL | opinion | · not_checked |  |  |  |  |
| Ranking in-house or state-lab substitution as the main local risk is opinion presented as the lead conclusion. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO does not test the in-house substitution ranking against the other candidates it lists: Verily and free networks or no purchase. | DEVIL | fact | ❔ unverifiable | 0.76 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 14% |
| The 'demonstrated' label in chunk 06 describes a general build-versus-buy option, not evidence that any of Biobot's 69 municipal and county programs has switched or could switch. | DEVIL | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_03` | Chunk 06 describes general alternative; no municipal switching shown among 69 programs. |
| The CEO calls the CDC loss a 'national contract loss' that does not show local buyers switching, yet counts it as one of the two 'concrete losses' in answer to a question about municipal and county deal losses. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) |  | Evidence does not contain the CEO's answer or question context. |
| The evidence includes a dated public report (2026-07) that the MWRA said its Biobot wastewater testing ended at the end of July 2026 as the state moved toward in-house testing. | VC | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The publicly reported Biobot contract with the MWRA was expiring and reported at $400,000 per month. | VC | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 95% |
| A replacement for the MWRA Biobot testing is expected to cost about one-quarter of the $400,000 per month figure, roughly $100,000 per month. | VC | forecast | · not_checked |  |  |  |  |
| The synthetic venture book has $14,384,000 in contracted ARR. | VC | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_revenue_and_retention_05`, `biobot_dummy_revenue_and_retention_07`, `biobot_dummy_pricing_and_unit_07` | JEV jev-1.14.0: evidence covers it 97% |
| Whether any part of the $14,384,000 contracted ARR reflects Massachusetts or similar state-level in-house substitution exposure cannot be determined from the information provided, because the synthetic book is not shown to break down by customer or state. | VC | opinion | · not_checked |  |  |  |  |
| The reported $400,000 per month MWRA contract is a real public record and is not part of the synthetic book, so it is not included in the $14,384,000 contracted ARR figure. | VC | opinion | · not_checked |  |  |  |  |
| The dated public report about the MWRA and the $400,000 per month contract is real-world external evidence, while the $14,384,000 contracted ARR figure comes from the synthetic book. | VC | opinion | · not_checked |  |  |  |  |
| Any exposure estimate for the synthetic book would be an opinion based on assumed customer mix rather than a verified record. | VC | opinion | · not_checked |  |  |  |  |
| The speaker cannot say how much of the $14,384,000 synthetic contracted ARR reflects Massachusetts or state-level in-house exposure. | CEO | opinion | · not_checked |  |  |  |  |
| The public evidence settles the speaker's earlier doubts about the reason for the MWRA loss and about in-house substitution, but only for MWRA. | CEO | opinion | · not_checked |  |  |  |  |
| The CDC loss is national and is not a local-buyer loss. | CEO | fact | ✅ supported | 0.75 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_02`, `biobot_public_contracts_customers_and_06` | CDC loss is a national contract (Verily replaced Biobot); MWRA is the separate state loss. |
| The evidenced local-buyer loss count is one (MWRA). | CEO | number | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03` | Only MWRA is evidenced as a lost local/state buyer; no others appear. |
| The speaker cannot verify any loss among the 69 municipal and county programs. | CEO | opinion | · not_checked |  |  |  |  |
| The synthetic ARR figure and the $36,000 and $96,000 averages come from biobot_dummy_commercial_contract_book_02 (synthetic, 2026-09-30). | CEO | fact | ✅ supported | 0.86 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_revenue_and_retention_05` | JEV jev-1.14.0: evidence covers it 94% |
| The speaker cannot see the cited contract book chunk in the current display. | CEO | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | Evidence does not say what the speaker can see in the display. |
| The MWRA figures are real public reporting. | CEO | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03` | MWRA notice and Boston Globe reporting are public sources. |
| The public MWRA figure is contract value, not net revenue or margin. | CEO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_contracts_customers_and_02`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 92% |
| Annualizing $400,000/month gives $4.8M per year. | CEO | number | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | $400,000 x 12 = $4.8M. |
| $4.8M per year would be about 33.4% of $14,384,000. | CEO | number | ❔ unverifiable | 0.00 | claude-sonnet-5-5 (escalated) |  | No valid supporting citation |
| The speaker cannot say whether the reported $400,000/month MWRA contract is inside or outside the synthetic contracted ARR book. | CEO | opinion | · not_checked |  |  |  |  |
| The $4.8M annualized comparison is an arithmetic comparison, not a finding. | CEO | opinion | · not_checked |  |  |  |  |
| The MWRA contract is probably outside the synthetic book, given that the local-program averages of $36,000 and $96,000 are far below a $4.8M annualized state contract. | CEO | opinion | · not_checked |  |  |  |  |
| Replacement at one-quarter of $400,000 is $100,000/month. | CEO | number | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 93% |
| The state still spends something on testing, but the speaker cannot verify that any of it goes to Biobot. | CEO | opinion | · not_checked |  |  |  |  |
| The 33% comparison applies only if MWRA were inside the synthetic book, which cannot be verified. | CEO | forecast | · not_checked |  |  |  |  |
| If MWRA were inside the book, about one-third of ARR would be gone. | CEO | forecast | · not_checked |  |  |  |  |
| If MWRA is outside the book, the synthetic ARR overstates nothing, but the real company has already lost a much larger contract than any single local program. | CEO | opinion | · not_checked |  |  |  |  |
| Other state-level buyers may face similar in-house substitution risk, but this cannot be sized from the evidence. | CEO | opinion | · not_checked |  |  |  |  |
| To confirm the analysis, the speaker needs the synthetic book's customer list by state and contract type, whether MWRA is included, expiry dates, and which other states have in-house plans. | CEO | opinion | · not_checked |  |  |  |  |
| The evidence shown gives no state-level breakdown of the synthetic book and no customer list. | CEO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_06` | Book gives counts by tier, no state breakdown; only an invented ID, no real customer list. |
| The Devil is right that the speaker overstated 'two concrete losses'. | CEO | opinion | · not_checked |  |  |  |  |
| The MWRA loss is evidenced as a Biobot customer loss with a stated cause in biobot_public_contracts_customers_and_06 (public source, 2026-07). | CEO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03` | JEV jev-1.14.0: evidence covers it 93% |
| The public source says the MWRA testing ended at the end of July 2026 as the state moved toward in-house testing. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 97% |
| The Boston Globe reported the MWRA contract at $400,000/month with a replacement at about one-quarter of that cost. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The MWRA contract was reported at $400,000 per month. | CEO | number | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The MWRA replacement cost is reported as about one-quarter of the original contract cost. | CEO | number | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 94% |
| The CEO inferred that the $400,000/month MWRA contract is 'probably outside the synthetic book' because the $36,000 and $96,000 averages look nothing like $4.8M. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's statement that 'if outside, the synthetic ARR overstates nothing' is wrong in a different way. | DEVIL | opinion | · not_checked |  |  |  |  |
| If MWRA's $4.8M was real Biobot revenue and sits outside the book, the book is not the company's whole contracted base and cannot answer the VC's exposure question. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO states that 'the state still spends something' on replacement testing at $100,000/month. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Replacement is about one-quarter of $400k/month, which implies $100k, but the CEO's statement isn't shown. |
| The source says only 'about one-quarter as much' for replacement testing, so $100,000/month is an expected cost and not a confirmed spend. | DEVIL | fact | ✅ supported | 0.80 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 91% |
| The 20 large-tier programs ($420,000 and $680,000) are worth 16 x $420,000 + 4 x $680,000 = $9,440,000. | DEVIL | number | 🧮 miscalculated | 1.00 | python |  | Arithmetic does not match the stated result |
| The 20 large-tier programs' $9,440,000 is 65.6% of the $14,384,000 ARR, which the CEO never examined. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_04` | 16x420k + 4x680k = 9.44M; 9.44/14.384 = 65.6%. CEO behavior unverified. |
| The 20 large-tier programs are the likeliest place for state-level substitution exposure. | DEVIL | forecast | · not_checked |  |  |  |  |
| I cannot verify from the evidence who the buyers of the 20 large-tier programs are. | DEVIL | opinion | · not_checked |  |  |  |  |
| The book may or may not contain MWRA; this cannot be verified either way since it lists no customers or states. | DEVIL | opinion | · not_checked |  |  |  |  |
| The synthetic book is not made only of the two tiers compared by the CEO. | DEVIL | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 94% |
| Chunk biobot_dummy_commercial_contract_book_04 (synthetic, dated 2026-09-30) shows four tiers: 28 programs at $36,000, 41 at $96,000, 16 at $420,000 and 4 at $680,000. | DEVIL | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 97% |
| The CEO compared MWRA only with the two local averages and ignored the 20 programs at $420,000 and $680,000. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02` | Tier counts match 16+4=20, but the CEO's comparison is not in the evidence. |
| Even the $420,000 and $680,000 tiers are far below $4.8M, so MWRA would not fit as a single line in the book. | DEVIL | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_revenue_and_retention_07` | Max tier is $680,000, far below $4.8M; no program exceeds $680,000. |
| The book shows no customer names or states, so a $4.8M state contract could be split across lines or be outside the book. | DEVIL | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | Book lists only counts by tier, with no names or states. |
| The CEO's reasoning supports neither inclusion nor exclusion of MWRA in the book, and the 'probably outside' inference rests on an incomplete comparison. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 33.4% figure is misleading as a scale check because it divides a real annualized contract value by a synthetic ARR with no stated link. | DEVIL | opinion | · not_checked |  |  |  |  |
| The source says the monthly figure is contract value, not net revenue. | DEVIL | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_dummy_commercial_contract_book_04`, `biobot_public_contracts_customers_and_06`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 92% |
| The synthetic contract book lists 16 state programs. | VC | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic contract book is dummy data and should not be treated as evidence of real revenue or real customer exposure. | VC | opinion | · not_checked |  |  |  |  |
| The MWRA contract ending and the state's move toward in-house testing are real public records, while the contract book figures are synthetic and the answers about the 16 programs are opinion. | VC | opinion | · not_checked |  |  |  |  |
| The synthetic contract book lists each of the 16 state programs at $420,000. | VC | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic contract book lists 4 federal or multi-region programs. | VC | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic contract book lists each of the 4 federal or multi-region programs at $680,000. | VC | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 97% |
| The synthetic contract book (biobot_dummy_commercial_contract_book_02) shows no customer names or states. | VC | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_06` | Book chunks give counts by tier; only invented ID SIM-MUN-001, no names or states. |
| A public report (biobot_public_contracts_customers_and_06, dated 2026-07) says the MWRA contract ended in July 2026. | VC | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03` | JEV jev-1.14.0: evidence covers it 97% |
| A public report (biobot_public_contracts_customers_and_06, dated 2026-07) says the state moved toward in-house testing. | VC | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_04`, `biobot_public_public_measurements_and_03` | JEV jev-1.14.0: evidence covers it 96% |
| The 16 state programs in the synthetic contract book cannot be named from the evidence shown, so it is unknown whether any of them have in-house testing plans, a procurement re-bid, or an expiry date in the next 12 months. | VC | opinion | · not_checked |  |  |  |  |
| Sizing the exposure from the 16 state programs would require the real (non-synthetic) contract register with customer names, states, contract values, expiry dates, procurement and re-bid status, and any in-house testing plans from state agencies. | VC | forecast | · not_checked |  |  |  |  |
| The speaker cannot name any of the 16 state programs. | CEO | opinion | · not_checked |  |  |  |  |
| The Devil cited a fuller four-tier count in chunk 04, which the speaker cannot see. | CEO | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | Evidence says nothing about who cited chunk 04 or visibility. |
| MWRA testing ended at the end of July 2026 as the state moved toward in-house testing. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| MWRA testing was reported at $400,000/month in public reporting (biobot_public_contracts_customers_and_06, 2026-07). | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The public source states the $400,000/month is contract value, not net revenue or margin. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 94% |
| The tiers and the $14,384,000 total are synthetic data. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_revenue_and_retention_07`, `biobot_dummy_pricing_and_unit_01`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 95% |
| MWRA reporting is real public reporting. | CEO | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_public_measurements_and_03`, `biobot_public_contracts_customers_and_06` | MWRA's own page and notice are public sources confirming the testing ended. |
| Any view of which states are at risk is opinion. | CEO | opinion | · not_checked |  |  |  |  |
| The 16 state programs are the likeliest place for similar substitution risk, since MWRA is a state-level buyer. | CEO | opinion | · not_checked |  |  |  |  |
| One case does not size the state-program substitution risk. | CEO | opinion | · not_checked |  |  |  |  |
| The maximum possible ceiling is 16 x $420,000 = $6,720,000. | CEO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| The speaker cannot say whether any state program has an in-house plan, a re-bid or an expiry in the next 12 months. | CEO | opinion | · not_checked |  |  |  |  |
| $6,720,000 is 46.7% of $14,384,000. | CEO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04` | 6,720,000/14,384,000 = 46.7%; 16×420k = 6.72M. |
| The 46.7% figure is a ceiling if every state program were lost, not a forecast, and the speaker does not claim that loss. | CEO | forecast | · not_checked |  |  |  |  |
| $400,000/month x 12 months = $4,800,000/year. | CEO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | $400,000 × 12 = $4.8M; arithmetic holds. |
| State programs may face MWRA-like in-house substitution risk; this is the speaker's opinion and is not evidenced. | CEO | forecast | · not_checked |  |  |  |  |
| The synthetic book is scenario data and whether MWRA is in it cannot be verified. | CEO | opinion | · not_checked |  |  |  |  |
| The tier counts of 28, 41, 16 and 4 are taken from the synthetic book. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_commercial_contract_book_01` | JEV jev-1.14.0: evidence covers it 97% |
| The most important assumption is that state programs resemble MWRA in substitution risk; this is untested and could be too high or too low. | CEO | forecast | · not_checked |  |  |  |  |
| The records needed are the contract book with customer, state, contract type, start and expiry dates, and renewal or re-bid terms; each state's lab capacity and in-house or budget announcements; whether MWRA is in the book; and win/loss and churn records with buyer-stated reasons. | CEO | opinion | · not_checked |  |  |  |  |
| The evidence shown contains no customer names, states, expiry dates or procurement status. | CEO | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06`, `biobot_dummy_procurement_and_sales_04` | Evidence names MWRA, Massachusetts, end-July 2026 expiry, and renewal funding status. |
| The Devil is right that the 'probably outside the book' inference compared MWRA only with the $36,000 and $96,000 tiers and ignored the $420,000 and $680,000 tiers. | CEO | opinion | · not_checked |  |  |  |  |
| Even the $420,000 and $680,000 tiers are far below $4.8M, so the book neither shows nor rules out MWRA. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker should not have said the synthetic ARR overstates nothing. | CEO | opinion | · not_checked |  |  |  |  |
| If MWRA was real Biobot revenue outside the book, the book is not the whole contracted base and cannot answer the exposure question. | CEO | opinion | · not_checked |  |  |  |  |
| The $100,000/month is the reported expected replacement cost ('about one-quarter'), not a confirmed spend. | CEO | fact | ✅ supported | 0.86 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 93% |
| The tiers come from the synthetic book biobot_dummy_commercial_contract_book_02 dated 2026-09-30, which lists 16 state programs at $420,000 and 4 federal or multi-region programs at $680,000. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 97% |
| The CEO's 'maximum possible ceiling' of 16 x $420,000 = $6,720,000 (46.7% of ARR) is not a ceiling. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO has no evidence that the 16 programs are state-level. | DEVIL | opinion | · not_checked |  |  |  |  |
| The only source describing the 16 programs as state-level is the VC's description of the book. | DEVIL | fact | ❌ contradicted | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | Scenario contract book chunks themselves describe 16 state programs. |
| The book lists no program names or states. | DEVIL | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_02`, `biobot_dummy_commercial_contract_book_01` | Book shows only tier counts and values, no program names or states. |
| The 'state programs resemble MWRA' premise is doubtful on the CEO's own numbers. | DEVIL | opinion | · not_checked |  |  |  |  |
| MWRA is at $4.8M a year. | DEVIL | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | $400,000/month annualizes to $4.8M; though the contract expired. |
| MWRA at $4.8M a year is 11.4 times a $420,000 tier. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06`, `biobot_dummy_commercial_contract_book_02` | 4.8M/420k = 11.43; arithmetic holds. |
| One MWRA-style loss is not comparable to one line of the book. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 46.7% figure reads as a bounded risk number but is an arithmetic sum of tier counts with no link to any expiry, re-bid or in-house plan. | DEVIL | opinion | · not_checked |  |  |  |  |
| Exposure cannot be bounded from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The ceiling should be the whole book, $14,384,000, until contract terms are seen. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO's ceiling assumes the only exposure sits in the 16 state programs. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) |  | CEO's ceiling assumption is not in the evidence. |
| The CEO relies on tier counts from a chunk it admits it cannot see, so the 16 and 4 counts are second-hand to it. | DEVIL | opinion | · not_checked |  |  |  |  |
| I cannot verify from the evidence shown which programs are state, federal or multi-region. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 4 federal or multi-region programs face procurement loss risk comparable to the CDC case. | DEVIL | forecast | · not_checked |  |  |  |  |
| The tier counts of 16 and 4 come from the VC's description and earlier turns, not from a chunk displayed here. | DEVIL | opinion | · not_checked |  |  |  |  |
| The whole-book ceiling of $14,384,000 is a logical bound, not a forecast. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's own evidence undermines the claim that the only exposure sits in the 16 state programs. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 4 federal or multi-region programs at $680,000 are exposed to procurement loss. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CDC contract went to Verily (biobot_public_competitors_alternatives_and_02, 2026-10-14). | DEVIL | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_02`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 96% |
| biobot_public_business_and_market_03 flags government contract concentration. | DEVIL | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_02`, `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 90% |
| Adding the 4 federal or multi-region programs gives 16 x $420,000 + 4 x $680,000 = $9,440,000. | DEVIL | number | 🧮 miscalculated | 1.00 | python |  | Arithmetic does not match the stated result |
| $9,440,000 is 65.6% of the synthetic ARR of $14,384,000. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_commercial_contract_book_04`, `biobot_dummy_revenue_and_retention_05` | 9,440,000/14,384,000 ≈ 65.6%; ARR confirmed $14,384,000. |
| The stated ceiling understates the maximum by $2,720,000. | DEVIL | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_revenue_and_retention_07` | $2.72M matches top-four ARR, but the stated ceiling isn't shown. |
| The speaker states that competition is closed enough and they are turning to the team. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for information about the founders' backgrounds, including how they met, how Biobot started as an MIT spin-out, and the key moments of its journey so far. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks which of the requested founder and company details are documented records and which are the responder's own recollection or opinion. | VC | opinion | · not_checked |  |  |  |  |
| Mariana Matus and Newsha Ghaeli met in Ghaeli's first week at MIT, when both were charged with making the Underworlds project real (biobot_case_p3_c2, case as of 2017). | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_case_p3_c2` | JEV jev-1.14.0: evidence covers it 96% |
| Before Biobot, Matus faced a similar commit-or-not decision with a smart toilet (biobot_case_p13_c3). | CEO | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_case_p13_c3`, `biobot_fact_matus_smart_toilet`, `biobot_case_p8_c6`, `biobot_case_p8_c4` | JEV jev-1.14.0: evidence covers it 92% |
| The founders tried to recruit a roboticist post-doc from Underworlds, who declined as a father of two young children. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_roboticist_declined` | JEV jev-1.14.0: evidence covers it 95% |
| Matus wanted to be careful with early hires and said they needed a mechanical engineer or business help, but felt they were a good team (biobot_case_p12_c1). | CEO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_hiring_gap` | JEV jev-1.14.0: evidence covers it 93% |
| The founders faced a trade-off between hardware development, which could yield proprietary IP, and a more flexible lab-based approach (biobot_case_p12_c1). | CEO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_case_p12_c2` | JEV jev-1.14.0: evidence covers it 93% |
| Public sources say the founders had worked together on MIT wastewater research before forming the company. | CEO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_case_p3_c2`, `biobot_public_history_and_operating_02`, `biobot_public_history_and_operating_01`, `biobot_case_p3_c3` | JEV jev-1.14.0: evidence covers it 92% |
| Public sources say Biobot used MIT DesignX, Sandbox, the MIT $100K competition and delta v. | CEO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_history_and_operating_02`, `biobot_fact_delta_v_admission`, `biobot_case_p10_c3` | JEV jev-1.14.0: evidence covers it 92% |
| Public sources say Biobot was on the 2017 delta v roster, pitched opioid monitoring at Demo Day and joined YC W18 (biobot_public_history_and_operating_02, as of 2018). | CEO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_case_p13_c3`, `biobot_public_history_and_operating_02`, `biobot_public_check_of_the_01`, `biobot_public_check_of_the_02` | JEV jev-1.14.0: evidence covers it 95% |
| Ghaeli wanted to lean into building relationships with mayors and executives. | CEO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_case_p13_c3`, `biobot_case_p11_c3` | JEV jev-1.14.0: evidence covers it 93% |
| Matus said they think differently but react alike on big things, with a 'low ego' approach of 'if it doesn't work, there is always another job' (biobot_case_p13_c3). | CEO | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_case_p13_c3` | JEV jev-1.14.0: evidence covers it 92% |
| The founders' team-dynamic quotes are recollections recorded in the case, not independent records. | CEO | opinion | · not_checked |  |  |  |  |
| Ghaeli arrived at MIT to a recently funded Underworlds proposal from Carlo Ratti. | CEO | fact | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_case_p3_c2` | Ratti welcomed her on day one with the recently funded Underworlds proposal. |
| The evidence shown does not include the incorporation date, founding equity, who holds the CEO title, later hires, or 2018-2026 milestones beyond YC. | CEO | fact | ✅ supported | 0.75 | claude-sonnet-5-5 (escalated) | `biobot_public_history_and_operating_02`, `biobot_case_p12_c1` | None of the shown chunks cover incorporation date, equity, CEO title, later hires, or post-YC milestones. |
| The complementary pairing of science and city-government relationships suits a municipal-facing business. | CEO | opinion | · not_checked |  |  |  |  |
| The evidence does not show that the complementary pairing has delivered results. | CEO | opinion | · not_checked |  |  |  |  |
| The evidence shows no sign of a technical or commercial leader beyond the two founders. | CEO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_case_p12_c1`, `biobot_fact_hiring_gap`, `biobot_fact_roboticist_declined` | Evidence shows only the two founders; the roboticist declined and a hiring gap is noted. |
| Key-person dependence is a risk that cannot be sized from the evidence. | CEO | opinion | · not_checked |  |  |  |  |
| The complementary-skills fit with municipal sales is an opinion, not an evidenced finding. | CEO | opinion | · not_checked |  |  |  |  |
| Key-person dependence on the two founders is an opinion; it cannot be sized from the evidence. | CEO | opinion | · not_checked |  |  |  |  |
| Ghaeli recalled the Underworlds proposal as 'blew my mind'. | CEO | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_case_p3_c2` | JEV jev-1.14.0: evidence covers it 95% |
| Ghaeli trained in architecture in Canada. | CEO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_case_p3_c2`, `biobot_fact_ghaeli_senseable_lab`, `biobot_case_p2_c2` | JEV jev-1.14.0: evidence covers it 96% |
| Ghaeli's architecture training counts as an engineering degree. | CEO | opinion | · not_checked |  |  |  |  |
| Ghaeli had never worked in biology. | CEO | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_case_p3_c2` | JEV jev-1.14.0: evidence covers it 93% |
| Ghaeli managed the six-department Underworlds effort and handled the 'urban' side: mappings, implementation and government contacts. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_case_p3_c2`, `biobot_fact_underworlds_scale` | JEV jev-1.14.0: evidence covers it 94% |
| Matus is described as the scientist with 'a giant vision' (biobot_case_p13_c3). | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_case_p13_c3` | JEV jev-1.14.0: evidence covers it 96% |
| The evidence shown does not verify Matus's degree, her exact role, or any dates for her background. | CEO | opinion | · not_checked |  |  |  |  |
| The CEO claims the founders' 'low ego' quote, the 'think differently but react alike' dynamic, and the Matus/Ghaeli role split are supported by biobot_case_p13_c3. | DEVIL | opinion | · not_checked |  |  |  |  |
| Matus and Ghaeli were charged with making Underworlds real. | DEVIL | fact | ✅ supported | 0.86 | jev:jev-latest | `biobot_case_p3_c2` | JEV jev-1.14.0: evidence covers it 93% |
| A roboticist post-doc declined to join the venture. | DEVIL | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_roboticist_declined` | JEV jev-1.14.0: evidence covers it 95% |
| Matus said they needed a mechanical engineer or business help but were a good team. | DEVIL | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_hiring_gap` | JEV jev-1.14.0: evidence covers it 94% |
| The displayed chunks do not show that Matus is 'the scientist with a giant vision'. | DEVIL | opinion | · not_checked |  |  |  |  |
| The displayed chunks do not show that Matus faced a smart-toilet decision. | DEVIL | opinion | · not_checked |  |  |  |  |
| The displayed chunks do not show that Matus and Ghaeli complement each other. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO states that Ghaeli trained in architecture in Canada. | DEVIL | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_case_p3_c2`, `biobot_case_p2_c2`, `biobot_fact_ghaeli_senseable_lab` | JEV jev-1.14.0: evidence covers it 93% |
| In the chunk, Ghaeli says an architecture degree in Canada is an engineering degree; this is her own recollection, not a checked record of her degree. | DEVIL | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_case_p3_c2` | It is Ghaeli's recalled statement, with no independent verification of the degree. |
| The CEO's 'complementary pairing' opinion rests on an unverified attribution of roles. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO says it will not fill gaps yet asserts the later 2017-2018 timeline and the founders' motives as fact, which is internally inconsistent. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO claims that 'Ghaeli wanted to lean into building relationships with mayors' is supported by biobot_case_p13_c3. | DEVIL | opinion | · not_checked |  |  |  |  |
| The spin-out narrative beyond what chunks biobot_case_p3_c2 and biobot_case_p12_c1 show should be treated as unconfirmed until the missing chunks are displayed. | DEVIL | opinion | · not_checked |  |  |  |  |
| Chunks biobot_case_p13_c3 and biobot_public_history_and_operating_02 may exist in the full evidence set but are not displayed to the auditor; this is an inference, not a claim that they do not exist. | DEVIL | forecast | · not_checked |  |  |  |  |
| Chunk biobot_case_p13_c3 is not in the evidence displayed to the auditor. | DEVIL | fact | ❔ unverifiable | 0.00 | claude-sonnet-5-5 (escalated) |  | No valid supporting citation |
| Only biobot_case_p12_c1 and biobot_case_p3_c2 are shown in the displayed evidence. | DEVIL | fact | ❌ contradicted | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_public_current_product_catalog_03`, `biobot_public_company_and_offering_03` | Several other chunks are displayed besides those two. |
| The CEO cites biobot_public_history_and_operating_02 for MIT DesignX, Sandbox, the $100K competition, delta v, Demo Day and YC W18. | DEVIL | opinion | · not_checked |  |  |  |  |
| Chunk biobot_public_history_and_operating_02 is not displayed in the evidence shown to the auditor. | DEVIL | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_history_and_operating_09`, `biobot_public_history_and_operating_04` | The chunk is not among the displayed evidence for this claim. |
| The auditor cannot verify the quotes or facts attributed to the undisplayed chunks from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO presents second-hand material as documented record. | DEVIL | opinion | · not_checked |  |  |  |  |
| Ghaeli met Matus in her first week at MIT. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_case_p3_c2` | JEV jev-1.14.0: evidence covers it 96% |
| The case chunks shown record that the founders wanted a mechanical engineer or someone for the business side. | VC | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_hiring_gap` | JEV jev-1.14.0: evidence covers it 95% |
| The case chunks shown record that a roboticist post-doc declined to join the team (biobot_case_p12_c1, as of 2017). | VC | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_roboticist_declined` | JEV jev-1.14.0: evidence covers it 97% |
| The recorded information about the founders' hiring wants and the declined roboticist post-doc is a 2017 record. | VC | fact | ✅ supported | 0.77 | jev:jev-latest | `biobot_fact_roboticist_declined` | JEV jev-1.14.0: evidence covers it 87% |
| Identifying the biggest current gaps in the team and the next hire to make is a question about today's situation rather than the 2017 record. | VC | opinion | · not_checked |  |  |  |  |
| The 2017 case record cannot by itself establish the team's current gaps. | VC | opinion | · not_checked |  |  |  |  |
| The next 12-month impact of a hire in sales, lab operations, or procurement and renewals is a forecast that cannot be verified from the case record. | VC | forecast | · not_checked |  |  |  |  |
| The speaker corrected a previous answer that cited biobot_case_p13_c3 and biobot_public_history_and_operating_02, which are not displayed. | CEO | fact | ❔ unverifiable | 0.81 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 12% |
| The speaker has no current internal knowledge in the evidence shown and cannot report current headcount, open roles or hiring plans. | CEO | opinion | · not_checked |  |  |  |  |
| A senior commercial leader owning procurement, renewals and re-bids is the next hire the speaker would want for municipal and state buyers. | CEO | opinion | · not_checked |  |  |  |  |
| The MWRA contract ended in July 2026 (biobot_public_contracts_customers_and_06, 2026-07). | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03` | JEV jev-1.14.0: evidence covers it 97% |
| The CDC contract reportedly went to Verily (biobot_public_competitors_alternatives_and_02, 2026-10-14). | CEO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_competitors_alternatives_and_02`, `biobot_public_business_and_market_04`, `biobot_public_business_and_market_03` | JEV jev-1.14.0: evidence covers it 94% |
| The MWRA and CDC outcomes point to renewal and procurement risk and are signals, not an internal diagnosis. | CEO | opinion | · not_checked |  |  |  |  |
| A lab-operations or mechanical hire is the second candidate given the 2017 record, ranked without evidence on today's lab constraints. | CEO | opinion | · not_checked |  |  |  |  |
| A renewals owner could track expiry dates, in-house testing plans and re-bid terms for each program within 12 months. | CEO | forecast | · not_checked |  |  |  |  |
| The revenue effect of the hire cannot be quantified from the available evidence. | CEO | opinion | · not_checked |  |  |  |  |
| The 2017 team gaps and the 2026 renewal events point to a commercial and procurement gap today; this is untested and the gap may already be filled or lie elsewhere. | CEO | forecast | · not_checked |  |  |  |  |
| A senior commercial leader for procurement and renewals as the next hire is opinion and not evidenced. | CEO | opinion | · not_checked |  |  |  |  |
| The 'giant vision' and 'low ego' quotes, the smart-toilet decision, the Ghaeli-and-mayors role split, and the delta v / Demo Day / YC W18 timeline should not be presented as documented and are unconfirmed. | CEO | opinion | · not_checked |  |  |  |  |
| The 2017 gaps may or may not persist in 2026; the current team cannot be verified. | CEO | forecast | · not_checked |  |  |  |  |
| The 12-month effect of the hire is a forecast with no supporting numbers. | CEO | forecast | · not_checked |  |  |  |  |
| MWRA and CDC outcomes are treated as signals of renewal risk, not as proof of a team gap. | CEO | opinion | · not_checked |  |  |  |  |
| Ghaeli met Matus in her first week at MIT and both were charged with making Underworlds real, according to biobot_case_p3_c2 (2017). | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_case_p3_c2` | JEV jev-1.14.0: evidence covers it 95% |
| Ghaeli said an architecture degree in Canada is an engineering degree; this is her own recollection and not a checked record of her degree. | CEO | fact | ✅ supported | 0.93 | claude-sonnet-5-5 (escalated) | `biobot_case_p3_c2` | Ghaeli's recalled quote says an architecture degree in Canada is an engineering degree. |
| A roboticist post-doc declined to join, according to biobot_case_p12_c1 (2017). | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_roboticist_declined` | JEV jev-1.14.0: evidence covers it 96% |
| Matus said they needed a mechanical engineer or someone on the business side but were a good team (biobot_case_p12_c1; biobot_fact_hiring_gap, 2017). | CEO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_hiring_gap` | JEV jev-1.14.0: evidence covers it 93% |
| The complementary-pairing view of the founding team is opinion and rests on no displayed evidence. | CEO | opinion | · not_checked |  |  |  |  |
| Documented team gaps in the 2017 record are a mechanical engineer and a business-side partner. | CEO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_hiring_gap` | JEV jev-1.14.0: evidence covers it 94% |
| It cannot be verified from the evidence whether either 2017 gap was filled, who is on the team today, or who holds which role. | CEO | opinion | · not_checked |  |  |  |  |
| The CEO's pick of a senior commercial leader for procurement and renewals as the next hire is the weakest point of the statement's subject. | DEVIL | opinion | · not_checked |  |  |  |  |
| The one documented gap is the 2017 record calling for a mechanical engineer or someone on the business side. | DEVIL | fact | ✅ supported | 0.79 | jev:jev-latest | `biobot_case_p12_c1`, `biobot_fact_hiring_gap` | JEV jev-1.14.0: evidence covers it 93% |
| Matus said 'either' in 2017, so the business-side hire was an alternative to a mechanical hire rather than a priority. | DEVIL | opinion | · not_checked |  |  |  |  |
| Moving from the 2017 quote to a 2026 commercial hire is a nine-year extrapolation. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO ranks lab or mechanical hiring second 'without evidence on today's lab constraints'. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | No evidence on a CEO ranking of lab/mechanical hiring or current lab constraints. |
| The CEO's ranking of lab or mechanical hiring has no evidence on either side, so the next hire cannot be ranked from the displayed evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's claim that a renewals owner could track expiry dates and re-bid terms is weak because that tracking requires contract records the CEO says it does not have. | DEVIL | opinion | · not_checked |  |  |  |  |
| If contract records exist, a hire is not needed to start tracking; if they do not exist, the problem is systems and data rather than a senior title. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify current headcount or who owns renewals, so the claimed gap may already be filled. | DEVIL | opinion | · not_checked |  |  |  |  |
| The MWRA and CDC outcomes may have causes unrelated to Biobot's commercial staffing. | DEVIL | forecast | · not_checked |  |  |  |  |
| The contract-tracking problem may be a records or systems issue rather than a headcount issue. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO's choice of a senior commercial leader for procurement and renewals rests on two events: MWRA ending in July 2026 and the CDC contract reportedly going to Verily. | DEVIL | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_02`, `biobot_public_contracts_customers_and_06` | Evidence confirms both events as procurement risks; the CEO's hiring rationale is not shown. |
| The MWRA ending in July 2026 and the CDC contract reportedly going to Verily do not show a team gap at Biobot. | DEVIL | opinion | · not_checked |  |  |  |  |
| The MWRA source, as the CEO described it earlier, says the state moved toward in-house testing. | DEVIL | opinion | · not_checked |  |  |  |  |
| The state moving toward in-house testing at MWRA is a buyer make-versus-buy decision, and a renewals owner may not have been able to prevent it. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CDC contract loss is a competitive procurement outcome. | DEVIL | opinion | · not_checked |  |  |  |  |
| The evidence does not say whether Biobot's sales team lost the CDC contract on price, product or process. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO infers a commercial-staffing gap from outcomes that have different causes. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO admits it cannot see customer churn or win/loss reasons. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker asks where Biobot will be in ten years if the company works, and what its end-state product is. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks whether Biobot's future is still wastewater surveillance sold to government agencies or something broader. | VC | opinion | · not_checked |  |  |  |  |
| The speaker states that government agencies can run their own wastewater testing, citing biobot_public_competitors_alternatives_and_06 dated 2026-10-22. | VC | fact | ✅ supported | 0.83 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 88% |
| The speaker asks what would remain defensible at scale given that government agencies can run their own wastewater testing. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks the respondent to separate documented records, current internal plans, and their own opinion. | VC | opinion | · not_checked |  |  |  |  |
| The speaker's ten-year vision is for Biobot to be a population-health data layer for cities and agencies, not only a wastewater testing vendor. | CEO | opinion | · not_checked |  |  |  |  |
| MWRA recorded the end of Biobot's Massachusetts data stream in July 2026. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The 2026 record shows that build-versus-buy (in-house testing) is a real alternative to Biobot's service. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker does not claim that testing alone is defensible. | CEO | opinion | · not_checked |  |  |  |  |
| Any moat for Biobot would have to come from a longitudinal data set across many jurisdictions, comparable benchmarks, analytics agencies adopt in their workflows, or service cheap or reliable enough that in-house testing is not worth it. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify whether Biobot holds any of these moat sources, including the size or exclusivity of its data or the terms of its data rights. | CEO | opinion | · not_checked |  |  |  |  |
| Proposed step 1: stop treating testing as the product and sell analytics and benchmarking that work even when an agency runs its own lab. | CEO | forecast | · not_checked |  |  |  |  |
| Proposed step 2: test whether agencies that insource still pay for analytics, using the MWRA case as the first test. | CEO | forecast | · not_checked |  |  |  |  |
| Proposed step 3: show that the data improves decisions, with buyer-stated outcomes. | CEO | forecast | · not_checked |  |  |  |  |
| Proposed step 4: only after that, widen to other buyers. | CEO | forecast | · not_checked |  |  |  |  |
| Before committing, the speaker would confirm the data rights in contracts, the customer mix, and win/loss reasons, none of which the speaker can see. | CEO | opinion | · not_checked |  |  |  |  |
| In the speaker's vision, wastewater surveillance would remain the sensor, with a decision system built on top offering trend signals, early warnings and measures of whether interventions worked. | CEO | forecast | · not_checked |  |  |  |  |
| The ten-year vision of a population-health data layer is the speaker's opinion, not an internal plan or documented record. | CEO | opinion | · not_checked |  |  |  |  |
| Agencies that insource testing might still pay for analytics, which is untested. | CEO | forecast | · not_checked |  |  |  |  |
| In the speaker's vision, governments would be the first buyers and other institutions possibly later. | CEO | forecast | · not_checked |  |  |  |  |
| The displayed evidence contains nothing showing Biobot is pursuing a population-health data layer strategy. | CEO | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_case_p1_c1`, `biobot_case_p9_c4` | Evidence shows population-health framing, data-driven public health, and data-selling vision. |
| The speaker cannot report Biobot's internal plans, roadmap or R&D budget. | CEO | opinion | · not_checked |  |  |  |  |
| In 2017 Biobot's founders were turning sewage into information about population health and wanted to turn urine and stool data into a business. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_case_p1_c1`, `biobot_case_p1_c2` | JEV jev-1.14.0: evidence covers it 94% |
| In 2017, the founders were told their idea was too broad and had no defensible intellectual property. | CEO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_case_p1_c1`, `biobot_fact_early_objections` | JEV jev-1.14.0: evidence covers it 93% |
| Ghaeli cited urbanization, GovTech and the push toward data-driven public-health decisions as market trends in 2017. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_case_p9_c4`, `biobot_fact_govtech_trend`, `biobot_fact_urban_share_claim` | JEV jev-1.14.0: evidence covers it 96% |
| Agencies can collect, analyze and publish wastewater signals without buying Biobot's complete service. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 95% |
| The displayed evidence shows that MWRA recorded the end of Biobot's Massachusetts data stream in July 2026 (biobot_public_competitors_alternatives_and_06, dated 2026-10-22). | DEVIL | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06`, `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 95% |
| The critics' view that Biobot lacks defensible intellectual property and scalable advantage applies as much to a software analytics layer as to testing. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO lists four moat hypotheses: longitudinal data, benchmarks, workflow adoption, and cost. | DEVIL | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) |  | Evidence does not show the CEO's stated moat hypotheses; it only gives scenario data. |
| The CEO admits it cannot verify that Biobot holds any of the four moat hypotheses. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_capacity_and_product_04` | Evidence shows no proof of moat but not what the CEO admitted. |
| A ten-year vision whose defensibility rests on zero verified assets, and whose first test case is a lost customer, is unsupported. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 'population health' framing in biobot_case_p1_c1 is a 2017 founder aspiration rather than evidence of a data-layer strategy. | DEVIL | opinion | · not_checked |  |  |  |  |
| An agency that has built its own lab is the buyer least likely to pay for a vendor's analytics on data it now owns. | DEVIL | forecast | · not_checked |  |  |  |  |
| Benchmarks require comparable data from many jurisdictions. | DEVIL | opinion | · not_checked |  |  |  |  |
| Insourcing agencies would stop supplying data to Biobot, which would shrink the data set that the CEO's step 1 depends on. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO's step 2 risks shrinking the data set that step 1 depends on. | DEVIL | opinion | · not_checked |  |  |  |  |
| The answer gives no evidence on pricing, willingness to pay, or data-sharing rights. | DEVIL | fact | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_ratti_who_pays` | Evidence includes pricing floor and willingness-to-pay uncertainty; data-sharing rights unaddressed. |
| The displayed evidence does not say why MWRA ended the Biobot data stream. | DEVIL | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence says state moved toward in-house testing, giving a reason for the end. |
| The answer therefore does not show any durable competitive position at ten years. | DEVIL | opinion | · not_checked |  |  |  |  |
| An insourcing agency is less likely to buy vendor analytics on data it owns. | DEVIL | forecast | · not_checked |  |  |  |  |
| Insourcing agencies may stop supplying data to Biobot benchmarks. | DEVIL | forecast | · not_checked |  |  |  |  |
| MWRA's reason for ending the data stream is not stated in the displayed evidence. | DEVIL | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence states the state moved toward in-house testing. |
| The displayed evidence does not say whether MWRA was offered analytics or would pay for analytics. | DEVIL | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_06` | No evidence of analytics being offered to or purchased by MWRA. |
| Calling MWRA 'the first test' of insourcing-plus-analytics is an inference, not an established fact. | DEVIL | opinion | · not_checked |  |  |  |  |
| The chunk biobot_public_competitors_alternatives_and_06 says CDC receives data from state and local departments as well as contractors. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The chunk biobot_public_competitors_alternatives_and_06 says agencies can 'collect, analyze, and publish' signals themselves. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The 'analyze' and 'publish' capabilities described in the chunk correspond to the analytics layer, indicating agencies may perform analytics in-house rather than buying it. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's cited evidence undercuts the claim that analytics is defensible once testing is insourced. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 2017 chunk biobot_case_p1_c1 quotes critics saying Biobot had 'no defensible intellectual property and limited scalability'. | DEVIL | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_case_p1_c1`, `biobot_fact_early_objections` | JEV jev-1.14.0: evidence covers it 94% |
| The speaker asks which of the four proposed steps is the riskiest and what specific, observable result would show it is achievable. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks which inputs come from documented records, which come from current internal knowledge, and which are opinion. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks how the benchmarking and longitudinal-data steps would work if an agency moves testing in-house and stops sending samples or data to Biobot. | VC | opinion | · not_checked |  |  |  |  |
| The four proposed steps are: sell analytics that work when an agency runs its own lab, test whether insourcing agencies still pay, prove the data improves decisions, and widen to other buyers. | VC | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) |  | Evidence does not mention these four proposed steps in the meeting answer. |
| An agency that moves testing in-house may stop sending samples or data to Biobot. | VC | forecast | · not_checked |  |  |  |  |
| The speaker withdraws the claim that MWRA was the 'first test' of the analytics pivot. | CEO | opinion | · not_checked |  |  |  |  |
| Step 1 is a build that the speaker cannot size. | CEO | opinion | · not_checked |  |  |  |  |
| An observable result showing step 2 is achievable would be at least one agency running its own lab signing a paid, multi-period analytics agreement, with the buyer stating in writing why it pays and what it uses the output for. | CEO | forecast | · not_checked |  |  |  |  |
| Pass and fail thresholds for step 2 (number of agencies, price, term) are not known to the speaker and should be set against actual pricing and cost data. | CEO | opinion | · not_checked |  |  |  |  |
| In 2017 critics said the idea had no defensible IP and limited scalability, and the founders framed their work as 'population health'. | CEO | fact | ✅ supported | 0.84 | jev:jev-latest | `biobot_case_p1_c1`, `biobot_fact_early_objections` | JEV jev-1.14.0: evidence covers it 89% |
| Biobot publicly describes public-sector, pharma and life-science use cases and modeled weekly county/ZIP data, but its paying customer base is not disclosed. | CEO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_company_and_offering_03`, `biobot_public_research_findings_and_03`, `biobot_public_current_product_catalog_12`, `biobot_public_current_product_catalog_06`, `biobot_public_history_and_operating_09` | JEV jev-1.14.0: evidence covers it 93% |
| Public sources do not establish Biobot's revenue, retention, active paying customers or backlog. | CEO | fact | ✅ supported | 0.86 | jev:jev-latest | `biobot_public_company_and_offering_03`, `biobot_public_research_findings_and_03`, `biobot_public_business_and_market_06` | JEV jev-1.14.0: evidence covers it 89% |
| The speaker has no current internal knowledge of pricing, data rights or willingness to pay. | CEO | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_research_findings_and_03` | Evidence says nothing about what the speaker knows internally; only public pricing gaps. |
| Benchmarking and longitudinal data would mostly not work from an agency's own samples, because a lab-owning agency may stop sending data. | CEO | forecast | · not_checked |  |  |  |  |
| Possible routes to benchmark data are contracts granting data-sharing or de-identified benchmarking rights, benchmarks built from public agency dashboards, or keeping the modeled national network as the benchmark base; all are unverified. | CEO | forecast | · not_checked |  |  |  |  |
| The speaker cannot verify that Biobot holds any data-sharing rights or sample sources needed for these routes. | CEO | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_agencies_control_access` | Evidence doesn't address the speaker's verification ability; agencies control sample access. |
| The evidence displayed to the speaker does not say why MWRA left or whether MWRA was offered analytics. | CEO | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03` | Displayed chunks say MWRA moved in-house but give no reasons or analytics offer. |
| If none of these data routes is available, the data-layer moat fails and the vision should be narrowed. | CEO | forecast | · not_checked |  |  |  |  |
| The most important assumption is that an agency owning its lab will still pay a vendor for analytics on data it owns; this is untested. | CEO | forecast | · not_checked |  |  |  |  |
| The Devil's inference that insourcing agencies are the buyers least likely to pay may be right. | CEO | opinion | · not_checked |  |  |  |  |
| Step 2 (insourcing agencies still paying) is the riskiest step. | CEO | opinion | · not_checked |  |  |  |  |
| A paid multi-period analytics agreement from a lab-owning agency, with buyer-stated reasons, would be the observable signal, with no thresholds known. | CEO | opinion | · not_checked |  |  |  |  |
| Contracts may or may not grant data-sharing or benchmarking rights; this is unverified. | CEO | opinion | · not_checked |  |  |  |  |
| MWRA's reason for leaving is not stated in the evidence displayed to the speaker. | CEO | opinion | · not_checked |  |  |  |  |
| The Devil's description of the competitors chunk is taken as stated, since that chunk is not displayed to the speaker. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot re-verify the cited chunk biobot_public_competitors_alternatives_and_06 because it is not displayed to them. | CEO | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_competitors_alternatives_and_06` | Claim concerns the speaker's display; the chunk appears in evidence here, but display unknown. |
| The Devil describes the chunk biobot_public_competitors_alternatives_and_06 as saying agencies can collect, analyze, and publish signals themselves. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The speaker's claim that analytics is defensible after insourcing is undercut by the Devil's reading of that chunk. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker's step 2 could shrink the data that step 1 depends on. | CEO | opinion | · not_checked |  |  |  |  |
| The analytics pivot is a hypothesis, not a plan. | CEO | opinion | · not_checked |  |  |  |  |
| Step 2, whether insourcing agencies still pay, is the riskiest step because it decides whether the pivot has any buyer. | CEO | opinion | · not_checked |  |  |  |  |
| Step 3, proving better decisions, matters only if step 2 holds. | CEO | opinion | · not_checked |  |  |  |  |
| The cited chunk biobot_public_company_and_offering_03 (dated 2026-10-21) says Biobot samples 'hundreds of US sites'. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_company_and_offering_03`, `biobot_public_current_product_catalog_06` | JEV jev-1.14.0: evidence covers it 96% |
| The CEO's routes (a) contractual data rights and (b) public dashboards are admitted to be unverified or weaker and non-standardized. | DEVIL | fact | ❔ unverifiable | 0.81 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 11% |
| The answer states that if none of the routes is available, 'the vision should be narrowed', which concedes that the moat has no verified basis. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_fact_early_objections` | Evidence doesn't contain the answer quote; characterization as conceding moat is interpretive. |
| The 'observable result' of one paid analytics agreement from a lab-owning agency has no threshold and does not test benchmarking. | DEVIL | opinion | · not_checked |  |  |  |  |
| One buyer paying does not show that a cross-jurisdiction data layer exists. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify the network's composition, ownership or data rights from the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| Modeled data is a weaker benchmark than measured data and cannot validate decision improvement. | DEVIL | forecast | · not_checked |  |  |  |  |
| Sites in the sampled network may include agency customers who could insource. | DEVIL | forecast | · not_checked |  |  |  |  |
| The evidence does not say who owns the sampling sites or the data rights. | DEVIL | opinion | · not_checked |  |  |  |  |
| The cited chunk biobot_public_company_and_offering_03 (dated 2026-10-21) says Biobot offers 'modeled weekly county/ZIP data'. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_company_and_offering_03`, `biobot_public_current_product_catalog_12`, `biobot_public_research_and_performance_05`, `biobot_public_current_product_catalog_06`, `biobot_public_history_and_operating_09` | JEV jev-1.14.0: evidence covers it 96% |
| The cited chunk biobot_public_company_and_offering_03 does not state who owns the sampling sites, whether the samples are Biobot's own or come from customers, or whether the network is still collecting. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's fallback route (c) keeps 'the modeled national network' as the benchmark base, and this route is the weakest point of the answer. | DEVIL | opinion | · not_checked |  |  |  |  |
| The route is circular because if the sampling sites include agency customers who later insource, as MWRA did, the benchmark base shrinks. | DEVIL | opinion | · not_checked |  |  |  |  |
| Modeled data is not a longitudinal measurement and therefore cannot prove that an analytics layer improves decisions (step 3). | DEVIL | opinion | · not_checked |  |  |  |  |
| Modeled data cannot serve as a comparable benchmark for agencies that run their own labs. | DEVIL | opinion | · not_checked |  |  |  |  |
| The evidence shown gives no accuracy or validation figures for the modeled data. | DEVIL | opinion | · not_checked |  |  |  |  |
| A benchmark made of modeled numbers cannot show an agency that its own measured data is out of line, because the reference carries its own uncertainty. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker asks how much the company is raising now, on what terms or valuation, from which investors, and how many months of runway it will fund. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks that documented records be separated from current internal plans and opinion, and that the speaker say if the figures cannot be seen. | VC | opinion | · not_checked |  |  |  |  |
| The evidence shown contains no current raise amount, instrument, valuation, terms or target investors. | CFO | opinion | · not_checked |  |  |  |  |
| Third-party databases show a round of roughly $13.4M to $13.45M dated 2025-06-06. | CFO | fact | ✅ supported | 0.84 | jev:jev-latest | `biobot_public_funding_and_investors_05`, `biobot_public_check_of_the_07` | JEV jev-1.14.0: evidence covers it 89% |
| Third-party databases label the 2025-06-06 round as Series A, Series B or Series A1 depending on the source. | CFO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_funding_and_investors_05`, `biobot_public_check_of_the_07` | JEV jev-1.14.0: evidence covers it 96% |
| Third-party databases show total funding of about $40.17M to $41.4M. | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_public_funding_and_investors_05` | Screenshots show $41.4M; CB Insights shows about $40.17M. |
| A secondary site gives a valuation of about $126.8M as of 2025-06-06, with no filing behind it. | CFO | fact | ✅ supported | 0.87 | jev:jev-latest | `biobot_public_funding_and_investors_05`, `biobot_public_business_and_market_06` | JEV jev-1.14.0: evidence covers it 89% |
| The public funding figures are unreconciled and should all be treated as estimates. | CFO | opinion | · not_checked |  |  |  |  |
| A screenshot-only lead says Biobot had a 2018 $2.5M seed round with 16 investors. | CFO | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_funding_and_investors_01` | $2.5M 2018 seed supported; the 16-investor screenshot lead isn't shown in evidence. |
| Government Technology confirms the $2.5M amount and Ekistic as lead investor for the 2018 seed, but not the investor count. | CFO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_funding_and_investors_01` | Report gives $2.5M and Ekistic lead; no investor count mentioned. |
| The screenshot-only lead is unverified and the speaker does not rely on it. | CFO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify the current cap table, the investors in the 2025 round, or whether grants are included in the totals. | CFO | opinion | · not_checked |  |  |  |  |
| Given open questions on MWRA and data rights, a raise should not be sized until win/loss, retention and contract data are confirmed. | CFO | opinion | · not_checked |  |  |  |  |
| The speaker will not state the terms of a current raise because the evidence does not contain them. | CFO | opinion | · not_checked |  |  |  |  |
| Any raise amount should be tied to milestones rather than to a target runway. | CFO | opinion | · not_checked |  |  |  |  |
| Cash of $12.8M divided by burn of $336,630 per month equals about 38.0 months. | CFO | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| Burn stays constant at the scenario level, with no working-capital changes. | CFO | forecast | · not_checked |  |  |  |  |
| The synthetic 2026 packet is scenario ground truth, not actual Biobot data. | CFO | opinion | · not_checked |  |  |  |  |
| Any new raise would be additional to the $12.8M of cash; the evidence does not say whether it is already included. | CFO | forecast | · not_checked |  |  |  |  |
| A synthetic scenario record dated 2026-09-30 reports unrestricted cash of $12.8M. | CFO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08` | JEV jev-1.14.0: evidence covers it 97% |
| A synthetic scenario record dated 2026-09-30 reports operating cash burn of about $336,630 per month. | CFO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 97% |
| At constant burn, the scenario's cash would last roughly 38 months, before working-capital changes, and this is not a forecast. | CFO | number | ✅ supported | 0.90 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 93% |
| On the scenario, no raise is needed to survive about three years. | CFO | forecast | · not_checked |  |  |  |  |
| Any raise in the scenario would be for growth or to buy extra runway, and the speaker does not know which. | CFO | opinion | · not_checked |  |  |  |  |
| Biobot's own announcements establish a $20M funding round in 2021. | CFO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_funding_and_investors_05`, `biobot_public_funding_and_investors_03`, `biobot_public_check_of_the_06` | JEV jev-1.14.0: evidence covers it 96% |
| Biobot's own announcements establish an undisclosed funding round in 2025. | CFO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_funding_and_investors_05`, `biobot_public_funding_and_investors_04` | JEV jev-1.14.0: evidence covers it 94% |
| The CFO claimed that no raise is needed to survive about three years. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | Runway of 38 months is shown, but the CFO's statement isn't in evidence. |
| The 38-month runway figure excludes working-capital changes. | DEVIL | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08` | JEV jev-1.14.0: evidence covers it 96% |
| The evidence does not say whether the $12.8M cash figure is net of the 2025 round proceeds or what restricted cash or debt exists. | DEVIL | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08`, `biobot_public_business_and_market_06` | Cash stated as unrestricted; no source covers 2025 proceeds, restricted cash or debt. |
| The CFO cannot say whether a raise has already closed. | DEVIL | opinion | · not_checked |  |  |  |  |
| The runway claim is a best case and cannot support a statement about whether a raise is needed. | DEVIL | opinion | · not_checked |  |  |  |  |
| If gross profit fell by a quarter, burn would rise by about $2.07M a year. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | 25% of $8,260,440 is about $2.065M, which raises burn by that amount. |
| If gross profit fell by a quarter, runway would drop to about 25 months. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | Burn $6.10M/yr; $12.8M divided by $508.7K a month is about 25.2 months. |
| The CFO's conclusion is inconsistent with the CFO's own advice not to size a raise until retention is confirmed, because that advice admits the revenue base is unverified. | DEVIL | opinion | · not_checked |  |  |  |  |
| Growjo estimates revenue of $3.5M (biobot_public_business_and_market_01, dated 2026-10-22). | DEVIL | fact | ✅ supported | 0.87 | jev:jev-latest | `biobot_public_business_and_market_01`, `biobot_dummy_revenue_and_retention_03` | JEV jev-1.14.0: evidence covers it 90% |
| The Growjo $3.5M revenue estimate is unverified. | DEVIL | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_business_and_market_01`, `biobot_public_business_and_market_06` | JEV jev-1.14.0: evidence covers it 91% |
| The synthetic gross profit of $8.26M plus $6.12M service cost implies about $14.4M revenue. | DEVIL | number | ✅ supported | 0.93 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 96% |
| The CFO's runway claim takes the synthetic 2026-09-30 record (biobot_dummy_pricing_and_unit_08) as constant burn. | DEVIL | opinion | · not_checked |  |  |  |  |
| The Growjo estimate and the synthetic implied revenue differ by about four times, and the CFO did not reconcile them. | DEVIL | opinion | · not_checked |  |  |  |  |
| Growjo's estimate is weak evidence, but the gap shows how fragile the revenue base is. | DEVIL | opinion | · not_checked |  |  |  |  |
| A 25% fall in gross profit ($8,260,440 x 25%) equals $2,065,110 lost gross profit per year. | DEVIL | number | ✅ supported | 0.97 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | 8,260,440 × 0.25 = 2,065,110. |
| Annual burn of $4,039,560 plus $2,065,110 equals $6,104,670, which is $508,723 per month. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| $12,800,000 divided by $508,723 per month is about 25.2 months of runway. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | 4,039,560+2,065,110=6,104,670/12=508,723; 12.8M/508,723≈25.16. |
| Implied revenue of $8,260,440 plus $6,123,560 equals $14,384,000. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| Implied revenue of $14,384,000 divided by Growjo's $3,500,000 is about 4.1x. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_01` | 14,384,000/3,500,000=4.11; Growjo revenue is $3.5M. |
| A 25% fall in gross profit is an illustrative stress case, not evidenced. | DEVIL | forecast | · not_checked |  |  |  |  |
| Service cost stays at $6,123,560 regardless of gross profit changes in the stress case, so cost is not scaled down. | DEVIL | forecast | · not_checked |  |  |  |  |
| Implied revenue equals gross profit plus service cost, as the synthetic record appears to define it. | DEVIL | forecast | · not_checked |  |  |  |  |
| The synthetic 2026-09-30 record shows monthly burn of $336,630. | DEVIL | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 97% |
| MWRA's loss reducing revenue is an inference; its size is not shown in the evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| The synthetic 2026-09-30 record shows annualized gross profit of $8,260,440. | DEVIL | number | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 96% |
| The $336,630 monthly burn depends on annualized gross profit of $8,260,440 holding. | DEVIL | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 96% |
| The CFO conceded that MWRA ended its Biobot data stream in July 2026. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_03` | MWRA ended in July 2026 is supported; the CFO conceding it is not shown. |
| The CFO conceded that retention, win/loss and contract data are unconfirmed. | DEVIL | fact | ❔ unverifiable | 0.86 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 8% |
| A lost customer is a direct threat to the gross profit that makes burn only $336,630. | DEVIL | opinion | · not_checked |  |  |  |  |
| If revenue falls, burn rises and the 38-month runway shrinks. | DEVIL | opinion | · not_checked |  |  |  |  |
| $8,260,440 x 25% = $2,065,110 lost gross profit per year; $4,039,560 + $2,065,110 = $6,104,670 burn per year = $508,723 per month; $12,800,000 / $508,723 = about 25.2 months; $8,260,440 + $6,123,560 = $14,384,000 implied revenue; $14,384,000 / $3,500,000 = about 4.1x | DEVIL | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The synthetic 2026-09-30 packet shows annualized gross profit of $8,260,440. | VC | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 98% |
| The synthetic 2026-09-30 packet shows cash burn of about $336,630 a month. | VC | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 98% |
| FY2024 gross profit was $2,300,000. | VC | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_historical_operating_trajectory_08` | JEV jev-1.14.0: evidence covers it 98% |
| FY2024 operating loss was $7,000,000. | VC | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_historical_efficiency_and_03` | JEV jev-1.14.0: evidence covers it 97% |
| Annualized gross profit of $8,260,440 is larger than FY2024 gross profit of $2,300,000. | VC | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_historical_operating_trajectory_08` | JEV jev-1.14.0: evidence covers it 98% |
| A monthly cash burn of about $336,630 corresponds to roughly $4,039,560 of annual cash burn. | VC | number | ✅ supported | 0.92 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 94% |
| A customer named MWRA left (could leave) in a way that is relevant to revenue concentration; the statement asks how much burn improvement depends on revenue that could leave as MWRA did. | VC | opinion | · not_checked |  |  |  |  |
| The question of what drove the improvement in burn between the two dates cannot be answered from the statement alone and is an opinion about missing analysis rather than a verified fact. | VC | opinion | · not_checked |  |  |  |  |
| If a raise were sized from this base, funding priorities and cuts would need to be decided, and the first milestone to slip would be the one most dependent on the revenue base, which is a forecast. | VC | forecast | · not_checked |  |  |  |  |
| FY2024 gross profit was $2.30M, equal to $6.35M revenue minus $4.05M service cost (synthetic scenario record, not actual Biobot data). | CFO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_historical_operating_trajectory_08` | JEV jev-1.14.0: evidence covers it 98% |
| Base runway of about 38.0 months ($12.8M divided by $336,630 per month) is a best case at constant burn, before working capital, and assumes the revenue base holds. | CFO | number | ✅ supported | 0.84 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08` | JEV jev-1.14.0: evidence covers it 89% |
| The earlier claim that no raise is needed to survive about three years was too strong and is corrected. | CFO | opinion | · not_checked |  |  |  |  |
| Growjo's $3.5M revenue estimate (unverified, weak source) has not been reconciled with the synthetic implied revenue of about $14.4M, a gap of roughly 4x. | CFO | number | ✅ supported | 0.75 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_01` | Growjo $3.5M unverified; implied $14.38M; ratio ~4.1x; no reconciliation shown. |
| Implied synthetic revenue of about $14,384,000 equals annualized gross profit plus service cost ($8,260,440 + $6,123,560). | CFO | number | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_revenue_and_retention_05` | JEV jev-1.14.0: evidence covers it 97% |
| Actual revenue should be confirmed before any raise is sized. | CFO | opinion | · not_checked |  |  |  |  |
| Funding retention and win/loss work, contract data-rights review, the paid-analytics test with a lab-owning agency, and the existing core service is the preferred use of any raise. | CFO | opinion | · not_checked |  |  |  |  |
| Sales, research and corporate expense not tied to those tests should be cut or deferred, but the evidence gives no breakdown of which lines or how much. | CFO | opinion | · not_checked |  |  |  |  |
| The paid analytics agreement from a lab-owning agency is the first milestone likely to slip, because it is the riskiest and depends on a buyer's decision with no threshold or pricing data. | CFO | forecast | · not_checked |  |  |  |  |
| The most important assumption is that gross profit holds near $8.26M; if it does not, burn and runway change quickly. | CFO | opinion | · not_checked |  |  |  |  |
| The synthetic packet is scenario ground truth, not actual Biobot data. | CFO | opinion | · not_checked |  |  |  |  |
| FY2024 opex was $9.30M and the FY2024 operating loss was $7.00M (synthetic scenario record). | CFO | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_historical_efficiency_and_03` | JEV jev-1.14.0: evidence covers it 97% |
| The FY2024 operating loss is compared with the 2026 annualized burn as like for like, with both before working-capital changes and financing. | CFO | opinion | · not_checked |  |  |  |  |
| Any raise would be on top of the $12.8M; the evidence does not say whether 2025 proceeds are included in that figure. | CFO | opinion | · not_checked |  |  |  |  |
| The 2026-09-30 annualized gross profit was $8,260,440 after service cost of $6,123,560 (synthetic scenario record). | CFO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 98% |
| The 2026-09-30 annualized opex was $12.3M and annualized burn was about $4,039,560 a year (synthetic scenario record). | CFO | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_05`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 98% |
| Annual burn fell by about $2.96M because gross profit rose by about $5.96M while opex rose by $3.0M. | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_pricing_and_unit_08` | GP 2.3→8.26M (+5.96M); opex 9.3→12.3M (+3.0M); burn 7.0→4.04M (−2.96M). |
| The records do not say why gross profit rose (price, volume, mix or new customers), so the improvement cannot be attributed. | CFO | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_pricing_and_unit_05` | Records give totals only, no price/volume/mix decomposition. |
| Revenue by customer is not visible in the evidence, so MWRA or any single customer's share of revenue cannot be sized. | CFO | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_06`, `biobot_public_contracts_customers_and_06` | No revenue-by-customer figures; MWRA contract value only reported, not revenue share. |
| In the Devil's stress case, a 25% fall in gross profit lifts annual burn to about $6.10M and cuts runway to about 25 months; this case is illustrative, not evidenced. | CFO | forecast | · not_checked |  |  |  |  |
| The stress case burn of $6,104,670 a year equals $508,723 per month, and $12.8M divided by that gives about 25.2 months of runway. | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | Stress case burn of $6,104,670 not in evidence; only base 4.04M. |
| FY2024 burn proxy: $2,300,000 - $9,300,000 = -$7,000,000; 2026 annualized: $8,260,440 - $12,300,000 = -$4,039,560; improvement = $7,000,000 - $4,039,560 = $2,960,440 per year; gross profit change = $8,260,440 - $2,300,000 = $5,960,440; opex change = $12,300,000 - $9,300,000 = $3,000,000; $5,960,440 - $3,000,000 = $2,960,440; base runway = $12,800,000 / $336,630 = about 38.0 months; stress: $8,260,440 x 25% = $2,065,110; $4,039,560 + $2,065,110 = $6,104,670 per year = $508,723 per month; $12,800,000 / $508,723 = about 25.2 months; implied revenue = $8,260,440 + $6,123,560 = $14,384,000 | CFO | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The CFO's attribution of the burn improvement is the weakest point in the answer. | DEVIL | opinion | · not_checked |  |  |  |  |
| A 3.6x jump in gross profit in under two years needs an identified driver. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO admits she cannot name a driver for the gross profit change (price, volume, mix, new customers). | DEVIL | opinion | · not_checked |  |  |  |  |
| Without an identified driver, the CFO cannot say whether the revenue is recurring. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's response to the VC's question about revenue that could leave uses a 25% stress case borrowed from the Devil. | DEVIL | fact | ❔ unverifiable | 0.91 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 7% |
| The 25% stress case is arbitrary and not evidenced. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO holds service cost fixed at $6.12M in the stress case. | DEVIL | fact | ✅ supported | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05` | $6,123,560 service cost used in burn; stress case itself not described. |
| Service cost rose from $4.05M to $6.12M. | DEVIL | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_pricing_and_unit_05` | FY2024 $4.05M; annualized 2026 $6,123,560. |
| Implied 2026 revenue is about $14.4M versus $6.35M previously, meaning the revenue base more than doubled. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03` | $14.384M vs $6.35M is 2.27x, more than doubled. |
| Revenue growth of about 2.3x makes dependence on a few large customers plausible. | DEVIL | forecast | · not_checked |  |  |  |  |
| The 'does NOT add up' flag matters because the improvement figure compares a loss before financing with a burn before working capital. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's answer states gross profit rose $5.96M and opex rose $3.0M. | DEVIL | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_pricing_and_unit_08` | Figures match computed differences; whether CFO stated them is not directly shown. |
| The $2.96M figure is a proxy rather than a measured cash change. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's 'fund retention and win/loss work' is not a sized use of funds, since no cost is given for it. | DEVIL | opinion | · not_checked |  |  |  |  |
| The question of what a smaller raise would fund is unanswered. | DEVIL | opinion | · not_checked |  |  |  |  |
| Gross profit growth is about 3.6x, calculated as $8,260,440 / $2,300,000. | DEVIL | number | ✅ supported | 0.92 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_03` | 8,260,440/2,300,000 = 3.59. |
| Implied 2026 revenue is $14,384,000, calculated as $8,260,440 + $6,123,560. | DEVIL | number | ✅ supported | 0.92 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05` | 8,260,440 + 6,123,560 = 14,384,000, matching contracted ARR. |
| Revenue growth is about 2.3x, calculated as $14,384,000 / $6,350,000. | DEVIL | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_revenue_and_retention_07` | 14.384/6.35 = 2.27x, about 2.3x. |
| Service cost growth is about 1.5x, calculated as $6,123,560 / $4,050,000. | DEVIL | number | ✅ supported | 0.92 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_historical_efficiency_and_03` | 6,123,560/4,050,000 = 1.51x. |
| The 2026 annualized figure may rest on a short period, since the evidence does not state the base period. | DEVIL | forecast | · not_checked |  |  |  |  |
| Large revenue growth could mean customer concentration, since no customer-level data is shown. | DEVIL | forecast | · not_checked |  |  |  |  |
| Implied revenue equals gross profit plus service cost, as the synthetic record appears to define it. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CFO's answer treats FY2024 and the 2026 annualized figures as like for like. | DEVIL | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_pricing_and_unit_08` | FY2024 actual vs annualized 2026 run-rate compared directly; different time bases. |
| FY2024 and the 2026 annualized figures are not comparable. | DEVIL | opinion | · not_checked |  |  |  |  |
| FY2024 is an audited-style full-year actual in the scenario record. | DEVIL | fact | ❔ unverifiable | 0.65 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_operating_trajectory_08` | Full-year amounts shown, but nothing mentions audit status. |
| The 2026 figure is an annualization of a single date, 2026-09-30. | DEVIL | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | 2026 figures annualized as of 2026-09-30 snapshot. |
| The evidence does not say what period the 2026 figure annualizes. | DEVIL | fact | ❌ contradicted | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | Annualization basis given: 886 sites, 24 samples/yr, as of 2026-09-30. |
| If the 2026 figure extrapolates a strong quarter or month, the $8.26M gross profit could be a run-rate peak rather than a sustainable level. | DEVIL | forecast | · not_checked |  |  |  |  |
| Gross profit rose from $2.30M to $8.26M, a 3.6x jump, in under two years. | DEVIL | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_pricing_and_unit_05` | 2.30M to 8.26M is 3.59x, end-2024 to Sept 2026. |
| $8,260,440 / $2,300,000 = about 3.6x gross profit growth; $8,260,440 + $6,123,560 = $14,384,000 implied 2026 revenue; $14,384,000 / $6,350,000 = about 2.3x revenue growth; $6,123,560 / $4,050,000 = about 1.5x service cost growth | DEVIL | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The statement asks what new capital would be spent on, split by category such as sales, research, sampling and lab operations, and corporate. | VC | opinion | · not_checked |  |  |  |  |
| The statement asks, for each spending category, which dated milestone it must reach before the next round. | VC | opinion | · not_checked |  |  |  |  |
| The statement asks for observable pass or fail evidence for each category's milestone. | VC | opinion | · not_checked |  |  |  |  |
| The statement asks that documented records be separated from internal plans and opinion, and that the speaker state whether these plans cannot be seen. | VC | opinion | · not_checked |  |  |  |  |
| The speaker cannot see any documented spend plan, budget split or dated milestones in the evidence. | CFO | opinion | · not_checked |  |  |  |  |
| Test 1 fails if revenue by customer or contract terms cannot be produced. | CFO | forecast | · not_checked |  |  |  |  |
| The evidence shows a 33-renewal cohort: 24 renewed with a dedicated budget line and 9 on time-limited grant money (biobot_dummy_procurement_and_sales_04, synthetic, 2026-09-30). | CFO | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_procurement_and_sales_04` | JEV jev-1.14.0: evidence covers it 98% |
| The pass test for Test 1 should also report how much of the 9 grant-funded renewals renewed again. | CFO | opinion | · not_checked |  |  |  |  |
| Test 2 (sales paid-analytics test with lab-owning agencies) has a proposed budget of $400,000. | CFO | forecast | · not_checked |  |  |  |  |
| Test 2 has an assumed milestone date of 2027-06-30. | CFO | forecast | · not_checked |  |  |  |  |
| Test 2 passes if at least one signed paid agreement is obtained with a stated price and term; this threshold is the speaker's own and not evidenced. | CFO | forecast | · not_checked |  |  |  |  |
| Test 2 fails if no paid agreement is signed or only free pilots are obtained. | CFO | forecast | · not_checked |  |  |  |  |
| The evidence states that a free pilot is not a paying program (biobot_dummy_historical_operating_trajectory_01). | CFO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_historical_operating_trajectory_01` | JEV jev-1.14.0: evidence covers it 97% |
| Test 3 (research validation of the modeled network against measured data) has a proposed budget of $250,000. | CFO | forecast | · not_checked |  |  |  |  |
| Test 3 has an assumed milestone date of 2027-06-30. | CFO | forecast | · not_checked |  |  |  |  |
| The evidence contains only one cost split: annualized sales, research and corporate opex of $12,300,000 as a single line. | CFO | fact | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_04`, `biobot_dummy_historical_efficiency_and_03` | Evidence also gives FY2024 and FY2025 opex figures, so it is not the only split. |
| Test 3 passes if published error figures against measured sites with an agreed tolerance are produced, and fails if no validation figures can be produced. | CFO | forecast | · not_checked |  |  |  |  |
| Sampling and lab operations and the core service receive no new capital and are funded by gross profit as already modeled; sizing is impossible without a breakdown. | CFO | opinion | · not_checked |  |  |  |  |
| The total of the three tests is $800,000. | CFO | number | ❔ unverifiable | 0.81 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 11% |
| $800,000 equals about 2.4 months of the current burn at $336,630 per month. | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | 800,000/336,630 ≈ 2.38 months using the stated monthly burn. |
| The $800,000 total is the cost of the tests only and is not a recommended raise. | CFO | opinion | · not_checked |  |  |  |  |
| Only the cash, burn and opex lines are documented in the evidence. | CFO | fact | ❔ unverifiable | 0.00 | claude-sonnet-5-5 (escalated) |  | No valid supporting citation |
| The speaker cannot verify actual spend by category, team size, or whether any of the proposed work is already underway. | CFO | opinion | · not_checked |  |  |  |  |
| Any raise sized later should follow these test results and be checked against the unreconciled gross profit base, since 2026 run-rate gross profit cannot be reconciled to a driver. | CFO | opinion | · not_checked |  |  |  |  |
| All category amounts, dates and pass thresholds are the speaker's assumptions. | CFO | opinion | · not_checked |  |  |  |  |
| The synthetic packet is treated as scenario ground truth, not actual Biobot data. | CFO | opinion | · not_checked |  |  |  |  |
| The evidence contains a $6,123,560 service cost (biobot_dummy_pricing_and_unit_08, synthetic, 2026-09-30). | CFO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_pricing_and_unit_05` | JEV jev-1.14.0: evidence covers it 97% |
| Monthly burn of $336,630 holds. | CFO | forecast | · not_checked |  |  |  |  |
| The evidence does not break opex into sales, research, sampling/lab or corporate categories. | CFO | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08`, `biobot_dummy_historical_efficiency_and_04` | Opex is only given as combined sales, research and corporate lump sums. |
| The amounts in the statement are the speaker's opinion and illustrative, not a plan, and the speaker will not state a raise size. | CFO | opinion | · not_checked |  |  |  |  |
| No raise is sized in the statement; the speaker would fund only the tests that decide whether to raise and size a raise from their results. | CFO | opinion | · not_checked |  |  |  |  |
| Test 1 (retention and win/loss with contract data-rights review, corporate category) has a proposed budget of $150,000. | CFO | forecast | · not_checked |  |  |  |  |
| Test 1 has an assumed milestone date of 2027-03-31. | CFO | forecast | · not_checked |  |  |  |  |
| Test 1 passes if every customer whose contract ended or was due in the cohort is coded renew or lost with a reason, and the data-rights status of each contract is documented. | CFO | forecast | · not_checked |  |  |  |  |
| The CFO's $800k test budget is claimed to equal about 2.4 months of the current burn. | DEVIL | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | The claim is that CFO asserted 2.4 months; arithmetic matches burn. |
| The evidence shows municipal and county buyers rely on appropriations and state buyers rely on grant mixes. | DEVIL | fact | ✅ supported | 0.92 | jev:jev-latest | `biobot_dummy_procurement_and_sales_04` | JEV jev-1.14.0: evidence covers it 94% |
| A buyer's budget cycle may not fit a 2027-06-30 milestone, a date the CFO also assumed. | DEVIL | forecast | · not_checked |  |  |  |  |
| The pass threshold of one signed agreement is the CFO's own and is weak. | DEVIL | opinion | · not_checked |  |  |  |  |
| A single contract would not show repeatable demand. | DEVIL | opinion | · not_checked |  |  |  |  |
| A single signed contract would pass the test even if the 9 grant-funded renewals were not renewed again. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CFO's item (4) assumes zero new capital for sampling and lab operations. | DEVIL | fact | ❔ unverifiable | 0.82 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 11% |
| The CFO admits she cannot see the breakdown of the $6.12M service cost or the $12.3M opex. | DEVIL | fact | ❔ unverifiable | 0.77 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 9% |
| The CFO cannot know that the tests would not draw on existing lab capacity. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $800k has no support as a cost and the milestone logic is not sized. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $800k cannot answer the VC's request for a spend split. | DEVIL | opinion | · not_checked |  |  |  |  |
| None of the $150k, $400k or $250k test budget figures has evidence behind it. | DEVIL | opinion | · not_checked |  |  |  |  |
| $150,000 + $400,000 + $250,000 = $800,000. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| $800,000 / $336,630 per month is about 2.4 months. | DEVIL | number | ✅ supported | 0.93 | claude-sonnet-5-5 (escalated) | `biobot_dummy_pricing_and_unit_08` | 800,000 / 336,630 = 2.38, about 2.4 months. |
| The CFO's category amounts, dates and thresholds are her own assumptions, not evidenced. | DEVIL | opinion | · not_checked |  |  |  |  |
| A paid-analytics test likely costs more than $400k. | DEVIL | forecast | · not_checked |  |  |  |  |
| Buyer budget cycles may not fit the 2027-06-30 milestone. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CFO admits the $150k, $400k and $250k figures are illustrative. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $800k total is presented as the cost of deciding whether to raise capital. | DEVIL | fact | ❔ unverifiable | 0.84 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 11% |
| The $800k total has no costing basis. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 38-month burn was previously conceded by the CFO to be a best case that is not stress-tested. | DEVIL | opinion | · not_checked |  |  |  |  |
| Framing the tests against burn makes them look cheap relative to a figure the CFO has called unreliable. | DEVIL | opinion | · not_checked |  |  |  |  |
| A credible paid-analytics test with a lab-owning agency needs sales cycle time, pilot support, and lab and analytic staff. | DEVIL | opinion | · not_checked |  |  |  |  |
| $400k for a single signed agreement is likely understated. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker was asked which single milestone the next investor will judge them on, and how likely it is to be hit by its date. | VC | opinion | · not_checked |  |  |  |  |
| The question asks the speaker to state a metric, a threshold and a date for that milestone. | VC | opinion | · not_checked |  |  |  |  |
| The median municipal or county sales cycle is 163 days. | VC | number | ✅ supported | 0.94 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_historical_efficiency_and_05`, `biobot_dummy_historical_efficiency_and_02`, `biobot_dummy_historical_efficiency_and_01` | JEV jev-1.14.0: evidence covers it 96% |
| The median state or federal sales cycle is 318 days. | VC | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03` | JEV jev-1.14.0: evidence covers it 97% |
| The question asks whether the milestone date leaves enough time to sign and show renewal given these sales cycle lengths. | VC | opinion | · not_checked |  |  |  |  |
| The question asks the speaker to separate documented-record evidence from their own estimate in the answer. | VC | opinion | · not_checked |  |  |  |  |
| The speaker's expected milestone for the next investor is a signed paid annual agreement from a lab-owning agency plus retention evidence by 2027-06-30; this is the speaker's opinion and not documented. | CFO | opinion | · not_checked |  |  |  |  |
| A 318-day state or federal cycle starting 2026-10-31 would close around 2027-09-14. | CFO | number | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | 2026-10-31 plus 318 days is 2027-09-15, so about 2027-09-14. |
| A 163-day signing from 2026-10-31 ends about 2027-04-12, so a first annual renewal would fall about a year later and cannot be shown by 2027-06-30. | CFO | number | ✅ supported | 0.75 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | 163 days from 2026-10-31 is 2027-04-12; renewal a year later is after June 2027. |
| A deal signed by 2027-06-30 would be evidence of sales, not retention. | CFO | opinion | · not_checked |  |  |  |  |
| The existing cohort contains 24 renewals with a dedicated budget line and 9 renewals on time-limited grant money, totaling 33 renewals. | CFO | number | ✅ supported | 0.95 | jev:jev-latest | `biobot_dummy_procurement_and_sales_04`, `biobot_dummy_revenue_and_retention_04` | JEV jev-1.14.0: evidence covers it 98% |
| The renewal count of 33 covers only renewals; non-renewals are not visible, so a renewal rate cannot be stated. | CFO | fact | ❌ contradicted | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_revenue_and_retention_04` | Evidence shows 35 due, 33 renewed, 2 churned: 94.3% renewal. |
| The earlier $400k test cost and 2.4-month comparison were unsupported illustrations based on a burn figure the speaker had called a best case, and the speaker withdraws them as costing. | CFO | opinion | · not_checked |  |  |  |  |
| The most important assumption is that discovery is already qualified for any target agency; if it is not, the clock starts later and the date is missed. | CFO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify any pipeline, whether lab-owning agencies are in discovery, or whether a raise has already closed. | CFO | opinion | · not_checked |  |  |  |  |
| The clock starts 2026-10-31 with discovery assumed already qualified; the evidence does not say any agency is in discovery. | CFO | forecast | · not_checked |  |  |  |  |
| The median cycle is applied to a single deal, and actual deals vary around the median. | CFO | forecast | · not_checked |  |  |  |  |
| The milestone threshold is at least 2 signed paid annual agreements with a stated price and term, corrected from an earlier threshold of 1. | CFO | opinion | · not_checked |  |  |  |  |
| A lab-owning agency buys on the municipal or county cycle; this is unknown and could instead be the state or federal cycle. | CFO | forecast | · not_checked |  |  |  |  |
| An annual agreement means renewal is due about 12 months after signing. | CFO | forecast | · not_checked |  |  |  |  |
| The synthetic packet is treated as scenario ground truth and not as actual Biobot data. | CFO | forecast | · not_checked |  |  |  |  |
| The speaker's threshold of 2 signed paid agreements, the date 2027-06-30, and the low likelihood of achieving them are the speaker's opinion and are not evidenced. | CFO | opinion | · not_checked |  |  |  |  |
| The speaker gives no percentage likelihood because the evidence contains no win-rate or pipeline data. | CFO | fact | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_market_sizing_scenario_04` | Evidence does contain pipeline data: 54 opportunities, $7.6M unweighted, $2.9M weighted. |
| The median sales cycle is 163 days for municipal or county buyers and 318 days for state or federal buyers, measured from qualified discovery to signed annual agreement. | CFO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_historical_efficiency_and_05` | JEV jev-1.14.0: evidence covers it 97% |
| A pilot is a separate paid stage and is not proof of renewal. | CFO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_procurement_and_sales_04` | JEV jev-1.14.0: evidence covers it 96% |
| From 2026-10-31 to 2027-06-30 is 242 days. | CFO | number | ❔ unverifiable | 0.76 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 15% |
| The 242-day window leaves 79 days of slack against the 163-day municipal or county median. | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | 242 minus 163 equals 79; municipal median is 163. |
| The 242-day window is 76 days short of the 318-day state or federal median. | CFO | number | ✅ supported | 0.90 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | 318 minus 242 equals 76; median confirmed. |
| 2026-10-31 to 2027-06-30 = 30 (Nov) + 31 (Dec) + 31 (Jan) + 28 (Feb) + 31 (Mar) + 30 (Apr) + 31 (May) + 30 (Jun) = 242 days; 242 - 163 = 79 days slack (municipal or county); 163 - 242 = -79 means signing fits; 318 - 242 = 76 days late (state or federal); 24 + 9 = 33 renewed contracts | CFO | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The CFO says 242 days fits a 163-day municipal median with 79 days of slack. | DEVIL | opinion | · not_checked |  |  |  |  |
| If no agency is qualified today, a 163-day cycle from a 2027-01 qualification lands around 2027-06-12 at the median, leaving almost no margin. | DEVIL | forecast | · not_checked |  |  |  |  |
| For state or federal buyers the 242-day window is missed. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO treats a lab-owning agency as municipal or county without evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| A lab-owning agency is plausibly state-level, which would place it on the 318-day cycle and make the milestone impossible. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's low likelihood conclusion is probably right, but her reasoning that one municipal deal fits is unsupported. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's milestone drops retention ('evidence of sales, not retention') while the metric still names retention evidence, so the metric and threshold do not match. | DEVIL | fact | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) |  | Evidence does not include the CFO's milestone or metric wording. |
| The CFO cites a 33-renewal cohort (24 dedicated budget, 9 grant) but cannot state a renewal rate. | DEVIL | opinion | · not_checked |  |  |  |  |
| The FY2022 cohort in the evidence shows 10 of 11 renewed, a 90.9% renewal rate (biobot_dummy_historical_contract_cohorts_04, 2022-12-31). | DEVIL | number | ✅ supported | 0.98 | jev:jev-latest | `biobot_dummy_historical_contract_cohorts_04` | JEV jev-1.14.0: evidence covers it 98% |
| The FY2022 renewal figure is four years old and cannot stand in for the 2026 renewal rate. | DEVIL | opinion | · not_checked |  |  |  |  |
| The municipal median procurement cycle is 163 days, and the state or federal cycle is 318 days (documented in the evidence). | DEVIL | number | ✅ supported | 0.97 | jev:jev-latest | `biobot_dummy_procurement_and_sales_03`, `biobot_dummy_historical_efficiency_and_05`, `biobot_dummy_historical_efficiency_and_02` | JEV jev-1.14.0: evidence covers it 97% |
| The CFO says the date leaves enough time to sign at most one municipal or county deal. | DEVIL | opinion | · not_checked |  |  |  |  |
| The span from 2026-10-31 to 2027-06-30 is 242 days. | DEVIL | number | ❔ unverifiable | 0.77 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 11% |
| 242 minus 163 equals 79 days of slack at the median. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| 318 minus 242 equals 76 days late for a state or federal buyer. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| A lab-owning agency may be a state-level buyer on the 318-day cycle. | DEVIL | forecast | · not_checked |  |  |  |  |
| Roughly half of deals take longer than the median cycle. | DEVIL | forecast | · not_checked |  |  |  |  |
| Discovery may not yet be qualified for any target agency, since the evidence shows no pipeline. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CFO's stated threshold is at least 2 signed agreements. | DEVIL | fact | ❔ unverifiable | 0.84 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 9% |
| The CFO's claim that 242 days fits the 163-day median conflicts with her own threshold of at least 2 signed agreements. | DEVIL | opinion | · not_checked |  |  |  |  |
| A median is the midpoint of a distribution, so about half of deals take longer than the median cycle. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) |  | Evidence states medians but not the statistical definition; general knowledge only. |
| The CFO gives no distribution of procurement cycle times, and the evidence shows only medians. | DEVIL | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | Only median cycle figures appear; no distribution given. |
| The 79 days of slack at the median is not a safety margin. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 2026-10-31 clock start assumes discovery is already qualified. | DEVIL | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_03` | Cycle is measured from qualified discovery, so the start implies qualification. |
| The CFO concedes she cannot see any pipeline. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker asks what the single biggest risk to the business is, choosing among access to sewer sites and samples, data privacy, regulation, team, or funding. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for an observable early-warning signal for the biggest risk, the threshold indicating it is materializing, and the planned response. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks that documented records be separated from the speaker's own estimate and that it be stated if the underlying data cannot be seen. | VC | opinion | · not_checked |  |  |  |  |
| The speaker's opinion is that revenue durability is the biggest risk, meaning buyers who pay with time-limited or appropriated money stop paying. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify the 2026 access, regulatory or team position from the evidence shown. | CEO | opinion | · not_checked |  |  |  |  |
| The early-warning signal proposed is the share of expiring contracts that renew, split by dedicated budget versus grant funding. | CEO | forecast | · not_checked |  |  |  |  |
| A grant-funded cohort failing to renew a second time, or a cohort where the number of contracts coded 'lost' is not reported, should trigger concern. | CEO | forecast | · not_checked |  |  |  |  |
| A single large customer leaving would be a second trigger, but revenue by customer is not visible to the speaker. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot state a renewal rate because non-renewals are not visible. | CEO | opinion | · not_checked |  |  |  |  |
| If the risk materializes, the response is to code every lost or at-risk contract with a reason, move grant-funded customers onto appropriated budget lines, and hold hiring and spend until the gross profit driver is explained. | CEO | forecast | · not_checked |  |  |  |  |
| The gross profit figure rose from $2.30M to $8.26M, and the driver of this jump is still unreconciled. | CEO | number | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_dummy_historical_efficiency_and_03`, `biobot_dummy_pricing_and_unit_05`, `biobot_dummy_pricing_and_unit_08` | Figures match, but driver is reconciled: ARR/site-based run-rate vs FY2024 reported; 'unreconciled' unsupported. |
| The speaker would not claim the risk is controlled until revenue by customer and contract terms can be produced. | CEO | opinion | · not_checked |  |  |  |  |
| Ranking revenue durability as the biggest risk is the speaker's opinion. | CEO | opinion | · not_checked |  |  |  |  |
| The renewal-cohort split and gross profit figures come from earlier turns, not from the chunks currently displayed. | CEO | opinion | · not_checked |  |  |  |  |
| Ratti flagged the 'who pays' risk in 2017 and it is still open. | CEO | opinion | · not_checked |  |  |  |  |
| The warning signal and threshold are the speaker's own proposals and are not evidenced. | CEO | opinion | · not_checked |  |  |  |  |
| Customer concentration is unverified because revenue by customer cannot be seen. | CEO | opinion | · not_checked |  |  |  |  |
| Ratti warned the founders in 2017 that 'you don't know who's going to pay for this data yet'. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_fact_ratti_who_pays`, `biobot_case_p12_c4`, `biobot_fact_buyer_unclear` | JEV jev-1.14.0: evidence covers it 96% |
| In 2017, selling to governments with tight budgets and burdensome purchasing was seen as hard. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_fact_access_and_procurement`, `biobot_case_p12_c4`, `biobot_case_p9_c5`, `biobot_case_p1_c3` | JEV jev-1.14.0: evidence covers it 97% |
| The only 2026 renewal evidence the speaker has is that 24 of 33 contracts renewed on a dedicated budget line and 9 renewed on time-limited grant money. | CEO | number | ✅ supported | 0.81 | jev:jev-latest | `biobot_dummy_procurement_and_sales_04` | JEV jev-1.14.0: evidence covers it 92% |
| The 2026 renewal evidence is from a synthetic source cited from earlier turns and not displayed in the current chunks. | CEO | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) |  | Claim is about what chunks display; evidence doesn't address provenance of turns. |
| The speaker ranks revenue durability above site access and privacy. | CEO | opinion | · not_checked |  |  |  |  |
| Site access is a relationship problem the team has handled before, as shown in a 2017 case. | CEO | opinion | · not_checked |  |  |  |  |
| The privacy page states that community data do not identify households. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_business_and_market_05` | JEV jev-1.14.0: evidence covers it 96% |
| The CEO ranks revenue durability as the biggest risk. | DEVIL | fact | ❔ unverifiable | 0.86 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 9% |
| The CEO cannot see revenue by customer. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO waves away customer concentration as 'unverified'. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's argument that access is handled rests on chunk biobot_case_p6_c2, which is not displayed and cannot be verified by the speaker. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's action plan of moving grant customers onto appropriated lines assumes buyers will convert. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 24 renewals show some grant customers converted, but nothing shows the remaining 9 can or will convert. | DEVIL | number | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_dummy_procurement_and_sales_04` | 24 dedicated-budget, 9 grant-funded renewals; no evidence on their conversion. |
| The public caveat biobot_public_research_findings_and_03 says public sources do not establish retention or active customer counts. | DEVIL | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_public_research_findings_and_03`, `biobot_public_company_and_offering_03` | JEV jev-1.14.0: evidence covers it 91% |
| The biggest risk is substitution and loss of large public customers. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's signal would detect customer loss and substitution too late. | DEVIL | opinion | · not_checked |  |  |  |  |
| Customer loss and in-house substitution being the larger risk is the speaker's inference and is not evidenced. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's cited chunk biobot_case_p6_c2 and the renewal cohort are not displayed, so they cannot be verified. | DEVIL | opinion | · not_checked |  |  |  |  |
| The displayed public source biobot_public_business_and_market_03 (dated 2024-05-01) says the CDC program was material in size. | DEVIL | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 93% |
| The public source does not show the full 2026 customer picture, as it states itself. | DEVIL | forecast | · not_checked |  |  |  |  |
| The displayed public source biobot_public_business_and_market_03 records a later Verily award. | DEVIL | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_competitors_alternatives_and_02` | JEV jev-1.14.0: evidence covers it 91% |
| The displayed public source biobot_public_business_and_market_03 says the MWRA confirms the Massachusetts Biobot data stream ended in July 2026. | DEVIL | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The CEO does not cite the public source biobot_public_business_and_market_03 in the meeting. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO relies on 2017 quotes about who pays and tight budgets, which are nine years old. | DEVIL | fact | ✅ supported | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_fact_ratti_who_pays`, `biobot_fact_access_and_procurement` | Both quotes dated 2017, about nine years before 2026. |
| The CEO's 24-of-33 renewal split is not displayed in the evidence block and cannot show non-renewals. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's warning signal 'Any grant-funded cohort that fails to renew a second time' fires only after a loss and is therefore lagging rather than early. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's warning signal 'Any cohort where lost contracts are not reported' measures disclosure rather than the business. | DEVIL | opinion | · not_checked |  |  |  |  |
| The Devil points to the Massachusetts loss as a materialized risk. | VC | opinion | · not_checked |  |  |  |  |
| The MWRA says its Biobot testing ended at the end of July 2026 as the state moved testing in-house. | VC | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_03`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The Boston Globe reported that the expiring Massachusetts Biobot contract was worth $400,000 per month. | VC | number | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 96% |
| The Boston Globe reported that a replacement for the expiring Massachusetts Biobot contract was expected to cost about one-quarter of the original contract value, roughly $100,000 per month. | VC | number | ✅ supported | 0.84 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 92% |
| The public sources cited do not establish current revenue or the number of active customers. | VC | opinion | · not_checked |  |  |  |  |
| The speaker concedes the Devil's point that the Massachusetts loss is documented, current and larger evidence than the speaker's 2017 quotes. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify revenue by customer, active customers or contract end dates from the evidence shown. | CEO | opinion | · not_checked |  |  |  |  |
| A customer-by-customer table of revenue, contract end date and in-house capability should be produced within 30 days before funding anything. | CEO | forecast | · not_checked |  |  |  |  |
| Offering large public customers a lower-priced tier or analytics layer could keep them buying even if they move sampling in-house. | CEO | forecast | · not_checked |  |  |  |  |
| The Globe's one-quarter replacement cost suggests price pressure, not only loss of customers. | CEO | opinion | · not_checked |  |  |  |  |
| The evidence has no cost data, so the speaker gives no dollar figure for the mitigation. | CEO | fact | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence does contain cost data: $400,000/month contract, replacement about one-quarter the cost. |
| A large customer's contract within 12 months of end with no renewal discussion, an RFP for in-house or competing testing, or lab staff hiring or equipment purchases would be a leading signal before expiry. | CEO | forecast | · not_checked |  |  |  |  |
| Escalation is triggered when a customer above an agreed revenue share shows a lab-build signal or has no renewal meeting 6 months before expiry. | CEO | forecast | · not_checked |  |  |  |  |
| The share-of-revenue cutoff for escalation is not evidenced and would be set once revenue by customer is known. | CEO | opinion | · not_checked |  |  |  |  |
| The Massachusetts case generalises to other public customers is the most important assumption, and nothing in the evidence shows it. | CEO | opinion | · not_checked |  |  |  |  |
| The reported replacement at one-quarter cost implies roughly $100,000 per month of retained spend. | CEO | number | ❌ contradicted | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Replacement is the state's in-house program; Biobot's testing ended, so spend isn't retained. |
| The speaker concedes that substitution of large public customers is at least co-equal with revenue durability. | CEO | opinion | · not_checked |  |  |  |  |
| The gap between the $400,000 per month contract and the roughly $100,000 per month replacement is about $300,000 per month in program value. | CEO | number | ✅ supported | 0.80 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 93% |
| The Globe's one-quarter replacement cost applies to the whole program and is not Biobot revenue or margin. | CEO | forecast | · not_checked |  |  |  |  |
| The MWRA case may not generalise to other public customers. | CEO | forecast | · not_checked |  |  |  |  |
| The lower-priced tier or analytics layer is the speaker's proposal, with no evidence buyers would pay for it. | CEO | opinion | · not_checked |  |  |  |  |
| Signal thresholds such as 6 and 12 months are the speaker's own proposals and are not evidenced. | CEO | opinion | · not_checked |  |  |  |  |
| Revenue by customer is not visible, so the share of revenue exposed to in-house build is unknown. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker's earlier answer relied on 2017 chunks (biobot_fact_ratti_who_pays, biobot_fact_access_and_procurement) for the 'who pays' and tight-budget risks. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker did not use the 2026 MWRA evidence in the earlier answer, which the speaker calls an omission. | CEO | opinion | · not_checked |  |  |  |  |
| MWRA says Biobot testing ended at the end of July 2026 as the state moved testing in-house. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The Boston Globe reported on 2026-07-04 that the expiring Biobot contract was $400,000 per month, with a replacement expected at about one-quarter of that cost. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The $400,000 per month figure is contract value, not Biobot net revenue or margin. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_dummy_commercial_contract_book_04`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 94% |
| The 2024-05-01 source records CDC program materiality and a later Verily award. | CEO | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 94% |
| The 2024-05-01 source states that CDC and Verily are not the full 2026 customer picture. | CEO | fact | ✅ supported | 0.88 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_03` | Source says these show risk, not the company's complete 2026 customer situation. |
| The CEO's mitigation is a lower-priced tier or analytics layer sold to customers who are leaving to build in-house. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $300,000/month gap is a weak input because the Globe figure covers the state's replacement program, not Biobot revenue or margin. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO assumes the one-quarter ratio applies to the whole replacement program. | DEVIL | opinion | · not_checked |  |  |  |  |
| A gap in contract value is not a loss of Biobot revenue, and the evidence does not show how much the state would have kept buying. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO concedes she cannot see revenue by customer, so she cannot say whether Massachusetts is an outlier or typical. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 30-day customer table delays any spend. | DEVIL | opinion | · not_checked |  |  |  |  |
| The first signal is only a guess at 'in-house capability' that nothing shows Biobot can observe. | DEVIL | opinion | · not_checked |  |  |  |  |
| The public source states it does not establish the full 2026 customer picture. | DEVIL | fact | ✅ supported | 0.83 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_research_findings_and_03` | JEV jev-1.14.0: evidence covers it 90% |
| $400,000/month x 1/4 = $100,000/month. | DEVIL | number | ✅ supported | 0.95 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | $400,000 times one-quarter equals $100,000 per month; arithmetic matches reported figures. |
| $400,000 - $100,000 = $300,000/month contract-value gap, which is not Biobot revenue. | DEVIL | calculation | ✅ supported | 1.00 | python |  | Arithmetic verified in code; its inputs are the speaker's own |
| A contract is always within 12 months of expiry at some point, so the 12-month signal would fire for every contract. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO's own quantification undercuts the mitigation: she says the Globe's one-quarter replacement cost suggests price pressure, then proposes cutting price. | DEVIL | opinion | · not_checked |  |  |  |  |
| If the buyer's in-house replacement costs about $100,000/month against $400,000/month, a cheaper Biobot tier has to compete with that $100,000/month figure. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO gives no price, cost or evidence that any buyer would pay for analytics, and she admits she cannot cost it. | DEVIL | opinion | · not_checked |  |  |  |  |
| The mitigation is an untested idea, not a plan. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 'within 12 months of end with no renewal discussion' signal fires for nearly every contract because contracts are always within 12 months of expiry at some point, so it does not discriminate. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 'share of revenue' cutoff in the leading signals is left undefined, so the trigger cannot fire. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 6-month meeting threshold likely comes too late, since a state moving in-house needs lead time for procurement, staffing and lab capacity. | DEVIL | forecast | · not_checked |  |  |  |  |
| The MWRA decision was probably made well before the July 2026 end date. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO could not cost the substitution mitigation in the meeting. | VC | opinion | · not_checked |  |  |  |  |
| The CEO could not see revenue by customer in the meeting. | VC | opinion | · not_checked |  |  |  |  |
| The smallest evidence needed to show current revenue held by customers who could move testing in-house is a customer-level revenue extract from billing or accounting records, cross-referenced with customer testing-capability information. | VC | forecast | · not_checked |  |  |  |  |
| The smallest evidence needed to show how much revenue the Massachusetts loss has already removed is a before-and-after customer revenue comparison from billing records covering the periods before and after the Massachusetts loss. | VC | forecast | · not_checked |  |  |  |  |
| Documented records should be kept separate from estimates when presenting this evidence. | VC | opinion | · not_checked |  |  |  |  |
| The speaker proposes producing a customer-level revenue ledger built from billing/invoice records and signed contracts, with one row per customer and columns for trailing-12-month recognised revenue, contract start and end dates, funding source, and an in-house testing capability or procurement flag. | CEO | forecast | · not_checked |  |  |  |  |
| The Globe's one-quarter replacement ratio describes the state's replacement, so applying it to the whole program was an assumption. | CEO | opinion | · not_checked |  |  |  |  |
| The 12-month no-renewal-discussion signal was non-discriminating because every contract passes through that window. | CEO | opinion | · not_checked |  |  |  |  |
| The earlier 6-month threshold was probably too late. | CEO | opinion | · not_checked |  |  |  |  |
| Proposed leading signals include a customer's RFP or procurement notice for testing, lab staffing or equipment purchases, or a falling sample volume, with the trigger set once the ledger gives revenue shares. | CEO | forecast | · not_checked |  |  |  |  |
| The MWRA record shows the move was announced as a shift to in-house testing, and the speaker would ask for the date MWRA first told Biobot. | CEO | opinion | · not_checked |  |  |  |  |
| If Biobot cannot observe in-house capability, only the revenue-at-risk upper bound (all public-sector revenue) can be reported rather than a real exposure. | CEO | opinion | · not_checked |  |  |  |  |
| Billing and contract records can be tied to individual customers and contract end dates. | CEO | forecast | · not_checked |  |  |  |  |
| In-house capability can be assessed from customer disclosures or public information. | CEO | forecast | · not_checked |  |  |  |  |
| The share-of-revenue trigger cutoff is to be set after the ledger exists; no cutoff is evidenced. | CEO | forecast | · not_checked |  |  |  |  |
| The Globe's one-quarter replacement cost applies to the state's program, not to Biobot revenue or margin. | CEO | forecast | · not_checked |  |  |  |  |
| The speaker proposes two numbers from the ledger: revenue from customers flagged as able to build in-house divided by total revenue, and Massachusetts revenue removed (MWRA contract trailing-12-month Biobot revenue and its share of total). | CEO | forecast | · not_checked |  |  |  |  |
| The speaker cannot see the ledger data from the evidence shown, so gives no figure and will not estimate the exposure. | CEO | opinion | · not_checked |  |  |  |  |
| MWRA says Biobot testing ended at the end of July 2026. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The Boston Globe, dated 2026-07-04, put the expiring MWRA contract at $400,000 per month, with a replacement at about one-quarter of that cost. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The $400,000 per month figure is contract value, not net revenue or margin. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_contracts_customers_and_02`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 95% |
| The 2024-05-01 source records CDC materiality and a later Verily award and does not give the full 2026 customer picture. | CEO | fact | ✅ supported | 0.88 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 93% |
| The 2017 sources on who pays and tight budgets are old and secondary. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker withdraws the $300,000 per month gap as a measure of Biobot's loss, because it is a gap in the state's program value rather than lost Biobot revenue. | CEO | opinion | · not_checked |  |  |  |  |
| The CEO's proposed ledger is the weakest point in the analysis. | DEVIL | opinion | · not_checked |  |  |  |  |
| Many customers could build in-house if the savings were large, so a capability flag would mark almost everyone or no one. | DEVIL | forecast | · not_checked |  |  |  |  |
| A price-gap flag would fit the evidence better than a capability flag. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO does not propose a price-gap flag. | DEVIL | fact | ❔ unverifiable | 0.88 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 9% |
| The CEO's second number, revenue removed by Massachusetts, is only a historical share and not an exposure metric. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO does not say whether the MWRA contract was Biobot's largest customer contract, and she cannot see it. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO withdrew the $300,000/month figure. | DEVIL | fact | ❔ unverifiable | 0.85 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | No evidence of a $300,000/month figure or any CEO withdrawal. |
| The CEO kept the same unverified link between the Globe's $400,000/month contract value and Biobot revenue. | DEVIL | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence says $400k/month is contract value, not revenue; CEO's behavior not covered. |
| The evidence indicates the Globe's $400,000/month figure is contract value, not revenue or margin. | DEVIL | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 94% |
| The 2024-05-01 source (biobot_public_business_and_market_03) gives no 2026 customer picture. | DEVIL | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_03` | The chunk itself cites MWRA's July 2026 ending, though it says it is not the complete 2026 picture. |
| The CEO never says who verifies the flags. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's proposed ledger cannot answer the VC's question of how much revenue sits with customers who could move testing in-house. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO never says whether customers would disclose their plans before they leave. | DEVIL | opinion | · not_checked |  |  |  |  |
| The billing and contract records may or may not exist, and the speaker cannot see whether they exist. | DEVIL | opinion | · not_checked |  |  |  |  |
| That many public customers could build in-house if the price gap were large is the speaker's inference and is not evidenced. | DEVIL | opinion | · not_checked |  |  |  |  |
| That most of Biobot's customers are public-sector is the speaker's inference, and the evidence does not give the current customer mix. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's central column is a 'flag for in-house capability' that she admits Biobot may not be able to observe. | DEVIL | fact | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06` | Evidence covers in-house substitution risk but nothing about the CEO's column or admissions. |
| The CEO's fallback bound is 'all public-sector revenue', which would be nearly the whole book if the customers are mostly public. | DEVIL | opinion | · not_checked |  |  |  |  |
| An upper bound of all public-sector revenue would say almost nothing about exposure. | DEVIL | opinion | · not_checked |  |  |  |  |
| A ledger based on trailing-12-month revenue is backward-looking and says nothing about which customers are about to leave. | DEVIL | opinion | · not_checked |  |  |  |  |
| Capability is the wrong test for in-house risk. | DEVIL | opinion | · not_checked |  |  |  |  |
| The MWRA record shows a state moving testing in-house. | DEVIL | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_business_and_market_04` | JEV jev-1.14.0: evidence covers it 95% |
| The Globe's one-quarter replacement cost points to a price-driven decision rather than a capability gap. | DEVIL | opinion | · not_checked |  |  |  |  |
| The Devil argues that a capability flag is the wrong test and a price-gap flag would fit the Massachusetts evidence better. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for a count of current customers paying a per-sample or monthly price that a cheaper in-house alternative could undercut by a similar margin, drawn from billing records, contract terms, or customer-stated pricing. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for the date MWRA first told Biobot it would move testing in-house. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks that documented records be separated from estimates and that the speaker state if these items cannot be seen. | VC | opinion | · not_checked |  |  |  |  |
| The assistant cannot see billing records, contract terms, per-sample or monthly prices, or customer-stated pricing in the evidence shown. | CEO | opinion | · not_checked |  |  |  |  |
| The assistant's trailing-12-month revenue ledger is backward-looking, the MWRA contract is a single case, and the $400,000 figure is not Biobot revenue. | CEO | opinion | · not_checked |  |  |  |  |
| A better ledger would add per customer: price per sample or per month, the customer's alternative price, the contract end date, and the date the customer first signalled a change. | CEO | forecast | · not_checked |  |  |  |  |
| Computing the share of revenue where the price gap is at or above roughly 75% would be a useful measure. | CEO | forecast | · not_checked |  |  |  |  |
| $400,000/month x 1/4 = $100,000/month. | CEO | number | ✅ supported | 0.93 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Contract $400,000/month; replacement about one-quarter the cost gives $100,000/month. |
| ($400,000 - $100,000) / $400,000 = 75% price gap. | CEO | number | ✅ supported | 0.92 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Replacement at one-quarter cost implies a 75% gap; arithmetic is correct. |
| The 75% price gap is the state's replacement program ratio, not a Biobot-specific gap. | CEO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Ratio comes from the state's expiring contract versus replacement program, not Biobot pricing generally. |
| Records needed include invoices and signed contracts for prices, MWRA correspondence for the notice date, and procurement notices or customer statements for alternative prices. | CEO | forecast | · not_checked |  |  |  |  |
| The Globe's one-quarter replacement cost applies to the state's program, not to Biobot revenue or margin. | CEO | forecast | · not_checked |  |  |  |  |
| A 75% price-gap threshold is borrowed from the MWRA case and may not generalise to other customers. | CEO | forecast | · not_checked |  |  |  |  |
| Billing records and contracts can be tied to per-customer prices and end dates (not verified). | CEO | forecast | · not_checked |  |  |  |  |
| The assistant cannot count how many Biobot customers pay a price that a cheaper in-house option could undercut, and declines to estimate it. | CEO | opinion | · not_checked |  |  |  |  |
| Customers disclose or publish their alternative costs; this is the assistant's proposal and is not evidenced, and if false only an upper bound on the price-gap share is possible. | CEO | opinion | · not_checked |  |  |  |  |
| The assistant cannot see the date MWRA first told Biobot it would move in-house. | CEO | opinion | · not_checked |  |  |  |  |
| MWRA testing with Biobot ended at the end of July 2026 as the state moved toward in-house testing. | CEO | fact | ✅ supported | 0.93 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The Boston Globe reported on 2026-07-04 a $400,000/month MWRA contract with a replacement expected at about one-quarter of that cost. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The $400,000/month figure is contract value, not Biobot net revenue or margin. | CEO | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_dummy_commercial_contract_book_04` | JEV jev-1.14.0: evidence covers it 94% |
| The 2024-05-01 source records CDC materiality and a later Verily award and does not give the full 2026 customer picture. | CEO | fact | ✅ supported | 0.87 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 92% |
| The assistant's previous answer relied only on dated records and the 2017 chunks (biobot_fact_ratti_who_pays, biobot_fact_access_and_procurement), which are old and secondary. | CEO | opinion | · not_checked |  |  |  |  |
| A capability flag is a weaker test than price, since the Globe's one-quarter ratio points to price rather than capability. | CEO | opinion | · not_checked |  |  |  |  |
| The CEO's 75% price-gap threshold is the weakest point of the proposal. | DEVIL | opinion | · not_checked |  |  |  |  |
| A single observation from one state is a poor cutoff for other customers. | DEVIL | opinion | · not_checked |  |  |  |  |
| A 75% threshold would miss customers who leave at a 30% or 50% saving. | DEVIL | forecast | · not_checked |  |  |  |  |
| The MWRA case shows only the outcome that Biobot lost, not the saving that would have triggered a switch. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO does not say whether the replacement cost includes labs, staff and capital, or only running costs. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $100,000 per month cost may not be comparable with Biobot's price. | DEVIL | opinion | · not_checked |  |  |  |  |
| The proposed ledger requires customers' alternative prices, which the CEO concedes customers may never disclose. | DEVIL | fact | ❔ unverifiable | 0.77 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 13% |
| If customers do not disclose alternative prices, the flag cannot be built and the measure fails. | DEVIL | forecast | · not_checked |  |  |  |  |
| The proposal rests on a number that was never validated and on data the CEO cannot show exists. | DEVIL | opinion | · not_checked |  |  |  |  |
| The proposal cannot answer the VC's question about current customers' exposure. | DEVIL | opinion | · not_checked |  |  |  |  |
| Implied gap of 75% holds only if the replacement has the same scope as the expiring contract. | DEVIL | number | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence gives no scope comparison, so the cost gap is like-for-like only if scope matches. |
| The CEO computes the 75% price gap as ($400,000 - $100,000) / $400,000. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence gives the one-quarter ratio; the CEO's specific computation isn't in evidence. |
| The replacement program may cover less scope than the Biobot contract. | DEVIL | forecast | · not_checked |  |  |  |  |
| Customers might switch at savings well below 75%. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO proposes flagging customers whose price gap is at or above 75%. | DEVIL | fact | ❔ unverifiable | 0.90 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 7% |
| The $400,000 per month figure is Biobot's expiring contract value. | DEVIL | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 94% |
| The $100,000 per month figure is the Globe's expectation for the state's replacement program, per biobot_public_contracts_customers_and_06 dated 2026-07. | DEVIL | fact | ✅ supported | 0.81 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 92% |
| The evidence does not say whether the replacement program covers the same scope, sample volume, frequency or sites as the Biobot contract. | DEVIL | fact | ✅ supported | 0.88 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence gives only contract value and approximate replacement cost; no scope or volume details. |
| The two inputs ($400,000 and $100,000) are not like-for-like. | DEVIL | opinion | · not_checked |  |  |  |  |
| An in-house replacement program may do less than the Biobot contract, so the 75% figure could reflect a scope reduction rather than a price gap for the same service. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO says the 75% figure may not generalise, yet uses it as the test. | DEVIL | opinion | · not_checked |  |  |  |  |
| The speaker asks what documented evidence could be shown, from signed contracts or procurement documents, of the scope of the expiring Massachusetts contract versus the in-house replacement program. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks whether that documentary record of scope (sites, sample frequency, sample volume) exists. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for documented records to be separated from estimates and for the speaker to state if the record cannot be seen. | VC | opinion | · not_checked |  |  |  |  |
| The statement sets aside a 75% figure from discussion. | VC | opinion | · not_checked |  |  |  |  |
| The speaker cannot show the referenced record from the evidence shown and cannot verify whether it exists. | CEO | opinion | · not_checked |  |  |  |  |
| The 2017 chunks biobot_fact_ratti_who_pays and biobot_fact_access_and_procurement are old and secondary sources. | CEO | opinion | · not_checked |  |  |  |  |
| The previous answer computed a 75% gap from the $400,000 and one-quarter figures. | CEO | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence gives $400k/month and one-quarter, but nothing about a previous answer's computation. |
| The 75% gap is not like-for-like because the evidence does not establish that the replacement covers the same scope. | CEO | opinion | · not_checked |  |  |  |  |
| The 75% figure may partly reflect a scope reduction rather than a price gap, so it is withdrawn as a threshold and as a measure of price sensitivity. | CEO | opinion | · not_checked |  |  |  |  |
| A single observation cannot establish at what saving other customers would switch. | CEO | opinion | · not_checked |  |  |  |  |
| The implied gap of 75% is ($400,000/month - $100,000/month) / $400,000/month. | CEO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | Evidence implies $100k/month (one-quarter of $400k) but doesn't show this formula was used. |
| The implied gap is 75%, valid only if scope is identical, which is not evidenced. | CEO | number | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_contracts_customers_and_06` | 75% follows from one-quarter; scope identity isn't evidenced, only the replacement's approximate cost. |
| The speaker proposes obtaining the signed MWRA contract and its scope-of-work or task orders for sites, sampling frequency and volumes. | CEO | forecast | · not_checked |  |  |  |  |
| The speaker proposes obtaining the state's procurement or budget documents for the in-house program and Biobot's own invoices for volumes delivered. | CEO | forecast | · not_checked |  |  |  |  |
| Like-for-like unit prices could only be compared after obtaining the scope and volume documents. | CEO | forecast | · not_checked |  |  |  |  |
| The speaker has no signed contract, procurement document or MWRA scope statement giving sites, sample frequency or sample volume for the expiring Massachusetts contract or the in-house replacement. | CEO | opinion | · not_checked |  |  |  |  |
| The most important assumption is that the replacement is comparable in scope to the Biobot service, which the evidence does not establish. | CEO | opinion | · not_checked |  |  |  |  |
| The replacement program may cover less scope than the Biobot contract, which is not evidenced either way. | CEO | forecast | · not_checked |  |  |  |  |
| If the replacement covers less scope, the Massachusetts case says little about price competitiveness and the loss would be logged as a procurement and substitution risk with an unknown cause. | CEO | forecast | · not_checked |  |  |  |  |
| Scope-of-work documents and invoices exist and can be obtained, which is not verified. | CEO | forecast | · not_checked |  |  |  |  |
| The Globe's one-quarter replacement cost applies to the state's program, not to Biobot revenue or margin. | CEO | forecast | · not_checked |  |  |  |  |
| The speaker will not estimate the scope difference between the Biobot contract and the in-house replacement. | CEO | opinion | · not_checked |  |  |  |  |
| MWRA says Biobot testing ended at the end of July 2026 as the state moved toward in-house testing. | CEO | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_public_measurements_and_03`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The Boston Globe (2026-07-04) reported the expiring Biobot contract at $400,000 per month. | CEO | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 97% |
| The Boston Globe (2026-07-04) reported the replacement expected to cost about one-quarter of the expiring contract value. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The source states the $400,000 per month figure is contract value, not net revenue or margin. | CEO | fact | ✅ supported | 0.96 | jev:jev-latest | `biobot_public_contracts_customers_and_06`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 96% |
| The biobot_public_business_and_market_03 record (2024-05-01) records CDC materiality and a later Verily award. | CEO | fact | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_02`, `biobot_public_competitors_alternatives_and_02` | JEV jev-1.14.0: evidence covers it 94% |
| The biobot_public_business_and_market_03 record states it does not give the full 2026 customer picture. | CEO | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 92% |
| The weakest point is the CEO's closing fallback, in which a reduced-scope replacement would be logged as a procurement and substitution risk with an unknown cause. | DEVIL | opinion | · not_checked |  |  |  |  |
| The customer chose to build and run its own testing program regardless of the scope of any replacement. | DEVIL | opinion | · not_checked |  |  |  |  |
| The substitution risk is documented in the evidence and is not unknown; only the price-sensitivity part is unknown. | DEVIL | opinion | · not_checked |  |  |  |  |
| A reduced-scope replacement could mean the customer found most of the Biobot service unnecessary, which would be a value problem rather than only a substitution problem. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO never considers the possibility that a reduced-scope replacement signals low perceived value of the service. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO's plan measures only one lost contract. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 2024-05-01 source flags government contract concentration. | DEVIL | fact | ✅ supported | 0.90 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_02` | JEV jev-1.14.0: evidence covers it 92% |
| The CEO gives no step to size how many other customers sit in the same position, so the answer does not address the VC's underlying exposure concern. | DEVIL | opinion | · not_checked |  |  |  |  |
| The author cannot verify from the evidence whether any of these records exist. | DEVIL | opinion | · not_checked |  |  |  |  |
| A state's budget or procurement documents give program-level cost rather than unit-price comparisons. | DEVIL | forecast | · not_checked |  |  |  |  |
| A reduced-scope replacement could signal low perceived value of the Biobot service. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO's closing fallback is a non-answer and is inconsistent with her own corrections. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO said that if the replacement covers less scope she would log the loss as a procurement and substitution risk with an unknown cause. | DEVIL | opinion | · not_checked |  |  |  |  |
| The VC asked what the record shows about scope, and the CEO said she cannot see it. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO proposed resolving the scope question by obtaining the signed contract, task orders, state budget papers, and Biobot invoices. | DEVIL | fact | ❔ unverifiable | 0.86 | jev:jev-latest |  | JEV jev-1.14.0: evidence covers it 8% |
| The CEO did not say who would release the proposed documents or that they would settle the matter. | DEVIL | opinion | · not_checked |  |  |  |  |
| A state's budget papers give a top-line program cost rather than a like-for-like unit-price comparison, so the proposed documents may not yield what the CEO says they will. | DEVIL | opinion | · not_checked |  |  |  |  |
| The MWRA record says the move was to in-house testing. | DEVIL | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_business_and_market_03`, `biobot_public_contracts_customers_and_06`, `biobot_public_business_and_market_04`, `biobot_public_public_measurements_and_03`, `biobot_public_competitors_alternatives_and_06` | JEV jev-1.14.0: evidence covers it 96% |
| The Globe's expected cost for the MWRA program is about one-quarter of the prior level. | DEVIL | number | ✅ supported | 0.91 | jev:jev-latest | `biobot_public_contracts_customers_and_06` | JEV jev-1.14.0: evidence covers it 93% |

## VC decision

**pass** — conviction 74%

| Rubric | Score | Why |
|---|---:|---|
| Team | 3/5 | The founders have a documented MIT research origin and a complementary 2017 skill split. They raised a 2025 Series A1 led by Valor and added two board directors. The 2017 record shows open gaps: they wanted a mechanical engineer or business help, and a roboticist declined to join. Nothing shown verifies today's team, CEO title or commercial leadership, and later-stage roles were not discussed with evidence. |
| Market | 2/5 | The only real price point is a reported $15,000/year municipal minimum. Across the 870-city 2017 target list, that gives about $13M at the floor. The $22B base rests on a founders' guess of a 20% public-health share. The 420-account list is synthetic. Top-down figures of $506M and $744M rest on unsourced shares and were withdrawn. The market looks small and fragmented unless state or federal budgets are proven. |
| Product | 3/5 | There is a real product: wastewater turned into public-health data, a granted sampler patent, and modeled weekly county/ZIP data with public-sector and pharma use cases. However, the patent does not give exclusivity. No head-to-head evidence shows Biobot's forecasts beat a qualified lab, and no accuracy or turnaround validation was produced. |
| Traction | 2/5 | Real evidence shows past anchor customers but both were lost. The CDC now names Verily as its national contractor and Biobot as the previous one. MWRA ended Biobot testing in July 2026 as the state moved in-house; that contract was reported at $400,000/month in contract value. Public sources do not establish current revenue, retention or active customers. The 89-contract, $14.384M book and the renewal cohorts are synthetic scenario data only. |
| Moat | 1/5 | State, local and university labs can collect, analyze and publish wastewater signals without Biobot. CDC takes data from non-Biobot sources, and WastewaterSCAN runs about 150 sites with Verily as its lab partner. The 2017 thesis itself was differentiation without proprietary technology. MWRA demonstrated build-versus-buy in practice. The CTO conceded the moat is a hypothesis. |
| Timing | 2/5 | The 2017 tailwinds were real: GovTech and data-driven public health. The 2026 signals run the other way. The national contract moved to Verily, a state moved in-house at roughly one-quarter of the cost, and government buyers have tight budgets and burdensome procurement. Post-pandemic surveillance budgets appear to be shifting toward insourcing. Whether the timing works depends on unproven analytics demand. |
| Business Model | 2/5 | The synthetic scenario shows a 57.4% cost-of-service margin, a 64.1% contribution on a $36,000 contract, $12.8M cash and about $336,630/month burn. None of these are real Biobot figures. Real-world signals are weak. Buyers rely on appropriations or time-limited grants (9 of 33 synthetic renewals were grant-funded). 'Who pays' has been open since 2017. Sales cycles are 163–318 days in the scenario, and the one real price comparison shows heavy price pressure. |

**For:**

- The company has proven it can win large public buyers. It was the CDC's national wastewater contractor and held an MWRA contract reported at $400,000/month.
- It holds a granted sampler patent (US11714030B2) and a modeled county/ZIP data product spanning public-sector and pharma use cases.
- Credible institutional backing: a 2025 Series A1 led by Valor Equity Partners, with Atreides and Hyperplane participating and two new directors.
- The founders have a strong MIT research origin and a documented history of earning city-official engagement.

**Against:**

- No real current revenue, customer count, retention or pricing was produced. Every commercial metric discussed (the 89-contract book, $14.384M ARR, margins, runway, renewals) is synthetic scenario data.
- The two largest documented relationships were lost in 2026. The CDC national contract went to Verily, and MWRA moved in-house at about one-quarter of the reported cost.
- Defensibility is unproven. Agencies and university labs can run the full workflow themselves, CDC has non-Biobot feeds, and Verily backs both a paid national contract and the free WastewaterSCAN network.
- The market looks small at real price points: about $13M if all 870 target cities paid the $15,000 floor. Revenue likely depends on concentrated state and federal contracts that are the most exposed to substitution.
- The proposed pivot to analytics for agencies that insource is an untested hypothesis. It may also shrink the data base it depends on.
- Exposure to MWRA-style substitution could not be sized, because no customer-level ledger, contract terms or MWRA scope documents exist in the evidence.

**Terms:** Pass at this time; no term sheet. I would reopen the discussion, possibly as a small participation in an insider-led extension with milestone-based tranches, only after full diligence. That diligence must include a real customer-level ledger (counterparty, ARR, start and expiry dates, funding source) and audited revenue reconciled to the public estimates. It must also include at least two signed, paid, multi-period analytics agreements from agencies that run their own labs.

**Would change my mind:**

- A real customer-by-customer revenue ledger showing current ARR, renewal and churn by segment, and how much revenue sits with state and federal buyers able to insource.
- The signed MWRA scope-of-work and invoices, compared with the in-house replacement, showing whether the loss was price, scope or value driven.
- Blinded head-to-head tests against a state lab showing a measurable accuracy, lead-time or cost-per-sample advantage.
- At least two signed paid analytics agreements from lab-owning agencies, each with a stated price, term and buyer-stated reason.
- Evidence of contractual data-sharing or benchmarking rights that make the multi-jurisdiction data set defensible.

**Checked claims the decision relies on:**

- `4a2867c30c8b-t88-c1` Mariana Matus and Newsha Ghaeli met in Ghaeli's first week at MIT, when both were charged with making the Underworlds project real (biobot_case_p3_c2, case as of 2017).
- `4a2867c30c8b-t88-c7` Ghaeli managed the six-department Underworlds effort and handled the 'urban' side: mappings, implementation and government contacts.
- `4a2867c30c8b-t88-c11` The founders tried to recruit a roboticist post-doc from Underworlds, who declined as a father of two young children.
- `4a2867c30c8b-t88-c12` Matus wanted to be careful with early hires and said they needed a mechanical engineer or business help, but felt they were a good team (biobot_case_p12_c1).
- `4a2867c30c8b-t88-c16` Public sources say Biobot was on the 2017 delta v roster, pitched opioid monitoring at Demo Day and joined YC W18 (biobot_public_history_and_operating_02, as of 2018).
- `4a2867c30c8b-t3-c10` On 2025-06-18 Biobot announced a Series A1 financing led by Valor Equity Partners, with new investor Atreides Management and Hyperplane participating.
- `4a2867c30c8b-t3-c11` Two directors joined Biobot's board alongside the Series A1 announcement.
- `4a2867c30c8b-t6-c1` One public source puts a minimum municipal charge for Biobot wastewater testing at $15,000 per year.
- `4a2867c30c8b-t6-c2` That same public source warns that the $15,000 minimum municipal charge cannot be multiplied by the cumulative number of tested sites to size revenue.
- `4a2867c30c8b-t8-c6` The bottom-up figure using $15,000 is a floor-price view in which 870 cities x $15,000 = $13.05M if every potential city bought one site.
- `4a2867c30c8b-t8-c8` Segments counted are US/Canadian municipal public-health and wastewater buyers: 4 beachhead cities, 71 cities over 500,000, and 870 cities total per case p.11 (2017).
- `4a2867c30c8b-t9-c7` The $22B base is the founders' own estimate of public-health spend, calculated as 20% of $110B.
- `4a2867c30c8b-t9-c10` The case states the 20% figure was a guess by Ghaeli and Matus.
- `4a2867c30c8b-t48-c10` The sales database holds 420 reachable accounts: 220 municipalities, 150 county or metro agencies, and 50 state or multi-region agencies.
- `4a2867c30c8b-t3-c1` Biobot turns wastewater into public-health data for cities.
- `4a2867c30c8b-t38-c2` Biobot is the assignee of US11714030B2, titled 'Modular system for upstream wastewater sampling', which was granted on 2023-08-01.
- `4a2867c30c8b-t38-c3` US11714030B2 protects a particular claim set on sampling hardware and does not give exclusivity over wastewater surveillance generally.
- `4a2867c30c8b-t38-c10` The synthetic scenario record states there is no head-to-head evidence that Biobot's forecasts beat another qualified lab.
- `4a2867c30c8b-t103-c14` Biobot publicly describes public-sector, pharma and life-science use cases and modeled weekly county/ZIP data, but its paying customer base is not disclosed.
- `4a2867c30c8b-t68-c5` CDC names Verily as the current national wastewater testing contractor and Biobot as the previous one.
- `4a2867c30c8b-t73-c6` The CDC national testing contract went to Verily, with Biobot named as the previous contractor.
- `4a2867c30c8b-t48-c1` MWRA ended Biobot testing at the end of July 2026 as the state moved to in-house testing.
- `4a2867c30c8b-t43-c9` The Boston Globe reported the expiring MWRA contract at $400,000 per month.
- `4a2867c30c8b-t43-c11` The $400,000 per month figure is contract value, not margin.
- `4a2867c30c8b-t103-c15` Public sources do not establish Biobot's revenue, retention, active paying customers or backlog.
- `4a2867c30c8b-t23-c1` The evidence shows a synthetic contract book of 89 contracts and $14,384,000 ARR (biobot_dummy_commercial_contract_book_04, 2026-09-30).
- `4a2867c30c8b-t23-c2` The synthetic contract book is labelled SYNTHETIC SCENARIO GROUND TRUTH, not actual Biobot data.
- `4a2867c30c8b-t38-c6` A public source dated 2026-10-22 states that state/local public-health labs and university programs can collect, analyze and publish wastewater signals without buying Biobot's full service.
- `4a2867c30c8b-t38-c4` In a 2017 case (p.12), the founders believed Biobot could differentiate on analytics without proprietary technology, so the original thesis was not a technology moat.
- `4a2867c30c8b-t38-c11` The synthetic scenario record states that site coverage alone is not proof of an exclusive moat.
- `4a2867c30c8b-t39-c6` CDC receives wastewater data from state and local health departments as well as from contractors.
- `4a2867c30c8b-t68-c10` WastewaterSCAN operates at nearly 150 sites and measures more than a dozen infectious diseases, with a public dashboard.
- `4a2867c30c8b-t69-c16` Verily is WastewaterSCAN's lab partner, per the source.
- `4a2867c30c8b-t38-c7` Records show MWRA ended Biobot's Massachusetts data stream in July 2026.
- `4a2867c30c8b-t98-c8` Ghaeli cited urbanization, GovTech and the push toward data-driven public-health decisions as market trends in 2017.
- `4a2867c30c8b-t48-c2` The Globe reported that the MWRA Biobot contract was $400,000 per month and that its replacement cost about one-quarter of that amount.
- `4a2867c30c8b-t128-c4` In 2017, selling to governments with tight budgets and burdensome purchasing was seen as hard.
- `4a2867c30c8b-t128-c3` Ratti warned the founders in 2017 that 'you don't know who's going to pay for this data yet'.
- `4a2867c30c8b-t28-c8` The scenario gross margin is 57.4%.
- `4a2867c30c8b-t13-c18` At the synthetic cost of $12,920 per contract, contribution is $23,080 (64.1%) before platform allocation.
- `4a2867c30c8b-t108-c3` A synthetic scenario record dated 2026-09-30 reports unrestricted cash of $12.8M.
- `4a2867c30c8b-t108-c4` A synthetic scenario record dated 2026-09-30 reports operating cash burn of about $336,630 per month.
- `4a2867c30c8b-t118-c11` The evidence shows a 33-renewal cohort: 24 renewed with a dedicated budget line and 9 on time-limited grant money (biobot_dummy_procurement_and_sales_04, synthetic, 2026-09-30).
- `4a2867c30c8b-t119-c10` The evidence shows municipal and county buyers rely on appropriations and state buyers rely on grant mixes.
- `4a2867c30c8b-t23-c9` In the scenario, the median signed-contract sales cycle is 163 days for municipal or county buyers and 318 days for state or federal buyers, measured from qualified discovery to signed annual agreement (biobot_dummy_procurement_and_sales_03, 2026-09-30).

## Cost and model calls

| Role | Tier | Model | Calls | Input | Output | Cache read | Latency (s) | Cost |
|---|---|---|---:|---:|---:|---:|---:|---:|
| vc_decision | decision | claude-opus-5-5 | 1 | 315,927 | 4,644 | 0 | 50.1 | $1.36603 |
| fact_check_hard | check_hard | claude-sonnet-5-5 | 30 | 396,345 | 24,666 | 27,840 | 185.6 | $1.04732 |
| devil | debate | claude-sonnet-5-5 | 30 | 174,013 | 22,927 | 67,048 | 229.0 | $0.59649 |
| vc | debate | claude-sonnet-5-5 | 30 | 215,896 | 6,945 | 61,712 | 104.0 | $0.51890 |
| ceo | debate | claude-sonnet-5-5 | 20 | 110,780 | 20,444 | 41,933 | 174.7 | $0.43990 |
| cfo | debate | claude-sonnet-5-5 | 8 | 41,944 | 9,881 | 13,944 | 85.4 | $0.19711 |
| fact_check | verdict | jev:jev-latest | 30 | 1,186,985 | 117,898 | 0 | 82.7 | $0.04985 |
| cto | debate | claude-sonnet-5-5 | 2 | 11,882 | 1,931 | 2,198 | 16.6 | $0.04901 |
| claim_extract | fast | claude-haiku-5-5 | 91 | 58,059 | 79,065 | 68,760 | 360.0 | $0.04612 |
| vc_opening | debate | claude-sonnet-5-5 | 1 | 875 | 95 | 0 | 3.4 | $0.00750 |
| route | route | jev:jev-latest | 13 | 6,559 | 558 | 0 | 1.7 | $0.00028 |

## Audit

- Retrievals: 708 (91 for answers, 617 for fact-checks)
- Turns with citations that were not retrieved: 1
- Cross-agent memory access attempts: 0

### Events

- `2026-10-23T15:57:05.385+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 1.0, "question_seq": 1}
- `2026-10-23T15:57:40.507+00:00` **route_decision** {"executive": "cfo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.73, "question_seq": 6}
- `2026-10-23T15:59:07.458+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.97, "question_seq": 16}
- `2026-10-23T16:00:16.913+00:00` **route_decision** {"executive": "cfo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 1.0, "question_seq": 26}
- `2026-10-23T16:02:18.043+00:00` **route_decision** {"executive": "cto", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 1.0, "question_seq": 36}
- `2026-10-23T16:03:38.661+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 1.0, "question_seq": 46}
- `2026-10-23T16:05:10.145+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.97, "question_seq": 56}
- `2026-10-23T16:06:39.200+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.97, "question_seq": 66}
- `2026-10-23T16:09:23.289+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.98, "question_seq": 86}
- `2026-10-23T16:10:42.656+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.92, "question_seq": 96}
- `2026-10-23T16:12:00.361+00:00` **route_decision** {"executive": "cfo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.97, "question_seq": 106}
- `2026-10-23T16:13:31.803+00:00` **route_decision** {"executive": "cfo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.89, "question_seq": 116}
- `2026-10-23T16:14:59.297+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.92, "question_seq": 126}
